# Beam SCST epoch 2 → VALID5000 → TEST5000 + fixed rerank

Học tiếp **đúng một epoch** từ full checkpoint của `ducanh3/notebook7bc83c4c76/version199`, giữ Adam và RNG hai GPU. Cursor epoch2 bắt đầu0, cumulative update3541→7082; không học lại epoch1 và không reset optimizer. Dùng hai tối ưu vừa kiểm tra tốc độ, giữ FP32/beam5/global32/loss/model như epoch1. Không tự epoch3 hay chọn checkpoint theo TEST.

**Kaggle: GPU T4×2, Internet ON, Secret `codex` hoặc `KAGGLE_API_TOKEN` có quyền đọc nguồn ducanh3.** Không cần gắn output199: notebook tải đúng full checkpoint bằng API và xác minh SHA. Gắn5Dataset; các đường dẫn ở đầu cell cấu hình:

- `ducanh3/csa-scst-full-karpathy-cache-v1`
- `vuthetam/mscoco-2014`
- `ducanh2403/visual-cache`
- `ducanh2403/csa-prompt-test-cache-v181`
- `ducanh2403/objectdetectionecache`

Cap mặc định **5,75 giờ wall**, dự phòng100 phút sau train. Không phải ETA hay cam kết quota6giờ đủ. Train dự kiến gần3giờ từ mẫu tốc độ, thời gian setup/Java có thể thay đổi. Mỗi100update và khi hết cap lưu full checkpoint; nếu epoch2 chưa đủ thì dừng, không chấm TEST bằng partial weights. Giữ **toàn bộ Output**, reviewZIP không chứa weights.

VALID top1 so với epoch1 cùng seed43. TEST sinh beam5 một lần, chấm top1 và rerank cố định **0,3 decoder + 0,4 CLIP + 0,3 object**, đủ8thang đo + Recall/NLL, so với TEST epoch1/output200 và CE179 đã lưu. Không tìm trọng số trên TEST. Caption/beam/scores được giữ trước scorer để recovery nếu lỗi; không tự train lại. Đây là TEST đã được dự án dùng trước trong phát triển.

`RESUME_ROOT` chỉ nhận **toàn bộ output epoch2** đã gắn ở input, kiểm tra SHA/identity rồi tiếp cursor cũ. Bắt đầu mới để trống. Nếu chỉ còn Save Version, gửi owner/notebook/version để làm API resume đúng nguồn; không điền output epoch1 vào trường này.

Agent chỉ kiểm tra syntax/metadata/NumPy/mock; API, khôi phục binary/CUDA, thời gian toàn lượt và metric mới chờ người dùng chạy.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,zipfile,shutil
from pathlib import Path

# Inputs: same full frozen TRAIN/VALID cache, plus the proven TEST/rerank inputs.
CACHE_ROOT=Path('/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
CHECKPOINT_ROOT='' # Blank: downloads only exact v199 full checkpoint.bin via API.
RESUME_ROOT='' # Optional whole previous epoch2 output, never epoch1 output here.
SECRET_NAMES=('codex','KAGGLE_API_TOKEN')
RUN_TAG='run1'
WALL_HOURS=5.75 # Invocation stop cap, not ETA or GPU quota billing.
EVALUATION_RESERVE_SECONDS=6000 # 100 minutes after training for VALID/TEST/rerank/CPU scorers.
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/199'
CHECKPOINT_SHA='5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c'
OUTPUT=Path(f'/kaggle/working/beam_scst_epoch2_seed43_v199_{RUN_TAG}')
for path in (CACHE_ROOT,COCO_JSON,VISUAL_CACHE,PROMPT_TEST_ROOT,DETECTIONS):
    if not path.exists():raise FileNotFoundError('Attach/correct input before setup: '+str(path))
if OUTPUT.exists():raise RuntimeError('Preserve output; choose a new RUN_TAG or session')
if not 2<=WALL_HOURS<=5.75:raise ValueError('Reviewed cap 2–5.75h, not ETA/quota')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<10*1024**3:
    raise RuntimeError('Need10GiB scratch/output for weights, TEST patches and scorers')
OUTPUT.mkdir();WORK=Path(tempfile.mkdtemp(prefix='beam-scst-epoch2-',dir='/tmp'))
SOURCE=WORK/'train_source';SOURCE.mkdir()
TEST_SOURCE=WORK/'test_source';TEST_SOURCE.mkdir()
CLI=WORK/'cli';DEPS=WORK/'deps';CHAIR=WORK/'chair'
CONFIG=WORK/'train_config.json';LOG=OUTPUT/'notebook.log';TOKENS=[]
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ.update(PYTHONPATH=str(SOURCE),OMP_NUM_THREADS='1',TOKENIZERS_PARALLELISM='false',
                  USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')
cfg=dict(semantic=True,output=str(OUTPUT),checkpoint_root=CHECKPOINT_ROOT,cache_root=str(CACHE_ROOT),
    dataset_json=str(COCO_JSON),metadata_root=str(WORK/'metadata'),chair=str(CHAIR),
    resume_root=RESUME_ROOT,train_seconds=3600,worker_seconds=900)


def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
    (OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')


def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line

TRAIN_PAYLOADS={'TEST199_LOCK.json': '{\n  "schema": "clip_patch_199_test_lock_v1",\n  "source_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/199",\n  "checkpoint_sha256": "5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7",\n  "training_commit": "9aaddf5694f72373b8ab2482fafe7d7992abbe5e",\n  "epoch": 10,\n  "updates": 177110,\n  "beam_width": 5,\n  "max_new_tokens": 20,\n  "length_normalization": false,\n  "reranking": false,\n  "expected_test_ids": [\n    42,\n    359,\n    636,\n    711,\n    810,\n    974,\n    1146,\n    1180,\n    1290,\n    1353,\n    1393,\n    1425,\n    1448,\n    1573,\n    1590,\n    1626,\n    1655,\n    1682,\n    1700,\n    1757,\n    2142,\n    2225,\n    2240,\n    2295,\n    2453,\n    2690,\n    2753,\n    2759,\n    2822,\n    2867,\n    2890,\n    3001,\n    3103,\n    3156,\n    3382,\n    3590,\n    3716,\n    3832,\n    4069,\n    4125,\n    4246,\n    4312,\n    4359,\n    4760,\n    4916,\n    4972,\n    5123,\n    5154,\n    5205,\n    5388,\n    5758,\n    6040,\n    6393,\n    6415,\n    6484,\n    6861,\n    6896,\n    6921,\n    7072,\n    7088,\n    7278,\n    7288,\n    7519,\n    7556,\n    7593,\n    7784,\n    7795,\n    8179,\n    8180,\n    8218,\n    8277,\n    8483,\n    8493,\n    8612,\n    8775,\n    8803,\n    8888,\n    9002,\n    9041,\n    9156,\n    9170,\n    9217,\n    9236,\n    9381,\n    9548,\n    9647,\n    9699,\n    9727,\n    9791,\n    9801,\n    9807,\n    9914,\n    10014,\n    10104,\n    10115,\n    10290,\n    10400,\n    10428,\n    10526,\n    10684,\n    10707,\n    10764,\n    10825,\n    10837,\n    10935,\n    11081,\n    11099,\n    11115,\n    11156,\n    11181,\n    11184,\n    11198,\n    11209,\n    11276,\n    11625,\n    12085,\n    12115,\n    12204,\n    12471,\n    12744,\n    12817,\n    12896,\n    12927,\n    13220,\n    13466,\n    13490,\n    13525,\n    13882,\n    13985,\n    14056,\n    14175,\n    14257,\n    14338,\n    14392,\n    14450,\n    14473,\n    14713,\n    14723,\n    14781,\n    15017,\n    15050,\n    15254,\n    15260,\n    15345,\n    15667,\n    16005,\n    16241,\n    16249,\n    16491,\n    16521,\n    16599,\n    16631,\n    16848,\n    16928,\n    16931,\n    17115,\n    17178,\n    17313,\n    17756,\n    17769,\n    17905,\n    17927,\n    17953,\n    18014,\n    18412,\n    18480,\n    18519,\n    18687,\n    18770,\n    18771,\n    18975,\n    19158,\n    19167,\n    19308,\n    19443,\n    19455,\n    19888,\n    19904,\n    20111,\n    20179,\n    20254,\n    20268,\n    20276,\n    20305,\n    20381,\n    20470,\n    20598,\n    20913,\n    20992,\n    21198,\n    21202,\n    21284,\n    21310,\n    21327,\n    21498,\n    21864,\n    21900,\n    21979,\n    22118,\n    22158,\n    22167,\n    22176,\n    22192,\n    22213,\n    22371,\n    22759,\n    22850,\n    22892,\n    22929,\n    23034,\n    23117,\n    23320,\n    23351,\n    23731,\n    23802,\n    23840,\n    24207,\n    24223,\n    24343,\n    24553,\n    24734,\n    24805,\n    25014,\n    25144,\n    25181,\n    25286,\n    25316,\n    25386,\n    25521,\n    25550,\n    25603,\n    25747,\n    25758,\n    25860,\n    25989,\n    26051,\n    26209,\n    26584,\n    26609,\n    26654,\n    26665,\n    26671,\n    26730,\n    26762,\n    26942,\n    27065,\n    27186,\n    27226,\n    27440,\n    27570,\n    27585,\n    27620,\n    27656,\n    27778,\n    27871,\n    28114,\n    28156,\n    28157,\n    28293,\n    28377,\n    28452,\n    28463,\n    28506,\n    28532,\n    28582,\n    28655,\n    28675,\n    28688,\n    28790,\n    28998,\n    29045,\n    29056,\n    29094,\n    29160,\n    29465,\n    29472,\n    29577,\n    29656,\n    30034,\n    30143,\n    30455,\n    30470,\n    30549,\n    30607,\n    30667,\n    30685,\n    30785,\n    30793,\n    30925,\n    30954,\n    31000,\n    31016,\n    31024,\n    31176,\n    31281,\n    31390,\n    31442,\n    31606,\n    31636,\n    31667,\n    31965,\n    31981,\n    32334,\n    32400,\n    32524,\n    32570,\n    32587,\n    32688,\n    32712,\n    32811,\n    32901,\n    32909,\n    32990,\n    33052,\n    33144,\n    33368,\n    33638,\n    33645,\n    33647,\n    33672,\n    33753,\n    33759,\n    33830,\n    33845,\n    33871,\n    33904,\n    33990,\n    34080,\n    34115,\n    34126,\n    34180,\n    34428,\n    34482,\n    34617,\n    34662,\n    34811,\n    34900,\n    34904,\n    35012,\n    35105,\n    35195,\n    35498,\n    35738,\n    35770,\n    35807,\n    35825,\n    35853,\n    35975,\n    36082,\n    36238,\n    36478,\n    36501,\n    36598,\n    36661,\n    37367,\n    37705,\n    37729,\n    37882,\n    38048,\n    38210,\n    38332,\n    38479,\n    38662,\n    38666,\n    38678,\n    38938,\n    39009,\n    39068,\n    39081,\n    39152,\n    39202,\n    39430,\n    39484,\n    39516,\n    39589,\n    39606,\n    39671,\n    39697,\n    39731,\n    39844,\n    39914,\n    40446,\n    40621,\n    40635,\n    40924,\n    40946,\n    41077,\n    41119,\n    41397,\n    41839,\n    41867,\n    42055,\n    42070,\n    42091,\n    42190,\n    42201,\n    42408,\n    42526,\n    42944,\n    42956,\n    42970,\n    43202,\n    43289,\n    43613,\n    43635,\n    43657,\n    43680,\n    44112,\n    44244,\n    44328,\n    44520,\n    44524,\n    44642,\n    44910,\n    44934,\n    45057,\n    45108,\n    45208,\n    45356,\n    45383,\n    45434,\n    45467,\n    45550,\n    45587,\n    45627,\n    45644,\n    45710,\n    45775,\n    45966,\n    46011,\n    46077,\n    46099,\n    46269,\n    46331,\n    46432,\n    46433,\n    46497,\n    46571,\n    46761,\n    46775,\n    46847,\n    47394,\n    47511,\n    47596,\n    47632,\n    47651,\n    47658,\n    47837,\n    47882,\n    47886,\n    47938,\n    48129,\n    48332,\n    48504,\n    48530,\n    48546,\n    48668,\n    48731,\n    48743,\n    48905,\n    48924,\n    49014,\n    49097,\n    49255,\n    49688,\n    49731,\n    49893,\n    49984,\n    50006,\n    50025,\n    50159,\n    50201,\n    50230,\n    50309,\n    50350,\n    50411,\n    50443,\n    50482,\n    51089,\n    51223,\n    51250,\n    51335,\n    51484,\n    51540,\n    51576,\n    51773,\n    51961,\n    52017,\n    52020,\n    52155,\n    52282,\n    52386,\n    52591,\n    52827,\n    53267,\n    53454,\n    53465,\n    53491,\n    53642,\n    53892,\n    54340,\n    54389,\n    54490,\n    54513,\n    54516,\n    54593,\n    54679,\n    54899,\n    54931,\n    54959,\n    55002,\n    55010,\n    55059,\n    55223,\n    55295,\n    55466,\n    55727,\n    55772,\n    55810,\n    56013,\n    56248,\n    56326,\n    56396,\n    56405,\n    56413,\n    56580,\n    56633,\n    56651,\n    56983,\n    57244,\n    57286,\n    57593,\n    57843,\n    58139,\n    58232,\n    58329,\n    58393,\n    58425,\n    58472,\n    58492,\n    58569,\n    58636,\n    58851,\n    59015,\n    59383,\n    59614,\n    59732,\n    59740,\n    60049,\n    60177,\n    60340,\n    60364,\n    60623,\n    60700,\n    60970,\n    61207,\n    61414,\n    61564,\n    61647,\n    61899,\n    62230,\n    62353,\n    62706,\n    62985,\n    63403,\n    63602,\n    63647,\n    63726,\n    64084,\n    64152,\n    64196,\n    64334,\n    64531,\n    64710,\n    65208,\n    65258,\n    65969,\n    65981,\n    66072,\n    66179,\n    66271,\n    66412,\n    66413,\n    66567,\n    66568,\n    66685,\n    66771,\n    66879,\n    67122,\n    67255,\n    67422,\n    67463,\n    67532,\n    67569,\n    67587,\n    67686,\n    67704,\n    67881,\n    68025,\n    68120,\n    68203,\n    68375,\n    68380,\n    68408,\n    68490,\n    68745,\n    68833,\n    68864,\n    68881,\n    69138,\n    69236,\n    69392,\n    69698,\n    69757,\n    69795,\n    69873,\n    69946,\n    70020,\n    70067,\n    70256,\n    70280,\n    70351,\n    70426,\n    71004,\n    71072,\n    71090,\n    71171,\n    71281,\n    71596,\n    71687,\n    71929,\n    72011,\n    72052,\n    72077,\n    72338,\n    72368,\n    72397,\n    72507,\n    72873,\n    73119,\n    73256,\n    73333,\n    73521,\n    73527,\n    73702,\n    74181,\n    74209,\n    74369,\n    74460,\n    74465,\n    74478,\n    74711,\n    74733,\n    74759,\n    74789,\n    75055,\n    75162,\n    75173,\n    75479,\n    75591,\n    75595,\n    75600,\n    75888,\n    75932,\n    75990,\n    76155,\n    76292,\n    76329,\n    76529,\n    76738,\n    76765,\n    76962,\n    77181,\n    77282,\n    77360,\n    77402,\n    77415,\n    77689,\n    77754,\n    77963,\n    77980,\n    78035,\n    78056,\n    78420,\n    78565,\n    78707,\n    78748,\n    78959,\n    78979,\n    79139,\n    79369,\n    79392,\n    79545,\n    79686,\n    79816,\n    79860,\n    79868,\n    79975,\n    79992,\n    80000,\n    80117,\n    80328,\n    80367,\n    80369,\n    80413,\n    80659,\n    80737,\n    80747,\n    81354,\n    81656,\n    81661,\n    81897,\n    81988,\n    81998,\n    82042,\n    82229,\n    82293,\n    82740,\n    82812,\n    82846,\n    82921,\n    82933,\n    82935,\n    83059,\n    83065,\n    83217,\n    83477,\n    83836,\n    83915,\n    83923,\n    84000,\n    84060,\n    84064,\n    84073,\n    84258,\n    84362,\n    84447,\n    84650,\n    84674,\n    84735,\n    84859,\n    84889,\n    84936,\n    85007,\n    85144,\n    85183,\n    85329,\n    85390,\n    85527,\n    85529,\n    85562,\n    85665,\n    85721,\n    85747,\n    86135,\n    86202,\n    86439,\n    86483,\n    86516,\n    86582,\n    86839,\n    86884,\n    87199,\n    87399,\n    87429,\n    87470,\n    87476,\n    87493,\n    87617,\n    87864,\n    87912,\n    88092,\n    88142,\n    88225,\n    88252,\n    88269,\n    88445,\n    88485,\n    88517,\n    88556,\n    88652,\n    88695,\n    88784,\n    88854,\n    89228,\n    89293,\n    89487,\n    89500,\n    89558,\n    89603,\n    89638,\n    89909,\n    90062,\n    90208,\n    90280,\n    90365,\n    90366,\n    90476,\n    90646,\n    90707,\n    91045,\n    91300,\n    91304,\n    91318,\n    91416,\n    91469,\n    91488,\n    91857,\n    92177,\n    92257,\n    92416,\n    92488,\n    92678,\n    92801,\n    93154,\n    93156,\n    93175,\n    93353,\n    93534,\n    93765,\n    93771,\n    93803,\n    93853,\n    93948,\n    93964,\n    94025,\n    94055,\n    94140,\n    94157,\n    94174,\n    94306,\n    94651,\n    94951,\n    95051,\n    95062,\n    95161,\n    95297,\n    95441,\n    95702,\n    95772,\n    95786,\n    96110,\n    96306,\n    96351,\n    96633,\n    96638,\n    96689,\n    96832,\n    96991,\n    97017,\n    97146,\n    97156,\n    97240,\n    97513,\n    97561,\n    97667,\n    97693,\n    97754,\n    97924,\n    97964,\n    98003,\n    98197,\n    98284,\n    98322,\n    98434,\n    98444,\n    99053,\n    99081,\n    99317,\n    99747,\n    99794,\n    99874,\n    99961,\n    100001,\n    100087,\n    100098,\n    100138,\n    100187,\n    100306,\n    100594,\n    100661,\n    100848,\n    100853,\n    100977,\n    101473,\n    101491,\n    101919,\n    101985,\n    102348,\n    102466,\n    102589,\n    102735,\n    102805,\n    103114,\n    103255,\n    103272,\n    103375,\n    103485,\n    103491,\n    103496,\n    103584,\n    104421,\n    104568,\n    104906,\n    105011,\n    105052,\n    105234,\n    105480,\n    105611,\n    105622,\n    105751,\n    105912,\n    105945,\n    106046,\n    106048,\n    106331,\n    106335,\n    106849,\n    106909,\n    106970,\n    107108,\n    107123,\n    107183,\n    107216,\n    107511,\n    107541,\n    107814,\n    107853,\n    108152,\n    108193,\n    108338,\n    108392,\n    108495,\n    108853,\n    109005,\n    109316,\n    109425,\n    109434,\n    109445,\n    109454,\n    109939,\n    110026,\n    110156,\n    110282,\n    110369,\n    110435,\n    110551,\n    110601,\n    110630,\n    110765,\n    110857,\n    110877,\n    110884,\n    111024,\n    111109,\n    111367,\n    111546,\n    111830,\n    111988,\n    112093,\n    112394,\n    112568,\n    112591,\n    112634,\n    112751,\n    112904,\n    112956,\n    112988,\n    113040,\n    113058,\n    113128,\n    113223,\n    113556,\n    113828,\n    113836,\n    113890,\n    113905,\n    114158,\n    114303,\n    114317,\n    114348,\n    114352,\n    114616,\n    114634,\n    114684,\n    114744,\n    114745,\n    114941,\n    115070,\n    115118,\n    115162,\n    115178,\n    115241,\n    115455,\n    115846,\n    116017,\n    116133,\n    116208,\n    116341,\n    116361,\n    116370,\n    116574,\n    116678,\n    116712,\n    116722,\n    117061,\n    117089,\n    117201,\n    117432,\n    117458,\n    117525,\n    117690,\n    117914,\n    117988,\n    118069,\n    118134,\n    118154,\n    118260,\n    118367,\n    118432,\n    119134,\n    119146,\n    119414,\n    119469,\n    119640,\n    120007,\n    120070,\n    120234,\n    120428,\n    120475,\n    120767,\n    120776,\n    120792,\n    120977,\n    121031,\n    121196,\n    121611,\n    121692,\n    121716,\n    121839,\n    121876,\n    122007,\n    122051,\n    122161,\n    122239,\n    122252,\n    122266,\n    122302,\n    122354,\n    122606,\n    122747,\n    122861,\n    122981,\n    123017,\n    123137,\n    123321,\n    123415,\n    123622,\n    123836,\n    123964,\n    124185,\n    124462,\n    124647,\n    124659,\n    124760,\n    124930,\n    124952,\n    125071,\n    125072,\n    125129,\n    125404,\n    125472,\n    125656,\n    125815,\n    125971,\n    125983,\n    126030,\n    126098,\n    126145,\n    126192,\n    126257,\n    126299,\n    126324,\n    126502,\n    126538,\n    126766,\n    126958,\n    127263,\n    127451,\n    127516,\n    127517,\n    127556,\n    127671,\n    127788,\n    127801,\n    127842,\n    127865,\n    128014,\n    128119,\n    128140,\n    128528,\n    128675,\n    128679,\n    128704,\n    128812,\n    129339,\n    129416,\n    129431,\n    129548,\n    129637,\n    129645,\n    129663,\n    129706,\n    129707,\n    129852,\n    129982,\n    130171,\n    130225,\n    130352,\n    130419,\n    130437,\n    130438,\n    130516,\n    130524,\n    130566,\n    130826,\n    130909,\n    131019,\n    131089,\n    131138,\n    131207,\n    131418,\n    131497,\n    131504,\n    131847,\n    132272,\n    132317,\n    132415,\n    132612,\n    132702,\n    132776,\n    132796,\n    133090,\n    133210,\n    133380,\n    133470,\n    133515,\n    133567,\n    133619,\n    133645,\n    133867,\n    134133,\n    134167,\n    134213,\n    134255,\n    134503,\n    134518,\n    134555,\n    134558,\n    134691,\n    134722,\n    134863,\n    134964,\n    135117,\n    135210,\n    135281,\n    135399,\n    135578,\n    135900,\n    136117,\n    136433,\n    136572,\n    136584,\n    136642,\n    136680,\n    136681,\n    136780,\n    136911,\n    137221,\n    137265,\n    137412,\n    137678,\n    137727,\n    137844,\n    137938,\n    137950,\n    137967,\n    138070,\n    138073,\n    138083,\n    138477,\n    138553,\n    138599,\n    138675,\n    138755,\n    138821,\n    138937,\n    138965,\n    138970,\n    139004,\n    139094,\n    139457,\n    139468,\n    139567,\n    139660,\n    140021,\n    140076,\n    140388,\n    140444,\n    140658,\n    140686,\n    140702,\n    140929,\n    140963,\n    141197,\n    141256,\n    141278,\n    141317,\n    141517,\n    141587,\n    141849,\n    141879,\n    141962,\n    142177,\n    142189,\n    142246,\n    142323,\n    142346,\n    142386,\n    142879,\n    142891,\n    143101,\n    143103,\n    143107,\n    143167,\n    143346,\n    143458,\n    143554,\n    143998,\n    144064,\n    144122,\n    144167,\n    144333,\n    144373,\n    144379,\n    144715,\n    144795,\n    144798,\n    144874,\n    145009,\n    145048,\n    145208,\n    145369,\n    145436,\n    145549,\n    145666,\n    145727,\n    145734,\n    145751,\n    145831,\n    145873,\n    146042,\n    146272,\n    146315,\n    146503,\n    146767,\n    146837,\n    146855,\n    146865,\n    146965,\n    147042,\n    147068,\n    147346,\n    147375,\n    147409,\n    147415,\n    147425,\n    147556,\n    147667,\n    147685,\n    147725,\n    147915,\n    147980,\n    148188,\n    148329,\n    148444,\n    148568,\n    148588,\n    148665,\n    148792,\n    148910,\n    148977,\n    149150,\n    149196,\n    149268,\n    149304,\n    149371,\n    149375,\n    149444,\n    149456,\n    149500,\n    150013,\n    150024,\n    150080,\n    150117,\n    150224,\n    150538,\n    150576,\n    150717,\n    150779,\n    151081,\n    151480,\n    151521,\n    151524,\n    151790,\n    151832,\n    151970,\n    152004,\n    152176,\n    152252,\n    152258,\n    152298,\n    152326,\n    152333,\n    152336,\n    152388,\n    152398,\n    152482,\n    152525,\n    152543,\n    152598,\n    152648,\n    152740,\n    153231,\n    153631,\n    153782,\n    153797,\n    153803,\n    153966,\n    154095,\n    154145,\n    154241,\n    154670,\n    154783,\n    154867,\n    154880,\n    154931,\n    154971,\n    155488,\n    155714,\n    155736,\n    155897,\n    156195,\n    156242,\n    156282,\n    156292,\n    156532,\n    156572,\n    156685,\n    156772,\n    156858,\n    156915,\n    157001,\n    157019,\n    157102,\n    157184,\n    157516,\n    157593,\n    157617,\n    157726,\n    157789,\n    157807,\n    157980,\n    158109,\n    158118,\n    158292,\n    158414,\n    158722,\n    158744,\n    159030,\n    159044,\n    159112,\n    159203,\n    159215,\n    159262,\n    159311,\n    159374,\n    159742,\n    160126,\n    160186,\n    160195,\n    160351,\n    160501,\n    160963,\n    161008,\n    161112,\n    161144,\n    161185,\n    161202,\n    161222,\n    161231,\n    161370,\n    161505,\n    161586,\n    161697,\n    161736,\n    161743,\n    161962,\n    162043,\n    162362,\n    162393,\n    162398,\n    162415,\n    162445,\n    162459,\n    162503,\n    162634,\n    162677,\n    162902,\n    162914,\n    163118,\n    163348,\n    163611,\n    163684,\n    163710,\n    163852,\n    164104,\n    164440,\n    164548,\n    164759,\n    164835,\n    165009,\n    165029,\n    165225,\n    165298,\n    165547,\n    165572,\n    165643,\n    165752,\n    166124,\n    166478,\n    166645,\n    167122,\n    167235,\n    167422,\n    167510,\n    167529,\n    167544,\n    167549,\n    167602,\n    167810,\n    167989,\n    168093,\n    168243,\n    168250,\n    168337,\n    168580,\n    168683,\n    168714,\n    168847,\n    168991,\n    169098,\n    169361,\n    169436,\n    169539,\n    169757,\n    169802,\n    169858,\n    170015,\n    170038,\n    170211,\n    170250,\n    170321,\n    170411,\n    170436,\n    170893,\n    171190,\n    171199,\n    171241,\n    171443,\n    171595,\n    171695,\n    171819,\n    172021,\n    172265,\n    172285,\n    172330,\n    172396,\n    172408,\n    172556,\n    172648,\n    172673,\n    172733,\n    172851,\n    172911,\n    172935,\n    173081,\n    173117,\n    173235,\n    173385,\n    173391,\n    173584,\n    173797,\n    173819,\n    174329,\n    174511,\n    174567,\n    174594,\n    174672,\n    175024,\n    175387,\n    175653,\n    175688,\n    175718,\n    175804,\n    175864,\n    175948,\n    176031,\n    176312,\n    176330,\n    176466,\n    176799,\n    176863,\n    176906,\n    177015,\n    177102,\n    177246,\n    177262,\n    177366,\n    177453,\n    177489,\n    177535,\n    177567,\n    177811,\n    177838,\n    177861,\n    177938,\n    178413,\n    178578,\n    178746,\n    178748,\n    178761,\n    178795,\n    178796,\n    178818,\n    179114,\n    179285,\n    179441,\n    179487,\n    179720,\n    179765,\n    179921,\n    179960,\n    180336,\n    180447,\n    180495,\n    180784,\n    180798,\n    181026,\n    181169,\n    181572,\n    181574,\n    181586,\n    181643,\n    181799,\n    181859,\n    181870,\n    181948,\n    182056,\n    182162,\n    182245,\n    182279,\n    182334,\n    182369,\n    182417,\n    182441,\n    182503,\n    182683,\n    182736,\n    182784,\n    183014,\n    183181,\n    183528,\n    183620,\n    183715,\n    183786,\n    183970,\n    184139,\n    184227,\n    184324,\n    184384,\n    184386,\n    184400,\n    184557,\n    184773,\n    184791,\n    184830,\n    184902,\n    185210,\n    185240,\n    185476,\n    185487,\n    185639,\n    185721,\n    185781,\n    185789,\n    185850,\n    185922,\n    185930,\n    185950,\n    186282,\n    186428,\n    186497,\n    186518,\n    186747,\n    186797,\n    186816,\n    187001,\n    187054,\n    187072,\n    187329,\n    187610,\n    187776,\n    187857,\n    188077,\n    188173,\n    188192,\n    188416,\n    188718,\n    188795,\n    188852,\n    188865,\n    189197,\n    189376,\n    189566,\n    189744,\n    189766,\n    189806,\n    189828,\n    190015,\n    190033,\n    190081,\n    190291,\n    190326,\n    190391,\n    190595,\n    190648,\n    190690,\n    190705,\n    190783,\n    191069,\n    191096,\n    191240,\n    191390,\n    191842,\n    191846,\n    191893,\n    192095,\n    192212,\n    192247,\n    192306,\n    192440,\n    192496,\n    192594,\n    192730,\n    192782,\n    192788,\n    192904,\n    192932,\n    193015,\n    193021,\n    193349,\n    193449,\n    193494,\n    193504,\n    193881,\n    193884,\n    194120,\n    194188,\n    194306,\n    194411,\n    194559,\n    194704,\n    194746,\n    194755,\n    194756,\n    195086,\n    195113,\n    195275,\n    195353,\n    195406,\n    195542,\n    195594,\n    195645,\n    195648,\n    195673,\n    195750,\n    195790,\n    195800,\n    195817,\n    196009,\n    196091,\n    196490,\n    196521,\n    196715,\n    196839,\n    196989,\n    196995,\n    197193,\n    197245,\n    197458,\n    197461,\n    197609,\n    197716,\n    197745,\n    198043,\n    198176,\n    198448,\n    198641,\n    199050,\n    199122,\n    199149,\n    199203,\n    199247,\n    199257,\n    199319,\n    199346,\n    199395,\n    199403,\n    199404,\n    199449,\n    199551,\n    199553,\n    199764,\n    199783,\n    199940,\n    199962,\n    199969,\n    200250,\n    200267,\n    200274,\n    200291,\n    200401,\n    200567,\n    200572,\n    200717,\n    200739,\n    200959,\n    200961,\n    201148,\n    201163,\n    201207,\n    201637,\n    201732,\n    201916,\n    201918,\n    202001,\n    202040,\n    202102,\n    202389,\n    202489,\n    202653,\n    202913,\n    202923,\n    202944,\n    203003,\n    203438,\n    203629,\n    203636,\n    203669,\n    203734,\n    204020,\n    204232,\n    204640,\n    204930,\n    204943,\n    205101,\n    205183,\n    205230,\n    205577,\n    205601,\n    205636,\n    205676,\n    205720,\n    205854,\n    205863,\n    205866,\n    206112,\n    206381,\n    206705,\n    206770,\n    206783,\n    206851,\n    206859,\n    207027,\n    207151,\n    207205,\n    207292,\n    207366,\n    207437,\n    207585,\n    207675,\n    207853,\n    207913,\n    208132,\n    208283,\n    208311,\n    208408,\n    208516,\n    208589,\n    208623,\n    208648,\n    208778,\n    208808,\n    208830,\n    208871,\n    209048,\n    209147,\n    209388,\n    209420,\n    209480,\n    209527,\n    209544,\n    209747,\n    209803,\n    209808,\n    209868,\n    209904,\n    209917,\n    209935,\n    210012,\n    210032,\n    210144,\n    210230,\n    210273,\n    210372,\n    210564,\n    210982,\n    211054,\n    211112,\n    211119,\n    211120,\n    211186,\n    211204,\n    211206,\n    211246,\n    211326,\n    211487,\n    211491,\n    211618,\n    211653,\n    211690,\n    211756,\n    212058,\n    212077,\n    212285,\n    212288,\n    212351,\n    212421,\n    212587,\n    212688,\n    212817,\n    213008,\n    213132,\n    213224,\n    213338,\n    213538,\n    213739,\n    213773,\n    214046,\n    214184,\n    214224,\n    214369,\n    214454,\n    214527,\n    214664,\n    214919,\n    214961,\n    215124,\n    215245,\n    215356,\n    215408,\n    215456,\n    215471,\n    215524,\n    215608,\n    215622,\n    215675,\n    215744,\n    215808,\n    215867,\n    216006,\n    216051,\n    216228,\n    216303,\n    216363,\n    216677,\n    216710,\n    216726,\n    217060,\n    217071,\n    217082,\n    217133,\n    217156,\n    217183,\n    217323,\n    217393,\n    217468,\n    217872,\n    217951,\n    217957,\n    218290,\n    218646,\n    218988,\n    218996,\n    219135,\n    219269,\n    219315,\n    219434,\n    219525,\n    219567,\n    219589,\n    219632,\n    219657,\n    219723,\n    219792,\n    219820,\n    219928,\n    220111,\n    220171,\n    220187,\n    220224,\n    220255,\n    220307,\n    220367,\n    220446,\n    220678,\n    220858,\n    221155,\n    221190,\n    221213,\n    221232,\n    221291,\n    221700,\n    221829,\n    221869,\n    222075,\n    222146,\n    222370,\n    222430,\n    222559,\n    222648,\n    222757,\n    222811,\n    222863,\n    222879,\n    222908,\n    223032,\n    223094,\n    223182,\n    223183,\n    223241,\n    223256,\n    223300,\n    223648,\n    223738,\n    223930,\n    224051,\n    224053,\n    224377,\n    224530,\n    224542,\n    224554,\n    224647,\n    224664,\n    224675,\n    224742,\n    224757,\n    224802,\n    224848,\n    224974,\n    225032,\n    225076,\n    225299,\n    225494,\n    225495,\n    225546,\n    225715,\n    225757,\n    225786,\n    225792,\n    225867,\n    225946,\n    226058,\n    226161,\n    226368,\n    226498,\n    226588,\n    226629,\n    226883,\n    227164,\n    227220,\n    227226,\n    227358,\n    227370,\n    227523,\n    227736,\n    227781,\n    227851,\n    227878,\n    227884,\n    228309,\n    228335,\n    228676,\n    228867,\n    228981,\n    229014,\n    229132,\n    229267,\n    229274,\n    229529,\n    229553,\n    229688,\n    229747,\n    229889,\n    229968,\n    230008,\n    230040,\n    230150,\n    230501,\n    230598,\n    230669,\n    230679,\n    230739,\n    230884,\n    230936,\n    230964,\n    230987,\n    231037,\n    231617,\n    231685,\n    231691,\n    231732,\n    231758,\n    231991,\n    232173,\n    232243,\n    232348,\n    232432,\n    232949,\n    233042,\n    233079,\n    233124,\n    233139,\n    233266,\n    233370,\n    233376,\n    233553,\n    233830,\n    233882,\n    233915,\n    233950,\n    233968,\n    233977,\n    234057,\n    234153,\n    234251,\n    234413,\n    234499,\n    234633,\n    234677,\n    234719,\n    234902,\n    234963,\n    235274,\n    235597,\n    235692,\n    235751,\n    235778,\n    235791,\n    235836,\n    235864,\n    236000,\n    236049,\n    236068,\n    236505,\n    236690,\n    236784,\n    236837,\n    237350,\n    237399,\n    237495,\n    237501,\n    237618,\n    237764,\n    237767,\n    237772,\n    237814,\n    238125,\n    238272,\n    238355,\n    238449,\n    238459,\n    238528,\n    238700,\n    238989,\n    239198,\n    239347,\n    239351,\n    239376,\n    239943,\n    240147,\n    240274,\n    240387,\n    240403,\n    240637,\n    240655,\n    240784,\n    240804,\n    241297,\n    241373,\n    241503,\n    241668,\n    242103,\n    242270,\n    242365,\n    242423,\n    242909,\n    242946,\n    243090,\n    243213,\n    243344,\n    243914,\n    243955,\n    243989,\n    244215,\n    244240,\n    244339,\n    244487,\n    244539,\n    244737,\n    245067,\n    245242,\n    245450,\n    245664,\n    246348,\n    246597,\n    246649,\n    246650,\n    246863,\n    246876,\n    247123,\n    247184,\n    247428,\n    247625,\n    247782,\n    247806,\n    248051,\n    248111,\n    248142,\n    248224,\n    248441,\n    248488,\n    248616,\n    248701,\n    248774,\n    248980,\n    249131,\n    249158,\n    249352,\n    249506,\n    249532,\n    249623,\n    249643,\n    249687,\n    249953,\n    249969,\n    250260,\n    250311,\n    250313,\n    250344,\n    250619,\n    251140,\n    251206,\n    251367,\n    251379,\n    251590,\n    251717,\n    251856,\n    251857,\n    251888,\n    251892,\n    252101,\n    252177,\n    252203,\n    252220,\n    252573,\n    252582,\n    252610,\n    252639,\n    252716,\n    252783,\n    252927,\n    253695,\n    253732,\n    253755,\n    253810,\n    253825,\n    253964,\n    254167,\n    254493,\n    254750,\n    254814,\n    254828,\n    254838,\n    254929,\n    254983,\n    255071,\n    255165,\n    255209,\n    255315,\n    255322,\n    255385,\n    255401,\n    255631,\n    255662,\n    255827,\n    256003,\n    256094,\n    256180,\n    256192,\n    256223,\n    256250,\n    256260,\n    256337,\n    256569,\n    256660,\n    256838,\n    256868,\n    256899,\n    256941,\n    256973,\n    257187,\n    257219,\n    257421,\n    257624,\n    257870,\n    257965,\n    257992,\n    258078,\n    258089,\n    258266,\n    258331,\n    258395,\n    258588,\n    258628,\n    258883,\n    258911,\n    258920,\n    258998,\n    259097,\n    259382,\n    259567,\n    259625,\n    259819,\n    259854,\n    259983,\n    259986,\n    260307,\n    260927,\n    260966,\n    261180,\n    261182,\n    261453,\n    261471,\n    261721,\n    261779,\n    261780,\n    261948,\n    262101,\n    262148,\n    262162,\n    262175,\n    262262,\n    262347,\n    262391,\n    262440,\n    262514,\n    262800,\n    262895,\n    263011,\n    263120,\n    263211,\n    263223,\n    263258,\n    263346,\n    263474,\n    263612,\n    263828,\n    263858,\n    263866,\n    264347,\n    264454,\n    265125,\n    265407,\n    265552,\n    265596,\n    265742,\n    265816,\n    265916,\n    266082,\n    266348,\n    266712,\n    266847,\n    266892,\n    266920,\n    266977,\n    267135,\n    267175,\n    267349,\n    267408,\n    267515,\n    267694,\n    267802,\n    267861,\n    267932,\n    268000,\n    268052,\n    268215,\n    268322,\n    268363,\n    268375,\n    268390,\n    268394,\n    268710,\n    268770,\n    268885,\n    268941,\n    268944,\n    269058,\n    269254,\n    269395,\n    269417,\n    269419,\n    269606,\n    269689,\n    269784,\n    269942,\n    270175,\n    270386,\n    270505,\n    270705,\n    270738,\n    270815,\n    271138,\n    271240,\n    271402,\n    271452,\n    271471,\n    271576,\n    271607,\n    271852,\n    271892,\n    271934,\n    272049,\n    272212,\n    272262,\n    272323,\n    273002,\n    273118,\n    273188,\n    273493,\n    273618,\n    273772,\n    273784,\n    273930,\n    274126,\n    274219,\n    274398,\n    274455,\n    274494,\n    275120,\n    275202,\n    275210,\n    275611,\n    275657,\n    275761,\n    275881,\n    276018,\n    276036,\n    276055,\n    276127,\n    276195,\n    276215,\n    276408,\n    276434,\n    276458,\n    277051,\n    277197,\n    277208,\n    277239,\n    277289,\n    277440,\n    277521,\n    277642,\n    277694,\n    278437,\n    278509,\n    278550,\n    278579,\n    278601,\n    278747,\n    278934,\n    279030,\n    279085,\n    279149,\n    279209,\n    279387,\n    279490,\n    279524,\n    279621,\n    279634,\n    279677,\n    279696,\n    279994,\n    280007,\n    280022,\n    280360,\n    280619,\n    280705,\n    280734,\n    280736,\n    280760,\n    280909,\n    281008,\n    281019,\n    281056,\n    281102,\n    281208,\n    281376,\n    281409,\n    281455,\n    281598,\n    281722,\n    281759,\n    281782,\n    281837,\n    281924,\n    281929,\n    281972,\n    282098,\n    282113,\n    282130,\n    282224,\n    282251,\n    282298,\n    282365,\n    282772,\n    282912,\n    283012,\n    283060,\n    283113,\n    283203,\n    283277,\n    283495,\n    283698,\n    283772,\n    283910,\n    283977,\n    284350,\n    284605,\n    285302,\n    285325,\n    285352,\n    285568,\n    285597,\n    285633,\n    285645,\n    285661,\n    285929,\n    286010,\n    286176,\n    286662,\n    286711,\n    286717,\n    286760,\n    286981,\n    287469,\n    287636,\n    287663,\n    287725,\n    287882,\n    287959,\n    288202,\n    288435,\n    288491,\n    288799,\n    288955,\n    289128,\n    289260,\n    289392,\n    289469,\n    289630,\n    289754,\n    289781,\n    289883,\n    290078,\n    290289,\n    290570,\n    290839,\n    290843,\n    290948,\n    291251,\n    291538,\n    291664,\n    291962,\n    291983,\n    292170,\n    292278,\n    292301,\n    292478,\n    292590,\n    292647,\n    292663,\n    292685,\n    292804,\n    292901,\n    292916,\n    293022,\n    293026,\n    293611,\n    293703,\n    293810,\n    293841,\n    293907,\n    294053,\n    294134,\n    294264,\n    294487,\n    294593,\n    294763,\n    294831,\n    294832,\n    294853,\n    294855,\n    294908,\n    295051,\n    295055,\n    295097,\n    295134,\n    295340,\n    295440,\n    295614,\n    295728,\n    295837,\n    296231,\n    296236,\n    296243,\n    296284,\n    296433,\n    296524,\n    296707,\n    296731,\n    297017,\n    297037,\n    297074,\n    297233,\n    297372,\n    297972,\n    298172,\n    298252,\n    298290,\n    298370,\n    298382,\n    298452,\n    298461,\n    298628,\n    298691,\n    298699,\n    298722,\n    298978,\n    299023,\n    299089,\n    299116,\n    299295,\n    299319,\n    299409,\n    299448,\n    299533,\n    299550,\n    299601,\n    299631,\n    299640,\n    299734,\n    299785,\n    299946,\n    299952,\n    300000,\n    300023,\n    300034,\n    300066,\n    300086,\n    300233,\n    300437,\n    300471,\n    300509,\n    300655,\n    300848,\n    301102,\n    301107,\n    301333,\n    301467,\n    301756,\n    301817,\n    301827,\n    301837,\n    301908,\n    301912,\n    302030,\n    302221,\n    302260,\n    302375,\n    302555,\n    302599,\n    302716,\n    302767,\n    302806,\n    302842,\n    302867,\n    303215,\n    303267,\n    303706,\n    303738,\n    303893,\n    304012,\n    304147,\n    304217,\n    304300,\n    304390,\n    304744,\n    304834,\n    305000,\n    305004,\n    305035,\n    305540,\n    305609,\n    305695,\n    305799,\n    305800,\n    305821,\n    306664,\n    306693,\n    306940,\n    307034,\n    307243,\n    307262,\n    307292,\n    307511,\n    307523,\n    307538,\n    307790,\n    307873,\n    307967,\n    307993,\n    308235,\n    308430,\n    308587,\n    308828,\n    308883,\n    308889,\n    308894,\n    309100,\n    309120,\n    309371,\n    309418,\n    309678,\n    309933,\n    310185,\n    310262,\n    310479,\n    310611,\n    310618,\n    310757,\n    310796,\n    310980,\n    311394,\n    311465,\n    311518,\n    311690,\n    311877,\n    311904,\n    311922,\n    311961,\n    312192,\n    312216,\n    312343,\n    312692,\n    312783,\n    312803,\n    312878,\n    313055,\n    313063,\n    313169,\n    313176,\n    313182,\n    313372,\n    313398,\n    313888,\n    314016,\n    314251,\n    314294,\n    314370,\n    314388,\n    314613,\n    314667,\n    314836,\n    314864,\n    314865,\n    314914,\n    315018,\n    315128,\n    315248,\n    315249,\n    315303,\n    315352,\n    315434,\n    315474,\n    315486,\n    315501,\n    315524,\n    315742,\n    315769,\n    315841,\n    315864,\n    315899,\n    315909,\n    316041,\n    316047,\n    316138,\n    316240,\n    316323,\n    316404,\n    316503,\n    316694,\n    316795,\n    317028,\n    317070,\n    317244,\n    317441,\n    317565,\n    317725,\n    317999,\n    318080,\n    318099,\n    318124,\n    318130,\n    318290,\n    318314,\n    318461,\n    318523,\n    318573,\n    318618,\n    318813,\n    319024,\n    319051,\n    319073,\n    319127,\n    319350,\n    319654,\n    319743,\n    319931,\n    319933,\n    320180,\n    320370,\n    320425,\n    320480,\n    320524,\n    320627,\n    320664,\n    321014,\n    321195,\n    321214,\n    321333,\n    321437,\n    321542,\n    321647,\n    321706,\n    321811,\n    321866,\n    321973,\n    322175,\n    322222,\n    322369,\n    322482,\n    322511,\n    322586,\n    322719,\n    322824,\n    322922,\n    322944,\n    322955,\n    323100,\n    323128,\n    323263,\n    323356,\n    323639,\n    324008,\n    324189,\n    324250,\n    324421,\n    324500,\n    324595,\n    324638,\n    324818,\n    324837,\n    325027,\n    325153,\n    325157,\n    325211,\n    325228,\n    325557,\n    325593,\n    325736,\n    325770,\n    325840,\n    325991,\n    326082,\n    326174,\n    326317,\n    326344,\n    326462,\n    326555,\n    326911,\n    326968,\n    327038,\n    327073,\n    327149,\n    327177,\n    327466,\n    327567,\n    327605,\n    327663,\n    327872,\n    327961,\n    328030,\n    328111,\n    328200,\n    328301,\n    328352,\n    328433,\n    328452,\n    328462,\n    329030,\n    329139,\n    329217,\n    329219,\n    329336,\n    329475,\n    329717,\n    329806,\n    330051,\n    330369,\n    330439,\n    330522,\n    330681,\n    330699,\n    330750,\n    330897,\n    331223,\n    331455,\n    331727,\n    331937,\n    332096,\n    332318,\n    332407,\n    332455,\n    332545,\n    332877,\n    333018,\n    333069,\n    333101,\n    333156,\n    333182,\n    333198,\n    333286,\n    333434,\n    333480,\n    333665,\n    333746,\n    333756,\n    333845,\n    334083,\n    334085,\n    334311,\n    334321,\n    334371,\n    334399,\n    334530,\n    334760,\n    334977,\n    335099,\n    335450,\n    335631,\n    335733,\n    335800,\n    335810,\n    335885,\n    335981,\n    336232,\n    336384,\n    336493,\n    336777,\n    336802,\n    337042,\n    337055,\n    337139,\n    337185,\n    337194,\n    337506,\n    337666,\n    337705,\n    337826,\n    338472,\n    338562,\n    338678,\n    338703,\n    339022,\n    339356,\n    339426,\n    339676,\n    339678,\n    339943,\n    340175,\n    340209,\n    340332,\n    340472,\n    340637,\n    340642,\n    340654,\n    340894,\n    341010,\n    341058,\n    341128,\n    341393,\n    341409,\n    341603,\n    341700,\n    341921,\n    342322,\n    342334,\n    342394,\n    342479,\n    342711,\n    342800,\n    342849,\n    343394,\n    343599,\n    343706,\n    343774,\n    343914,\n    343954,\n    344003,\n    344197,\n    344308,\n    344397,\n    344909,\n    344930,\n    345029,\n    345071,\n    345136,\n    345139,\n    345245,\n    345288,\n    345389,\n    345941,\n    346011,\n    346067,\n    346140,\n    346638,\n    346641,\n    346788,\n    346865,\n    346934,\n    347018,\n    347172,\n    347236,\n    347535,\n    347650,\n    347666,\n    347740,\n    347879,\n    347950,\n    348012,\n    348519,\n    348648,\n    348669,\n    348701,\n    348708,\n    348730,\n    348793,\n    348877,\n    348881,\n    348896,\n    348905,\n    348991,\n    349184,\n    349185,\n    349437,\n    349579,\n    349776,\n    349804,\n    350054,\n    350084,\n    350099,\n    350111,\n    350262,\n    350334,\n    350522,\n    350648,\n    350789,\n    350959,\n    350974,\n    351017,\n    351034,\n    351149,\n    351183,\n    351335,\n    351351,\n    351530,\n    351620,\n    352027,\n    352080,\n    352127,\n    352228,\n    352476,\n    352694,\n    352744,\n    352761,\n    352901,\n    352937,\n    353001,\n    353136,\n    353231,\n    353536,\n    353644,\n    353695,\n    353830,\n    354165,\n    354368,\n    354507,\n    354527,\n    354533,\n    354559,\n    354657,\n    354685,\n    354736,\n    354929,\n    355137,\n    355197,\n    355221,\n    355228,\n    355272,\n    355276,\n    355370,\n    355441,\n    355453,\n    355513,\n    355905,\n    355971,\n    356424,\n    356500,\n    356505,\n    356623,\n    356708,\n    356950,\n    357096,\n    357265,\n    357322,\n    357331,\n    357459,\n    357478,\n    357542,\n    357633,\n    357978,\n    358039,\n    358055,\n    358090,\n    358136,\n    358254,\n    358301,\n    358342,\n    358494,\n    358664,\n    358817,\n    358976,\n    358982,\n    359086,\n    359136,\n    359184,\n    359270,\n    359465,\n    359676,\n    359715,\n    359791,\n    359838,\n    359855,\n    360101,\n    360143,\n    360170,\n    360181,\n    360318,\n    360346,\n    360504,\n    360610,\n    360611,\n    360678,\n    360730,\n    360762,\n    360818,\n    360943,\n    360960,\n    361027,\n    361123,\n    361147,\n    361177,\n    361400,\n    361472,\n    361481,\n    361527,\n    361551,\n    361586,\n    361739,\n    361751,\n    361763,\n    361804,\n    361860,\n    361919,\n    361924,\n    361992,\n    362122,\n    362159,\n    362275,\n    362343,\n    362368,\n    362831,\n    363100,\n    363188,\n    363508,\n    363594,\n    363793,\n    364113,\n    364166,\n    364340,\n    364592,\n    364783,\n    365094,\n    365121,\n    365139,\n    365325,\n    365685,\n    365745,\n    365851,\n    365945,\n    366031,\n    366096,\n    366111,\n    366157,\n    366508,\n    366529,\n    366630,\n    366830,\n    366852,\n    366950,\n    367018,\n    367329,\n    367579,\n    367843,\n    368088,\n    368117,\n    368218,\n    368220,\n    368222,\n    368505,\n    368528,\n    368961,\n    368994,\n    369243,\n    369373,\n    369506,\n    369541,\n    369771,\n    369913,\n    369934,\n    369979,\n    370138,\n    370175,\n    370678,\n    370749,\n    370819,\n    370839,\n    371004,\n    371058,\n    371135,\n    371176,\n    371250,\n    371395,\n    371822,\n    371945,\n    371957,\n    371958,\n    372191,\n    372252,\n    372384,\n    372471,\n    372495,\n    372577,\n    372633,\n    373007,\n    373089,\n    373119,\n    373333,\n    373422,\n    373846,\n    374061,\n    374083,\n    374369,\n    374574,\n    374677,\n    375078,\n    375205,\n    375232,\n    375285,\n    375324,\n    375407,\n    375426,\n    375440,\n    375484,\n    375769,\n    375786,\n    375840,\n    375915,\n    376177,\n    376236,\n    376278,\n    376321,\n    376322,\n    376372,\n    376493,\n    376677,\n    376891,\n    376900,\n    377000,\n    377111,\n    377239,\n    377282,\n    377314,\n    377368,\n    377730,\n    377738,\n    377882,\n    377946,\n    377984,\n    378098,\n    378506,\n    378545,\n    378657,\n    378667,\n    378673,\n    378701,\n    378709,\n    378765,\n    378814,\n    378860,\n    378962,\n    378968,\n    379108,\n    379158,\n    379190,\n    379332,\n    379475,\n    379529,\n    379734,\n    379922,\n    380011,\n    380143,\n    380162,\n    380300,\n    380493,\n    380639,\n    380734,\n    380854,\n    381032,\n    381051,\n    381123,\n    381262,\n    381360,\n    381413,\n    381587,\n    382041,\n    382089,\n    382214,\n    382696,\n    382743,\n    382999,\n    383018,\n    383211,\n    383282,\n    383337,\n    383443,\n    383513,\n    383676,\n    384012,\n    384213,\n    384333,\n    384462,\n    384723,\n    384822,\n    384981,\n    385005,\n    385194,\n    385378,\n    385508,\n    385514,\n    385580,\n    385598,\n    386164,\n    386204,\n    386227,\n    386613,\n    386739,\n    386764,\n    386766,\n    386967,\n    387576,\n    387776,\n    388027,\n    388161,\n    388225,\n    388481,\n    388487,\n    388528,\n    388531,\n    388643,\n    388658,\n    388677,\n    388770,\n    388983,\n    388988,\n    389283,\n    389378,\n    389464,\n    389480,\n    389644,\n    390301,\n    390475,\n    390759,\n    390795,\n    391006,\n    391016,\n    391199,\n    391214,\n    391365,\n    391400,\n    391895,\n    391915,\n    391940,\n    392212,\n    392640,\n    392878,\n    392892,\n    393004,\n    393031,\n    393068,\n    393145,\n    393271,\n    393338,\n    393410,\n    393583,\n    393647,\n    393777,\n    393869,\n    393942,\n    394240,\n    394608,\n    394801,\n    395124,\n    395185,\n    395290,\n    395382,\n    395402,\n    395621,\n    395644,\n    395752,\n    396051,\n    396209,\n    396350,\n    396542,\n    396691,\n    396692,\n    396693,\n    396779,\n    397041,\n    397042,\n    397045,\n    397404,\n    398005,\n    398007,\n    398067,\n    398385,\n    398606,\n    398810,\n    398882,\n    399049,\n    399178,\n    399205,\n    399284,\n    399296,\n    399703,\n    399769,\n    399790,\n    399851,\n    399921,\n    399957,\n    399991,\n    400235,\n    400317,\n    400453,\n    400886,\n    400985,\n    401028,\n    401249,\n    401591,\n    401623,\n    401797,\n    401808,\n    401860,\n    401862,\n    401935,\n    402000,\n    402248,\n    402330,\n    402588,\n    402674,\n    402685,\n    402786,\n    402788,\n    403122,\n    403180,\n    403378,\n    403385,\n    403421,\n    403500,\n    403821,\n    403863,\n    403864,\n    403937,\n    404088,\n    404128,\n    404338,\n    404388,\n    404461,\n    404462,\n    404568,\n    404608,\n    404613,\n    404886,\n    404922,\n    404931,\n    405249,\n    405365,\n    405432,\n    405440,\n    405531,\n    405614,\n    405660,\n    405811,\n    405815,\n    406070,\n    406155,\n    406182,\n    406329,\n    406377,\n    406395,\n    406647,\n    406744,\n    406899,\n    406908,\n    406976,\n    407042,\n    407072,\n    407260,\n    407298,\n    407524,\n    407532,\n    407717,\n    407915,\n    407943,\n    408049,\n    408057,\n    408120,\n    408327,\n    408364,\n    408449,\n    408542,\n    408621,\n    408757,\n    408774,\n    408789,\n    408818,\n    408859,\n    408950,\n    409052,\n    409088,\n    409100,\n    409119,\n    409217,\n    409331,\n    409667,\n    409701,\n    410002,\n    410101,\n    410231,\n    410278,\n    410428,\n    410482,\n    410597,\n    410612,\n    410641,\n    410670,\n    410724,\n    410855,\n    410880,\n    410885,\n    410924,\n    410979,\n    411093,\n    411138,\n    411405,\n    411438,\n    411630,\n    411666,\n    411685,\n    411821,\n    411845,\n    411941,\n    411968,\n    412184,\n    412240,\n    412355,\n    412419,\n    412464,\n    412483,\n    412584,\n    412604,\n    412676,\n    412767,\n    412807,\n    412813,\n    412978,\n    413056,\n    413096,\n    413120,\n    413419,\n    413734,\n    413852,\n    413955,\n    414071,\n    414078,\n    414228,\n    414261,\n    414374,\n    414385,\n    414664,\n    414884,\n    414934,\n    415020,\n    415153,\n    415196,\n    415432,\n    415727,\n    415741,\n    415746,\n    415748,\n    415770,\n    415880,\n    416184,\n    416331,\n    416405,\n    416534,\n    416668,\n    416746,\n    416968,\n    416991,\n    417164,\n    417281,\n    417284,\n    417416,\n    417586,\n    417696,\n    418106,\n    418143,\n    418281,\n    418523,\n    418563,\n    419158,\n    419228,\n    419344,\n    419371,\n    419723,\n    419859,\n    419860,\n    419884,\n    419971,\n    420069,\n    420113,\n    420313,\n    420422,\n    420546,\n    420579,\n    420666,\n    420669,\n    420958,\n    420963,\n    421072,\n    421109,\n    421195,\n    421361,\n    421471,\n    421564,\n    421923,\n    421979,\n    421999,\n    422157,\n    422200,\n    422212,\n    422221,\n    422326,\n    422517,\n    422545,\n    422676,\n    422998,\n    423005,\n    423141,\n    423309,\n    423337,\n    423576,\n    423618,\n    423830,\n    423971,\n    424172,\n    424174,\n    424208,\n    424220,\n    424422,\n    424432,\n    424585,\n    424776,\n    424812,\n    424880,\n    424975,\n    424998,\n    425000,\n    425100,\n    425151,\n    425475,\n    425522,\n    425526,\n    425573,\n    425609,\n    425620,\n    425906,\n    426166,\n    426172,\n    426203,\n    426532,\n    426578,\n    426878,\n    427111,\n    427338,\n    427376,\n    427401,\n    427646,\n    427666,\n    427802,\n    427997,\n    428064,\n    428291,\n    428447,\n    428867,\n    429063,\n    429108,\n    429142,\n    429174,\n    429283,\n    429593,\n    429623,\n    429633,\n    429833,\n    429834,\n    429913,\n    430160,\n    430286,\n    430420,\n    430469,\n    430762,\n    430856,\n    430961,\n    431197,\n    431521,\n    431545,\n    431570,\n    431615,\n    431660,\n    431832,\n    431891,\n    431996,\n    432120,\n    432234,\n    432519,\n    432534,\n    432547,\n    432570,\n    432588,\n    432605,\n    432657,\n    432799,\n    432993,\n    433136,\n    433243,\n    433574,\n    433625,\n    433804,\n    433883,\n    433892,\n    433904,\n    433924,\n    434129,\n    434177,\n    434279,\n    434463,\n    434510,\n    434804,\n    434900,\n    434915,\n    435034,\n    435037,\n    435069,\n    435270,\n    435324,\n    435807,\n    435922,\n    436044,\n    436127,\n    436141,\n    436252,\n    436348,\n    436350,\n    436385,\n    436426,\n    436467,\n    436676,\n    436858,\n    436865,\n    436992,\n    437126,\n    437325,\n    437599,\n    437789,\n    438038,\n    438221,\n    438226,\n    438232,\n    438269,\n    438373,\n    438535,\n    438721,\n    438738,\n    438774,\n    438850,\n    438915,\n    438985,\n    438999,\n    439015,\n    439092,\n    439180,\n    439326,\n    439398,\n    439472,\n    439589,\n    439651,\n    439774,\n    439781,\n    439834,\n    440067,\n    440354,\n    440514,\n    440871,\n    440895,\n    441009,\n    441028,\n    441147,\n    441240,\n    441292,\n    441363,\n    441453,\n    441472,\n    441473,\n    441496,\n    441553,\n    442165,\n    442214,\n    442305,\n    442463,\n    442667,\n    442836,\n    442861,\n    442942,\n    442993,\n    443299,\n    443537,\n    443583,\n    443689,\n    443887,\n    443949,\n    443969,\n    444302,\n    444444,\n    444722,\n    444746,\n    444927,\n    445041,\n    445233,\n    445512,\n    445668,\n    445812,\n    445908,\n    446459,\n    446522,\n    446597,\n    446799,\n    447080,\n    447087,\n    447314,\n    447407,\n    447460,\n    447548,\n    447787,\n    447789,\n    448263,\n    448278,\n    448365,\n    448626,\n    448627,\n    448697,\n    448786,\n    448787,\n    448795,\n    448871,\n    448897,\n    448923,\n    449114,\n    449180,\n    449191,\n    449274,\n    449338,\n    449403,\n    449485,\n    449634,\n    449675,\n    449828,\n    449872,\n    449879,\n    450037,\n    450355,\n    450370,\n    450399,\n    450404,\n    450559,\n    450724,\n    450728,\n    450762,\n    450845,\n    451095,\n    451234,\n    451305,\n    451431,\n    451621,\n    451683,\n    451751,\n    451872,\n    451896,\n    451972,\n    451990,\n    452078,\n    452084,\n    452113,\n    452122,\n    452414,\n    452623,\n    452663,\n    452724,\n    452784,\n    452881,\n    452899,\n    453013,\n    453283,\n    453297,\n    453302,\n    453400,\n    453622,\n    453649,\n    453860,\n    453895,\n    453903,\n    454102,\n    454195,\n    454252,\n    454296,\n    454444,\n    454577,\n    454607,\n    454693,\n    454776,\n    454907,\n    454957,\n    454978,\n    455157,\n    455170,\n    455219,\n    455275,\n    455290,\n    455299,\n    455464,\n    455588,\n    455675,\n    455698,\n    455716,\n    455859,\n    455974,\n    456046,\n    456192,\n    456519,\n    456521,\n    456552,\n    456611,\n    456690,\n    457054,\n    457249,\n    457683,\n    457717,\n    457718,\n    457900,\n    458045,\n    458052,\n    458119,\n    458137,\n    458177,\n    458240,\n    458549,\n    458558,\n    458560,\n    458567,\n    458721,\n    458750,\n    458755,\n    459141,\n    459374,\n    459437,\n    459590,\n    459665,\n    459757,\n    459922,\n    460243,\n    460251,\n    460346,\n    460361,\n    460379,\n    460390,\n    460454,\n    460461,\n    460505,\n    460621,\n    460682,\n    461130,\n    461278,\n    461378,\n    461884,\n    462026,\n    462031,\n    462105,\n    462213,\n    462345,\n    462364,\n    462565,\n    462640,\n    462755,\n    462784,\n    462814,\n    462929,\n    462982,\n    463084,\n    463203,\n    463211,\n    463226,\n    463283,\n    463325,\n    463398,\n    463454,\n    463459,\n    463542,\n    463610,\n    463640,\n    463653,\n    463825,\n    463918,\n    464092,\n    464286,\n    464357,\n    465069,\n    465087,\n    465357,\n    465418,\n    465424,\n    465463,\n    465537,\n    465550,\n    465585,\n    465591,\n    465735,\n    465862,\n    465911,\n    465973,\n    466052,\n    466156,\n    466259,\n    466392,\n    466422,\n    466456,\n    466818,\n    466882,\n    466986,\n    467031,\n    467062,\n    467067,\n    467135,\n    467197,\n    467437,\n    467467,\n    467477,\n    467540,\n    467578,\n    467721,\n    467755,\n    467887,\n    468197,\n    468302,\n    468337,\n    468363,\n    468541,\n    468704,\n    468933,\n    468935,\n    468993,\n    468997,\n    469002,\n    469085,\n    469192,\n    469577,\n    469618,\n    469777,\n    469793,\n    469803,\n    469961,\n    469996,\n    470005,\n    470366,\n    470417,\n    470513,\n    470623,\n    470952,\n    470984,\n    471013,\n    471015,\n    471179,\n    471497,\n    471566,\n    471641,\n    471642,\n    471869,\n    472088,\n    472101,\n    472109,\n    472143,\n    472216,\n    472295,\n    472415,\n    472557,\n    472732,\n    472762,\n    472795,\n    472833,\n    472854,\n    473069,\n    473133,\n    473171,\n    473237,\n    473318,\n    473340,\n    473406,\n    473573,\n    473728,\n    473754,\n    474384,\n    474402,\n    474519,\n    474608,\n    474675,\n    474751,\n    474934,\n    474974,\n    475053,\n    475150,\n    475396,\n    475398,\n    475403,\n    475667,\n    475906,\n    475923,\n    476065,\n    476109,\n    476215,\n    476282,\n    476339,\n    476406,\n    476552,\n    476754,\n    476925,\n    476939,\n    477087,\n    477194,\n    477202,\n    477477,\n    477483,\n    477526,\n    477867,\n    477924,\n    478077,\n    478155,\n    478282,\n    478528,\n    478621,\n    478766,\n    478858,\n    478922,\n    479030,\n    479213,\n    479508,\n    479562,\n    479612,\n    479832,\n    479948,\n    480000,\n    480021,\n    480215,\n    480605,\n    480643,\n    480720,\n    480747,\n    480951,\n    481187,\n    481252,\n    481314,\n    481325,\n    481398,\n    481446,\n    482021,\n    482022,\n    482081,\n    482160,\n    482436,\n    482562,\n    482690,\n    482907,\n    482994,\n    483108,\n    483149,\n    483275,\n    483311,\n    483348,\n    483545,\n    483722,\n    483767,\n    484029,\n    484042,\n    484080,\n    484145,\n    484175,\n    484301,\n    484303,\n    484312,\n    484415,\n    484434,\n    484551,\n    484587,\n    484681,\n    484695,\n    484938,\n    485485,\n    485494,\n    485731,\n    486046,\n    486568,\n    486573,\n    487076,\n    487141,\n    487217,\n    487222,\n    487236,\n    487349,\n    487375,\n    487450,\n    487525,\n    487685,\n    487741,\n    487774,\n    487898,\n    487957,\n    488198,\n    488219,\n    488401,\n    488539,\n    488697,\n    488723,\n    488736,\n    489611,\n    489624,\n    489739,\n    489763,\n    489861,\n    490008,\n    490337,\n    490413,\n    490509,\n    490620,\n    490643,\n    490688,\n    490741,\n    490878,\n    491064,\n    491090,\n    491102,\n    491497,\n    491653,\n    491757,\n    491765,\n    491793,\n    491835,\n    491902,\n    491947,\n    491985,\n    492067,\n    492132,\n    492282,\n    492420,\n    492469,\n    492506,\n    492552,\n    492683,\n    492814,\n    492817,\n    493284,\n    493623,\n    493797,\n    494077,\n    494404,\n    494439,\n    494566,\n    494578,\n    494608,\n    494808,\n    494957,\n    495081,\n    495160,\n    495183,\n    495349,\n    495377,\n    495519,\n    495985,\n    496207,\n    496213,\n    496525,\n    496541,\n    496575,\n    496597,\n    496604,\n    497014,\n    497034,\n    497067,\n    497094,\n    497334,\n    497438,\n    497719,\n    497907,\n    498439,\n    498555,\n    498583,\n    498665,\n    499095,\n    499181,\n    499198,\n    499200,\n    499266,\n    499423,\n    499537,\n    499705,\n    499755,\n    499937,\n    500019,\n    500211,\n    500446,\n    500703,\n    500780,\n    500825,\n    500844,\n    501116,\n    501122,\n    501420,\n    501429,\n    501491,\n    501614,\n    501652,\n    502090,\n    502240,\n    502419,\n    502558,\n    502671,\n    502766,\n    502852,\n    502854,\n    502936,\n    502959,\n    503005,\n    503137,\n    503183,\n    503311,\n    503470,\n    503595,\n    503826,\n    503980,\n    504005,\n    504074,\n    504101,\n    504152,\n    504248,\n    504293,\n    504341,\n    504414,\n    504415,\n    504747,\n    504790,\n    504811,\n    504813,\n    504824,\n    504891,\n    504958,\n    505636,\n    505650,\n    505739,\n    506201,\n    506310,\n    506357,\n    506416,\n    506458,\n    506489,\n    506568,\n    506577,\n    506872,\n    507065,\n    507081,\n    507147,\n    507352,\n    507686,\n    507921,\n    508302,\n    508370,\n    508586,\n    508672,\n    508917,\n    509270,\n    509497,\n    509656,\n    509807,\n    509819,\n    510657,\n    510955,\n    511145,\n    511583,\n    512116,\n    512183,\n    512337,\n    512467,\n    512476,\n    512576,\n    512760,\n    512776,\n    512785,\n    512793,\n    513041,\n    513060,\n    513064,\n    513073,\n    513507,\n    513567,\n    513611,\n    513699,\n    514147,\n    514366,\n    514402,\n    514437,\n    514468,\n    514787,\n    515056,\n    515241,\n    515303,\n    515347,\n    515531,\n    515540,\n    515668,\n    515727,\n    515755,\n    515760,\n    516020,\n    516143,\n    516178,\n    516316,\n    516408,\n    516416,\n    516601,\n    516750,\n    516766,\n    516786,\n    516813,\n    516875,\n    517296,\n    517610,\n    517629,\n    517687,\n    517967,\n    518234,\n    518255,\n    518375,\n    518530,\n    518551,\n    518725,\n    518729,\n    518873,\n    519039,\n    519046,\n    519208,\n    519359,\n    519555,\n    519569,\n    519611,\n    519631,\n    519744,\n    519764,\n    519815,\n    520012,\n    520043,\n    520108,\n    520109,\n    520430,\n    520482,\n    520508,\n    520525,\n    520530,\n    520787,\n    520871,\n    520918,\n    520933,\n    521131,\n    521142,\n    521306,\n    521427,\n    521879,\n    522273,\n    522427,\n    522452,\n    522489,\n    522889,\n    522909,\n    523175,\n    523194,\n    523816,\n    523978,\n    524002,\n    524011,\n    524064,\n    524173,\n    524637,\n    524638,\n    525118,\n    525170,\n    525272,\n    525297,\n    525373,\n    525376,\n    525568,\n    525616,\n    525619,\n    525646,\n    525763,\n    525777,\n    525899,\n    525903,\n    526030,\n    526126,\n    526222,\n    526256,\n    526257,\n    526319,\n    526364,\n    526414,\n    526555,\n    526560,\n    526645,\n    526737,\n    526806,\n    526955,\n    526972,\n    527164,\n    527248,\n    527313,\n    527407,\n    527464,\n    527535,\n    527575,\n    527625,\n    527631,\n    527828,\n    527846,\n    527906,\n    527961,\n    527962,\n    528011,\n    528575,\n    528604,\n    528729,\n    528862,\n    528905,\n    528962,\n    528984,\n    529256,\n    529578,\n    529592,\n    529626,\n    529631,\n    529668,\n    529691,\n    529813,\n    529950,\n    529981,\n    530162,\n    530207,\n    530226,\n    530317,\n    530383,\n    530706,\n    530913,\n    531163,\n    531264,\n    531414,\n    531602,\n    531622,\n    531707,\n    531739,\n    531797,\n    531875,\n    532085,\n    532211,\n    532295,\n    532403,\n    532568,\n    532580,\n    532610,\n    532695,\n    532780,\n    532855,\n    532867,\n    533013,\n    533281,\n    533407,\n    533485,\n    533493,\n    533979,\n    534121,\n    534162,\n    534444,\n    534533,\n    534605,\n    534733,\n    534983,\n    535080,\n    535183,\n    535245,\n    535262,\n    535292,\n    535306,\n    535313,\n    535578,\n    535602,\n    535748,\n    536028,\n    536038,\n    536158,\n    536183,\n    536429,\n    536589,\n    536615,\n    536653,\n    536725,\n    536791,\n    537007,\n    537053,\n    537574,\n    537589,\n    537939,\n    537982,\n    537991,\n    538092,\n    538122,\n    538451,\n    538695,\n    538819,\n    538934,\n    538976,\n    539079,\n    539124,\n    539355,\n    539423,\n    539678,\n    539768,\n    539784,\n    539904,\n    539926,\n    540187,\n    540280,\n    540531,\n    540636,\n    540714,\n    540774,\n    540806,\n    540860,\n    541071,\n    541202,\n    541474,\n    541478,\n    541587,\n    541627,\n    541664,\n    541965,\n    542077,\n    542089,\n    542150,\n    542325,\n    542388,\n    542509,\n    542625,\n    542634,\n    542755,\n    543065,\n    543300,\n    543322,\n    543631,\n    544365,\n    544444,\n    544825,\n    544883,\n    544975,\n    545007,\n    545039,\n    545111,\n    545129,\n    545385,\n    545441,\n    545597,\n    545612,\n    545734,\n    545756,\n    545796,\n    545867,\n    546052,\n    546067,\n    546095,\n    546126,\n    546222,\n    546325,\n    546378,\n    546500,\n    546773,\n    546854,\n    546964,\n    546965,\n    546976,\n    547155,\n    547258,\n    547502,\n    547601,\n    547839,\n    548014,\n    548361,\n    548423,\n    548538,\n    548703,\n    548795,\n    549012,\n    549136,\n    549261,\n    549797,\n    550013,\n    550051,\n    550084,\n    550322,\n    550349,\n    550365,\n    550422,\n    550452,\n    550529,\n    550576,\n    550627,\n    550707,\n    550968,\n    551107,\n    551338,\n    551654,\n    551944,\n    551974,\n    551987,\n    552217,\n    552503,\n    552564,\n    552654,\n    552837,\n    553078,\n    553097,\n    553154,\n    553221,\n    553420,\n    553442,\n    553447,\n    553547,\n    553558,\n    553576,\n    553678,\n    553776,\n    553788,\n    553869,\n    553879,\n    553942,\n    553968,\n    553990,\n    554037,\n    554154,\n    554241,\n    554291,\n    554445,\n    554500,\n    554595,\n    554618,\n    554749,\n    554859,\n    555337,\n    555387,\n    555763,\n    556073,\n    556083,\n    556120,\n    556158,\n    556205,\n    556341,\n    556349,\n    556545,\n    556562,\n    556608,\n    556653,\n    556726,\n    556901,\n    557130,\n    557135,\n    557346,\n    557501,\n    557552,\n    557564,\n    557731,\n    557732,\n    557998,\n    558013,\n    558064,\n    558083,\n    558137,\n    558235,\n    558388,\n    558406,\n    558483,\n    558585,\n    559047,\n    559364,\n    559440,\n    559550,\n    559656,\n    559665,\n    559773,\n    559884,\n    560010,\n    560218,\n    560439,\n    560591,\n    560623,\n    560718,\n    560744,\n    560804,\n    560880,\n    560918,\n    561014,\n    561100,\n    561403,\n    561491,\n    561670,\n    561750,\n    561856,\n    561928,\n    561938,\n    562084,\n    562121,\n    562207,\n    562345,\n    562356,\n    562381,\n    562419,\n    562448,\n    562582,\n    562592,\n    562628,\n    562778,\n    562843,\n    562897,\n    563175,\n    563311,\n    563665,\n    563797,\n    563927,\n    564088,\n    564091,\n    564163,\n    564280,\n    564382,\n    564612,\n    565012,\n    565031,\n    565186,\n    565198,\n    565239,\n    565331,\n    565389,\n    566314,\n    566529,\n    566634,\n    566644,\n    566717,\n    567011,\n    567013,\n    567329,\n    567565,\n    567657,\n    567787,\n    567863,\n    567886,\n    567922,\n    567944,\n    568040,\n    568107,\n    568115,\n    568132,\n    568318,\n    568325,\n    568337,\n    568961,\n    569059,\n    569096,\n    569289,\n    569849,\n    569931,\n    569969,\n    569976,\n    570107,\n    570138,\n    570465,\n    570521,\n    570573,\n    570678,\n    570688,\n    570736,\n    570755,\n    570756,\n    570834,\n    571449,\n    571518,\n    571641,\n    571746,\n    571786,\n    571804,\n    571821,\n    571876,\n    571990,\n    572055,\n    572182,\n    572226,\n    572233,\n    572370,\n    572408,\n    572427,\n    572574,\n    572733,\n    572767,\n    572784,\n    573067,\n    573223,\n    573626,\n    573756,\n    573881,\n    573896,\n    573962,\n    574138,\n    574256,\n    574280,\n    574316,\n    574633,\n    574666,\n    574672,\n    574804,\n    575018,\n    575135,\n    575179,\n    575194,\n    575367,\n    575389,\n    575410,\n    575643,\n    575904,\n    575915,\n    575916,\n    576084,\n    576222,\n    576363,\n    576369,\n    576463,\n    576566,\n    576576,\n    576714,\n    576820,\n    577024,\n    577033,\n    577160,\n    577277,\n    577385,\n    577403,\n    577417,\n    577524,\n    577526,\n    577853,\n    577866,\n    578093,\n    578314,\n    578492,\n    578698,\n    578736,\n    578740,\n    578848,\n    578977,\n    578993,\n    579056,\n    579240,\n    579307,\n    579548,\n    579664,\n    579759,\n    580591,\n    580613,\n    580778,\n    580829,\n    580835,\n    580931,\n    581010,\n    581033,\n    581253,\n    581332,\n    581357,\n    581482,\n    581886,\n    581929\n  ],\n  "test_id_sha256": "18a173e42d695050f9700423634f3caee4401a03ba9481320f57044806a8bc95",\n  "selection": "checkpoint fixed by VALID before TEST; rank0 raw beam cumulative logprob",\n  "prior_project_test_exposure": true\n}\n', 'TEST199_PROTOCOL.md': '# Fixed checkpoint199 TEST5000 — 09/11/2026\n\nUser explicitly approved TEST. Freeze best=endepoch10 from ducanh2403/2-vi-cai-kia-qua-nhieu/199, SHA5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7. No training/SCST/optimizer restore or model selection. Same source model, beam5/max20/raw logprob/no lengthnormalization/rerank as VALID; priorprojectTESTexposure disclosed. Strict5000KarpathyTEST IDs frozen from historicalTESTsnapshot and COCOdatasetSHA; reference captions used only for metrics/NLL, never generation or selecting candidates. TRAIN-only vocabularyexactSHA same as199. No prompt/bank/detector/ROI needed.\n\nOnly originalvisual-cache H5 +dataset_coco.json. Sequential selectedTESTrows to temporary local196patchfloat16 array~1.4GiB, originalprofile/shape/IDs/filename/finite/duplicate coverage/bytes checks; no TRAIN/VALIDfeature reads. AllfullH5headerIDs may be inspected. Store references/vocabulary separately, then2T4 independentworkers fixed2500images/rank, batched8images beam5 fromexactcheckpoint; no testtuning. Save captions/IDs/rawscores beforeCPUmetrics; NLL usesallTESTreferencecaptionevents in separateteacherforced loop after captionssaved. Keep lock/inputseals.\n\nIf no optionallocalcheckpointroot, downloadONLYbest.bin fromexactown199 via enabledKaggleSecret codex or KAGGLE_API_TOKEN; pinnedCLIrevision/exactversionroutingmockpreflight, fileSHA check, no latest/foreignnotebook/cache download. No tokenlogging. Samepackage runtimeTorch2.11cu128 checked; do not reinstallTorch. GPUphase uses no optimizer/backward; CUDA2T4/NCCL guard. Author only NumPy/mock/AST; user runs realIO/CUDA/officialscorers.\n\nOne notebook(parent2hstopcap, notETA/billing): setup/test/data/strictcheckpointlock, metricenvironment/syntheticPTB/METEOR/SPICE smoke, full5000decode, teacherNLL, savecaptions. CPUphases basicBLEU1/4/CIDEr/ROUGE-L, METEOR, SPICE, CHAIRs/i/recall, independentphaseoutputs thencombine. Reuse proven pycocoevalcap1.2 Java8/CoreNLP3.6 environment helper. StanfordmodelarchiveSHA recorded notindependentlypinned, same limitation aspriorprojectevaluation; no fullCoreNLPsubsets claims. SPICE save perimageAllF only(finite), notundefinedNaNsubcategories. All8required beforecomplete; failurepreservesreview/savedcaptions so no needautomaticmodelrerun. No model/scorer runlocally. Keep downloadedweights/output; zip excludesbinary/features. No Word edits. Record newresults inexistingSheet onlyafter userrun.\n', 'TEST199_BUILD_V2_PROTOCOL.md': '# TEST199 V2 — build compatibility fix, 10/11/2026\n\nV1retry downloaded exact199/SHA5061e305..., completed5000TESTstage559.44s and metricenvironment/synthetic smoke; no captionworker completed. Bothranks stopped at exactTorchstringguard: trained2.11.0+cu128 vs current2.11.0+cu130. Earlier5data/static tests andCUDA2T4runtime passed. Data/model/secret are not the stoppingcause this time. Do not conflate this with stuckCancelling sessions.\n\nV2 changes only environment-compatibility policy: accept exactly observedTorch2.11.0 CUDAwheel buildscu128/cu130, require trainingversioncu128, rejectCPU/otherpublicreleases/unreviewedbuilds. This is not blanket removal of guards or an assurance of identical floatingpoint/beam results. Record bothversions/buildchanged incheckpointpreflight,eachworker andfinalresult. Same checkpointSHA,epoch10,sourcemodel/beam/rank0/no lengthnormalization/no rerank, noTESTtuning/training. Strictstate_dictload, finite/pathlikelihood rescoringonfirstGPUbeam batch andcomplete2×2500IDsremainmandatory. RuntimepreflightbeforeCoreNLPdownload/TESTstage catches futureunsupportedversions early. No Torch reinstall/downgrade oroptimizer use. CPUmetadataunit teststestedlocally, userrunsGPU/officialscorers.\n\nFreshoutputclip_patch_199_test5000_v2, same2hstopcap/twoinputs/T4x2/secretnames. Reuse exactalreadydownloaded199bestbin atknownoldWorkingpath ifpresentandSHAvalid, elseonlyown199download; no broadpathscan/latest fallback. ExistingcompleteTEST features remaininoldtmp but notautoresumed/copied acrossruns. Original5000TESTref/patchprofile/metadata sealsunchanged, onlyTESTfeaturesstaged~1.4GiB. Eightmetrics+Recall/NLLrequired; captions/phaseoutputspreservedonfailure. DeliverV2sourcepinnednotebook andrecordlimits. References: PyTorchstate_dict/map_location tutorial supportsportableloading generally, not bitwiseproof acrosstheseCUDAwheels: https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html .\n', 'SMOKE_SOURCE.json': '{\n  "archive_sha256": "d6353b2adca467be2f2435e12fe31e317c17b2e9267efeccb2415c02e4d9b45a",\n  "version": 192,\n  "feature_profile": {\n    "model_id": "openai/clip-vit-base-patch16",\n    "output": "vision_model.last_hidden_state",\n    "preprocessing": "bilinear",\n    "image_size": [\n      224,\n      224\n    ],\n    "antialias": true,\n    "dataset_json_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "model_revision": null,\n    "mean": [\n      0.48145466,\n      0.4578275,\n      0.40821073\n    ],\n    "std": [\n      0.26862954,\n      0.26130258,\n      0.27577711\n    ],\n    "compute_dtype": "float32",\n    "storage_dtype": "float16",\n    "autocast": false\n  },\n  "vocabulary_sha256": "cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd",\n  "raw_selected_row_sha256": {\n    "32682": "4f0972bede6f3bb2d0c08c47d765afa30057cc221140207bc7160a03d0f4a684",\n    "432898": "d89b6dc04255ca587f3f137e3a9196662a21627a2fb72e0f6f1a2b1632f00c6a",\n    "337246": "ee4d8be1fdd6324d73c82626caacbb13291e9d0be8c413ac65c1b31fdf61e123",\n    "411871": "60e6cbcb2e8ecfbb8e8fda6e1b7d0b1e03ab9e4e82178d137268a6fe91bf0660",\n    "31280": "5914069fda1b6e2c9958f5585826a8a4ba59c60a0f6dab6c63d405b585a08321",\n    "303260": "689df55ff0b69f08e4c7b630c790626dbc46519a6d8a72aee621e394e5a1d30f",\n    "478665": "9b95376163897b507cd9c483b57c966785988309b8d0b2d9da8a8e8b853c8aff",\n    "155540": "23ea87d964d0e3bf9934643ee2e8a2476a002e1991de66844c2e171f57c44c86",\n    "456236": "504a5b6d1c2566a61ff05a4dfa8075f63bfe80b455bda716947405b890a46af5",\n    "419116": "b2ad30e5f0102825028f3e7b70e949145848e6a5b360dc8483430e572837c05e",\n    "28646": "bd0baba12de7a000172abc624f369e201aded42ddd13fa1fb0ba2a379f293f1e",\n    "407121": "b6da110d89d9abff9b8c7d8da1864679bae822c694a36873ed2d005ef50ba196",\n    "482626": "06bb06afcc51ff92985509b8029cf599125176bfc22824e6f2e3c62d9624dff1",\n    "97599": "fcf34e149aad871d4e126b0eea222c2488036327c314c11fd7d24de1757dff5c",\n    "64183": "5bf007a2a52b9187b35ee12a7d28f85240b413422ae45126da5f128f8191d9bc",\n    "301082": "cefd9cdfc4fc68ba10e60681d4663a44de16a2df299ba5c0d2b703b54accc39a",\n    "472445": "1596f5ff8636e0529eca9be23d19b843d380c08c8cbde769eeb5d2d77915834d",\n    "429839": "a379e5940de7f0f5c2909903e1bb54b2687020baa2687fb3eb37e8c1adfd1559",\n    "389404": "07eef654ba4fd4a50905844c9d7e7fbd0469373446bad55f60189e55a394cb70",\n    "290659": "5770ddce8b60f020f48eee897eddd7e2418892e0880df87469ee27a7af193f0c",\n    "189504": "1a0835b57030ceef86516f456d03a84a4b9295b19053899c4e7b0340c16b5ee9",\n    "77479": "a12f2e4b4e22ce32f1f8ea4643dca24a2fb2d521da9ba62a67fc03aee7ac8f37",\n    "89781": "ede9ef7ad497d3fa0fc37935dc9002c70cdf14bd0e82f73802e2699ef8a86afb",\n    "363241": "e6da8fdb3ced00e02d3e7c118335e26bc02cea08bf8f3a7890b4416f7db85206",\n    "85914": "ff153115805e6dca1999bbdcd1a9268a0d18470ccb37b91a3e80f62f8031b2e0",\n    "486986": "b513f8beb6d3fd3f6e436d6b40ee80bb2d9a28869e70d6bdf2289f6a86e1512b",\n    "288340": "460f64ad77ad1303e7c8012385c0be56cbf151e7252fc57aeca14e33da3538a6",\n    "145665": "082240910a659472d8d1503b536c2f6baa868dc37203f098cbbbba1f6e8e8336",\n    "139871": "847040a30c148ea4f01591e745729e4d075f404b5494e3e9097903ff0aecc9fa",\n    "403943": "b25bb9e0c857ea39eb2e8d3634d75b9b7aa4d271d83d9b69998b6f89d65cffbb",\n    "285089": "a9fd8e346ec568a224e0a01e29b607c76b5f042045bdc66bae9739de897a6b47",\n    "272870": "b915d0fbe0a93d3675d8141353d387f7d39e1d1d8f2000bfac27c4e0904cee78",\n    "268254": "07a77188db83d8eddc8d47e83ae995ec6ec5b87bfffde835c65e332ca4a1fa28",\n    "147205": "f2d2e6b54e6065560932a338cce11c125198a2cdf42e0fb375643558c009d7c6",\n    "382399": "e650037023ee0db7429fc0ea749600a0fcc6ea0f7a69f58044b1181227013a70",\n    "509551": "38d0f717c1ad38243517160030693f45e45cb827d270de4fd3f1bd124d2ddf11",\n    "119752": "577f55b46c63b329501d7dda4c7b161f625523972131d52ed5115d30ab4089cd",\n    "18728": "a887339f668fc1b736467460b9a41c5dc70674067ddd80a052054aafe31c5b3f",\n    "38118": "001275230bcbb0c24018eea685c6e40b5481941cc4e0bf64bb978ca45cfb4a89",\n    "561958": "e9decb01c27886db3e3b243ce1fd272df01ef969ab00152fcf0f976c0f86eebc",\n    "69140": "f0896a56b0a095ff5b9559ca38017bdd6de6f1e11bdf6bf87cf1c3a4328332f9",\n    "370401": "2163a3a4c2e5643c4ed63c82b3b3df7d5fba2e32ade0769f52f7b1b85c9e6bf3",\n    "487688": "076b0a3d8b8e7bfd8f1ca8288d8de71df4a4d79ec54cbb872084a9bf91046344",\n    "80104": "8c5bf66413c21fa5729ad8e9df2584917451ea72b34e1e4036a1b66f86f7f27e",\n    "217154": "faf53359d1b8b4c332a661e43bc4dd9e76ec1e329f77588c19372ed5b6d6a5fe",\n    "383777": "32c9529c7f0a9e050636f58f1dcdf80ce1d099a861345c4f9d7bb1b730547d8f",\n    "153184": "2a98c73d16c2ea6e1ab4a9835b8290f70ecc2ad36105d7829072253d69dcec9c",\n    "476279": "7cfc9b15701d560be56a7a7ba2730efa19c0915dd0fbeb2fc018d3ffbce12cca",\n    "267035": "0b7dee801c80fcbcb357a603f836617f0c164ea6aa176c0f8dc1b67a2cc129cc",\n    "412285": "bed0a09b9e4e29fdc07feb0b401d50a3a7b4d3beaed79c449d36a21db1916448",\n    "565929": "e7fbfca42adcbbfa609256b22b3f14d184b73abd4667af6ed8f9aa45eb472567",\n    "536366": "b0f31343c9f89e3623f4abfb1843d9f16c970c5bfcfa35f12dfbee615b5ca5ee",\n    "53529": "526350c7cf239cf7080e0c77033e1c614cd720a721a8e5b37857cfbd548cbdef",\n    "370095": "0fa0a4e6f69f674f5d576fc985e23c17ecef61331ac7c13da5354dedd4ce8473",\n    "544201": "224ebbcc427b71a9f37d35921852c14eb50cec0d3dbd4f5093af11611c6dc8ea",\n    "250127": "940755f90a2216df6b238cf0baee6ab3b1676b4381afbcb0b77cb6841ae52154",\n    "286018": "8d7e5ff4be8db785d241e83296a2b35da5f16d76c2edf8ad9b87f0fd0f26bf5f",\n    "453138": "3bb160671644a71a7f83e47ecddd4cbd8a707c63939e11d4fe6e6cb04f5ede35",\n    "50960": "759230405b0f97f220ec373efd0cca40a14ed905b24f0724ed00aaf6b2e5c5ed",\n    "563816": "de8011585b27e6a7a8a4aa7cb8c9a575c2c918f620ffde69952ec8e3fb9a17cc",\n    "174070": "38ce4f6926e8a0d2265a4e60e8e782e7ace86020870abd7719e784a4be4b6e6d",\n    "227048": "6b965dd3ff339be2719fd2042cbbafe8d2046462982de66f677cf0f9a1a7d1e5",\n    "306264": "49d95a63c71231913b76c27a91842656ddaeb272be59b3c0c91a9f3a68068fcb",\n    "392506": "ffc232d28f84c8fcab9690e6bed0ecfb306444fa56bd5263af2e4c6f9c09bb14",\n    "478522": "0277c5202643818175bafe7ab9cd5a774cc722bb3cae6a925afd288b2f95fce2",\n    "482305": "c6b9641c33e001ed21e5b209ff61ad448703210d1179fa5d130590f249eff433",\n    "36660": "3c955f36af644e6b9fa1f9bb6777e1fc160d5c641ca06fca57748a95834c3f51",\n    "562054": "653a29c0a5fbcc4bfd38300db0b67ebe3e66b85c9cf1f6a72a48d770e87f8a7f",\n    "385036": "c442b80a951ad81476535b2e5320f42cb2ebac72c4f2cdd4cb7405e0ffbdbf69",\n    "88983": "5bae2afab82d78a76dbb513b51230b23b0d2190ac6f61232d74a21d5f55dd7a4",\n    "459967": "7b564fde4dbe14dc7efd3707f0969a69ef79be40df12d6f5d1f00752b4cbd1cb",\n    "78188": "d21a74c2be034ee627e291dbe41f2fe08b31df835f0f2a93eb2d7c9dea7251cb",\n    "31796": "286054c8d47c3c3c28e527da6d945b7c9fd365885ce9f3e9884a9dfdb0cd7b11",\n    "542934": "22ef616cb1d25196de45977bf4d93182d22d065cec801df776394e982c8282ee",\n    "53328": "ec3b47a11d8f4af37ff7ebe8f7841373ff84a036c5e61b147f808d4c20a55f1a",\n    "569464": "ae159349ac86d1225775a581eace05b22e66159a946282b081089c97fe875126",\n    "491107": "0d063f8fb7fa60154ce214a5eb2ecc74c2a335b43d10efee0a63ec64c3bd7f33",\n    "416718": "4aae966096065a6faacb5c85a7aff2a4de12a94f039a8713deb60a72ecfebe61",\n    "547348": "5ffa5553cc67f7625bdb78fd3110f57a9eeb8af9159b48ea3e06c748ed318595",\n    "316963": "60087279f2bce95ad97bc2afb3c61abaddb5009d29ca5e3f881052ba1d4a464d",\n    "241534": "bc0b9483f40c5d58139ecae5b9bdb7c37f76cdb608684d705b9796c6c52c96ca",\n    "466797": "8625d03a9c83aaa92de966151889291280177f9d0c3e80330c733c3245c985e9",\n    "179650": "cbe934031183aee846a69adc4364373ab975816c59d2f2caf887b59ade6d895c",\n    "171376": "84a2952378f31cb2dd6cb935bebe219b7450af3689fcdc729e19d2f6031a0bab",\n    "186361": "de2590e41e5de51972e92dc2fe1619846fb5ed19330001b9758c1f825c8e7e45",\n    "279065": "a28482bd2fbe26924d4dcc80d735ceac25d30b992520cec826a6284f2ebeb3b7",\n    "486769": "dd7893205a7cd98a73270bbfb779a84c50a055772b9385b34437e410e018d406",\n    "406077": "ee2b9a80957525c4be1c054520ee87ce46c42fdc2e049451fbc163efa4e73de8",\n    "393675": "3fbd0a08ef9c5476b65de9a20cb4d2212c22420f2a7620cab76df09c781cbf45",\n    "189404": "18dc511dea6b0f8f66ddeaff9f3f6b26c928291bec4c25ee0479d7f5fbb00352",\n    "225539": "e6bd24516d71042ecedadb6c624e6ec54741536d047e0f69802eba262591b509",\n    "259833": "5c98bd0b96058b3fc64cc70f2bbe8c6120a22c7a71f88b08a690230b687546fb",\n    "292687": "f450f30803a502456ddeac74abfd898aad3da34f51e4ad1cf860559f6ef558d9",\n    "74517": "8395fd7b519c94f6e239534132a60aad072c7743cc836d560e6a0fa6c5176101",\n    "211208": "060943ab2bbcb7431df8119117b2d5b6e795732b9bd282a07a78a3cdef398955",\n    "368764": "6bb9c9e3d7a5eb3e3ffa997fc1983c30df3ad9dc604dcc325805b079b2d2dd16",\n    "497202": "cac06747ddc81c708d036e1837b39bd541c5699326ad48b5fcdb201b1e38bd01",\n    "497194": "9eaf904f5e6ffbef6bf5ffa5668e6b6dfaad733cbe0b64ed9e4353bb6f4561df",\n    "356921": "2fd5e27440cfb784b3ca14078bc283145f306f074808f3f3f03cec9fdf4e107f",\n    "5253": "5fed259e1a3659302bbb3ee8df74814740ca88c4b19870e4207df2863b9ea842",\n    "359722": "38d782ef8a184a5b4dba8f29cfa9c0cf438e5d4562e49f04d4cb7323c9b2efa6",\n    "395080": "8df14202ff0389850117a50775232ba42c7d5a8a8d47f63f99c37fa5e182c0c5",\n    "464174": "8998e350b40e938ed9f1fbd60fc9ffbd62970d264b2e21962ffff72e6769f903",\n    "139124": "6661afac5fa65f09431deb907f5abacac611e8381ee0f5a154b4b97ae174917c",\n    "245373": "6bf9dbe1cd0ee154f3c170dd4adeeff58aa04e5b628363e25535d4358fa46702",\n    "459346": "a02e0ffd68574f4625a79cf5a2eded1f2a7a17956d6bfb902da0a7efc60bc860",\n    "43407": "5dd6840ada8fa0b501acded3d74535dbd9f73a3cd3e6ff3b1bf947db7d36025f",\n    "55683": "f00f2523752a5273261b472ce265902357951ce9ffe32b18dfe3b23a3c10f0f2",\n    "407349": "59b887aa59bde8d76c67083a6d30f0f61a49ab77f2b752d629098e8bfe7e5027",\n    "11146": "e111073e78e24c6c20e7ed56183ddc806fdf0ced6aa3399116040d7d8500d47d",\n    "263651": "6fa1196b96c263565e5e8f76f0f9ba41ebca368891de1a6f50f45f5d37f5dfe2",\n    "579999": "9de985d18b004a488116603a3cb9e7c5112650421aeba6bf69fb8115ba72ff62",\n    "33352": "79394e7529ab1bbed029706f1a1844a50e932949717277b8e8c9df002ae42d7d",\n    "322315": "4a04ba23de6c6464e25fb68f622a2f43cb60bd9c23548ce3bc2963c50235262a",\n    "231521": "0003718a0ff1cb129b9377214b5e3283e835b3873ccb09564c07c39bd0ee9877",\n    "495712": "0db186721fe287df0eafd939879206506c7e05d61a42c787a561e1949b9c32e0",\n    "369212": "bc08cae636606e45f805d1c2bad9b80f83f90686f4b647624037826a9eb5746d",\n    "105237": "fbcd2d82f2b37d270fe5bf287d19547d542ee61c07b45752c204fffd8ce864c1",\n    "435920": "05f9d38444da410b6e9eab98ae8f685812fc7be14e8e56f304776dd49304456f",\n    "463114": "e16c17b27ff9895711c61fb04c72d0961c14ea8b240154544be1d6ca7d2242f5",\n    "238757": "90ae7e5258ad4b099064d5cc371972af23d159b6ad64448064c92175e8231407",\n    "398350": "9c6f70d7e72a98e033556248b254d47571e431c3837bbfb6370e11a46ecc38cd",\n    "314398": "6e54d746a29ff0dba5e82aa7b7b94f45928e1388d61c245c0604a4e75fe454f7",\n    "168287": "aee2b04ad009674a9c44c14267caca84b6216f9bf5359d4b8bc40c594c0bc749",\n    "233477": "5fcc321c6730d01a37837123e80b795811811891e3f2dc6bce18d06d9f0ac4f3",\n    "111624": "22814586ba0f161d35719abf3685dafade49baa6a9ccdd6e29b8cf90fa94d47c",\n    "558467": "e53daa3f50d29e5bfe1ec1e76c605368ba99779dd88f21e205670d07149f808a",\n    "217583": "972c15c8c69f4a21174cf66f8c9274d1e3af126bd08dc3876d17e3e42ca0864f",\n    "307464": "6e55ddae11b94dae3cc4f1067f28762e2b975db7ed28a0bd9bd58f08ad36d761",\n    "59552": "12b54f59f37125fa7789ea3a87432d304ffeb35804d5d21d73d265c7fc0d5912",\n    "579487": "8e6f65a3c8f0245737f074fe74ab61ae26cd7054840d986524b0a17abe428723",\n    "55559": "803be71f1af01445123cf3483660fb33b9399d43ae8212657e304439365619e8",\n    "171235": "1ba179c48dc7a62511c04ec9118d68451734f5166e65fc466590c4fa8d1a8011",\n    "4739": "d126881cb4d4ad5b683fbbeeeb285578e69f6f26a3c6bb1ef7f7406177ac7373",\n    "327841": "313c12d2d62edf53757de271074fb2831b0feee469af1d3269ecbefd7db13e1e",\n    "362865": "bcac7359ec47abe2914615fbfd5d683bb9f7957eca8b0553b0603ddd54b8bd12",\n    "406807": "e5981d92ea1f470412c83c1b499d4c8534c85941c2f5961454917284cfceea4d",\n    "123412": "44b8a36d2e4ddb16b6bf69043f9a156bb0fd4431b42d02011e04c8734100d221",\n    "113521": "b257bca5cf25c2914fc01b035ca6d00c72d13ea7aaa28865660ef6991b1aa59a",\n    "535113": "c6226b18fdef1dc0c473db9f3bd57ca4b2565355722e985cf943203416a8842b",\n    "350842": "7fd1a882bdd8e2f711b5ad26d4b41df4cfe23d61a2685210e1cc8f473204ecf1",\n    "8965": "781d3cbf28b7e0695c18b431c5df9fa4ecbb0593c7f1e08549151989cce00a9d",\n    "224695": "6a97091dd2a20aa714a08b3577b7fe758670cc572e3530cf28c6a8971f203261",\n    "469330": "ea707cf0377e818e21858186ac93daa5b30ec9c7843aef4abdd7fd0195be3b32",\n    "380733": "5541aed252bdf0854d4e7ff7b6bffbf2e22c81ca70007dc11665479a5bc5418e",\n    "86654": "6146a3ee767e412dae76e08747ff8f562a2f56d46690fb864df1a93d77ea1265",\n    "524834": "10184f1af56d0ee9cf8014f64a6874bd3fbcbe9052b1bc3955e59e244bfbd625",\n    "78548": "e9ebd75e3dae6e53e89f6ce5f49339f89f77d34bfbd271370d63a4a445f00382",\n    "404361": "5adfce34fb5bdeca86677c8cf93ccc184d4e6b1bf6bef59cf39ca20b3ef57ae7",\n    "253965": "509f85733f7f4cec3dccd4af3a9d8a4acc3a1eccdee60b11fabe292ec672c834",\n    "211325": "55684be7e4281d3b5b8e831a4987e4cef197b66b0c067389e08617eeb8db0ed4",\n    "264511": "bc37b1fff14ba9c1f1931c7eee96d6bbe622f465774f8afac5809ce3fa6eead2",\n    "117313": "fafb2b65b5118fd092da1abf7d6f2838515208142cdfd640a3d25abe4e7c613d",\n    "152286": "0b7d627b3c0afbc51f242f80ca51150f19a2a865b91d5aa59acfb165df40aed2",\n    "10784": "df5d08e192fa176f411eddbd3241deb60dbf8e9b7294355db756168628578a6e",\n    "560202": "8c9734a9bf8e1000523b4502521133d2552fb59a5c9a6d0ce1418e8743296fd0",\n    "24072": "8afea17a960b65f86cf318c666a6d89ba464837b67bc03a1c54798937a2ce1f8",\n    "279753": "612bba5d9808482be78bf16ba8cbbb3cabb9e711d14946a55e43d28cc4124b7a",\n    "326454": "3a2beb93368b54160b2e72d95e1762a50841f3f57b8e25de84fc777e406cd459",\n    "115374": "690e459d0bae6a9181b55518cd013b6f09964cb90d911d282d498715f8586f4b",\n    "227207": "8c75307a0c8e0fd2f487801394e40555ddee74155749d6dfba089fc20b4ad1f3",\n    "240904": "8c16a2a3d6c56fec7dad6d3d2fa54d671a5f60831bf22ff29cfd3b605fded912",\n    "193709": "cf7902733b74dd735f19b7c63949e6a63a10279fc724bc45e01d7d237caa37ba",\n    "155999": "db99f6666ed948626ff9ca94de0194e9ffbaf9ff07551ee3d91f822900c9a980",\n    "19552": "9205f8a8ce4be3f6cc7d23f9d18904b19c5fa65b949ef570648fb0342135d1a9",\n    "171959": "0718103de104d41f85be7c4204ffe853e9be25b49bcac85b398ee1d5bba20e40",\n    "257690": "99afa3066a9d6a12a3dc3e0f61f1ca5201e64c3b2fd90f82ae9f01ea81665efb",\n    "164655": "f4cdd483a2607de5952d81abe726835b7383389442c96983edf5cfbafbf82875",\n    "137824": "b4a30482bc33ce62a267c27800676ce89c6262b955b3071d9df14a6d380a5c97",\n    "335534": "e4e0f8a7e941949172c82e0c11e734ec1783fd43d771d12d5416edbec65d0aeb",\n    "101904": "136768c0d4ccbc03f50bd4ed9f99a24a3a8679398de862b44461a85091b5a2f9",\n    "265594": "d60f8bf27b58d0aaa97ab154c68db345fa3c40c81ae242c54e5d4840b401e6e5",\n    "64036": "66a3532a8a00170ed703a775ec9c2319dff29350daa8d67cefc20df56aa2e568",\n    "439251": "3e2f2b06d0f7220f8d879322d22ad9acd6368f3c9e2513e0238a572a81988de4",\n    "278494": "47b59f6a2881eb7800722197fbeff640b15a3cdb36841bcc5948b60870f9e268",\n    "359156": "38e4636720465cd0db1b777e103576084c87be61c260e06bd90e9b5cf5b744ec",\n    "813": "da5ef9621fa605e9c4140c54a9985090bbae0aad74898cb07b080b18f5d7e140",\n    "264838": "4f455500ff17caaa0c75d77aea8e555384e0f33bc2f5b8233e91cc581e53b335",\n    "483305": "04572bb4ac0685ebfe5ae2516f95b37b728a33af81706e8bcc0433942db9d5c7",\n    "321355": "1fc5723a2d671a2cbc09fa1befe8b4eb1d2e84e260cb2fd2bad92da96920c521",\n    "181208": "38fed0968423114fbafc2c586e2a1a42671f208d5577e26621afd1dc3193a776",\n    "136668": "23e948daf896b9251092212724bba1fffe82d2e9128f790ecb8116ac16affefd",\n    "43522": "809d3e98020abf9cf28ce99d062129845ad648e3543dd73bbef6b3e10c35967f",\n    "191473": "99028c8ba241596742f0ff65a6106ddb5413eaa2515e51abc9ba7d15f6c8e595",\n    "546511": "d6b798e6a09f1b79f811f6be6bd20c372d72595d4df28f6fc2344f4b14ac4e72",\n    "517139": "49fa03f0e7bbb6863f01cf53a99a5249db68fe6533d4b7566279a17c733d7c2f",\n    "120670": "cce1f2ffa0d26a64626146e747187c501ce1f6fe88f6cbf3f3977bbedd171265",\n    "30508": "44660d6e08cfbda6d2fc92d7d1051910e8f72565256687781751b4d96e27134f",\n    "120376": "86fe41ad39a83ed6178214fb2266a72b63f7cd80d2cdc7c26a7700e016783dc7",\n    "365729": "3ec3e0defb1530990db938816d2554ae8803792041fb0b6e0d1e4196e2820163",\n    "210086": "db6de2befe3187ad14650a4b95798ac4445ba7fb6a8d35a96d0f9124b6164ffc",\n    "221685": "f3547bc0d50c213f8cf972cbd7ceb815da1ae93334aa6d512d98254e2ef6c211",\n    "352523": "bf9d65de0bdd004ad294d6ddfff1f1b692e96805dd9fa90d206bf4ee052fa058",\n    "274463": "f1be1082bc800b2f23cbf8380eae0c4d94dda82e02a877b9fe11aebac621ccc1",\n    "528563": "eee2cd5ca9e99240dfad7808775d65e1783f14831d9fee4f6bfbeca9f79adafc",\n    "68701": "cb266574e16e6b115f7b4bc6c821ac472171e38e718e3db0d5c9ab7df1207136",\n    "128695": "11707b3ba6526c3c0cbf78d7ca9aaad21517301e7c12683a27afd935107383c0",\n    "289591": "2bd84b4bfc7e380a3d7a8f5cf9a537390d80e71cbf51d64beb8f6b6346be308b",\n    "467502": "6d921c5a0fcae3763fd942727da078470adda92cc83dc706d73dba2cba21927c",\n    "139932": "25ee5860b12b0607d8b043d83342bfd402e9e464e46254e95cd979d7c899205d",\n    "96897": "396949c41f0e3dec799d7f5972682dc1961c5d4c1d71b28255404bb6335f88e5",\n    "420400": "67342100dde68202712ba843e250d26e5542f68c436c2d8c3e3652fcf7024774",\n    "265237": "e7594a46cefd002a46ffac9734b4745d01d6c3f5163ed052481fbf25683b0735",\n    "285486": "36596ab1d982f33b479f6def8c94a889d5665838af17f26f839b4d82e5bfccdd",\n    "474373": "4d6ef18d4c8f45bb2fde2da474bdb698f253819e98bf6f889bba63e468ad7a6e",\n    "215805": "433e57ddb152276a25cc44a0e7ecac4eddcfb5bbf8caa77ca1f3da926c4c02ce",\n    "309024": "97b168ace9f5fab89539fab8aa49b4f8b2e4983bcc98a95aa5e0cf7e03b8cbc3",\n    "310751": "1bbb7d4eaf75541a7f9778aef000824842f83de0f1f68c31e10544323211894d",\n    "202918": "c55da768e68f92a5f3b24b1bee417eb7b0e598e3272b5c83c2d24c5c8b90c752",\n    "48432": "a48d3e7652c5d39f9023503ba84847571d59a006d2ff382659392c514c4a3e89",\n    "31434": "635c6afb4aad6c99599bc9d6f5f404b147fc1008545df1ee7a2eb7d923b50070",\n    "120529": "1641e51fa6f9f0ff57e003dc2c38c6edf3ca599be1d84579a909ab63a1fdf79d",\n    "326533": "2cb43975310d4faf304c6edf0592c07f26e574713216ba367ed6be8c89e309af",\n    "388779": "7e719c3bf20c0d11dc64e100ae4eb8e887caf2227bc4cba503aa1bd841d73d03",\n    "279810": "879589c20efbc41634157f6dff652494e80abc67a862e646d6b88459055a85f5",\n    "362718": "0bfeb72953d5f50f42c04fc55a88ce2868c3a7144e7fe51a9a655c0b5a013a2d",\n    "280642": "b0832207bc65fbb395628cef07335b9cd149fd8af37868461d070a472d7078f7",\n    "533019": "fd3ae938dab4ca770bdae70109517e65cd97f272d411bea1a61522de083b5f50",\n    "244646": "bdfc06549972082fd2f034fd66f1fae055d7db6d654430678f8b8b004fb8d793",\n    "272653": "a5a12eb2f24e83c4dd95abdab37844a06ee65389ab5f59f3c5abe2da6bf7b3e5",\n    "155198": "0d46f5a028b01380b8ae307e6e9cfe7b6f6fc5526318f33623f2e9cf115e80fa",\n    "499913": "03c214fbed87a845c6a72393564e1c3db924af88a5e263682b2977ce42948ce8",\n    "378911": "7195b91a872d412651ce971eaed1a1fb3bd27375bc8b9be9bb9bf5fcb1b7c128",\n    "441544": "b9b43a9e89e788d165bd9af7c16eeef70a115ba85ac5d09fa0171e9f4d6510f1",\n    "502748": "1e93412ac2376517780fe1dc32208805ae35839dcca843775d5d484ead5189a8",\n    "363951": "12bd9289cf80055058ee4e28c4e9b5dc8dfd5a21bea764cc0d3f0349c74a8fca",\n    "534443": "7709d6517a5663b9256617be43423f8a1f5007f655b21e412c4b2f5c70464b61",\n    "481810": "8fdae48c50c02a621eb78c80758bafe4c654a9f3f8436e1e6f6ef01cda389ecb",\n    "80084": "cf7fb3705a16258c357c0f19b033ef1087d58119c4ec1cef482493d8cd835c21",\n    "104047": "dd8fbd0f25debcd7201346d5d9726fa64f353e8ddcecf13369afb3f834ba18e3",\n    "364508": "751a0b7bc43ef3059e3121861d971f3303c2a7f4f5b276d16ca3c6bd650915da",\n    "389463": "e2a6d6c2498931351d2a297bceface58f40ff5cd776bd9b34ed0f74871e63cd7",\n    "551364": "4270511196de99fc8af0fbb8b153d6f38682f05892736d9825af5b48176f4c15",\n    "98915": "d7c8c2787ec9c8c1612cc8d03bdf6e28a6e883893ce66bb3f109bed664d9cb84",\n    "124501": "830e655d6b7009957288a85630e3fbb6a12ea08b774d8098947f650bcc159b28",\n    "564870": "8a859e1765dde7e6ddd70a67eea4a38480c7ad6b71193be84bdd0174e1a26dc2",\n    "90359": "554822fe63f0d15342730bdc13d5ca2eb67ca7b57cf4eed87a2a0be660766676",\n    "459328": "4dae4247f1d6dbb48fc15b7b45dc2285f82e82e104a1535e0c5f81a374a83fb8",\n    "549366": "67d2d90dba293c145a3e0579bab8caaa2db1fda41c839969413ccf0547491b7c",\n    "286492": "48e6db001fd929193b3e5b7dc4742ff3b9257c082f96477785ad6eabc593a179",\n    "412680": "95e13575633c3887100ff588166d795a5ca0d11e42f81658382e0d856e41e2a7",\n    "21204": "93892a0dcb29eeb1c62584f5a9ab33b983ca39f1823bdf2257dcfc2998ec6b1d",\n    "504888": "b9f3c97cc6ed82c7e31d10b6c55e5eddf75a176f12d2e73fa486c39172547fd2",\n    "480575": "81e3a9e5976d2de66824499b386c852f39a525ad47b3173448812877ed1a15b8",\n    "38628": "8214e538094c1503b962000205e912b1969cbb75192568baffb068b558cbf75c",\n    "188375": "be115512826f0472f81d768e629c8ab6c3fe7d1a0f224b2453471f3b9f57bd0e",\n    "94549": "ca2f2d3e44dc68c9a949db0a654e54ad5240bc9e55068437f46b65ce2e9bc291",\n    "156943": "266b92daf557112a6cb77308323038cf8962443b27276ae0e19858e929111b38",\n    "499997": "0ef21190d80183275b2e26e0c6392565f261e5655a55bde9c1fe2a3586301b03",\n    "455328": "524cfa1c030f05c0fd5c488fbb68848e11882bb0be37e3b34e7c9db517eedf24",\n    "150981": "307429a56936fb1be5369c9e1f10fd6e69af299a1532f2952ba74a23bbae76a6",\n    "239504": "f03693e11fcf2784d3dc6cfdd196397cc283392acfa2d266fae05e158b234148",\n    "319731": "df5a06025b554cbd12e866715cb0dfaea78d2e3f24e95ef6319826808eac0815",\n    "52929": "813cb2627d2bfb32d064bbb910c615161010c0758841e2175d1482b405bfee78",\n    "95178": "9acf4ffc1a10e28c8f001c3d9b9aa9584be27704f8a424734122abd49a15bf4c",\n    "104588": "094574f2af03d16942b841094e6c5904c1aefb39aecf62e06d821c995c00bfa5"\n  }\n}\n', 'test199_data.py': '"""Fixed TEST199 metadata/staging contracts; no neural imports."""\nfrom collections import Counter\nimport hashlib\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom full_data import encode_tokens\nfrom patch_data import DATA_SHA,COUNTS\n\nHERE=Path(__file__).resolve().parent;TEST_SIZE=5000\nCHECKPOINT_SHA=\'5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7\'\nVERSION=\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\'\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu128\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):\n        raise RuntimeError(\'Unreviewed Torch runtime; expected training2.11.0+cu128 and CUDA inference2.11.0+cu128/cu130. Preserve log; do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Observed CUDA builds of the same publicTorch2.11.0; strictweights and GPU beam-path checks required.\',\n        bitwise_cross_build_equivalence_established=False)\n\ndef validate_lock(lock):\n    if lock[\'source_version\']!=VERSION or lock[\'checkpoint_sha256\']!=CHECKPOINT_SHA or lock[\'epoch\']!=10 or lock[\'updates\']!=177110:raise ValueError(\'Wrong fixed199 checkpoint\')\n    if lock[\'beam_width\']!=5 or lock[\'max_new_tokens\']!=20 or lock[\'length_normalization\'] or lock[\'reranking\']:raise ValueError(\'Changed TEST decoding rule\')\n    ids=lock[\'expected_test_ids\']\n    if len(ids)!=TEST_SIZE or ids!=sorted(set(ids)) or any(type(x)!=int for x in ids):raise ValueError(\'Wrong fixed TEST cohort\')\n    import json\n    if hashlib.sha256(json.dumps(ids,separators=(\',\',\':\')).encode()).hexdigest()!=lock[\'test_id_sha256\']:raise ValueError(\'Changed cohort SHA\')\n    return lock\n\ndef lock():return validate_lock(read(HERE/\'TEST199_LOCK.json\'))\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted([*root.rglob(\'best.bin\'),*root.rglob(\'checkpoint.bin\')],key=lambda p:(p.name!=\'best.bin\',str(p)))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Need exact199 checkpoint bytes; no latest/other version fallback\')\n    return matches[0]\n\ndef selected_test_rows(ds,locked):\n    if dict(Counter(r[\'split\'] for r in ds[\'images\']))!=COUNTS:raise ValueError(\'Wrong Karpathy split counts\')\n    if len({int(r[\'cocoid\']) for r in ds[\'images\']})!=len(ds[\'images\']):raise ValueError(\'Duplicate source COCO IDs\')\n    rows=sorted([r for r in ds[\'images\'] if r[\'split\']==\'test\'],key=lambda r:int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in rows]!=locked[\'expected_test_ids\']:raise ValueError(\'Wrong TEST IDs\')\n    return rows\n\ndef prepare(cfg,locked=None,weight_path=None):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False);locked=lock() if locked is None else locked;weight_path=checkpoint(cfg[\'checkpoint_root\']) if weight_path is None else weight_path\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO dataset JSON\')\n    ds=read(cfg[\'dataset_json\']);rows=selected_test_rows(ds,locked);wanted={int(r[\'cocoid\']):i for i,r in enumerate(rows)};words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    gold=read(HERE/\'SMOKE_SOURCE.json\');save(out/\'vocabulary.json\',words)\n    if sha(out/\'vocabulary.json\')!=gold[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from199\')\n    save(out/\'test_references.json\',[dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]) for r in rows])\n    sequences=[];caption_image=[]\n    for i,r in enumerate(rows):\n        for s in r[\'sentences\']:sequences.append(encode_tokens(s[\'tokens\'],lookup));caption_image.append(i)\n    np.save(stage/\'sequences.npy\',np.asarray(sequences,np.int32));np.save(stage/\'caption_image.npy\',np.asarray(caption_image,np.int32));np.save(stage/\'ids.npy\',np.asarray([r[\'cocoid\'] for r in rows],np.int64))\n    patches=np.lib.format.open_memmap(stage/\'patches.npy\',mode=\'w+\',dtype=np.float16,shape=(len(rows),196,768));seen=set();sources=[]\n    try:\n        for path in sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\')):\n            with h5py.File(path,\'r\',rdcc_nbytes=2*1024**2) as f:\n                text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n                import json\n                if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n                meta=json.loads(text(f.attrs[\'metadata_json\']))\n                if {k:meta.get(k) for k in gold[\'feature_profile\']}!=gold[\'feature_profile\']:raise ValueError(\'Feature profile differs from199\')\n                ids=f[\'imgids\'][:];features=f[\'features\']\n                if features.shape!=(len(ids),197,768) or features.dtype!=np.float16:raise ValueError(\'Invalid original H5 shape/dtype\')\n                positions=[i for i,x in enumerate(ids) if int(x) in wanted]\n                for start in range(0,len(positions),128):\n                    pos=positions[start:start+128];raw=features[pos]\n                    if not np.isfinite(raw).all():raise ValueError(\'Nonfinite TEST patches\')\n                    for j,p in enumerate(pos):\n                        coco=int(ids[p]);image=wanted[coco]\n                        if coco in seen or text(f[\'filename\'][p])!=rows[image][\'filename\']:raise ValueError(\'Repeated TEST ID/filename mismatch\')\n                        values=raw[j,1:];patches[image]=values\n                        if not np.array_equal(patches[image].view(np.uint16),values.view(np.uint16)):raise ValueError(\'TEST patch bytes changed\')\n                        seen.add(coco)\n                    print(\'TEST patches staged\',len(seen),\'/\',len(rows),round(time.monotonic()-started),\'s\',flush=True)\n                sources.append(dict(path=str(path),bytes=path.stat().st_size,selected_TEST_rows=len(positions)))\n        if seen!=set(wanted):raise ValueError(\'Missing TEST features\')\n        patches.flush()\n    finally:patches._mmap.close()\n    receipt=dict(status=\'PATCH_TEST199_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),feature_profile=gold[\'feature_profile\'],sources=sources,\n        dataset_sha256=DATA_SHA,training=False,TRAIN_VALID_features_read=False,seconds=time.monotonic()-started,\n        stage_files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},\n        output_files_sha256={n:sha(out/n) for n in (\'vocabulary.json\',\'test_references.json\')})\n    save(out/\'stage_receipt.json\',receipt);save(out/\'test_lock.json\',dict(**locked,checkpoint=str(weight_path),stage_receipt_sha256=sha(out/\'stage_receipt.json\')))\n\ndef load(cfg,verify=True,validator=None):\n    out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);r=read(out/\'stage_receipt.json\');locked=(validator or validate_lock)(read(out/\'test_lock.json\'))\n    if r[\'status\']!=\'PATCH_TEST199_STAGE_COMPLETE\' or r[\'images\']!=TEST_SIZE or r[\'TRAIN_VALID_features_read\']:raise ValueError(\'Invalid TEST stage scope\')\n    if locked[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\'):raise ValueError(\'Changed TEST receipt\')\n    if verify:\n        for n,h in r[\'stage_files_sha256\'].items():\n            if Path(n).name!=n or sha(stage/n)!=h:raise ValueError(\'Changed staged TEST \'+n)\n        for n,h in r[\'output_files_sha256\'].items():\n            if Path(n).name!=n or sha(out/n)!=h:raise ValueError(\'Changed TEST output \'+n)\n    return {n:np.load(stage/(n+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for n in (\'patches\',\'ids\',\'sequences\',\'caption_image\')},read(out/\'vocabulary.json\'),r,locked\n', 'run_test199.py': '"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import semantic_test_data as module\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from semantic_patch_model import SemanticPatchCaptioner\n            from semantic_data import SemanticContext\n            model=SemanticPatchCaptioner(len(words),ck[\'model\'][\'noun_bank\'],ck[\'model\'][\'verb_bank\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'TEST decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==d.CHECKPOINT_SHA for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong TEST references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=tok.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete TEST scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CLIP_PATCH_BANK_PROMPT_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n', 'test_test199.py': '"""TEST199 NumPy/mock metadata checks; never import Torch/scorers."""\nimport ast\nimport copy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport test199_data as d\nimport run_test199 as run\n\nclass Checks(unittest.TestCase):\n    def test_observed_cuda_build_change_requires_same_framework_release(self):\n        self.assertFalse(d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu128\')[\'cuda_wheel_changed\'])\n        policy=d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu130\');self.assertTrue(policy[\'cuda_wheel_changed\']);self.assertFalse(policy[\'bitwise_cross_build_equivalence_established\'])\n        for current in (\'2.11.0+cpu\',\'2.11.0\',\'2.12.0+cu130\',\'2.11.0+cu129\'):\n            with self.assertRaises(RuntimeError):d.runtime_policy(\'2.11.0+cu128\',current)\n        with self.assertRaises(RuntimeError):d.runtime_policy(\'2.10.0+cu128\',\'2.11.0+cu130\')\n    def test_frozen_checkpoint_and_decoding_rule(self):\n        gold=d.lock();self.assertEqual(len(gold[\'expected_test_ids\']),5000)\n        for key,value in [(\'beam_width\',4),(\'max_new_tokens\',21),(\'reranking\',True),(\'source_version\',d.VERSION.replace(\'/199\',\'/200\')),(\'checkpoint_sha256\',\'0\'*64)]:\n            wrong=copy.deepcopy(gold);wrong[key]=value\n            with self.assertRaises(ValueError):d.validate_lock(wrong)\n    def test_checkpoint_accepts_exact_file_or_both_names_and_rejects_other(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/\'best.bin\').write_bytes(b\'fixture\');(root/\'checkpoint.bin\').write_bytes(b\'fixture\')\n            with patch.object(d,\'CHECKPOINT_SHA\',d.sha(root/\'best.bin\')):\n                self.assertEqual(d.checkpoint(root),root/\'best.bin\');self.assertEqual(d.checkpoint(root/\'checkpoint.bin\'),root/\'checkpoint.bin\')\n                (root/\'best.bin\').write_bytes(b\'wrong\');(root/\'checkpoint.bin\').write_bytes(b\'wrong\')\n                with self.assertRaises(ValueError):d.checkpoint(root)\n    def test_only_TEST_features_exact_bytes_vocab_and_seal(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);out=root/\'out\';out.mkdir();rawdir=root/\'raw\';rawdir.mkdir();(rawdir/\'one.h5\').write_bytes(b\'fake header\')\n            rows=[dict(cocoid=(i+1)*10,split=s,filename=f\'{i}.jpg\',sentences=[dict(tokens=[\'a\',\'dog\'],raw=\'a dog\')]) for i,s in enumerate((\'train\',\'test\',\'train\',\'val\',\'test\'))]\n            dataset=root/\'coco.json\';d.save(dataset,dict(images=rows));(root/\'best.bin\').write_bytes(b\'fixed checkpoint\')\n            words=[\'<pad>\',\'<bos>\',\'<eos>\',\'<unk>\',\'a\',\'dog\'];d.save(root/\'expected_vocab.json\',words);profile=dict(model_id=\'openai/clip-vit-base-patch16\')\n            locked=d.lock();locked.update(checkpoint_sha256=d.sha(root/\'best.bin\'),expected_test_ids=[20,50],test_id_sha256=hashlib.sha256(b\'[20,50]\').hexdigest())\n            raw=np.full((5,197,768),np.nan,np.float16);raw[1]=3;raw[4]=7;raw[:,0]=19;reads=[]\n            class Features:\n                shape=raw.shape;dtype=raw.dtype\n                def __getitem__(self,pos):\n                    if set(pos)-{1,4}:raise AssertionError(\'TRAIN/VALID row read\')\n                    reads.extend(pos);return raw[pos]\n            class Handle(dict):\n                attrs=dict(complete=True,id_key=\'coco_id\',metadata_json=json.dumps(profile))\n                def __enter__(self):return self\n                def __exit__(self,*a):pass\n            h=Handle(features=Features(),imgids=np.arange(1,6)*10,filename=np.array([f\'{i}.jpg\'.encode() for i in range(5)]))\n            original=d.read\n            def read(p):return dict(feature_profile=profile,vocabulary_sha256=d.sha(root/\'expected_vocab.json\')) if Path(p).name==\'SMOKE_SOURCE.json\' else original(p)\n            cfg=dict(output=str(out),stage=str(root/\'stage\'),dataset_json=str(dataset),visual_cache=str(rawdir),checkpoint_root=str(root/\'best.bin\'))\n            with patch.object(d,\'TEST_SIZE\',2),patch.object(d,\'CHECKPOINT_SHA\',locked[\'checkpoint_sha256\']),patch.object(d,\'DATA_SHA\',d.sha(dataset)),patch.object(d,\'COUNTS\',dict(train=2,val=1,test=2)),patch.object(d,\'lock\',return_value=locked),patch.object(d,\'vocabulary\',return_value=words),patch.object(d,\'read\',side_effect=read),patch.dict(sys.modules,{\'h5py\':SimpleNamespace(File=lambda *a,**k:h)}):\n                d.prepare(cfg);data,got,receipt,_=d.load(cfg)\n                try:\n                    self.assertEqual(reads,[1,4]);self.assertEqual(got,words);np.testing.assert_array_equal(data[\'patches\'][0],raw[1,1:]);np.testing.assert_array_equal(data[\'patches\'][1],raw[4,1:]);self.assertFalse(receipt[\'TRAIN_VALID_features_read\'])\n                finally:\n                    for v in data.values():v._mmap.close()\n                (out/\'test_references.json\').write_bytes(b\'changed\')\n                with self.assertRaises(ValueError):d.load(cfg)\n    def test_all_eight_metrics_required_before_complete(self):\n        with tempfile.TemporaryDirectory() as td:\n            out=Path(td);d.save(out/\'scoring_lock.json\',dict(files_sha256={}));d.save(out/\'chair_counts.json\',{});d.save(out/\'environment.json\',dict(fixture=True))\n            parts={\'basic\':dict(BLEU_1=.7,BLEU_4=.3,CIDEr=1.1,ROUGE_L=.5),\'meteor\':dict(METEOR=.2),\'spice\':dict(SPICE=.15),\'chair\':dict(CHAIRs=.05,CHAIRi=.03,Recall=.4)}\n            for mode,metrics in parts.items():d.save(out/(mode+\'.json\'),dict(images=5000,metrics=metrics,input_sha256={}))\n            with patch.object(run,\'inputs\',return_value=(out,[],[],[dict(NLL_sum=6,NLL_tokens=3,runtime_policy=d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu130\'))])),patch(\'builtins.print\'):\n                run.combine(dict(output=str(out)));self.assertEqual(d.read(out/\'result.json\')[\'metrics\'][\'NLL\'],2.)\n                parts[\'spice\']={};d.save(out/\'spice.json\',dict(images=5000,metrics={},input_sha256={}))\n                with self.assertRaises(ValueError):run.combine(dict(output=str(out)))\n    def test_source_parse_no_training_no_neural_import_during_audit(self):\n        folder=Path(__file__).parent\n        for n in (\'test199_data.py\',\'run_test199.py\'):ast.parse((folder/n).read_text(encoding=\'utf-8\'))\n        source=(folder/\'run_test199.py\').read_text(encoding=\'utf-8\')\n        for bad in (\'backward(\', \'torch.optim\', \'optimizer.step\', \'model.train(\'):self.assertNotIn(bad,source)\n        self.assertNotIn(\'torch\',sys.modules);self.assertNotIn(\'pycocoevalcap\',sys.modules)\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'patch_model.py': '"""User-run neural baseline: one patch memory, standard causal decoder."""\nimport math\nimport torch\nfrom torch import nn\n\nclass PatchCaptioner(nn.Module):\n    def __init__(self,vocab,dim=512,heads=8,layers=4,hidden=2048,dropout=.1,maximum=64):\n        super().__init__();self.dim=dim\n        self.project=nn.Sequential(nn.Linear(768,dim),nn.LayerNorm(dim))\n        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)\n        p=torch.arange(maximum,dtype=torch.float32)[:,None];scale=torch.exp(torch.arange(0,dim,2)*(-math.log(10000)/dim))\n        pos=torch.zeros(maximum,dim);pos[:,0::2]=torch.sin(p*scale);pos[:,1::2]=torch.cos(p*scale)\n        self.register_buffer(\'positions\',pos,persistent=True);self.dropout=nn.Dropout(dropout)\n        layer=nn.TransformerDecoderLayer(dim,heads,hidden,dropout,batch_first=True,norm_first=False)\n        self.decoder=nn.TransformerDecoder(layer,layers,norm=nn.LayerNorm(dim));self.head=nn.Linear(dim,vocab)\n        for module in self.modules():\n            if isinstance(module,nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:nn.init.zeros_(module.bias)\n            elif isinstance(module,nn.Embedding):nn.init.normal_(module.weight,std=.02);module.weight.data[0].zero_()\n            elif isinstance(module,nn.MultiheadAttention):nn.init.xavier_uniform_(module.in_proj_weight);nn.init.zeros_(module.in_proj_bias)\n\n    def encode(self,patches):\n        if patches.ndim!=3 or patches.shape[1:]!=(196,768):raise ValueError(\'Wrong patch shape\')\n        return self.project(patches)\n\n    def decode(self,memory,prefix):\n        if prefix.ndim!=2 or len(prefix)!=len(memory):raise ValueError(\'Wrong memory/prefix shape\')\n        if prefix.shape[1]>len(self.positions) or (prefix==0).all(1).any():raise ValueError(\'Empty or too long prefix\')\n        text=self.dropout(self.embedding(prefix)*math.sqrt(self.dim)+self.positions[:prefix.shape[1]])\n        causal=torch.ones(prefix.shape[1],prefix.shape[1],dtype=torch.bool,device=prefix.device).triu(1)\n        output=self.decoder(text,memory,tgt_mask=causal,tgt_key_padding_mask=prefix==0)\n        return self.head(output)\n\n    def forward(self,patches,prefix):return self.decode(self.encode(patches),prefix)\n\n    @torch.no_grad()\n    def greedy(self,patches,maximum=20):\n        ids=torch.ones(len(patches),1,dtype=torch.long,device=patches.device);done=torch.zeros(len(patches),dtype=torch.bool,device=patches.device)\n        for _ in range(maximum):\n            word=self(patches,ids)[:,-1].argmax(-1);word=word.masked_fill(done,0)\n            ids=torch.cat((ids,word[:,None]),1);done|=word==2\n            if done.all():break\n        return ids\n', 'patch_beam.py': '"""User-run batched beam search; no model import at module load."""\ndef beam_search(model,memory,width=5,maximum=20):\n    import torch\n    if model.training or not 1<=width<=model.head.out_features-2:raise ValueError(\'Beam requires eval model/valid width\')\n    b,n,d=memory.shape;vocab=model.head.out_features;device=memory.device\n    seq=torch.ones((b,width,1),dtype=torch.long,device=device);score=torch.full((b,width),-torch.inf,device=device);score[:,0]=0\n    done=torch.zeros((b,width),dtype=torch.bool,device=device);expanded=memory[:,None].expand(-1,width,-1,-1).reshape(b*width,n,d)\n    for _ in range(maximum):\n        logits=model.decode(expanded,seq.reshape(b*width,-1))[:,-1].float();logp=logits.log_softmax(-1).view(b,width,vocab)\n        logp[:,:,:2]=-torch.inf;logp=logp.masked_fill(done[:,:,None],-torch.inf);logp[:,:,0]=torch.where(done,torch.zeros_like(score),logp[:,:,0])\n        score,flat=(score[:,:,None]+logp).flatten(1).topk(width,dim=1)\n        parent=flat//vocab;token=flat%vocab\n        seq=torch.cat((seq.gather(1,parent[:,:,None].expand(-1,-1,seq.shape[-1])),token[:,:,None]),-1)\n        done=done.gather(1,parent)|(token==2)\n        if done.all():break\n    return seq,score\n\ndef check_paths(model,memory,seq,scores):\n    import torch\n    from torch.nn import functional as F\n    b,k,length=seq.shape;vocab=model.head.out_features\n    if not torch.isfinite(scores).all() or (scores[:,:-1]<scores[:,1:]).any() or not (seq[:,:,0]==1).all():raise ValueError(\'Invalid beam paths/scores\')\n    target=seq[:,:,1:];before_eos=(target==2).cumsum(-1)-(target==2).long();mask=before_eos==0\n    if ((target<2)&mask).any() or ((target!=0)&~mask).any():raise ValueError(\'Invalid BOS/PAD/EOS path\')\n    mem=memory[:,None].expand(-1,k,-1,-1).reshape(b*k,*memory.shape[1:]);prefix=seq[:,:,:-1].reshape(b*k,-1)\n    logits=model.decode(mem,prefix).float();logp=F.log_softmax(logits,-1);logp[:,:,:2]=-torch.inf\n    # Exclude post-EOS PAD without multiplying negative infinity by zero.\n    per=logp.gather(-1,target.reshape(b*k,-1,1)).squeeze(-1).masked_fill(~mask.reshape(b*k,-1),0).sum(-1).view(b,k)\n    torch.testing.assert_close(scores,per,rtol=1e-4,atol=2e-4)\n\ndef text_from_ids(ids,words):\n    out=[]\n    for word in ids[1:]:\n        word=int(word)\n        if word==2:break\n        if word>1:out.append(words[word])\n    return \' \'.join(out)\n', 'patch_data.py': '"""TRAIN-only raw-patch smoke stage; importing never imports a neural library."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom data import vocabulary,encode\nfrom real_bank_data import sha,save,read\n\nDATA_SHA=\'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d\'\nCOUNTS=dict(train=82783,restval=30504,val=5000,test=5000)\n\ndef selected_rows(dataset,count=256):\n    from collections import Counter\n    rows=dataset[\'images\']\n    if dict(Counter(r[\'split\'] for r in rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    ids=[int(r[\'cocoid\']) for r in rows]\n    if len(set(ids))!=len(ids):raise ValueError(\'Duplicate COCO ID\')\n    pool=[r for r in rows if r[\'split\'] in (\'train\',\'restval\')]\n    if not 1<=count<=len(pool):raise ValueError(\'Invalid TRAIN smoke size\')\n    return sorted(pool,key=lambda r:(hashlib.sha256(f"patch-baseline/42/{r[\'cocoid\']}".encode()).hexdigest(),int(r[\'cocoid\'])))[:count]\n\ndef check_features(raw):\n    if raw.ndim!=3 or raw.shape[1:]!=(197,768) or not np.isfinite(raw).all():raise ValueError(\'Expected finite raw [B,197,768]\')\n    if raw.dtype.kind!=\'f\':raise ValueError(\'Expected floating raw features\')\n    return raw[:,1:,:].astype(np.float16)\n\ndef batch(data,events):\n    events=np.asarray(events,dtype=np.int64)\n    if events.ndim!=1 or not len(events) or np.any(events<0) or np.any(events>=len(data[\'sequences\'])):raise ValueError(\'Invalid caption event\')\n    seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n    return np.asarray(data[\'patches\'][data[\'caption_image\'][events]],np.float32),seq\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False)\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    dataset=read(cfg[\'dataset_json\']);rows=selected_rows(dataset);words=vocabulary(dataset)\n    if len(words)!=10000:raise ValueError(\'Wrong TRAIN vocabulary size\')\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)};found=set();profile=None;shards=[]\n    patches=np.empty((len(rows),196,768),np.float16);row_hashes={}\n    paths=sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))\n    if not paths:raise FileNotFoundError(\'Add raw visual-cache Dataset; ROI134 cannot replace raw patches\')\n    for path in paths:\n        if time.monotonic()-started>cfg[\'prepare_seconds\']:raise TimeoutError(\'Raw-cache scan/staging cap exceeded\')\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not bool(f.attrs.get(\'complete\',False)) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete or ambiguous cache IDs\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']))\n            expected=dict(model_id=\'openai/clip-vit-base-patch16\',output=\'vision_model.last_hidden_state\',preprocessing=\'bilinear\',image_size=[224,224],antialias=True,dataset_json_sha256=DATA_SHA)\n            if any(meta.get(k)!=v for k,v in expected.items()):raise ValueError(\'Raw feature profile differs\')\n            np.testing.assert_allclose(meta[\'mean\'],[.48145466,.4578275,.40821073],rtol=0,atol=1e-8)\n            np.testing.assert_allclose(meta[\'std\'],[.26862954,.26130258,.27577711],rtol=0,atol=1e-8)\n            current={k:meta.get(k) for k in (*expected,\'model_revision\',\'mean\',\'std\',\'compute_dtype\',\'storage_dtype\',\'autocast\')}\n            if profile is not None and current!=profile:raise ValueError(\'Mixed raw extraction metadata\')\n            profile=current;ids=f[\'imgids\'][:];features=f[\'features\'];names=f[\'filename\']\n            if features.shape!=(len(ids),197,768) or features.dtype.kind!=\'f\' or features.dtype.itemsize not in (2,4) or len(names)!=len(ids):raise ValueError(\'Raw cache shape/dtype differs\')\n            positions=[i for i,v in enumerate(ids) if int(v) in wanted]\n            if positions:\n                raw=features[positions];values=check_features(raw)\n                for pos,full,value in zip(positions,raw,values):\n                    coco=int(ids[pos]);j=wanted[coco]\n                    if coco in found or text(names[pos])!=rows[j][\'filename\']:raise ValueError(\'Duplicate/mismatched selected raw ID\')\n                    found.add(coco);patches[j]=value;row_hashes[str(coco)]=hashlib.sha256(np.ascontiguousarray(full).tobytes()).hexdigest()\n            shards.append(dict(name=path.name,bytes=path.stat().st_size,shape=list(features.shape),selected_rows=len(positions)))\n        print(\'Raw patch scan:\',path.name,\'TRAIN extracted\',len(found),\'/\',len(rows),flush=True)\n    if found!=set(wanted):raise ValueError(\'Missing selected TRAIN features\')\n    sequences=[];caption_image=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:\n            sequences.append(encode(sentence[\'tokens\'],words));caption_image.append(j)\n    arrays=dict(patches=patches,sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(caption_image,np.int32),coco_ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64))\n    for k,v in arrays.items():np.save(stage/(k+\'.npy\'),v)\n    save(stage/\'vocabulary.json\',words)\n    manifest=dict(status=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),split=\'TRAIN_only\',test_used=False,validation_used=False,\n        dataset_sha256=DATA_SHA,feature_profile=profile,shards=shards,raw_selected_row_sha256=row_hashes,\n        files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},seconds=time.monotonic()-started,\n        limits=\'Inherited producer/revision/preprocessing; no full H5 payload hash or new CLIP extraction. Only selected TRAIN feature rows read.\')\n    save(stage/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\' or m[\'images\']!=256 or m[\'test_used\'] or m[\'validation_used\']:raise ValueError(\'Wrong stage scope\')\n    for n,h in m[\'files_sha256\'].items():\n        if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'patches\',\'sequences\',\'caption_image\',\'coco_ids\')}\n    return data,read(root/\'vocabulary.json\'),m\n', 'full_data.py': '"""Full metadata, schedules and lazy raw H5 reader; no neural imports."""\nfrom collections import OrderedDict,Counter\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport unicodedata\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom patch_data import DATA_SHA,COUNTS,check_features\n\nTRAIN_IMAGES=113287;TRAIN_EVENTS=566747;VALID_IMAGES=5000;VALID_EVENTS=25010\nSTEPS=(TRAIN_EVENTS+31)//32;EPOCHS=10\n\ndef encode_tokens(tokens,lookup):\n    if not isinstance(tokens,list) or not all(isinstance(t,str) and t for t in tokens):raise ValueError(\'Invalid caption tokens\')\n    x=[lookup.get(unicodedata.normalize(\'NFKC\',t).lower(),3) for t in tokens[:52]]\n    return [1]+x+[2]+[0]*(52-len(x))\n\ndef learning_rate(step):\n    if type(step)!=int or not 1<=step<=EPOCHS*STEPS:raise ValueError(\'Invalid update\')\n    if step<=STEPS:return 8e-6+(1e-4-8e-6)*(step-1)/(STEPS-1)\n    ratio=(step-STEPS)/(9*STEPS)\n    return 1e-5+.5*(1e-4-1e-5)*(1+math.cos(math.pi*ratio))\n\ndef order_for_epoch(data,epoch):\n    if type(epoch)!=int or not 1<=epoch<=10:raise ValueError(\'Invalid epoch\')\n    image=np.asarray(data[\'caption_image\'][:TRAIN_EVENTS])\n    if (np.diff(image)<0).any() or len(np.unique(image))!=TRAIN_IMAGES:raise ValueError(\'TRAIN caption mapping differs\')\n    if image.min()!=0 or image.max()!=TRAIN_IMAGES-1:raise ValueError(\'Heldout/foreign TRAIN image index\')\n    return np.random.default_rng(42+epoch).permutation(TRAIN_EVENTS)\n\ndef validate_state(s):\n    epoch=s[\'epoch\'];cursor=s[\'batch_cursor\'];phase=s[\'phase\']\n    if type(epoch)!=int or type(cursor)!=int or not 1<=epoch<=11 or not 0<=cursor<=STEPS or phase not in (\'train\',\'validate\'):raise ValueError(\'Invalid cursor\')\n    if epoch==11 and (cursor or phase!=\'train\'):raise ValueError(\'Invalid completed cursor\')\n    if phase==\'validate\' and cursor!=STEPS:raise ValueError(\'Premature validation\')\n    if s[\'updates\']!=(epoch-1)*STEPS+cursor or s[\'caption_events\']!=(epoch-1)*TRAIN_EVENTS+min(cursor*32,TRAIN_EVENTS):raise ValueError(\'Exposure mismatch\')\n    if [r[\'epoch\'] for r in s[\'history\']]!=list(range(1,epoch)):raise ValueError(\'Incomplete history\')\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();root=Path(cfg[\'stage\']);root.mkdir(parents=True,exist_ok=False);out=Path(cfg[\'output\'])\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    ds=read(cfg[\'dataset_json\']);all_rows=ds[\'images\']\n    if dict(Counter(r[\'split\'] for r in all_rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    mapping={int(r[\'cocoid\']):r for r in all_rows}\n    if len(mapping)!=len(all_rows):raise ValueError(\'Duplicate COCO ID\')\n    train=[r for r in all_rows if r[\'split\'] in (\'train\',\'restval\')];val=[r for r in all_rows if r[\'split\']==\'val\'];rows=train+val\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)}\n    index=np.full((len(rows),2),-1,np.int32);seen=set();profiles=[];sources=[]\n    for number,path in enumerate(sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))):\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']));gold=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'feature_profile\']\n            profile={k:meta.get(k) for k in gold}\n            if profile!=gold:raise ValueError(\'Cache extraction profile differs from smoke192\')\n            ids=f[\'imgids\'][:];names=f[\'filename\'][:];features=f[\'features\']\n            if features.shape!=(len(ids),197,768) or features.dtype!=np.dtype(\'float16\') or len(names)!=len(ids):raise ValueError(\'Raw H5 shape differs\')\n            for i,(coco,name) in enumerate(zip(ids,names)):\n                coco=int(coco)\n                if coco in seen or coco not in mapping or text(name)!=mapping[coco][\'filename\']:raise ValueError(\'H5 ID/filename mismatch\')\n                seen.add(coco)\n                if coco in wanted:index[wanted[coco]]=(number,i)\n            sources.append(dict(path=str(path.resolve()),bytes=path.stat().st_size,shape=list(features.shape),dtype=str(features.dtype),chunks=features.chunks,compression=features.compression))\n        print(\'Full H5 metadata:\',number+1,\'mapped\',int((index[:,0]>=0).sum()),\'/\',len(rows),flush=True)\n    if seen!=set(mapping) or (index<0).any() or len(train)!=TRAIN_IMAGES or len(val)!=VALID_IMAGES:raise ValueError(\'Missing/foreign raw cache coverage\')\n    words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    sequences=[];images=[];refs=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:sequences.append(encode_tokens(sentence[\'tokens\'],lookup));images.append(j)\n        if j>=TRAIN_IMAGES:refs.append(dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]))\n        if (j+1)%20000==0:print(\'Caption metadata:\',j+1,\'/\',len(rows),flush=True)\n    arrays=dict(sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(images,np.int32),ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64),locations=index)\n    if arrays[\'sequences\'].shape!=(TRAIN_EVENTS+VALID_EVENTS,54) or int((arrays[\'caption_image\']<TRAIN_IMAGES).sum())!=TRAIN_EVENTS:raise ValueError(\'Wrong full caption count\')\n    for n,v in arrays.items():np.save(root/(n+\'.npy\'),v)\n    save(root/\'vocabulary.json\',words);save(root/\'validation_references.json\',refs)\n    if sha(root/\'vocabulary.json\')!=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from smoke\')\n    manifest=dict(status=\'FULL_PATCH_METADATA_COMPLETE\',schema=\'raw_h5_patch_stream_v1\',dataset_sha256=DATA_SHA,sources=sources,feature_profile=gold,\n        train_images=TRAIN_IMAGES,train_caption_events=TRAIN_EVENTS,valid_images=VALID_IMAGES,valid_caption_events=VALID_EVENTS,\n        files_sha256={p.name:sha(p) for p in root.iterdir() if p.is_file()},test_features_used=False,feature_arrays_staged=False,seconds=time.monotonic()-started)\n    save(root/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root,verify=True):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'FULL_PATCH_METADATA_COMPLETE\' or m[\'train_caption_events\']!=TRAIN_EVENTS or m[\'test_features_used\']:raise ValueError(\'Wrong stage scope\')\n    if verify:\n        for n,h in m[\'files_sha256\'].items():\n            if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage file \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'sequences\',\'caption_image\',\'ids\',\'locations\')}\n    return data,read(root/\'vocabulary.json\'),m\n\nclass FeatureReader:\n    def __init__(self,root,capacity=256):\n        self.data,_,self.meta=load_stage(root,verify=False);self.handles={};self.cache=OrderedDict();self.capacity=capacity\n        self.golden=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'raw_selected_row_sha256\']\n        self.local=None;self.local_rows=None\n        if self.meta.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            spec=self.meta[\'local_features\']\n            if spec[\'file\']!=\'local_patches.npy\' or spec[\'rows\']!=\'local_rows.npy\':raise ValueError(\'Unexpected local payload names\')\n            self.local=np.load(Path(root)/spec[\'file\'],mmap_mode=\'r\',allow_pickle=False)\n            self.local_rows=np.load(Path(root)/spec[\'rows\'],mmap_mode=\'r\',allow_pickle=False)\n            if self.local.shape!=(len(self.data[\'ids\']),196,768) or self.local.dtype!=np.float16 or self.local_rows.shape!=(len(self.data[\'ids\']),):raise ValueError(\'Invalid local patch mapping/shape\')\n    def get(self,image):\n        image=int(image)\n        if image in self.cache:self.cache.move_to_end(image);return self.cache[image]\n        if self.local is not None:\n            row=int(self.local_rows[image])\n            if not 0<=row<len(self.local):raise ValueError(\'Invalid local row\')\n            value=self.local[row]\n            if not np.isfinite(value).all():raise ValueError(\'Nonfinite local feature\')\n            self.cache[image]=value\n            if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n            return value\n        import h5py\n        file,row=map(int,self.data[\'locations\'][image]);source=self.meta[\'sources\'][file]\n        if file not in self.handles:\n            path=Path(source[\'path\'])\n            if path.stat().st_size!=source[\'bytes\']:raise ValueError(\'Raw file size changed\')\n            self.handles[file]=h5py.File(path,\'r\',rdcc_nbytes=2*1024**2)\n        raw=self.handles[file][\'features\'][row];coco=str(int(self.data[\'ids\'][image]))\n        if coco in self.golden and hashlib.sha256(np.ascontiguousarray(raw).tobytes()).hexdigest()!=self.golden[coco]:raise ValueError(\'Smoke raw row replay differs \'+coco)\n        value=check_features(raw[None])[0];self.cache[image]=value\n        if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n        return value\n    def close(self):\n        self.cache.clear()\n        for f in self.handles.values():f.close()\n        self.handles.clear()\n        if self.local is not None:self.local._mmap.close();self.local=None\n        if self.local_rows is not None:self.local_rows._mmap.close();self.local_rows=None\n\nclass CaptionBatches:\n    def __init__(self,root,events,rank=None,cursor=0,batch_size=32):\n        self.root=str(root);self.events=np.asarray(events,np.int64);self.rank=rank;self.cursor=cursor;self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.events)+self.batch_size-1)//self.batch_size-self.cursor\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;ev=self.events[start:start+self.batch_size]\n        if self.rank is not None:ev=ev[self.rank*16:(self.rank+1)*16]\n        if not len(ev):raise ValueError(\'Empty distributed caption batch\')\n        d=self.reader.data;image=d[\'caption_image\'][ev];x=np.stack([self.reader.get(i) for i in image]).astype(np.float32)\n        seq=np.asarray(d[\'sequences\'][ev],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return x,seq,ev.copy()\n\nclass ImageBatches:\n    def __init__(self,root,images,batch_size=8):self.root=str(root);self.images=np.asarray(images,np.int64);self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.images)+self.batch_size-1)//self.batch_size\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        indices=self.images[item*self.batch_size:(item+1)*self.batch_size]\n        return np.stack([self.reader.get(i) for i in indices]).astype(np.float32),indices.copy()\n\ndef loader(dataset,workers=2):\n    from torch.utils.data import DataLoader\n    return DataLoader(dataset,batch_size=None,num_workers=workers,pin_memory=True,multiprocessing_context=\'spawn\' if workers else None,prefetch_factor=2 if workers else None)\n', 'make_test199_notebook.py': '"""Author fixed199 TEST notebook; never submit/execute inference."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nHERE=Path(__file__).resolve().parent;ROOT=HERE.parents[1];PREFIX=\'experiments/clip_patch_baseline/\'\nsys.path.insert(0,str(ROOT/\'experiments/csa_paper\'))\nfrom make_smoke_notebook import RUNNER,cell\n\ndef make(revision,semantic=False):\n    revision=subprocess.check_output([\'git\',\'rev-parse\',revision],cwd=ROOT,text=True).strip()\n    mapping={PREFIX+n:n for n in (\'TEST199_LOCK.json\',\'TEST199_PROTOCOL.md\',\'TEST199_BUILD_V2_PROTOCOL.md\',\'SMOKE_SOURCE.json\',\'test199_data.py\',\'run_test199.py\',\'test_test199.py\',\'patch_model.py\',\'patch_beam.py\',\'patch_data.py\',\'full_data.py\',\'make_test199_notebook.py\')}\n    mapping.update({\'experiments/csa_paper/\'+n:n for n in (\'data.py\',\'real_bank_data.py\',\'cuda_runtime_guard.py\',\'saved_quality_data.py\',\'score_prompt_test_extra.py\')})\n    if semantic:\n        mapping.update({PREFIX+n:n for n in (\'semantic_test_data.py\',\'semantic_data.py\',\'semantic_patch_model.py\',\'test_semantic_test.py\',\'BANK_PROMPT_TEST_PROTOCOL.md\')})\n        mapping.update({\'experiments/csa_paper/model.py\':\'bank_layers.py\',\'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json\':\'PROMPT_TEST_PREPARATION_RESULT.json\'})\n    hashes={n:hashlib.sha256(subprocess.check_output([\'git\',\'show\',revision+\':\'+n],cwd=ROOT)).hexdigest() for n in mapping}\n    setup=f"""import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil\nfrom pathlib import Path\nREVISION={revision!r}\nMAPPING={mapping!r}\nHASHES={hashes!r}\n"""+"""VISUAL_CACHE=Path(\'/kaggle/input/datasets/ducanh2403/visual-cache\')\nCOCO_JSON=Path(\'/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json\')\nCHECKPOINT_ROOT=\'\' # Optional exact199 mounted file/root. Blank downloads ONLY own199/best.bin.\nSECRET_NAMES=(\'codex\',\'KAGGLE_API_TOKEN\') # Enable the matching Kaggle Secret; token is never logged.\nWALL_HOURS=2 # STOP cap including CPU metrics, not ETA or billing.\nOUTPUT=Path(\'/kaggle/working/clip_patch_199_test5000_v2\')\nfor p in (VISUAL_CACHE,COCO_JSON):\n    if not p.exists():raise FileNotFoundError(\'Attach/correct original inputs: \'+str(p))\nif OUTPUT.exists():raise RuntimeError(\'Preserve existing captions/output; use fresh session/output path\')\nif min(shutil.disk_usage(\'/tmp\').free,shutil.disk_usage(\'/kaggle/working\').free)<6*1024**3:raise RuntimeError(\'Need6GiB free for TEST patches/checkpoint/CPU scorer assets\')\nOUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix=\'clip-patch199-test-\',dir=\'/tmp\'));DEPS=SOURCE/\'deps\';CLI=SOURCE/\'cli\';CHAIR=SOURCE/\'chair\';LOG=OUTPUT/\'notebook.log\'\nPARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]\nos.environ[\'PYTHONPATH\']=str(SOURCE);os.environ[\'OMP_NUM_THREADS\']=\'1\'\nCONFIG=SOURCE/\'config.json\'\ncfg=dict(output=str(OUTPUT),stage=str(SOURCE/\'test_patches\'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)\ndef write_config():\n    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/\'protocol.json\').write_text(json.dumps(cfg,indent=2))\nwrite_config()\ndef redact(line):\n    for token in TOKENS:line=line.replace(token,\'[REDACTED]\')\n    return line\ndef download(url,expected,path):\n    remaining=PARENT_DEADLINE-time.monotonic()\n    if remaining<=0:raise TimeoutError(\'Parent deadline before download\')\n    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()\n    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError(\'Changed pinned input: \'+path.name)\n    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)\n"""\n    runner=RUNNER.replace(\'def run(args, timeout=1800):\',\'def run(args, timeout=1800, environment=None):\').replace(\'cwd=SOURCE, stdout=\',\'cwd=SOURCE, env=environment, stdout=\')\n    runner=runner.replace("print(\'Running:\',", "if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError(\'TEST parent cap\')\\n    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())\\n    print(\'Running:\',").replace("else: print(line, end=\'\', flush=True);", "else: line=redact(line); print(line, end=\'\', flush=True);").replace(\'Smoke active;\',\'TEST/CPU metrics active;\').replace(\'Smoke wall cap exceeded\',\'TEST parent/phase cap exceeded\')\n    routing="""import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,\'build_kaggle_client\') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',path=directory)\n    request=endpoint.call_args.args[0]\n    assert (request.user_name,request.kernel_slug,request.version_label)==(\'ducanh2403\',\'2-vi-cai-kia-qua-nhieu\',\'v199\')\nprint(\'Exact own199 API routing mock PASS; no network.\',flush=True)\n"""\n    if semantic:\n        routing=routing.replace(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',\'ducanh2403/kltnnnnnff/9\').replace("(\'ducanh2403\',\'2-vi-cai-kia-qua-nhieu\',\'v199\')","(\'ducanh2403\',\'kltnnnnnff\',\'v9\')").replace(\'Exact own199\',\'Exact own9\')\n    execution=\'ROUTING_PREFLIGHT=\'+repr(routing)+\'\\n\'+"""complete=False\ntry:\n    for path,name in MAPPING.items():download(\'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/\'+REVISION+\'/\'+path,HASHES[path],SOURCE/name)\n    (OUTPUT/\'source_receipt.json\').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))\n    run([sys.executable,\'-u\',SOURCE/\'test_test199.py\'],120)\n    run([sys.executable,\'-u\',SOURCE/\'cuda_runtime_guard.py\',\'--output\',OUTPUT],60)\n    sys.path.insert(0,str(SOURCE))\n    from test199_data import checkpoint,VERSION,CHECKPOINT_SHA\n    existing=Path(\'/kaggle/working/clip_patch_199_test5000_v1/inputs199/clip_patch_baseline_full_xe10_v3/best.bin\')\n    available=CHECKPOINT_ROOT or (str(existing) if existing.is_file() else \'\')\n    if available:\n        weights=checkpoint(available);origin=\'mounted or existing exact199 bytes\'\n    else:\n        run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--target\',CLI,\'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip\'],600)\n        env=os.environ.copy();env[\'PYTHONPATH\']=str(CLI)\n        run([sys.executable,\'-u\',\'-c\',ROUTING_PREFLIGHT],120,env)\n        from kaggle_secrets import UserSecretsClient\n        token=None\n        for name in SECRET_NAMES:\n            try:value=UserSecretsClient().get_secret(name).strip()\n            except Exception:continue\n            if value:token=value;break\n        if not token:raise RuntimeError(\'Enable own Kaggle API Secret codex or KAGGLE_API_TOKEN; no token is printed.\')\n        TOKENS.append(token)\n        for key in (\'KAGGLE_API_TOKEN\',\'KAGGLE_USERNAME\',\'KAGGLE_KEY\'):env.pop(key,None)\n        env[\'KAGGLE_API_TOKEN\']=token;env[\'PYTHONUNBUFFERED\']=\'1\'\n        target=OUTPUT/\'inputs199\';target.mkdir()\n        # CLI main entry works even on builds without kaggle.__main__.\n        run([sys.executable,\'-u\',\'-c\',\'from kaggle.cli import main; main()\',\'kernels\',\'output\',VERSION,\'-p\',str(target),\'--file-pattern\',r\'(^|/)best[.]bin$\'],900,env)\n        weights=checkpoint(target);origin=\'exact own SaveVersion199 API\'\n    cfg[\'checkpoint_root\']=str(weights);write_config()\n    (OUTPUT/\'checkpoint_source.json\').write_text(json.dumps(dict(version=VERSION,sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'checkpoint_preflight\',\'--config\',CONFIG],120)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'pycocoevalcap==1.2\',\'nltk==3.9.2\',\'future\'],600)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85\'],600)\n    os.environ[\'PYTHONPATH\']=str(DEPS)+os.pathsep+str(SOURCE)\n    run([sys.executable,\'-c\',"import nltk;from pattern.en import singularize;assert singularize(\'umbrellas\')==\'umbrella\';assert nltk.download(\'punkt\');assert nltk.download(\'punkt_tab\')"],300)\n    base=\'https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/\'\n    for name,digest in {\'chair.py\':\'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d\',\'chair.pkl\':\'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\'}.items():download(base+name,digest,CHAIR/name)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'environment\',\'--config\',CONFIG],1200)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'smoke\',\'--config\',CONFIG],600)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'prepare\',\'--config\',CONFIG],1800)\n    cfg[\'worker_seconds\']=min(1800,PARENT_DEADLINE-time.monotonic()-1800);write_config()\n    if cfg[\'worker_seconds\']<300:raise TimeoutError(\'Insufficient budget before TEST inference; no automatic extension\')\n    launch=[sys.executable,\'-u\',\'-m\',\'torch.distributed.run\',\'--standalone\',\'--nnodes=1\',\'--nproc_per_node=2\',\'--max_restarts=0\']\n    run([*launch,SOURCE/\'run_test199.py\',\'worker\',\'--config\',CONFIG],cfg[\'worker_seconds\']+30)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'tokenize\',\'--config\',CONFIG],180)\n    for mode,cap in ((\'basic\',180),(\'meteor\',600),(\'spice\',1800),(\'chair\',600),(\'combine\',120)):\n        run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',mode,\'--config\',CONFIG],cap)\n    complete=json.loads((OUTPUT/\'result.json\').read_text())[\'status\']==\'CLIP_PATCH_199_TEST5000_COMPLETE\'\nexcept BaseException:\n    (OUTPUT/\'failure.txt\').write_text(redact(traceback.format_exc()));raise\nfinally:\n    (OUTPUT/\'run_status.json\').write_text(json.dumps(dict(complete=complete,training=False,test_tuning=False,source_version=\'199\')))\n    review=OUTPUT.with_name(OUTPUT.name+\'_review.zip\')\n    with zipfile.ZipFile(review,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in OUTPUT.rglob(\'*\'):\n            if p.is_file() and p.suffix in (\'.json\',\'.jsonl\',\'.log\',\'.txt\'):z.write(p,str(p.relative_to(OUTPUT)))\n    print(\'SEND REVIEW:\',review,flush=True);print(\'KEEP OUTPUT:\',OUTPUT,flush=True)\n    print(\'TEST COMPLETE; no training/tuning.\' if complete else \'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.\',flush=True)\n"""\n    intro="""# Baseline199 — TEST5000 V2, sửa tương thích CUDA\n\n**T4×2**, InternetON, Save & Run All. Chỉ gắn `ducanh2403/visual-cache` và `vuthetam/mscoco-2014`. Không cần detector/ROI/bank/prompt/testcachecũ. Chỉ stage5000TESTpatch~1,4GiB, không chép full33GiB và không chạyCLIP lại.\n\nCheckpointbestepoch10 cốđịnh từ **ducanh2403/2-vi-cai-kia-qua-nhieu/199**, SHA5061e305...; CHECKPOINT_ROOT tùychọn nếuđãcófile/DS, đểtrống thì tảiONLYbest.bin từđúng199. Bật Secret **codex** hoặc **KAGGLE_API_TOKEN** của tàikhoảnđó. Khônglatest/phiênbảnkhác/foreignnotebook API; khônglộtoken.\n\nBeam5/max20/rawlogprob/rank0 giữnhưVALID, khôngrerank/lengthnormalization/training/tuning. CaptionthamchiếuTEST chỉdùng chấmđiểm vàNLL saukhisinhcaption. Haiworker2500ảnh/rank, lưucaptiontrướcCPUmetrics. ChấmBLEU-1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall,NLL); thiếuphase thìkhôngđánhcomplete. Giữảnh/split/model/sốcaptioncốđịnh; priorprojectTESTexposure vẫnđượcghi.\n\nV2 cho phép Torch2.11.0 CUDAcu128/cu130 đã quan sát, ghi rõ khác biệtbuild và không cam kết caption giốngbitwise. Kiểm checkpoint/runtime trước khi stage; strictweights và GPUbeam-pathcheck vẫn bắt buộc. Không cài lại Torch. Nếu filecheckpoint199 cũ còn trongWorking, tự dùng lại sau kiểmSHA.\n\nCap2giờgồmcảdownload/scorerCPU, khôngphảiETA/quota. StanfordCoreNLP/JavaMeteorspicecóthểcầnthờigian. MỗiCPUphase lưukếtquảriêng; nếulỗi, giữoutputcaption đểphụchồiCPU, khôngtựchạymôhìnhlại. Gửi **clip_patch_199_test5000_v2_review.zip**. Tácgiảchỉkiểmmetadata/NumPy/mock/AST; chưa chạyCUDA/scorerslocal.\n"""\n    if semantic:\n        setup=setup.replace("CHECKPOINT_ROOT=\'\'", "PROMPT_TEST_ROOT=Path(\'/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181\')\\nCHECKPOINT_ROOT=\'\'")\n        setup=setup.replace(\'exact199\',\'exact bank+prompt best9\').replace(\'own199\',\'own kltnnnnnff/9\').replace(\'clip_patch_199_test5000_v2\',\'clip_patch_bank_prompt_test5000_v1\')\n        setup=setup.replace(\'for p in (VISUAL_CACHE,COCO_JSON):\',\'for p in (VISUAL_CACHE,COCO_JSON,PROMPT_TEST_ROOT):\')\n        setup=setup.replace(\'cfg=dict(output=\',"cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=")\n        execution=execution.replace(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',\'ducanh2403/kltnnnnnff/9\')\n        execution=execution.replace("\'2-vi-cai-kia-qua-nhieu\', \'v199\'", "\'kltnnnnnff\', \'v9\'")\n        execution=execution.replace(\'from test199_data import checkpoint,VERSION,CHECKPOINT_SHA\',\'from semantic_test_data import checkpoint,VERSION,CHECKPOINT_SHA,prompt_inputs\')\n        execution=execution.replace("Path(\'/kaggle/working/clip_patch_199_test5000_v1/inputs199/clip_patch_baseline_full_xe10_v3/best.bin\')", "Path(\'/kaggle/working/clip_patch_bank_prompt_xe10_v1/best.bin\')")\n        execution=execution.replace("SOURCE/\'test_test199.py\'],120)","SOURCE/\'test_test199.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_semantic_test.py\'],120)")\n        execution=execution.replace("cfg[\'checkpoint_root\']=str(weights);write_config()", "cfg[\'checkpoint_root\']=str(weights);write_config()\\n    prompt_inputs(cfg) # exact TEST181 seal/scope check before expensive setup/stage")\n        execution=execution.replace("TOKENS.append(token)","if token.startswith(\'sk-\'):raise RuntimeError(\'Use a Kaggle API Secret, not an OpenAI key.\')\\n        TOKENS.append(token)")\n        execution=execution.replace(\'exact199 bytes\',\'exact best9 bytes\').replace(\'exact own SaveVersion199 API\',\'exact own kltnnnnnff/9 API\')\n        execution=execution.replace(\'CLIP_PATCH_199_TEST5000_COMPLETE\',\'CLIP_PATCH_BANK_PROMPT_TEST5000_COMPLETE\').replace("source_version=\'199\'","source_version=\'ducanh2403/kltnnnnnff/9\'")\n        intro=\'\'\'# Bank + prompt, best epoch9 — TEST5000\n\n**T4 ×2**, Internet ON, Save & Run All. Attach THREE Datasets:\n- `ducanh2403/visual-cache`\n- `vuthetam/mscoco-2014`\n- `ducanh2403/csa-prompt-test-cache-v181`\n\nEnable your **Kaggle API** Secret `codex` or `KAGGLE_API_TOKEN` (not an OpenAI key). Blank CHECKPOINT_ROOT downloads only best.bin from **ducanh2403/kltnnnnnff/9**, exact SHA7c11be52. User confirmed notebook slug/version; default owner is own ducanh2403. Missing access fails early; no latest/other checkpoint fallback. Alternatively mount the exact knownSHA file/root and set CHECKPOINT_ROOT. No API token printed.\n\nBest epoch9 is159399updates; the full experiment ran10epochs but epoch10 has different weights. Keep epoch9 chosen on VALID before TEST. Model frozen, no training/selection/reranking. Frozen TRAIN noun/verb banks are inside the checkpoint; no bank130 or prompt175 dataset needed. TEST181 only supplies258 fixed prompt embeddings +instance links. Uses exact same ROI-box→14×14patch pooling/masks as training; old ROI feature arrays and old181 vocabulary are NOT used. No CLIP/UPT inference.\n\nOnly5000TEST patches staged (~1.4GiB), no full33GiB copy or TRAIN/VALID feature reads. Same5000IDs/rawreferences, TRAIN vocabulary and beam5/max20/rawlogprob/rank0 as baseline199. Captions saved before teacherNLL and CPUscoring. Metrics: BLEU1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall,NLL). Complete only when all eight metrics finished. Previous project TEST exposure disclosed; no TEST tuning.\n\nParent2h STOP cap inherited from baselineTEST, not ETA/billing. Latest answer confirmed cache path but did not supply remaining GPU hours; check account quota before launch. GPUweights/runtime/firstbeam-path guards before full decode. No Torch reinstall. Preserve caption/output on errors; no automatic model rerun. Send **clip_patch_bank_prompt_test5000_v1_review.zip**. Author checks NumPy/mock/AST only; models/CUDA/scorers run only by user.\n\'\'\'\n    nb=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(display_name=\'Python 3\',language=\'python\',name=\'python3\'),kaggle=dict(isInternetEnabled=True,isGpuEnabled=True)),cells=[cell(\'markdown\',intro),cell(\'code\',setup),cell(\'code\',runner),cell(\'code\',execution)])\n    p=HERE/(\'CLIP_PATCH_BANK_PROMPT_TEST5000_2GPU.ipynb\' if semantic else \'CLIP_PATCH_199_TEST5000_V2_2GPU.ipynb\');p.write_text(json.dumps(nb,ensure_ascii=False,indent=1)+\'\\n\',encoding=\'utf-8\');print(p.name,p.stat().st_size,\'bytes\',len(mapping),\'pins\',revision)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--revision\',required=True);p.add_argument(\'--semantic\',action=\'store_true\');a=p.parse_args();make(a.revision,a.semantic)\n', 'data.py': '"""Pure NumPy/data contracts; importing this file never imports a model."""\nfrom collections import Counter\nimport re\nimport unicodedata\n\nimport numpy as np\n\nPAD, BOS, EOS, UNK = range(4)\nSPECIALS = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\n\n\ndef roi_pool(patches, objects, width, height, cap=150):\n    """Pixel xyxy -> fractional-area-weighted raw patch features, no class inputs."""\n    patches = np.asarray(patches, dtype=np.float32)\n    if patches.ndim != 3 or patches.shape[0] != patches.shape[1]:\n        raise ValueError(\'Expected square patch grid [G,G,C], without CLS\')\n    if not np.isfinite(patches).all() or not np.isfinite([width, height]).all():\n        raise ValueError(\'Nonfinite patch or dimensions\')\n    if width <= 0 or height <= 0 or cap < 1 or not isinstance(objects, list):\n        raise ValueError(\'Invalid dimensions, cap, or detector record\')\n    if not objects:\n        return patches.mean(axis=(0, 1), keepdims=False)[None], np.array([[0, 0, width, height]], np.float32), [\'full_frame_fallback\']\n    checked = []\n    for index, obj in enumerate(objects):\n        box = np.asarray(obj[\'bbox\'], dtype=np.float32)\n        score = float(obj[\'conf\'])\n        if box.shape != (4,) or not np.isfinite(box).all() or not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Malformed detection\')\n        box = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if box[2] <= box[0] or box[3] <= box[1]:\n            raise ValueError(\'Nonpositive detection area; do not turn corruption into empty detection\')\n        checked.append((-score, index, box))\n    selected = sorted(checked, key=lambda item: item[:2])[:cap]\n    grid = patches.shape[0]\n    axis = np.arange(grid, dtype=np.float32)\n    pooled, boxes = [], []\n    for _, _, box in selected:\n        x0, y0, x1, y1 = box / np.array([width, height, width, height]) * grid\n        wx = np.maximum(0, np.minimum(axis + 1, x1) - np.maximum(axis, x0))\n        wy = np.maximum(0, np.minimum(axis + 1, y1) - np.maximum(axis, y0))\n        weights = (wy[:, None] * wx[None, :]).astype(np.float32)\n        pooled.append(np.einsum(\'ij,ijc->c\', weights, patches) / weights.sum())\n        boxes.append(box)\n    return np.stack(pooled), np.stack(boxes), [\'predicted_roi\'] * len(pooled)\n\n\ndef train_captions(dataset):\n    for image in dataset[\'images\']:\n        if image[\'split\'] in (\'train\', \'restval\'):\n            for sentence in image[\'sentences\']:\n                tokens = sentence[\'tokens\']\n                if not isinstance(tokens, list) or not all(isinstance(t, str) and t for t in tokens):\n                    raise ValueError(\'Malformed caption tokens\')\n                yield [unicodedata.normalize(\'NFKC\', t).lower() for t in tokens]\n\n\ndef vocabulary(dataset, limit=10000):\n    if limit < 4:\n        raise ValueError(\'Vocabulary limit includes four specials\')\n    counts = Counter(t for caption in train_captions(dataset) for t in caption if t not in SPECIALS)\n    return SPECIALS + sorted(counts, key=lambda word: (-counts[word], word))[:limit - 4]\n\n\ndef encode(tokens, words, maximum=54):\n    if maximum < 2:\n        raise ValueError(\'Sequence must retain BOS/EOS\')\n    lookup = {word: i for i, word in enumerate(words)}\n    content = [lookup.get(unicodedata.normalize(\'NFKC\', t).lower(), UNK) for t in tokens[:maximum - 2]]\n    return [BOS] + content + [EOS] + [PAD] * (maximum - len(content) - 2)\n\n\ndef bank_words(tagged_lemmas, minimum=5):\n    """Input must come from pinned POS model applied to TRAIN captions only."""\n    counts = {\'noun\': Counter(), \'verb\': Counter()}\n    for lemma, pos in tagged_lemmas:\n        word = unicodedata.normalize(\'NFKC\', lemma).lower()\n        kind = \'noun\' if pos in (\'NOUN\', \'PROPN\') else \'verb\' if pos == \'VERB\' else None\n        if kind and re.fullmatch(\'[a-z]+\', word):\n            counts[kind][word] += 1\n    return {kind: [(w, c[w]) for w in sorted(c, key=lambda w: (-c[w], w)) if c[w] >= minimum]\n            for kind, c in counts.items()}\n', 'real_bank_data.py': '"""Safe real-input contracts. No neural imports or execution."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\ndef validate_arrays(a, images, vocab):\n    n = len(images); total = len(a[\'features\'])\n    shapes = dict(features=(total,768), boxes=(total,4), scores=(total,), offsets=(n+1,), ids=(n,))\n    types = dict(features=np.float16, boxes=np.float32, scores=np.float32, offsets=np.int64, ids=np.int64)\n    if set(a) != set(shapes): raise ValueError(\'Region members differ\')\n    for key in shapes:\n        if a[key].shape != shapes[key] or a[key].dtype != types[key] or not np.isfinite(a[key]).all():\n            raise ValueError(\'Invalid region array: \'+key)\n    offsets = a[\'offsets\']\n    if offsets[0] != 0 or offsets[-1] != total or not ((np.diff(offsets)>=1)&(np.diff(offsets)<=150)).all():\n        raise ValueError(\'Invalid offsets\')\n    for i, m in enumerate(images):\n        if int(a[\'ids\'][i]) != m[\'coco_id\']: raise ValueError(\'Image order differs\')\n        b = a[\'boxes\'][offsets[i]:offsets[i+1]]\n        if len(b)!=len(m[\'region_kinds\']) or m[\'width\']<=0 or m[\'height\']<=0: raise ValueError(\'Region metadata differs\')\n        if not ((b[:,0]>=0)&(b[:,1]>=0)&(b[:,2]<=m[\'width\'])&(b[:,3]<=m[\'height\'])&(b[:,2]>b[:,0])&(b[:,3]>b[:,1])).all():\n            raise ValueError(\'Invalid boxes\')\n        if any(k not in (\'predicted_roi\',\'full_frame_fallback\') for k in m[\'region_kinds\']): raise ValueError(\'Unknown ROI kind\')\n        for seq in m[\'sequences\']:\n            if len(seq)!=54 or seq[0]!=1 or seq.count(2)!=1 or any(type(x)!=int or x<0 or x>=len(vocab) for x in seq):\n                raise ValueError(\'Invalid target sequence\')\n            end=seq.index(2)\n            if 0 in seq[:end] or any(seq[end+1:]): raise ValueError(\'Invalid EOS/PAD sequence\')\n\ndef load_prepared(root):\n    root=Path(root); p=read(HERE/\'REAL_BANK_INPUT_PACKAGE.json\')\n    if not root.is_dir():\n        raise FileNotFoundError(\'Add Input csa-real-v126-lemma-fixed or correct PREPARED_ROOT; directory not mounted: \'+str(root))\n    if not (root/\'corrected_assets_receipt.json\').is_file():\n        candidates=list(root.rglob(\'corrected_assets_receipt.json\'))\n        if len(candidates)!=1:\n            raise ValueError(f\'Expected one corrected_assets_receipt.json under {root}; found {len(candidates)}. Add the combined csa-real-v126-lemma-fixed Dataset, not only the semantic-bank Dataset.\')\n        root=candidates[0].parent\n    receipt=read(root/\'corrected_assets_receipt.json\')\n    for field, expected in [(\'original_files_sha256\',p[\'original_files_sha256\']),(\'corrected_overlay_sha256\',p[\'overlay_sha256\'])]:\n        if receipt.get(field)!=expected: raise ValueError(\'Input receipt differs: \'+field)\n    if receipt.get(\'selected_bank_file\')!=p[\'selected_bank_file\'] or receipt.get(\'original_kernel_version\')!=p[\'original_kernel_version\']:\n        raise ValueError(\'Selected bank/version differs\')\n    for name, expected in {**p[\'original_files_sha256\'],**p[\'overlay_sha256\']}.items():\n        if not (root/name).is_file() or sha(root/name)!=expected: raise ValueError(\'Missing/changed input: \'+name)\n    with np.load(root/\'regions.npz\', allow_pickle=False) as f: arrays={k:f[k] for k in f.files}\n    metadata=read(root/\'pilot_metadata.json\'); vocab=read(root/\'vocabulary.json\'); bank=read(root/p[\'selected_bank_file\'])\n    images=metadata[\'images\']\n    if len(images)!=320 or sum(x[\'group\']==\'train\' for x in images)!=256 or sum(x[\'group\']==\'val\' for x in images)!=64:\n        raise ValueError(\'Pilot groups differ\')\n    validate_arrays(arrays,images,vocab)\n    if len(bank[\'noun\'])!=5269 or len(bank[\'verb\'])!=1594 or len(vocab)!=10000: raise ValueError(\'Bank/vocabulary size differs\')\n    for group in (\'train\',\'val\'):\n        if [images[i][\'coco_id\'] for i in p[group+\'_indices\']]!=p[group+\'_ids\']: raise ValueError(\'Smoke selection differs\')\n    return root,p,arrays,images,vocab,bank\n\ndef batch(arrays, images, indices):\n    lengths=[int(arrays[\'offsets\'][i+1]-arrays[\'offsets\'][i]) for i in indices]\n    x=np.zeros((len(indices),max(lengths),768),np.float32); valid=np.zeros(x.shape[:2],bool)\n    seq=np.array([images[i][\'sequences\'][0] for i in indices],np.int64)\n    seq=seq[:,:int((seq!=0).sum(1).max())]\n    for j,i in enumerate(indices):\n        lo,hi=arrays[\'offsets\'][i:i+2]; x[j,:lengths[j]]=arrays[\'features\'][lo:hi]; valid[j,:lengths[j]]=True\n    return x,valid,seq\n', 'cuda_runtime_guard.py': '"""Metadata-only CUDA preflight; Torch is imported only in the user-run probe."""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef validate_runtime(info):\n    if not info[\'cuda_build\'] or not info[\'set_device_symbol\']:\n        raise RuntimeError(\'Loaded PyTorch lacks CUDA support. Select Accelerator GPU T4 x2 and restart the Kaggle session. If T4 x2 was already selected, send cuda_runtime.json; do not blindly reinstall Torch.\')\n    if not info[\'cuda_available\'] or info[\'device_count\']!=2:\n        raise RuntimeError(\'Exactly two CUDA GPUs must be visible. Select GPU T4 x2, restart the session and check Save Version settings. See cuda_runtime.json.\')\n    if not info[\'nccl_available\']:raise RuntimeError(\'This PyTorch runtime lacks NCCL. Send cuda_runtime.json before changing packages.\')\n\n\ndef check_runtime(output,rank=None):\n    import torch\n    import torch.distributed as dist\n    info=dict(torch_version=str(torch.__version__),torch_file=str(torch.__file__),cuda_build=torch.version.cuda,\n        set_device_symbol=hasattr(torch._C,\'_cuda_setDevice\'),cuda_available=bool(torch.cuda.is_available()),\n        device_count=int(torch.cuda.device_count()),nccl_available=bool(dist.is_nccl_available()),models_executed=False)\n    path=Path(output)/(\'cuda_runtime.json\' if rank is None else f\'cuda_runtime_rank{rank}.json\')\n    path.write_text(json.dumps(info,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(\'CUDA runtime:\',json.dumps(info),flush=True)\n    validate_runtime(info)\n    print(\'CUDA runtime PASS: two GPUs and NCCL; no model loaded.\',flush=True)\n    return info\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args();check_runtime(a.output)\n', 'saved_quality_data.py': '"""Saved text contracts only; no models or metric dependencies."""\nimport hashlib\nimport json\nimport math\n\ndef check_rows(rows,expected):\n    if len(rows)!=len(expected) or len({x[\'coco_id\'] for x in rows})!=len(rows) or {x[\'coco_id\'] for x in rows}!=set(expected):\n        raise ValueError(\'Missing/duplicate/foreign caption IDs\')\n    if any(not isinstance(x[\'caption\'],str) for x in rows):raise ValueError(\'Non-text caption\')\n\ndef common_refs(references,bundle):\n    all_refs={x[\'coco_id\']:x[\'references\'] for x in references}\n    ids=bundle[\'cohort_ids\']\n    if len(ids)!=len(set(ids)) or len(ids)!=4000:raise ValueError(\'Expected4000unique IDs\')\n    if not set(ids)<=set(all_refs):raise ValueError(\'Old4000 not contained in new5000\')\n    selected=[dict(coco_id=i,references=all_refs[i][:5]) for i in sorted(ids)]\n    if any(len(x[\'references\'])!=5 for x in selected):raise ValueError(\'Missing reference\')\n    digest=hashlib.sha256(json.dumps(selected,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n    if digest!=bundle[\'reference_sha256\']:raise ValueError(\'Old/new reference identities differ\')\n    order=bundle.get(\'legacy_order\',sorted(ids))\n    if len(order)!=len(ids) or set(order)!=set(ids):raise ValueError(\'Wrong legacy input order\')\n    by_id={x[\'coco_id\']:x for x in selected}\n    return [by_id[i] for i in order]\n\ndef chair_counts(chair,rows):\n    sentences=chair[\'sentences\'];ids={x[\'coco_id\'] for x in rows};mapping={x[\'coco_id\']:x[\'caption\'] for x in rows}\n    if len(sentences)!=len(ids) or {x[\'image_id\'] for x in sentences}!=ids:raise ValueError(\'CHAIR ID coverage\')\n    if any(x[\'caption\']!=mapping[x[\'image_id\']] for x in sentences):raise ValueError(\'CHAIR captions differ\')\n    counts=dict(images=len(ids),hallucinated_captions=sum(bool(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        hallucinated_mentions=sum(len(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        object_mentions=sum(len(x[\'mscoco_generated_words\']) for x in sentences),\n        recalled_unique=sum(len(set(x[\'mscoco_generated_words\'])&set(x[\'mscoco_gt_words\'])) for x in sentences),\n        reference_unique=sum(len(set(x[\'mscoco_gt_words\'])) for x in sentences))\n    metric=dict(CHAIRs=counts[\'hallucinated_captions\']/len(ids),CHAIRi=counts[\'hallucinated_mentions\']/max(1,counts[\'object_mentions\']),\n                Recall=counts[\'recalled_unique\']/max(1,counts[\'reference_unique\']))\n    if any(not math.isfinite(v) or abs(v-chair[\'overall_metrics\'][k])>1e-12 for k,v in metric.items()):raise ValueError(\'CHAIR count aggregation differs\')\n    return metric,counts\n\ndef align_cider_scores(references,values,rows):\n    if len(references)!=len(values):raise ValueError(\'CIDEr score/reference length mismatch\')\n    indexed={r[\'coco_id\']:float(v) for r,v in zip(references,values)}\n    if len(indexed)!=len(references) or {r[\'coco_id\'] for r in rows}!=set(indexed):raise ValueError(\'CIDEr score IDs differ\')\n    return [indexed[r[\'coco_id\']] for r in rows]\n', 'score_prompt_test_extra.py': '"""USER-RUN CPU scorers on exact saved TEST captions; no model imports."""\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport tarfile\nimport urllib.request\nimport zipfile\n\nHERE = Path(__file__).resolve().parent\nJRE_URL = (\'https://github.com/adoptium/temurin8-binaries/releases/download/\'\n           \'jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz\')\nJRE_SHA = \'6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e\'\nKEYS = (\'Bleu_1\', \'Bleu_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\', \'CHAIRs\', \'CHAIRi\')\n\n\ndef read(p):\n    return json.loads(Path(p).read_text(encoding=\'utf-8\'))\n\n\ndef sha(p):\n    with Path(p).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef save(p, value):\n    p = Path(p)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_suffix(p.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    os.replace(tmp, p)\n\n\ndef download(url, dest):\n    print(\'Downloading:\', url, flush=True)\n    total = 0\n    with urllib.request.urlopen(url, timeout=90) as r, Path(dest).open(\'wb\') as f:\n        while chunk := r.read(1024 * 1024):\n            f.write(chunk)\n            total += len(chunk)\n            if total % (25 * 1024 * 1024) == 0:\n                print(\'Downloaded\', total // (1024 * 1024), \'MiB\', flush=True)\n\n\ndef verify(root, arm, manifest):\n    """Accept nesting, never a different payload/version or incomplete cohort."""\n    spec = manifest[\'arms\'][arm]\n    files = {}\n    for name, digest in spec[\'payload_sha256\'].items():\n        matches = [p for p in Path(root).rglob(name) if p.is_file() and sha(p) == digest]\n        if len(matches) != 1:\n            raise ValueError(f\'Need one exact {name} from {spec["version"]}; found {len(matches)}\')\n        files[name] = matches[0]\n    result, lock = read(files[\'result.json\']), read(files[\'test_lock.json\'])\n    if (result[\'status\'] != \'PROMPT_TEST5000_COMPLETE\' or result[\'arm\'] != arm\n            or result[\'images\'] != manifest[\'images\'] or result[\'training\']\n            or result[\'checkpoint_sha256\'] != spec[\'checkpoint_sha256\']\n            or lock[\'checkpoint_sha256\'] != spec[\'checkpoint_sha256\']\n            or sha(files[\'test_references.json\']) != lock[\'references_sha256\']\n            or read(files[\'run_status.json\']) != dict(complete=True, arm=arm, training=False, test_tuning=False)):\n        raise ValueError(\'Wrong source identity/status\')\n    refs = read(files[\'test_references.json\'])\n    ids = {r[\'coco_id\'] for r in refs}\n    if len(refs) != manifest[\'images\'] or len(ids) != len(refs):\n        raise ValueError(\'Duplicate/missing TEST reference IDs\')\n    if any(type(i) is not int for i in ids) or any(\n            not isinstance(r[\'references\'], list) or len(r[\'references\']) < 5\n            or any(not isinstance(t, str) or not t.strip() for t in r[\'references\']) for r in refs):\n        raise ValueError(\'Wrong reference structure\')\n    captions = {}\n    for policy in manifest[\'policies\']:\n        rows = read(files[policy + \'_captions.json\'])\n        if (len(rows) != len(ids) or {r[\'coco_id\'] for r in rows} != ids\n                or any(not isinstance(r[\'caption\'], str) or not r[\'caption\'].strip() for r in rows)\n                or result[\'metrics\'][policy] != spec[\'metrics\'][policy]):\n            raise ValueError(\'Changed/incomplete captions or historical metrics: \' + policy)\n        captions[policy] = rows\n    return refs, captions, files\n\n\ndef environment(out):\n    from importlib.metadata import version\n    if version(\'pycocoevalcap\') != \'1.2\':\n        raise ValueError(\'Expected pycocoevalcap==1.2\')\n    assets = out / \'assets\'\n    assets.mkdir()\n    archive = assets / \'java8.tar.gz\'\n    download(JRE_URL, archive)\n    if sha(archive) != JRE_SHA:\n        raise ValueError(\'Changed pinned Java8 asset\')\n    with tarfile.open(archive) as t:\n        t.extractall(assets / \'jre\', filter=\'data\')\n    java = list((assets / \'jre\').rglob(\'bin/java\'))\n    if len(java) != 1:\n        raise ValueError(\'Missing pinned Java executable\')\n    spice_dir = Path(importlib.util.find_spec(\'pycocoevalcap.spice.spice\').origin).parent\n    lib = spice_dir / \'lib\'\n    lib.mkdir(exist_ok=True)\n    archive = assets / \'corenlp.zip\'\n    download(\'https://downloads.cs.stanford.edu/nlp/software/stanford-corenlp-full-2015-12-09.zip\', archive)\n    for suffix in (\'.jar\', \'-models.jar\'):\n        name = \'stanford-corenlp-3.6.0\' + suffix\n        member = \'stanford-corenlp-full-2015-12-09/\' + name\n        with zipfile.ZipFile(archive) as z, z.open(member) as src, (lib / name).open(\'wb\') as dst:\n            shutil.copyfileobj(src, dst)\n    package = spice_dir.parent\n    save(out / \'environment.json\', dict(pycocoevalcap=version(\'pycocoevalcap\'), java=str(java[0]),\n        java_archive_sha256=JRE_SHA, corenlp_archive_sha256=sha(archive),\n        corenlp_version=\'3.6.0\', model_digest_pinned_independently=False,\n        evaluator_files={str(p.relative_to(package)): sha(p) for p in sorted(package.rglob(\'*\'))\n                         if p.is_file() and p.suffix in (\'.py\', \'.jar\', \'.gz\')}))\n\n\ndef java_path(out):\n    env = read(out / \'environment.json\')\n    os.environ[\'PATH\'] = str(Path(env[\'java\']).parent) + os.pathsep + os.environ[\'PATH\']\n    os.environ[\'JAVA_HOME\'] = str(Path(env[\'java\']).parent.parent)\n\n\ndef checked_point(value, length, scores):\n    value = float(value)\n    if not math.isfinite(value) or len(scores) != length:\n        raise ValueError(\'Nonfinite metric or incomplete per-image coverage\')\n    return value\n\n\ndef smoke(out):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    tok = PTBTokenizer()\n    refs = tok.tokenize({1: [dict(caption=\'a person riding a bicycle on a road\')]})\n    hyp = tok.tokenize({1: [dict(caption=\'a person riding a bicycle on a road\')]})\n    points = {\'Bleu_1\': float(Bleu(4).compute_score(refs, hyp)[0][0])}\n    for name, factory in [(\'METEOR\', Meteor), (\'SPICE\', Spice)]:\n        scorer = factory()\n        try:\n            point, scores = scorer.compute_score(refs, hyp)\n            points[name] = checked_point(point, 1, scores)\n            if points[name] <= 0:\n                raise ValueError(\'Exact-match smoke has no positive score: \' + name)\n        finally:\n            del scorer\n    save(out / \'smoke.json\', dict(passed=True, scores=points))\n    print(\'Actual CPU PTB/BLEU/METEOR/SPICE smoke PASS\', points, flush=True)\n\n\ndef tokenize(out, root, arm, manifest):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    refs, captions, files = verify(root, arm, manifest)\n    tok = PTBTokenizer()\n    reference = tok.tokenize({r[\'coco_id\']: [dict(caption=t) for t in r[\'references\']] for r in refs})\n    save(out / \'references_ptb.json\', reference)\n    for policy, rows in captions.items():\n        save(out / (policy + \'_ptb.json\'), tok.tokenize({r[\'coco_id\']: [dict(caption=r[\'caption\'])] for r in rows}))\n    save(out / \'input_receipt.json\', dict(arm=arm, source_version=manifest[\'arms\'][arm][\'version\'],\n        images=len(refs), payload_sha256={n: sha(p) for n, p in files.items()},\n        tokenized_sha256={p.name: sha(p) for p in out.glob(\'*_ptb.json\')},\n        training=False, decoding=False, reranking=False))\n\n\ndef metric(out, arm, manifest, mode):\n    receipt = read(out / \'input_receipt.json\')\n    if receipt[\'arm\'] != arm or receipt[\'payload_sha256\'] != manifest[\'arms\'][arm][\'payload_sha256\']:\n        raise ValueError(\'Input receipt changed\')\n    for n, digest in receipt[\'tokenized_sha256\'].items():\n        if sha(out / n) != digest:\n            raise ValueError(\'Tokenized captions changed\')\n    reference = read(out / \'references_ptb.json\')\n    for policy in manifest[\'policies\']:\n        print(\'CPU scoring:\', policy, mode, flush=True)\n        hyp = read(out / (policy + \'_ptb.json\'))\n        if len(reference) != manifest[\'images\'] or set(reference) != set(hyp):\n            raise ValueError(\'Metric cohort differs\')\n        if mode == \'basic\':\n            from pycocoevalcap.bleu.bleu import Bleu\n            from pycocoevalcap.cider.cider import Cider\n            from pycocoevalcap.rouge.rouge import Rouge\n            b, _ = Bleu(4).compute_score(reference, hyp)\n            c, cs = Cider().compute_score(reference, hyp)\n            r, rs = Rouge().compute_score(reference, hyp)\n            scores = dict(Bleu_1=float(b[0]), Bleu_4=float(b[3]),\n                CIDEr=checked_point(c, len(hyp), cs), ROUGE_L=checked_point(r, len(hyp), rs))\n            for k in (\'Bleu_4\', \'CIDEr\', \'ROUGE_L\'):\n                if abs(scores[k] - manifest[\'arms\'][arm][\'metrics\'][policy][k]) > 1e-7:\n                    raise ValueError(\'Historical metric replay differs: \' + policy + \'/\' + k)\n        else:\n            if mode == \'meteor\':\n                from pycocoevalcap.meteor.meteor import Meteor\n                scorer = Meteor()\n            elif mode == \'spice\':\n                from pycocoevalcap.spice.spice import Spice\n                scorer = Spice()\n            else:\n                raise ValueError(mode)\n            try:\n                point, per_image = scorer.compute_score(reference, hyp)\n                scores = {mode.upper(): checked_point(point, len(hyp), per_image)}\n            finally:\n                del scorer\n        if any(not math.isfinite(v) for v in scores.values()):\n            raise ValueError(\'Nonfinite metric\')\n        save(out / f\'{policy}_{mode}.json\', dict(images=len(hyp), metrics=scores,\n            tokenized_sha256=receipt[\'tokenized_sha256\']))\n\n\ndef combine(out, arm, manifest):\n    points = {}\n    receipt = read(out / \'input_receipt.json\')\n    for policy in manifest[\'policies\']:\n        row = {}\n        for mode in (\'basic\', \'meteor\', \'spice\'):\n            part = read(out / f\'{policy}_{mode}.json\')\n            if part[\'images\'] != 5000 or part[\'tokenized_sha256\'] != receipt[\'tokenized_sha256\']:\n                raise ValueError(\'Partial/changed metric phase\')\n            row.update(part[\'metrics\'])\n        row.update({k: manifest[\'arms\'][arm][\'metrics\'][policy][k] for k in (\'CHAIRs\', \'CHAIRi\')})\n        if set(row) != set(KEYS) or not all(math.isfinite(v) for v in row.values()):\n            raise ValueError(\'Need all eight metrics\')\n        points[policy] = row\n    result = dict(status=\'PROMPT_TEST_EIGHT_METRICS_COMPLETE\', arm=arm, images=5000,\n        source_version=manifest[\'arms\'][arm][\'version\'], metrics=points,\n        display_x100={p: {k: 100*v for k, v in row.items()} for p, row in points.items()},\n        chair_provenance=\'Reused audited scores, exact caption/reference/result SHA verified; not recomputed.\',\n        inputs=receipt, environment=read(out / \'environment.json\'), training=False, decoding=False,\n        reranking=False, test_tuning=False, prior_project_test_exposure=True)\n    save(out / \'result.json\', result)\n    with (out / \'metrics_x100.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=[\'arm\', \'policy\', *KEYS])\n        w.writeheader()\n        for policy, row in result[\'display_x100\'].items():\n            w.writerow(dict(arm=arm, policy=policy, **row))\n    print(\'COMPLETE — saved captions only:\', result[\'display_x100\'], flush=True)\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'environment\', \'smoke\', \'tokenize\', \'basic\', \'meteor\', \'spice\', \'combine\'])\n    p.add_argument(\'--arm\', choices=[\'CONTROL\', \'PROMPT\'], required=True)\n    p.add_argument(\'--input\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    args = p.parse_args()\n    manifest = read(HERE / \'PROMPT_TEST_EXTRA_INPUTS.json\')\n    if args.mode == \'environment\':\n        environment(args.output)\n    else:\n        java_path(args.output)\n        if args.mode == \'smoke\':\n            smoke(args.output)\n        elif args.mode == \'tokenize\':\n            tokenize(args.output, args.input, args.arm, manifest)\n        elif args.mode == \'combine\':\n            combine(args.output, args.arm, manifest)\n        else:\n            metric(args.output, args.arm, manifest, args.mode)\n', 'semantic_test_data.py': '"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS,record_arrays,checked_root\n\nHERE=Path(__file__).resolve().parent\nCHECKPOINT_SHA=\'7c11be52fce9cd90dadc55fa514d4017d540f40f630232a68bfe4306c754158b\'\nVERSION=\'ducanh2403/kltnnnnnff/9\'\n\n\ndef validate_lock(value):\n    baseline=shared.lock()\n    if any(value[k]!=baseline[k] for k in (\'expected_test_ids\',\'test_id_sha256\',\'beam_width\',\'max_new_tokens\',\'length_normalization\',\'reranking\')):raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\']!=VERSION or value[\'checkpoint_sha256\']!=CHECKPOINT_SHA or value[\'epoch\']!=9 or value[\'updates\']!=159399 or value[\'method\']!=\'bank_prompt_patch_v1\':raise ValueError(\'Need fixed bank+prompt best9, not endpoint10\')\n    return value\n\n\ndef lock():\n    value=shared.lock();value.update(schema=\'bank_prompt_best9_test_v1\',source_version=VERSION,checkpoint_sha256=CHECKPOINT_SHA,\n        training_commit=\'f5f6eb05d7c89917b0719637dae4e948ad73f531\',epoch=9,updates=159399,method=\'bank_prompt_patch_v1\')\n    return validate_lock(value)\n\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Missing exact version9 best.bin SHA7c11be52; no final/latest fallback\')\n    return matches[0]\n\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu130\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Same publicTorch2.11.0 observed CUDA wheels; strictstate and actual GPU beam-path check required.\',bitwise_cross_build_equivalence_established=False)\n\n\ndef validate_checkpoint_state(ck):\n    s,i=ck[\'state\'],ck[\'identity\'];expected=lock()\n    if s[\'epoch\']!=10 or s[\'batch_cursor\']!=0 or s[\'phase\']!=\'train\' or s[\'updates\']!=159399 or s[\'caption_events\']!=5100723 or s[\'best_epoch\']!=9 or len(s[\'history\'])!=9:raise ValueError(\'Wrong best9 snapshot cursor\')\n    if i[\'source\'][\'commit\']!=expected[\'training_commit\'] or i[\'method\']!=expected[\'method\'] or i[\'epochs\']!=10 or i[\'global_batch\']!=32:raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if not {\'noun_bank\',\'verb_bank\'}<=set(ck[\'model\']):raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\n\ndef prompt_inputs(cfg):\n    inventory=read(HERE/\'PROMPT_TEST_PREPARATION_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    # The old ROI feature array/vocabulary are not inputs to the new patch model.\n    wanted={n:inventory[n] for n in (\'receipt.json\',\'summary.json\',\'prompt_links.jsonl\',\'prompt_texts.json\',\'prompt_embeddings.npy\')}\n    root=checked_root(cfg[\'prompt_test_root\'],\'receipt.json\',wanted);texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if texts!=sorted(set(texts)) or len(texts)!=258 or vectors.shape!=(258,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,rtol=0,atol=2e-6):raise ValueError(\'Changed TEST181 text space\')\n    rows=[json.loads(line) for line in (root/\'prompt_links.jsonl\').read_text(encoding=\'utf-8\').splitlines()]\n    expected=lock()[\'expected_test_ids\']\n    if len(rows)!=5000 or sorted(r[\'coco_id\'] for r in rows)!=expected or any(r[\'group\']!=\'test\' for r in rows):raise ValueError(\'Wrong/duplicate prompt TEST IDs\')\n    return rows,texts,vectors,wanted\n\n\ndef prepare(cfg):\n    rows,texts,vectors,seals=prompt_inputs(cfg)\n    for row in rows:record_arrays(row,texts)\n    shared.prepare(cfg,locked=lock(),weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg,rows,texts,vectors,seals)\n\n\ndef stage_prompts(cfg,rows,texts,vectors,seals):\n    encoded=[record_arrays(row,texts) for row in rows]\n    stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);ids=np.load(stage/\'ids.npy\',allow_pickle=False);lookup={r[\'coco_id\']:j for j,r in enumerate(rows)}\n    order=[lookup[int(i)] for i in ids]\n    for key in ARRAYS:np.save(stage/(\'sem_\'+key+\'.npy\'),np.stack([encoded[j][key] for j in order]))\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    receipt=read(out/\'stage_receipt.json\');receipt[\'method\']=\'bank_prompt_patch_v1\';receipt[\'prompt181_files_sha256\']=seals\n    receipt[\'stage_files_sha256\'].update({p.name:sha(p) for p in stage.glob(\'sem_*.npy\')});save(out/\'stage_receipt.json\',receipt)\n    saved=read(out/\'test_lock.json\');saved[\'stage_receipt_sha256\']=sha(out/\'stage_receipt.json\');save(out/\'test_lock.json\',saved)\n    save(out/\'semantic_test_stage.json\',dict(images=5000,objects=sum(len(r[\'objects\']) for r in rows),relations=sum(len(r[\'relations\']) for r in rows),\n        text_vectors=258,geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\',TRAIN_VALID_features_read=False,old_ROI_features_used=False,old_vocabulary_used=False))\n\n\ndef load(cfg,verify=True):\n    result=shared.load(cfg,verify=verify,validator=validate_lock)\n    if result[2].get(\'method\')!=\'bank_prompt_patch_v1\':raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n', 'semantic_data.py': '"""Pure NumPy fixed-bank/prompt175 to patch-grid preparation; no models."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import load_stage,TRAIN_IMAGES,VALID_IMAGES\n\nHERE=Path(__file__).resolve().parent\nARRAYS=dict(object_text_id=((8,),np.int16),object_box=((8,4),np.float32),object_weights=((8,196),np.float32),object_valid=((8,),bool),\n    relation_text_id=((3,),np.int16),subject_weights=((3,196),np.float32),target_weights=((3,196),np.float32),relation_geometry=((3,8),np.float32),relation_valid=((3,),bool))\n\ndef box_weights(value):\n    b=np.asarray(value,np.float64)\n    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError(\'Invalid normalized xyxy box\')\n    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)\n    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)\n    if weights.sum()<=0:raise ValueError(\'Box has no patch support\')\n    return (weights/weights.sum()).astype(np.float32)\n\ndef record_arrays(row,texts):\n    expected={\'coco_id\',\'filename\',\'group\',\'roi_count\',\'source_mode\',\'objects\',\'relations\',\'counts\'}\n    if set(row)!=expected or len(row[\'objects\'])>8 or len(row[\'relations\'])>3:raise ValueError(\'Unexpected prompt fields/caps\')\n    out={k:np.zeros(shape,dtype=dtype) for k,(shape,dtype) in ARRAYS.items()};lookup={t:i for i,t in enumerate(texts)};objects={}\n    for slot,obj in enumerate(row[\'objects\']):\n        instance=obj[\'instance_id\']\n        if not instance or instance in objects or obj[\'text\'] not in lookup:raise ValueError(\'Invalid/duplicate object instance/text\')\n        if not 0<=obj[\'roi\']<row[\'roi_count\']:raise ValueError(\'Invalid original ROI provenance\')\n        # Pool the SAME matched YOLO rectangle used by legacy ROI features, not the ROI index.\n        w=box_weights(obj[\'roi_bbox\']);box_weights(obj[\'bbox\'])\n        objects[instance]=w;out[\'object_weights\'][slot]=w;out[\'object_box\'][slot]=obj[\'bbox\'];out[\'object_text_id\'][slot]=lookup[obj[\'text\']];out[\'object_valid\'][slot]=True\n    for slot,rel in enumerate(row[\'relations\']):\n        s,o=rel[\'subject_instance\'],rel[\'object_instance\']\n        if s==o or s not in objects or o not in objects or rel[\'text\'] not in lookup:raise ValueError(\'Missing/collapsed relation endpoints/text\')\n        g=np.asarray(rel[\'geometry\'],np.float32)\n        if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid relation geometry\')\n        out[\'subject_weights\'][slot]=objects[s];out[\'target_weights\'][slot]=objects[o];out[\'relation_text_id\'][slot]=lookup[rel[\'text\']];out[\'relation_geometry\'][slot]=g;out[\'relation_valid\'][slot]=True\n    return out\n\ndef checked_root(mount,receipt_name,expected):\n    found=[p.parent for p in Path(mount).rglob(receipt_name) if sha(p)==expected[receipt_name]]\n    if len(found)!=1:raise ValueError(\'Need one exact semantic cache root: \'+receipt_name)\n    root=found[0]\n    for n,h in expected.items():\n        if Path(n).name!=n or not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed semantic payload \'+n)\n    return root\n\ndef fixed_banks(mount):\n    package=read(HERE/\'SAVE_BANK_V130_PACKAGE.json\');root=checked_root(mount,\'cache_receipt.json\',package[\'files\']);bank=read(HERE/\'bank_words_corrected.json\');manifest=read(root/\'cache_manifest.json\')\n    if sha(HERE/\'bank_words_corrected.json\')!=\'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac\' or bank[\'source\']!=\'TRAIN only, train+restval\' or manifest[\'source_bank_sha256\']!=sha(HERE/\'bank_words_corrected.json\'):raise ValueError(\'Bank supervision/provenance changed\')\n    arrays={}\n    with np.load(root/\'semantic_banks.npz\',allow_pickle=False) as f:\n        for kind in (\'noun\',\'verb\'):\n            words=[x[\'word\'] for x in bank[kind]];v=f[kind+\'_vectors\']\n            if f[kind+\'_words\'].tolist()!=words or v.shape!=(len(words),512) or v.dtype!=np.float32 or not np.isfinite(v).all() or not np.allclose(np.linalg.norm(v,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Bad bank vector order/space\')\n            arrays[kind]=v.copy()\n    return arrays,package,manifest\n\ndef checked_inputs(cfg):\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'];root=checked_root(cfg[\'prompt_root\'],\'cache_receipt.json\',inv[\'cache_files\']);cohort=read(root/\'cohort.json\');texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False);prompt_manifest=read(root/\'cache_manifest.json\')\n    if len(texts)!=410 or texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Wrong frozen175 text vectors\')\n    banks,bank_package,bank_manifest=fixed_banks(cfg[\'bank_root\'])\n    if (prompt_manifest[\'model\'],prompt_manifest[\'revision\'])!=(bank_manifest[\'model\'],bank_manifest[\'revision\']):raise ValueError(\'Bank/prompt CLIP text spaces differ\')\n    if cohort[\'test\'] or len(cohort[\'train\'])!=TRAIN_IMAGES or len(cohort[\'val\'])!=VALID_IMAGES or len(set(cohort[\'train\']+cohort[\'val\']))!=TRAIN_IMAGES+VALID_IMAGES:raise ValueError(\'Prompt source scope differs\')\n    return inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package\n\n\ndef stage_semantics(cfg,deadline):\n    started=time.monotonic();stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);data,_,meta=load_stage(stage,verify=False);count=TRAIN_IMAGES+VALID_IMAGES\n    if len(data[\'ids\'])!=count or meta.get(\'semantic\'):raise ValueError(\'Expected new full patch stage\')\n    inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package=checked_inputs(cfg)\n    if set(cohort[\'train\'])!=set(map(int,data[\'ids\'][:TRAIN_IMAGES])) or set(cohort[\'val\'])!=set(map(int,data[\'ids\'][TRAIN_IMAGES:])):raise ValueError(\'Prompt/full patch split differs\')\n    arrays={k:np.lib.format.open_memmap(stage/(\'sem_\'+k+\'.npy\'),mode=\'w+\',dtype=dtype,shape=(count,)+shape) for k,(shape,dtype) in ARRAYS.items()}\n    for value in arrays.values():value[:]=0\n    index={int(i):j for j,i in enumerate(data[\'ids\'])};seen=set();totals={g:Counter() for g in (\'train\',\'val\')}\n    try:\n        with (root/\'prompt_links.jsonl\').open(encoding=\'utf-8\') as stream:\n            for line in stream:\n                if time.monotonic()>deadline:raise TimeoutError(\'Semantic stage cap; not complete training input\')\n                row=json.loads(line);i=row[\'coco_id\'];j=index.get(i)\n                if type(i)!=int or i in seen or j is None or row[\'group\']!=(\'train\' if j<TRAIN_IMAGES else \'val\'):raise ValueError(\'Foreign/repeated/leaked prompt image\')\n                encoded=record_arrays(row,texts)\n                for key,a in encoded.items():arrays[key][j]=a\n                seen.add(i);totals[row[\'group\']].update(images=1,objects=len(row[\'objects\']),relations=len(row[\'relations\']),with_object=bool(row[\'objects\']),with_relation=bool(row[\'relations\']))\n                if len(seen)%10000==0:print(\'Semantic patch inputs\',len(seen),\'/\',count,flush=True)\n        if seen!=set(index):raise ValueError(\'Missing semantic coverage\')\n        for value in arrays.values():value.flush()\n    finally:\n        for value in arrays.values():value._mmap.close()\n        for value in data.values():value._mmap.close()\n    for kind,v in banks.items():np.save(stage/(\'sem_\'+kind+\'_bank.npy\'),v)\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    names=[\'sem_\'+n+\'.npy\' for n in ARRAYS]+[\'sem_noun_bank.npy\',\'sem_verb_bank.npy\',\'sem_text_vectors.npy\']\n    seals={n:sha(stage/n) for n in names}\n    if time.monotonic()>deadline:raise TimeoutError(\'Semantic sealing cap\')\n    meta[\'files_sha256\'].update(seals);meta[\'semantic\']=dict(method=\'bank_prompt_patch_v1\',prompt_cache=inv[\'cache_files\'],bank_cache=bank_package[\'files\'],text_model=prompt_manifest[\'model\'],text_revision=prompt_manifest[\'revision\'],geometry=\'14x14 fractional area on matched ROI box; Resize224x224, no center crop\',counts={g:dict(c) for g,c in totals.items()},seconds=time.monotonic()-started,TRAIN_VALID_only=True,test_used=False,files_sha256=seals)\n    save(stage/\'manifest.json\',meta);save(out/\'stage_summary.json\',meta);save(out/\'semantic_stage.json\',meta[\'semantic\'])\n    print(\'Semantic stage COMPLETE; no CLIP/UPT/model execution.\',flush=True)\n\nclass SemanticContext:\n    def __init__(self,stage):\n        self.arrays={k:np.load(Path(stage)/(\'sem_\'+k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in ARRAYS}\n        self.text=np.load(Path(stage)/\'sem_text_vectors.npy\',mmap_mode=\'r\',allow_pickle=False)\n    def images(self,indices,device):\n        import torch\n        p={k:np.array(v[indices],copy=True) for k,v in self.arrays.items() if k not in (\'object_text_id\',\'relation_text_id\')}\n        for role in (\'object\',\'relation\'):\n            p[role+\'_text\']=self.text[self.arrays[role+\'_text_id\'][indices]]*p[role+\'_valid\'][...,None]\n        return {k:torch.from_numpy(np.ascontiguousarray(v)).to(device) for k,v in p.items()}\n    def events(self,data,events,device):return self.images(data[\'caption_image\'][events],device)\n', 'semantic_patch_model.py': '"""User-run bank/prompt patch memory; unchanged baseline causal decoder."""\nimport torch\nfrom torch import nn\nfrom patch_model import PatchCaptioner\nfrom bank_layers import Config,SemanticAggregation,Alignment\nfrom memory_fusion import fuse_memory\n\n\nclass BankMemory(nn.Module):\n    def __init__(self):\n        super().__init__();self.aggregate=SemanticAggregation(Config(vocab=10000,dropout=0))\n        self.align=Alignment();self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,bank):\n        prototypes,_=self.aggregate(bank)\n        selected,_=self.align(visual,prototypes,torch.ones(visual.shape[:2],dtype=torch.bool,device=visual.device))\n        return visual+self.attention(visual,selected,selected,need_weights=False)[0]\n\n\nclass PromptMemory(nn.Module):\n    def __init__(self,input_dim):\n        super().__init__();self.slot=nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,512))\n        self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,slot_inputs,support,valid):\n        slots=self.slot(slot_inputs)\n        allowed=support.transpose(1,2).gt(0)&valid[:,None,:]\n        active=allowed.any(-1)\n        # Give inactive queries one finite key, then zero their entire residual.\n        safe=allowed.clone();safe[:,:,0]|=~active\n        mask=(~safe)[:,None].expand(-1,8,-1,-1).reshape(-1,visual.shape[1],slots.shape[1])\n        delta=self.attention(visual,slots,slots,attn_mask=mask,need_weights=False)[0]\n        return visual+delta.masked_fill(~active[...,None],0)\n\n\nclass SemanticPatchCaptioner(nn.Module):\n    def __init__(self,vocab,nouns,verbs,strength_gate=False):\n        super().__init__()\n        # Construct the baseline FIRST; never reinitialize its modules recursively.\n        self.base=PatchCaptioner(vocab)\n        self.register_buffer(\'noun_bank\',nouns);self.register_buffer(\'verb_bank\',verbs)\n        self.object_bank=BankMemory();self.relation_bank=BankMemory()\n        self.object_prompt=PromptMemory(512+512+4)\n        self.relation_prompt=PromptMemory(512+512+512+8)\n        self.gate=nn.Linear(1024,512);nn.init.zeros_(self.gate.weight);nn.init.zeros_(self.gate.bias)\n        self.strength_gate=nn.Linear(1024,1) if strength_gate else None\n        if self.strength_gate is not None:\n            nn.init.zeros_(self.strength_gate.weight);nn.init.zeros_(self.strength_gate.bias)\n    @property\n    def project(self):return self.base.project\n    @property\n    def head(self):return self.base.head\n    def memories(self,patches,p,visual=None):\n        if visual is None:visual=self.base.encode(patches)\n        mo=self.object_bank(visual,self.noun_bank);mr=self.relation_bank(mo,self.verb_bank)\n        obj=torch.cat((p[\'object_text\'],torch.bmm(p[\'object_weights\'],mo),p[\'object_box\']),-1)\n        rel=torch.cat((p[\'relation_text\'],torch.bmm(p[\'subject_weights\'],mo),torch.bmm(p[\'target_weights\'],mo),p[\'relation_geometry\']),-1)\n        vo=self.object_prompt(mo,obj,p[\'object_weights\'],p[\'object_valid\'])\n        vr=self.relation_prompt(mr,rel,p[\'subject_weights\']+p[\'target_weights\'],p[\'relation_valid\'])\n        return vo,vr\n    def encode(self,patches,p):\n        if self.strength_gate is not None:return self.encode_with_gate(patches,p)[0]\n        vo,vr=self.memories(patches,p);gate=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        return gate*vo+(1-gate)*vr\n    def encode_with_gate(self,patches,p):\n        if self.strength_gate is None:raise ValueError(\'Strength-gated model required\')\n        visual=self.base.encode(patches);vo,vr=self.memories(patches,p,visual)\n        branch=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        delta=branch*(vo-visual)+(1-branch)*(vr-visual)\n        strength=self.strength_gate(torch.cat((visual,delta),-1)).sigmoid()\n        return fuse_memory(visual,vo,vr,branch,strength),strength\n    def decode(self,memory,prefix):return self.base.decode(memory,prefix)\n    def forward(self,patches,prefix,p):return self.decode(self.encode(patches,p),prefix)\n', 'test_semantic_test.py': '"""Author-safe fixed bank/prompt TEST checks; no Torch/model/scorer execution."""\nimport ast\nimport copy\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport semantic_test_data as d\nfrom semantic_data import SemanticContext\nimport run_test199 as runner\n\n\ndef row(coco):\n    return dict(coco_id=coco,filename=f\'{coco}.jpg\',group=\'test\',roi_count=9,source_mode=\'object_only\',counts={},relations=[],\n        objects=[dict(instance_id=\'person:4\',roi=7,text=\'a person\',bbox=[0,0,.5,1],roi_bbox=[0,0,.5,1])])\n\n\nclass Checks(unittest.TestCase):\n    def test_fixed_checkpoint_epoch9_and_decode(self):\n        lock=d.lock();self.assertEqual(lock[\'updates\'],159399);self.assertEqual(len(lock[\'expected_test_ids\']),5000)\n        self.assertEqual(lock[\'expected_test_ids\'],d.shared.lock()[\'expected_test_ids\'])\n        for k,v in ((\'epoch\',10),(\'updates\',177110),(\'beam_width\',4),(\'reranking\',True),(\'checkpoint_sha256\',\'0\'*64),(\'expected_test_ids\',[1])):\n            bad=copy.deepcopy(lock);bad[k]=v\n            with self.assertRaises(ValueError):d.validate_lock(bad)\n\n    def test_best_snapshot_not_final_epoch10(self):\n        ck=dict(state=dict(epoch=10,batch_cursor=0,phase=\'train\',updates=159399,caption_events=5100723,best_epoch=9,history=[{}]*9),\n            identity=dict(source=dict(commit=d.lock()[\'training_commit\']),method=\'bank_prompt_patch_v1\',epochs=10,global_batch=32),model=dict(noun_bank=None,verb_bank=None))\n        d.validate_checkpoint_state(ck)\n        bad=copy.deepcopy(ck);bad[\'state\'].update(epoch=11,updates=177110)\n        with self.assertRaises(ValueError):d.validate_checkpoint_state(bad)\n        bad=copy.deepcopy(ck);bad[\'identity\'][\'method\']=\'plain_patch_v1\'\n        with self.assertRaises(ValueError):d.validate_checkpoint_state(bad)\n\n    def test_exact_weights_lookup_and_runtime(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/\'best.bin\').write_bytes(b\'best9\');(root/\'checkpoint.bin\').write_bytes(b\'final10\')\n            with patch.object(d,\'CHECKPOINT_SHA\',d.sha(root/\'best.bin\')):\n                self.assertEqual(d.checkpoint(root),root/\'best.bin\')\n                with self.assertRaises(ValueError):d.checkpoint(root/\'checkpoint.bin\')\n        for v in (\'2.11.0+cpu\',\'2.12.0+cu130\'):\n            with self.assertRaises(RuntimeError):d.runtime_policy(\'2.11.0+cu130\',v)\n        self.assertFalse(d.runtime_policy(\'2.11.0+cu130\',\'2.11.0+cu130\')[\'cuda_wheel_changed\'])\n\n    def test_stage_geometry_joins_logical_TEST_ids_and_seals(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);stage=root/\'stage\';stage.mkdir();out=root/\'out\';out.mkdir();np.save(stage/\'ids.npy\',np.array([20,10]))\n            d.save(out/\'stage_receipt.json\',dict(stage_files_sha256={}));d.save(out/\'test_lock.json\',{})\n            a=row(10);b=row(20);b[\'objects\'][0].update(roi_bbox=[.5,0,1,1])\n            vectors=np.zeros((1,512),np.float32);vectors[0,0]=1\n            d.stage_prompts(dict(stage=str(stage),output=str(out)),[a,b],[\'a person\'],vectors,{\'fixture\':\'hash\'})\n            context=SemanticContext(stage)\n            left=context.arrays[\'object_weights\'][0,0].reshape(14,14);self.assertEqual(float(left[:,:7].sum()),0)\n            self.assertAlmostEqual(float(left[:,7:].sum()),1,places=6)\n            np.testing.assert_array_equal(context.text,vectors)\n            receipt=d.read(out/\'stage_receipt.json\');self.assertEqual(receipt[\'method\'],\'bank_prompt_patch_v1\')\n            for n,h in receipt[\'stage_files_sha256\'].items():self.assertEqual(d.sha(stage/n),h)\n            self.assertEqual(d.read(out/\'test_lock.json\')[\'stage_receipt_sha256\'],d.sha(out/\'stage_receipt.json\'))\n            for v in (*context.arrays.values(),context.text):v._mmap.close()\n\n    def test_prompt181_payloads_and_ID_scope_without_ROI_or_old_vocab(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);ids=d.lock()[\'expected_test_ids\'];rows=[row(i) for i in ids]\n            d.save(root/\'receipt.json\',{});d.save(root/\'summary.json\',{})\n            texts=[\'a person\']+[f\'z{i:03}\' for i in range(257)];d.save(root/\'prompt_texts.json\',texts)\n            vectors=np.zeros((258,512),np.float32);vectors[:,0]=1;np.save(root/\'prompt_embeddings.npy\',vectors)\n            def write_rows():\n                (root/\'prompt_links.jsonl\').write_text(\'\'.join(__import__(\'json\').dumps(r)+\'\\n\' for r in rows),encoding=\'utf-8\')\n            write_rows();inventory=dict(cache_files={p.name:d.sha(p) for p in root.iterdir()});original=d.read\n            def reader(path):return dict(transport_inventory=inventory) if Path(path).name==\'PROMPT_TEST_PREPARATION_RESULT.json\' else original(path)\n            with patch.object(d,\'read\',side_effect=reader):\n                got,_,_,_=d.prompt_inputs(dict(prompt_test_root=str(root)));self.assertEqual(len(got),5000)\n                rows[0][\'coco_id\']=rows[1][\'coco_id\'];write_rows();inventory[\'cache_files\'][\'prompt_links.jsonl\']=d.sha(root/\'prompt_links.jsonl\')\n                with self.assertRaises(ValueError):d.prompt_inputs(dict(prompt_test_root=str(root)))\n            self.assertFalse((root/\'test_regions.npz\').exists());self.assertFalse((root/\'vocabulary.json\').exists())\n\n    def test_shared_scorer_complete_requires_all8_for_new_method(self):\n        with tempfile.TemporaryDirectory() as td:\n            out=Path(td);d.save(out/\'scoring_lock.json\',dict(files_sha256={}));d.save(out/\'chair_counts.json\',{});d.save(out/\'environment.json\',{})\n            parts={\'basic\':dict(BLEU_1=.7,BLEU_4=.3,CIDEr=1.2,ROUGE_L=.5),\'meteor\':dict(METEOR=.2),\'spice\':dict(SPICE=.2),\'chair\':dict(CHAIRs=.04,CHAIRi=.03,Recall=.4)}\n            for mode,m in parts.items():d.save(out/(mode+\'.json\'),dict(images=5000,metrics=m,input_sha256={}))\n            with patch.object(runner,\'inputs\',return_value=(out,[],[],[dict(NLL_sum=6,NLL_tokens=3,runtime_policy=d.runtime_policy(\'2.11.0+cu130\',\'2.11.0+cu130\'))])),patch(\'builtins.print\'):\n                runner.combine(dict(output=str(out),semantic=True));self.assertEqual(d.read(out/\'result.json\')[\'epoch\'],9)\n                self.assertEqual(d.read(out/\'result.json\')[\'checkpoint_sha256\'],d.CHECKPOINT_SHA)\n                d.save(out/\'spice.json\',dict(images=5000,metrics={},input_sha256={}))\n                with self.assertRaises(ValueError):runner.combine(dict(output=str(out),semantic=True))\n\n    def test_no_training_or_neural_import_source(self):\n        here=Path(__file__).parent\n        for n in (\'semantic_test_data.py\',\'run_test199.py\',\'test199_data.py\',\'make_test199_notebook.py\'):ast.parse((here/n).read_text(encoding=\'utf-8\'))\n        source=(here/\'run_test199.py\').read_text(encoding=\'utf-8\')\n        for bad in (\'backward(\', \'optimizer.step\', \'torch.optim\', \'model.train(\'):self.assertNotIn(bad,source)\n        self.assertNotIn(\'torch\',sys.modules);self.assertNotIn(\'pycocoevalcap\',sys.modules)\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'BANK_PROMPT_TEST_PROTOCOL.md': '# Bank + prompt — fixed epoch9 TEST5000, 10/11/2026\n\nUser approved TEST evaluation after reviewing VALID. Freeze best.bin epoch9 SHA7c11be52fce9cd90dadc55fa514d4017d540f40f630232a68bfe4306c754158b, training sourcef5f6eb0, SaveVersion9 at notebook kltnnnnnff (user confirmed slug/version, owner defaults to own ducanh2403; editable source route, exact weight SHA mandatory). Best snapshot has159399updates/9completedepochs even though run finished10epochs; do not demand177110updates or load finalepoch10.\n\nSame5000KarpathyTEST IDs/raw references/TRAIN vocabulary/CLIP-B16 patch bytes as baseline199, beam5/max20/rawlogprob/rank0/no lengthnormalization. One fixed learned model, no zero/shuffle branches or reranking/grid/selection/training. Three datasets: visual-cache, mscoco-2014 and csa-prompt-test-cache-v181 user reconfirmed. TRAIN bank vectors are checkpoint buffers; no bank130/prompt175 datasets needed at inference. Decode only uses image patches and frozen TEST181 object/relation phrases + geometric links. Remap stored matched ROI boxes to fractional14x14 patch pools/support with exact training record_arrays; resolve endpoints by instance IDs. Do not use old ROI feature arrays or old181 vocabulary, which differs from new baseline vocabulary. Verify needed181 payloads against historical seals and258x512 normalized vectors; all5000IDs exactlyonce/groupTEST. One unseen phrase is encoded by the same frozen text encoder in181, no caption-reference prompt creation or new encoder execution.\n\nReuse proven TEST199 patch staging/CPU scorer phases; allow explicit lock/validator injection while keeping baseline defaults/tests. New model constructed with noun/verb bank buffers from exact checkpoint and strict fullstate load. Verify saved state/method/source/core initialization/semantic source metadata and CUDA2.11.0observedcu128/cu130; training wascu130. Beam-path check on actual GPU firstbatch. Captions persisted before teacherNLL and CPU BLEU1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall). Onlycomplete if allphases/5000IDs/seals pass. TESTreferences onlyforNLL/scoring, nevercaptionselection. Previous project TEST exposure disclosed; no tuning on this run.\n\nFresh notebook default parent2h stopcap as baselineTEST, not ETA/billing. Current remaining GPU hours not supplied in latest answer; no increase in previous TEST cap or training authorized. Source checkpoint/API preflight and TESTprompt cache checks precede expensive stage. T4x2, exactown APIsecret unless user mounts knownSHA weights. Never use chat-exposed OpenAI token; Kaggle Secret is separate. No localTorch/models/autograd/scorers/Kaggle runs; NumPy/mock/AST/source closure author checks only. Preserve captions/checkpoints/log on failure, no automatic retries/redecode. Keep source version9 best+latest; no Word edits.\n', 'bank_layers.py': '"""CSA equations/conventions from IMPLEMENTATION_PROTOCOL; self-written, no C0."""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass Config:\n    vocab: int\n    dim: int = 512\n    heads: int = 8\n    hidden: int = 2048\n    prototypes: int = 150\n    layers: int = 3\n    dropout: float = .2\n    raw_dim: int = 768\n\n\ndef zero_pad(x, valid):\n    return x.masked_fill(~valid[..., None], 0)\n\n\nclass FFN(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(c.dim, c.hidden), nn.ReLU(), nn.Dropout(c.dropout),\n                                 nn.Linear(c.hidden, c.dim), nn.Dropout(c.dropout))\n        self.norm = nn.LayerNorm(c.dim, eps=1e-5)\n\n    def forward(self, x):\n        return self.norm(x + self.net(x))\n\n\nclass SemanticAggregation(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.wa = nn.Linear(c.dim, c.prototypes, bias=False)\n        self.mlp = nn.Sequential(nn.Linear(c.prototypes, c.prototypes), nn.ReLU(),\n                                 nn.Linear(c.prototypes, c.prototypes))\n        self.wb = nn.Linear(c.dim, c.dim, bias=False)\n\n    def forward(self, bank):\n        weights = self.mlp(self.wa(bank)).float().softmax(dim=0)\n        return weights.T.to(bank.dtype) @ self.wb(bank), weights\n\n\nclass Alignment(nn.Module):\n    def forward(self, visual, prototypes, valid):\n        if prototypes.ndim == 2:\n            prototypes = prototypes.unsqueeze(0).expand(visual.shape[0], -1, -1)\n        cosine = torch.bmm(F.normalize(prototypes.float(), dim=-1, eps=1e-6),\n                           F.normalize(visual.float(), dim=-1, eps=1e-6).transpose(1, 2))\n        weights = (cosine / .1).masked_fill(~valid[:, None, :], -torch.inf).softmax(dim=-1)\n        relevance = (weights * cosine).sum(dim=-1)\n        selected = relevance.softmax(dim=-1)[..., None] * prototypes\n        return selected.to(visual.dtype), weights\n\n\nclass Fusion(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.align = Alignment()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.cross_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm_s, self.norm_c = nn.LayerNorm(c.dim), nn.LayerNorm(c.dim)\n        self.ff_s, self.ff_c = FFN(c), FFN(c)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate = nn.Linear(2 * c.dim, 1, bias=False)\n\n    def forward(self, visual, prototypes, valid):\n        selected, _ = self.align(visual, prototypes, valid)\n        context = self.self_attn(visual, visual, visual, key_padding_mask=~valid, need_weights=False)[0]\n        own = zero_pad(self.norm_s(visual + self.drop(context)), valid)\n        own = zero_pad(self.ff_s(own), valid)\n        context = self.cross_attn(own, selected, selected, need_weights=False)[0]\n        cross = zero_pad(self.norm_c(own + self.drop(context)), valid)\n        cross = zero_pad(self.ff_c(cross), valid)\n        gate = self.gate(torch.cat((own, cross), dim=-1)).sigmoid()\n        return zero_pad(gate * own + (1 - gate) * cross, valid), selected\n\n\nclass DecoderLayer(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.object_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.relation_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm = nn.LayerNorm(c.dim)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate_o, self.gate_r = nn.Linear(2 * c.dim, c.dim), nn.Linear(2 * c.dim, c.dim)\n        self.ffn = FFN(c)\n\n    def forward(self, x, previous_word, vo, vr, text_valid, region_valid):\n        causal = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).triu(1)\n        h = self.self_attn(x, x, x, attn_mask=causal, key_padding_mask=~text_valid, need_weights=False)[0]\n        h = zero_pad(self.norm(x + self.drop(h)), text_valid)\n        co = self.object_attn(h, vo, vo, key_padding_mask=~region_valid, need_weights=False)[0]\n        cr = self.relation_attn(h, vr, vr, key_padding_mask=~region_valid, need_weights=False)[0]\n        go = self.gate_o(torch.cat((co, previous_word), dim=-1)).sigmoid()\n        gr = self.gate_r(torch.cat((cr, previous_word), dim=-1)).sigmoid()\n        return zero_pad(self.ffn(go * co + gr * cr), text_valid)\n\n\nclass CSACaptioner(nn.Module):\n    def __init__(self, c, nouns, verbs):\n        super().__init__()\n        if c.dim % 2 or c.dim % c.heads or c.vocab < 4 or c.layers < 1 or c.prototypes < 1:\n            raise ValueError(\'Invalid model dimensions\')\n        for bank in (nouns, verbs):\n            if bank.ndim != 2 or bank.shape[1] != c.dim or not len(bank) or not torch.isfinite(bank).all():\n                raise ValueError(\'Empty, nonfinite or incompatible semantic bank\')\n        self.c = c\n        self.register_buffer(\'nouns\', nouns.detach().float().clone())\n        self.register_buffer(\'verbs\', verbs.detach().float().clone())\n        self.project = nn.Linear(c.raw_dim, c.dim)\n        self.noun_agg, self.verb_agg = SemanticAggregation(c), SemanticAggregation(c)\n        self.object1, self.object2, self.relation = Fusion(c), Fusion(c), Fusion(c)\n        self.embedding = nn.Embedding(c.vocab, c.dim, padding_idx=0)\n        self.decoder = nn.ModuleList([DecoderLayer(c) for _ in range(c.layers)])\n        self.head = nn.Linear(c.dim, c.vocab)\n        self.apply(self._init)\n        for module in self.modules():\n            if isinstance(module, nn.MultiheadAttention):\n                nn.init.xavier_uniform_(module.in_proj_weight)\n                nn.init.zeros_(module.in_proj_bias)\n        nn.init.normal_(self.embedding.weight, std=c.dim ** -.5)\n        with torch.no_grad():\n            self.embedding.weight[0].zero_()\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, nn.Linear):\n            nn.init.xavier_uniform_(module.weight)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.LayerNorm):\n            nn.init.ones_(module.weight)\n            nn.init.zeros_(module.bias)\n\n    def encode(self, regions, valid):\n        if valid.dtype != torch.bool or regions.ndim != 3 or valid.shape != regions.shape[:2] or not valid.any(dim=1).all():\n            raise ValueError(\'Every image requires at least one valid region\')\n        if not torch.isfinite(regions).all():\n            raise ValueError(\'Nonfinite visual feature\')\n        v = zero_pad(self.project(regions), valid)\n        nouns, _ = self.noun_agg(self.nouns)\n        verbs, _ = self.verb_agg(self.verbs)\n        v, selected = self.object1(v, nouns, valid)\n        vo, _ = self.object2(v, selected, valid)\n        vr, _ = self.relation(vo, verbs, valid)\n        return vo, vr\n\n    def decode(self, ids, vo, vr, valid):\n        if ids.ndim != 2 or ids.shape[1] == 0 or not (ids[:, 0] == 1).all():\n            raise ValueError(\'Prefix must start with BOS, with nonempty time dimension\')\n        raw = self.embedding(ids)\n        position = torch.arange(ids.shape[1], device=ids.device, dtype=torch.float32)[:, None]\n        frequency = torch.exp(torch.arange(0, self.c.dim, 2, device=ids.device).float() * (-math.log(10000) / self.c.dim))\n        pe = torch.empty(ids.shape[1], self.c.dim, device=ids.device)\n        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)\n        text_valid = ids != 0\n        x = zero_pad(raw * math.sqrt(self.c.dim) + pe.to(raw.dtype), text_valid)\n        for layer in self.decoder:\n            x = layer(x, raw, vo, vr, text_valid, valid)\n        return self.head(x)\n\n    def forward(self, regions, valid, ids):\n        return self.decode(ids, *self.encode(regions, valid), valid)\n\n\ndef xe_loss(logits, target):\n    mask = target != 0\n    if not mask.any():\n        raise ValueError(\'XE batch has no target tokens\')\n    nll = F.cross_entropy(logits.flatten(0, 1), target.flatten(), reduction=\'none\', ignore_index=0).view_as(target)\n    return (nll * mask).sum() / mask.sum()\n\n\n@torch.no_grad()\ndef beam_search(model, vo, vr, valid, beams=5, maximum=20):\n    """Per-image beams; no hidden length normalization, BOS/PAD cannot be emitted."""\n    if model.training or maximum < 1 or not 1 <= beams <= model.c.vocab - 2:\n        raise ValueError(\'Beam search requires eval mode and valid width/length\')\n    results = []\n    for i in range(len(vo)):\n        seq = torch.ones(1, 1, dtype=torch.long, device=vo.device)\n        scores = torch.zeros(1, device=vo.device)\n        finished = torch.zeros(1, device=vo.device, dtype=torch.bool)\n        for _ in range(maximum):\n            count = len(seq)\n            logits = model.decode(seq, vo[i:i+1].expand(count, -1, -1),\n                                  vr[i:i+1].expand(count, -1, -1), valid[i:i+1].expand(count, -1))[:, -1]\n            logp = logits.float().log_softmax(-1)\n            logp[:, :2] = -torch.inf\n            logp[finished] = -torch.inf\n            logp[finished, 0] = 0\n            best, flat = (scores[:, None] + logp).flatten().topk(beams)\n            parent, token = flat // model.c.vocab, flat % model.c.vocab\n            seq = torch.cat((seq[parent], token[:, None]), dim=1)\n            finished = finished[parent] | (token == 2)\n            scores = best\n            if finished.all():\n                break\n        results.append((seq, scores))\n    return results\n\n\ndef beam_mean_loss(model, regions, valid, sequences, rewards, maximum=20):\n    """Gradient rescore; rewards [B,K] are externally computed CIDEr-D and detached."""\n    b, k, length = sequences.shape\n    if rewards.shape != (b, k) or length < 2 or length > maximum + 1:\n        raise ValueError(\'Invalid beam/reward shape\')\n    flat = sequences.reshape(b * k, length)\n    logits = model(regions.repeat_interleave(k, 0), valid.repeat_interleave(k, 0), flat[:, :-1])\n    target = flat[:, 1:]\n    # Include first EOS, exclude every later position (even malformed non-PAD tokens).\n    eos_before = (target == 2).cumsum(dim=1) - (target == 2).long()\n    mask = (target != 0) & (eos_before == 0)\n    logp = logits.float().log_softmax(-1).gather(-1, target[..., None]).squeeze(-1)\n    score = (logp * mask).sum(-1).reshape(b, k) / maximum\n    advantage = rewards.detach() - rewards.detach().mean(dim=1, keepdim=True)\n    return -(advantage * score).mean()\n', 'PROMPT_TEST_PREPARATION_RESULT.json': '{\n  "status": "TEST_PREPARATION_METADATA_REVIEW_PASS",\n  "checks": 48,\n  "summary": {\n    "status": "PROMPT_TEST_PREPARATION_COMPLETE",\n    "test_images": 5000,\n    "caption_generation": false,\n    "caption_training": false,\n    "caption_scoring": false,\n    "with_objects": 4865,\n    "with_relations": 1136,\n    "objects": 14593,\n    "relations": 2228,\n    "texts": 258,\n    "reused_texts": 257,\n    "new_texts": [\n      "a person hugging an elephant"\n    ],\n    "source175_files": {\n      "cohort.json": "d97eaa8cecd07a4de825e3be99926f751f610e103ff9582c2b482e53d2c9980d",\n      "prompt_texts.json": "35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2",\n      "prompt_links.jsonl": "e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0",\n      "summary.json": "28d650c5403fcf851e9a8f5883824ceba2d46144a9d2222be36cb056e07b5122",\n      "review_examples.json": "17b4ed35145338ac3fe65b28b293e4a11d8ac39fba1199ff3a46b29219329df7",\n      "prompt_embeddings.npy": "f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c",\n      "mapping_receipt.json": "134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d",\n      "reuse_report.json": "de76c927e7839bfb553ca09ea3373474ba2afeae6b12b7f1e9959d8ed41358ef",\n      "cache_manifest.json": "05f40a246b19cb37717a4aadb7bcb20f2e0dd0ebabe60e983d83b9f328e9f56a",\n      "cache_receipt.json": "72c532cd73b832c5cdf8f0ff747a882413e006fa84670f61a6a54abd272613cb",\n      "run_status.json": "fc1f7837f14785888c4ff30c193ca71b72ce34931f93c918cc23a9945089d8e9",\n      "source_receipt.json": "7df1b07aa9e8ef821e4608f406c355142d3435dbd7aa5bbf805daf4e6afd5365",\n      "notebook.log": "5c656006b117092d6cc19ef472d18e7fb264a6f1a00876be16f694af87facd74",\n      "cache_plan.json": "4f1fd4c729fe72db06de781ad2a99af8755eef225cae23ecd989778438aeb057"\n    },\n    "new_encoding": {\n      "spec": {\n        "id": "openai/clip-vit-base-patch16",\n        "revision": "57c216476eefef5ab752ec549e440a49ae4ae5f3",\n        "files_sha256": {\n          "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n          "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n          "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n          "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n          "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n          "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n          "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n          "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n        }\n      },\n      "files_sha256": {\n        "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n        "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n        "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n        "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n        "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n        "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n        "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n        "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n      },\n      "loading_info": {\n        "missing_keys": [],\n        "unexpected_keys": [],\n        "mismatched_keys": [],\n        "error_msgs": []\n      },\n      "max_tokens": 7,\n      "seconds": 14.177083713999991\n    }\n  },\n  "transport_inventory": {\n    "cache_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/181",\n    "cache_files": {\n      "test_regions.npz": "7fb385e1a03c5241c4d65cc240d0434e041284a9a829198c5167f0725be57b1a",\n      "test_metadata.json": "4271b46b7ab67117433bfdf84f6eb9ba66515ada9b3c4ae4d2e507bc2cede018",\n      "vocabulary.json": "924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7",\n      "test_receipt.json": "7a212068c7e73f6f0791906c1ec81d56e3178fd62317bdf0a5e528350d95055d",\n      "prompt_links.jsonl": "54e25cd17c9842c07fb4ef0b3da4ed4ad276571d3711d51bf0b3ac1a86cc3c55",\n      "prompt_texts.json": "afdad5d42734adfbd38883f1fa7a22aaba1b76ee8cd7510635fd3e8b22d44553",\n      "prompt_embeddings.npy": "7119776b70380376a343dd9130bed6d2e5fc19594d83cf4bdd724a7343a4fb35",\n      "summary.json": "3c9b863b73d9efdeda7f5979d25c829c74d805505a856c965141dc275a0e8847",\n      "source_coverage.json": "b5f66a8bcb40027dcdd11b0d71c92551d68a11f91a6daf56e1157cdd0b5a7a3c",\n      "receipt.json": "f86708aa99a3a07087a55164cd3fa099d01e01eef15a91da38540dc28a5c3742"\n    },\n    "source_review_zip_sha256": "44f752a96e5fd228f704f45a0cc7de0160b97a4636ad958ec4c59a2e77b1de8a"\n  },\n  "limitations": [\n    "NPZ/metadata/vector binaries absent review ZIP; their payload hashes are user-run evidence.",\n    "Slot packing used dummy vectors; real258x512 vector validation occurs during transport.",\n    "No author models/scorers/API; no new TEST captions or accuracy scores."\n  ]\n}\n', 'csa_patch_model.py': '"""Paper-structure CSA encoder/parallel decoder; CLIP-patch + UPT extensions disclosed."""\nimport torch\nfrom torch import nn\nfrom bank_layers import CSACaptioner, Config\nfrom semantic_patch_model import PromptMemory\n\n\nclass CSAPatchCaptioner(nn.Module):\n    def __init__(self, vocab, nouns, verbs):\n        super().__init__()\n        self.core = CSACaptioner(Config(vocab=vocab), nouns, verbs)\n        self.object_prompt = PromptMemory(512 + 512 + 4)\n        self.relation_prompt = PromptMemory(512 + 512 + 512 + 8)\n\n    @property\n    def head(self):\n        return self.core.head\n\n    def streams(self, patches, p):\n        if patches.ndim != 3 or patches.shape[1:] != (196, 768):\n            raise ValueError(\'Expected196CLIPpatches, not ROI indices/features\')\n        valid = torch.ones(patches.shape[:2], dtype=torch.bool, device=patches.device)\n        vo, vr = self.core.encode(patches, valid)\n        obj = torch.cat((p[\'object_text\'], torch.bmm(p[\'object_weights\'], vo), p[\'object_box\']), -1)\n        rel = torch.cat((p[\'relation_text\'], torch.bmm(p[\'subject_weights\'], vo),\n                         torch.bmm(p[\'target_weights\'], vo), p[\'relation_geometry\']), -1)\n        return (self.object_prompt(vo, obj, p[\'object_weights\'], p[\'object_valid\']),\n                self.relation_prompt(vr, rel, p[\'subject_weights\'] + p[\'target_weights\'], p[\'relation_valid\']))\n\n    def encode(self, patches, p):\n        # Packaging for existing beam expansion only; decode restores both streams.\n        return torch.cat(self.streams(patches, p), dim=1)\n\n    def decode(self, memory, prefix):\n        if memory.ndim != 3 or memory.shape[1:] != (392, 512):\n            raise ValueError(\'Expected packaged pair of196x512memories\')\n        vo, vr = memory.split(196, dim=1)\n        valid = torch.ones(vo.shape[:2], dtype=torch.bool, device=vo.device)\n        return self.core.decode(prefix, vo, vr, valid)\n\n    def forward(self, patches, prefix, p):\n        return self.decode(self.encode(patches, p), prefix)\n', 'memory_fusion.py': '"""Shared neural/NumPy fusion algebra and saved gate diagnostics, no Torch import."""\nimport math\nimport numpy as np\n\n\ndef fuse_memory(visual,objects,relations,branch,strength):\n    delta=branch*(objects-visual)+(1-branch)*(relations-visual)\n    return visual+strength*delta\n\n\ndef update_gate_stats(state,values):\n    a=np.asarray(values,dtype=np.float64).reshape(-1)\n    if not len(a) or not np.isfinite(a).all() or (a<0).any() or (a>1).any():raise ValueError(\'Invalid semantic strength\')\n    state=state or dict(count=0,total=0.,minimum=1.,maximum=0.,below_point1=0,above_point9=0)\n    return dict(count=state[\'count\']+len(a),total=state[\'total\']+float(a.sum()),minimum=min(state[\'minimum\'],float(a.min())),maximum=max(state[\'maximum\'],float(a.max())),\n        below_point1=state[\'below_point1\']+int((a<.1).sum()),above_point9=state[\'above_point9\']+int((a>.9).sum()))\n\n\ndef merge_gate_stats(states):\n    if not states or any(s[\'count\']<=0 or not math.isfinite(s[\'total\']) for s in states):raise ValueError(\'Missing semantic gate diagnostics\')\n    count=sum(s[\'count\'] for s in states)\n    return dict(patches=count,mean=sum(s[\'total\'] for s in states)/count,minimum=min(s[\'minimum\'] for s in states),maximum=max(s[\'maximum\'] for s in states),\n        fraction_below_point1=sum(s[\'below_point1\'] for s in states)/count,fraction_above_point9=sum(s[\'above_point9\'] for s in states)/count,\n        interpretation=\'Learned residual strength, not a calibrated confidence or grounding accuracy.\')\n', 'csa_per_caption_test_data.py': '"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read, save, sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS, record_arrays, checked_root\nHERE = Path(__file__).resolve().parent\nCHECKPOINT_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nVERSION = \'ducanh3/notebook7bc83c4c76/179\'\n\ndef validate_lock(value):\n    baseline = valid_cohort()\n    if any((value[k] != baseline[k] for k in (\'expected_test_ids\', \'test_id_sha256\', \'beam_width\', \'max_new_tokens\', \'length_normalization\', \'reranking\'))):\n        raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\'] != VERSION or value[\'checkpoint_sha256\'] != CHECKPOINT_SHA or value[\'epoch\'] != 8 or (value[\'updates\'] != 141688) or (value[\'method\'] != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'):\n        raise ValueError(\'Need fixed seed43 best epoch8\')\n    return value\n\ndef valid_cohort():\n    import hashlib\n    value = shared.lock()\n    value.update(expected_test_ids=VALID_IDS, test_id_sha256=hashlib.sha256(json.dumps(VALID_IDS, separators=(\',\', \':\')).encode()).hexdigest())\n    return value\n\ndef lock():\n    value = valid_cohort()\n    value.update(schema=\'csa_per_caption_seed_test_v1\', source_version=VERSION, checkpoint_sha256=CHECKPOINT_SHA, training_commit=\'5eb012279e6fd0b5e47a56710ed5e3efa7449d2f\', epoch=8, updates=141688, method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\')\n    return validate_lock(value)\n\ndef checkpoint(root):\n    root = Path(root)\n    files = [root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches = [p for p in files if p.is_file() and sha(p) == CHECKPOINT_SHA]\n    if not matches:\n        raise ValueError(\'Missing exact seed43 best.bin SHAa51e7ec2\')\n    return matches[0]\n\ndef runtime_policy(training, current):\n    allowed = (\'2.11.0+cu128\', \'2.11.0+cu130\')\n    if training not in allowed or current not in allowed:\n        raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training, inference_torch=current, cuda_wheel_changed=training != current, policy=\'Recorded training and inference Torch builds; FP32 and strict checkpoint checks.\', bitwise_cross_build_equivalence_established=False)\n\ndef validate_checkpoint_state(ck):\n    s, i = (ck[\'state\'], ck[\'identity\'])\n    expected = lock()\n    if s[\'epoch\'] != 9 or s[\'batch_cursor\'] != 0 or s[\'phase\'] != \'train\' or (s[\'updates\'] != 141688) or (s[\'caption_events\'] != 4533976) or (s[\'best_epoch\'] != 8) or (len(s[\'history\']) != 8):\n        raise ValueError(\'Wrong seed43 best snapshot cursor\')\n    if i[\'source\'][\'commit\'] != expected[\'training_commit\'] or i[\'method\'] != expected[\'method\'] or i[\'epochs\'] != 10 or (i[\'global_batch\'] != 32):\n        raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if i.get(\'seed\') != 43:\n        raise ValueError(\'Wrong training seed\')\n    if not {\'core.nouns\', \'core.verbs\'} <= set(ck[\'model\']):\n        raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\ndef configure_valid(cfg):\n    global VALID_IDS\n    if sha(cfg[\'dataset_json\']) != shared.DATA_SHA:\n        raise ValueError(\'Changed Karpathy metadata\')\n\n    def val_rows(ds, locked):\n        rows = sorted((r for r in ds[\'images\'] if r[\'split\'] == \'val\'), key=lambda r: int(r[\'cocoid\']))\n        if [int(r[\'cocoid\']) for r in rows] != locked[\'expected_test_ids\']:\n            raise ValueError(\'Wrong VALID IDs\')\n        return rows\n    VALID_IDS = sorted((int(r[\'cocoid\']) for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'val\'))\n    if len(VALID_IDS) != 5000 or len(set(VALID_IDS)) != 5000:\n        raise ValueError(\'Need VALID5000\')\n    shared.selected_test_rows = val_rows\n\ndef prompt_inputs(cfg):\n    configure_valid(cfg)\n    inv = read(HERE / \'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    wanted = {n: inv[n] for n in (\'cache_receipt.json\', \'cohort.json\', \'prompt_links.jsonl\', \'prompt_texts.json\', \'prompt_embeddings.npy\')}\n    root = checked_root(cfg[\'prompt_test_root\'], \'cache_receipt.json\', wanted)\n    cohort = read(root / \'cohort.json\')\n    if cohort[\'test\'] or sorted(cohort[\'val\']) != VALID_IDS:\n        raise ValueError(\'Wrong prompt175 VALID cohort\')\n    texts = read(root / \'prompt_texts.json\')\n    vectors = np.load(root / \'prompt_embeddings.npy\', allow_pickle=False)\n    if texts != sorted(set(texts)) or vectors.shape != (410, 512) or vectors.dtype != np.float32 or (not np.isfinite(vectors).all()):\n        raise ValueError(\'Changed prompt175 embeddings\')\n    with (root / \'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        rows = [r for r in map(json.loads, stream) if r[\'group\'] == \'val\']\n    if sorted((r[\'coco_id\'] for r in rows)) != VALID_IDS:\n        raise ValueError(\'Wrong VALID prompt links\')\n    return (rows, texts, vectors, wanted)\n\ndef prepare(cfg):\n    rows, texts, vectors, seals = prompt_inputs(cfg)\n    for row in rows:\n        record_arrays(row, texts)\n    shared.prepare(cfg, locked=lock(), weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg, rows, texts, vectors, seals)\n\ndef stage_prompts(cfg, rows, texts, vectors, seals):\n    encoded = [record_arrays(row, texts) for row in rows]\n    stage = Path(cfg[\'stage\'])\n    out = Path(cfg[\'output\'])\n    ids = np.load(stage / \'ids.npy\', allow_pickle=False)\n    lookup = {r[\'coco_id\']: j for j, r in enumerate(rows)}\n    order = [lookup[int(i)] for i in ids]\n    for key in ARRAYS:\n        np.save(stage / (\'sem_\' + key + \'.npy\'), np.stack([encoded[j][key] for j in order]))\n    np.save(stage / \'sem_text_vectors.npy\', vectors)\n    receipt = read(out / \'stage_receipt.json\')\n    receipt[\'method\'] = \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'\n    receipt[\'prompt175_files_sha256\'] = seals\n    receipt[\'stage_files_sha256\'].update({p.name: sha(p) for p in stage.glob(\'sem_*.npy\')})\n    save(out / \'stage_receipt.json\', receipt)\n    saved = read(out / \'test_lock.json\')\n    saved[\'stage_receipt_sha256\'] = sha(out / \'stage_receipt.json\')\n    save(out / \'test_lock.json\', saved)\n    save(out / \'semantic_test_stage.json\', dict(images=5000, objects=sum((len(r[\'objects\']) for r in rows)), relations=sum((len(r[\'relations\']) for r in rows)), text_vectors=410, geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\', TRAIN_TEST_features_read=False, old_ROI_features_used=False, old_vocabulary_used=False))\n\ndef load(cfg, verify=True):\n    result = shared.load(cfg, verify=verify, validator=validate_lock)\n    if result[2].get(\'method\') != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':\n        raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n\ndef bind_verified_cache(cfg, cache_source_sha):\n    configure_valid(cfg)\n    out = Path(cfg[\'output\'])\n    cached = read(out/\'test_lock.json\')\n    expected = lock()\n    same = (\'expected_test_ids\', \'test_id_sha256\', \'beam_width\', \'max_new_tokens\',\n            \'length_normalization\', \'reranking\', \'training_commit\', \'method\')\n    if any(cached[k] != expected[k] for k in same):\n        raise ValueError(\'Cache cohort, decoder or architecture differs\')\n    if cached[\'checkpoint_sha256\'] != cache_source_sha:\n        raise ValueError(\'Wrong original cache source\')\n    rebound = dict(cached, **expected)\n    rebound[\'checkpoint\'] = cfg[\'checkpoint_root\']\n    validate_lock(rebound)\n    save(out/\'cache_checkpoint_binding.json\', dict(\n        original_cache_source_sha256=cache_source_sha, xe_checkpoint_sha256=CHECKPOINT_SHA,\n        checkpoint_source=VERSION, original_lock=cached, bound_lock=rebound,\n        cache_payloads_changed=False, training_references_scope=\'TRAIN/restval only\'))\n    save(out/\'test_lock.json\', rebound)\n', 'run_csa_per_caption_test.py': '"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import csa_per_caption_test_data as module\n        module.configure_valid(cfg)\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg)\n    if cfg.get(\'evaluate_pilot\'):\n        receipt=read(Path(cfg[\'output\'])/\'pilot_checkpoint_receipt.json\')\n        path=Path(cfg[\'output\'])/\'pilot_checkpoint.bin\'\n        if receipt[\'updates\']!=cfg[\'eval_update\'] or receipt[\'source_sha256\']!=d.CHECKPOINT_SHA or sha(path)!=receipt[\'sha256\']:\n            raise ValueError(\'Wrong SCST-only checkpoint\')\n        ck=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if ck[\'updates\']!=cfg[\'eval_update\'] or ck[\'source_sha256\']!=d.CHECKPOINT_SHA:\n            raise ValueError(\'Pilot state differs\')\n        return ck,d.runtime_policy(read(Path(cfg[\'output\'])/\'training_runtime.json\')[\'torch_version\'],str(torch.__version__))\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from csa_patch_model import CSAPatchCaptioner\n            from semantic_data import SemanticContext\n            model=CSAPatchCaptioner(len(words),ck[\'model\'][\'core.nouns\'],ck[\'model\'][\'core.verbs\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'VALID decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'VALID decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] if cfg.get(\'evaluate_pilot\') else d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong VALID references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=read(cfg[\'cached_reference_ptb\']);hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'],files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete VALID scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CSA_PER_CAPTION_BEST9_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n', 'scst_cache.py': '"""CPU-build and GPU-consume reusable SCST input cache."""\nfrom collections import Counter, OrderedDict\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport tarfile\n\nfrom real_bank_data import read, save, sha\n\nSCHEMA = \'csa_scst_inputs_v1\'\nSOURCE_SHA = \'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef\'\nVALID_META = (\'test_lock.json\', \'stage_receipt.json\', \'test_references.json\', \'vocabulary.json\')\nASSETS = (\'java8.tar.gz\', \'chair.py\', \'chair.pkl\', \'reference_ptb.json\')\nREWARD = (\'reward_refs.jsonl.gz\', \'reward_df.jsonl.gz\')\n\n\ndef build_reward(stage):\n    from captioning.scst import TrainingCider\n\n    stage = Path(stage)\n    refs = {int(i): v for i, v in read(stage/\'training_references.json\').items()}\n    scorer = TrainingCider(refs, reference_cache_size=2048)\n    for name, values in zip(REWARD, (scorer.references.items(), scorer.df.items())):\n        with gzip.open(stage/name, \'wt\', encoding=\'utf8\') as out:\n            for row in values:\n                out.write(json.dumps(row, ensure_ascii=False)+\'\\n\')\n\n\ndef load_reward(stage):\n    from captioning.scst import TrainingCider\n\n    stage = Path(stage)\n    with gzip.open(stage/REWARD[0], \'rt\', encoding=\'utf8\') as stream:\n        refs = {int(i): tokens for i, tokens in map(json.loads, stream)}\n    with gzip.open(stage/REWARD[1], \'rt\', encoding=\'utf8\') as stream:\n        df = Counter({tuple(key): count for key, count in map(json.loads, stream)})\n    if len(refs) != 113287 or not df:\n        raise ValueError(\'Wrong cached TRAIN reward\')\n    scorer = TrainingCider.__new__(TrainingCider)\n    scorer.references, scorer.df = refs, df\n    scorer.log_count = math.log(len(refs))\n    scorer.reference_cache_size = 2048\n    scorer.vectors = OrderedDict()\n    return scorer\n\n\ndef seal(root, train, valid, meta, images):\n    root, train, valid, meta = map(Path, (root, train, valid, meta))\n    if images not in (4000, 20000, 40000):\n        raise ValueError(\'Unsupported cache schedule size\')\n    (root/\'train\').mkdir()\n    for name in (\'pilot_ids.npy\', \'pilot_sequences.npy\', \'pilot_patches.npy\', *REWARD):\n        shutil.move(train/name, root/\'train\'/name)\n    for path in train.glob(\'sem_*.npy\'):\n        shutil.move(path, root/\'train\'/path.name)\n    (root/\'valid\').mkdir()\n    for path in valid.glob(\'*.npy\'):\n        shutil.move(path, root/\'valid\'/path.name)\n    for name in VALID_META:\n        shutil.copyfile(meta/name, root/name)\n    lock = read(root/\'test_lock.json\')\n    lock.pop(\'checkpoint\', None)  # The consumer binds the actual mounted binary.\n    if lock[\'checkpoint_sha256\'] != SOURCE_SHA:\n        raise ValueError(\'Wrong checkpoint lock\')\n    save(root/\'test_lock.json\', lock)\n    files = {p.relative_to(root).as_posix(): sha(p) for p in sorted(root.rglob(\'*\')) if p.is_file()}\n    save(root/\'scst_cache_receipt.json\', dict(schema=SCHEMA, source_sha256=SOURCE_SHA,\n         train_images=images, train_start=0, train_stop=images, validation_images=5000, test_used=False, files_sha256=files))\n\n\ndef mount(cfg, root, destination, linked_patches=None):\n    root, destination = Path(root), Path(destination)\n    manifest = read(root/\'scst_cache_receipt.json\')\n    if (manifest[\'schema\'] != SCHEMA or manifest[\'source_sha256\'] != SOURCE_SHA\n            or manifest[\'train_images\'] not in (4000, 20000, 40000)\n            or manifest.get(\'train_start\', 0) != 0\n            or manifest[\'validation_images\'] != 5000 or manifest[\'test_used\']):\n        raise ValueError(\'Wrong SCST cache identity\')\n    required = set((*VALID_META, *ASSETS, *(f\'train/{n}\' for n in REWARD),\n                    \'train/pilot_patches.npy\', \'train/pilot_ids.npy\', \'train/pilot_sequences.npy\',\n                    \'train/sem_text_vectors.npy\', \'valid/patches.npy\'))\n    if not required <= manifest[\'files_sha256\'].keys():\n        raise ValueError(\'Incomplete SCST cache\')\n    for name, digest in manifest[\'files_sha256\'].items():\n        path = root/name\n        target = path.resolve()\n        linked = (linked_patches or {}).get(name)\n        if (not target.is_relative_to(root.resolve()) and target != linked) or sha(path) != digest:\n            raise ValueError(\'Changed cache file: \'+name)\n    for name in VALID_META:\n        shutil.copyfile(root/name, destination/name)\n    lock = read(destination/\'test_lock.json\')\n    lock[\'checkpoint\'] = cfg[\'checkpoint_root\']\n    save(destination/\'test_lock.json\', lock)\n    cfg.update(stage=str(root/\'valid\'), train_stage=str(root/\'train\'),\n               cached_reference_ptb=str(root/\'reference_ptb.json\'))\n    save(destination/\'scst_cache_receipt.json\', manifest)\n    chair = Path(cfg[\'chair\'])\n    chair.mkdir()\n    for name in (\'chair.py\', \'chair.pkl\'):\n        shutil.copyfile(root/name, chair/name)\n    jre = destination/\'jre\'\n    with tarfile.open(root/\'java8.tar.gz\') as t:\n        t.extractall(jre, filter=\'data\')\n    java, = jre.rglob(\'bin/java\')\n    save(destination/\'environment.json\', dict(java=str(java), cached=True))\n\n\ndef seal_train_shard(root, train, start, stop):\n    if (start, stop) not in ((40000,80000),(80000,113287)):\n        raise ValueError(\'Wrong full-cache shard range\')\n    root, train = Path(root), Path(train)\n    dest = root/\'train\'\n    dest.mkdir(parents=True)\n    for name in (\'pilot_ids.npy\',\'pilot_sequences.npy\',\'pilot_patches.npy\'):\n        shutil.move(train/name,dest/name)\n    for path in train.glob(\'sem_*.npy\'):\n        shutil.move(path,dest/path.name)\n    files = {p.relative_to(root).as_posix():sha(p) for p in sorted(dest.iterdir())}\n    save(root/\'scst_train_shard_receipt.json\',dict(schema=SCHEMA,source_sha256=SOURCE_SHA,\n         train_images=stop-start,train_start=start,train_stop=stop,validation_images=0,\n         test_used=False,files_sha256=files))\n', 'scst_dataset_mount.py': "import hashlib, json, shutil, zipfile\nfrom pathlib import Path\nfrom real_bank_data import read, sha\n\ndef materialize_part0(dataset, local, expected_parts, expected_ids):\n    dataset, local = Path(dataset).resolve(), Path(local)\n    manifest = read(dataset/'cache_manifest.json')\n    if (manifest['schema']!='csa_scst_full_dataset_v1' or manifest['train_images']!=113287\n            or manifest['valid_images']!=5000 or manifest['test_used']\n            or manifest['ordered_train_ids_sha256']!=expected_ids or len(manifest['parts'])!=3):\n        raise ValueError('Wrong full cache Dataset manifest')\n    for part, expected in zip(manifest['parts'],expected_parts,strict=True):\n        if any(part.get(key)!=value for key,value in expected.items()):\n            raise ValueError('Wrong source part identity in Dataset manifest')\n    part = manifest['parts'][0]\n    bundle = dataset/'cache_metadata_zip.bin'\n    if sha(bundle)!=manifest['bundle_sha256']:\n        raise ValueError('Changed cache metadata bundle')\n    patch_names = {'train/pilot_patches.npy':'part0_train_pilot_patches.npy',\n                   'valid/patches.npy':'part0_valid_patches.npy'}\n    links = {}\n    with zipfile.ZipFile(bundle) as archive:\n        name = 'part0/scst_cache_receipt.json'\n        if part['receipt_member']!=name:\n            raise ValueError('Unexpected original receipt path')\n        raw = archive.read(name)\n        if hashlib.sha256(raw).hexdigest()!=expected_parts[0]['receipt_sha256']:\n            raise ValueError('Changed original Part 0 receipt')\n        files=json.loads(raw)['files_sha256']\n        if set(files)!=set(part['files']):\n            raise ValueError('Incomplete Part 0 file map')\n        local.mkdir()\n        (local/'scst_cache_receipt.json').write_bytes(raw)\n        for name,digest in files.items():\n            entry=part['files'][name]\n            target=local/name\n            if not target.resolve().is_relative_to(local.resolve()) or entry['sha256']!=digest:\n                raise ValueError('Wrong mapped file: '+name)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            if name in patch_names:\n                if entry.get('file')!=patch_names[name]:\n                    raise ValueError('Wrong mounted patch file')\n                source=(dataset/patch_names[name]).resolve(strict=True)\n                if not source.is_relative_to(dataset) or not source.is_file():\n                    raise ValueError('Wrong mounted patch path')\n                target.symlink_to(source)\n                links[name]=source\n            else:\n                member='part0/'+name\n                if entry.get('bundle')!=bundle.name or entry.get('member')!=member:\n                    raise ValueError('Wrong bundled member')\n                with archive.open(member) as source, target.open('xb') as dest:\n                    shutil.copyfileobj(source,dest,length=8*1024**2)\n    return links\n", 'captioning/scst.py': '"""SCST primitives, independent of CLIP and the training data loader.\n\nReward uses CIDEr-D (clipping, sigma=6, x10), fixed training-image IDF,\nand NLTK Treebank normalization. Published metrics still use official PTB.\n"""\nfrom collections import Counter, OrderedDict\nimport math\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\ndef normalize(text):\n    from nltk.tokenize import TreebankWordTokenizer\n    punctuation = {"\'\'", "\'", \'``\', \'`\', \'.\', \'?\', \'!\', \',\', \':\', \'-\', \'--\', \'...\', \';\',\n                   \'(\', \')\', \'[\', \']\', \'{\', \'}\'}\n    return \' \'.join(w for w in TreebankWordTokenizer().tokenize(text.lower())\n                    if w not in punctuation)\n\n\ndef ngrams(text):\n    words = text.split()\n    return Counter(tuple(words[i:i+n]) for n in range(1, 5)\n                   for i in range(len(words)-n+1))\n\n\nclass TrainingCider:\n    """Cached reference vectors; never recompute document frequency per batch."""\n    def __init__(self, references, reference_cache_size=None):\n        if reference_cache_size is not None and reference_cache_size < 1:\n            raise ValueError(\'Reference cache size must be positive or None.\')\n        self.reference_cache_size = reference_cache_size\n        if len(references) < 2 or any(len(v) != 5 for v in references.values()):\n            raise ValueError(\'Need at least two training images with five references each.\')\n        self.references = {k: [normalize(s) for s in refs] for k, refs in references.items()}\n        self.df = Counter()\n        for refs in self.references.values():\n            self.df.update(set(g for ref in refs for g in ngrams(ref)))\n        self.log_count = math.log(len(references))\n        self.vectors = ({k: [self.vector(ref) for ref in refs]\n                         for k, refs in self.references.items()} if reference_cache_size is None else OrderedDict())\n\n    def reference_vectors(self, image_id):\n        if image_id not in self.references: raise KeyError(image_id)\n        if image_id not in self.vectors:\n            self.vectors[image_id] = [self.vector(ref) for ref in self.references[image_id]]\n        if self.reference_cache_size is not None:\n            self.vectors.move_to_end(image_id)\n            while len(self.vectors) > self.reference_cache_size: self.vectors.popitem(last=False)\n        return self.vectors[image_id]\n\n    def vector(self, text):\n        vectors, norms, length = [{} for _ in range(4)], [0.]*4, 0\n        for g, count in ngrams(text).items():\n            weight = count * (self.log_count - math.log(max(1, self.df.get(g, 0))))\n            vectors[len(g)-1][g] = weight\n            norms[len(g)-1] += weight**2\n            if len(g) == 2:\n                length += count\n        return vectors, [math.sqrt(n) for n in norms], length\n\n    def score(self, ids, captions):\n        if len(ids) != len(captions):\n            raise ValueError(\'Reward IDs and captions differ.\')\n        scores = []\n        for image_id, text in zip(ids, captions):\n            hyp, hn, hl = self.vector(normalize(text))\n            score = 0.\n            reference_vectors = self.reference_vectors(image_id)\n            for ref, rn, rl in reference_vectors:\n                similarity = 0.\n                for n in range(4):\n                    if hn[n] and rn[n]:\n                        similarity += sum(min(v, ref[n].get(g, 0.))*ref[n].get(g, 0.)\n                                          for g, v in hyp[n].items()) / (hn[n]*rn[n])\n                score += similarity/4 * math.exp(-((hl-rl)**2)/72.)\n            scores.append(10.*score/len(reference_vectors))\n        return np.asarray(scores, dtype=np.float32)\n\n\n@torch.no_grad()\ndef rollout(model, memory, padding, tokenizer, sample, max_len=30):\n    """No dropout: sample and greedy use the same distribution as rescoring.\n\n    Keep PAD/BOS probabilities, matching the existing beam decoder policy.\n    The EOS action counts; actions after the first EOS do not count.\n    """\n    sequence = torch.full((memory.size(0), 1), tokenizer.bos_idx,\n                          dtype=torch.long, device=memory.device)\n    active = torch.ones(memory.size(0), dtype=torch.bool, device=memory.device)\n    actions, masks = [], []\n    for _ in range(max_len-1):\n        logits = model.decoder(sequence, memory, memory_key_padding_mask=padding)[:, -1].float()\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError(\'Nonfinite rollout logits.\')\n        token = (torch.multinomial(logits.softmax(-1), 1).squeeze(1) if sample\n                 else logits.argmax(-1))\n        token = torch.where(active, token, tokenizer.pad_idx)\n        actions.append(token)\n        masks.append(active.clone())\n        sequence = torch.cat([sequence, token[:, None]], dim=1)\n        active = active & token.ne(tokenizer.eos_idx)\n        if not active.any():\n            break\n    return sequence, torch.stack(masks, dim=1)\n\n\ndef policy_loss(logits, actions, mask, advantage):\n    """Standard sequence-sum REINFORCE; greedy reward has no gradient."""\n    if logits.shape[:2] != actions.shape or actions.shape != mask.shape:\n        raise ValueError(\'Policy action/mask shape mismatch.\')\n    selected = F.log_softmax(logits.float(), -1).gather(-1, actions[..., None]).squeeze(-1)\n    return -(selected*mask).sum(1).mul(advantage.detach()).mean()\n\n\ndef decode(sequence, tokenizer):\n    captions = []\n    for row in sequence.detach().cpu().tolist():\n        words = []\n        for token in row[1:]:\n            if token == tokenizer.eos_idx:\n                break\n            if token not in (tokenizer.pad_idx, tokenizer.bos_idx):\n                words.append(tokenizer.idx2word.get(token, tokenizer.UNK))\n        captions.append(\' \'.join(words))\n    return captions\n', 'captioning/__init__.py': '', 'CACHE_PARTS.json': '{\n  "parts": [\n    {\n      "part": 0,\n      "receipt_name": "scst_cache_receipt.json",\n      "receipt_sha256": "3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510",\n      "archive_sha256": "64eb7d3c958fca4996d717c300191022176e5d688ddd4ba6ea94a5f1a30d3946",\n      "train_start": 0,\n      "train_stop": 40000,\n      "ids_sha256": "f1fd1507bb3dc6b5dd4ddc57a83e723e2610ec7a45e7a6b05813fa633e519480",\n      "source_sha256": "b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef",\n      "notebook_version": "ducanh3/notebook7bc83c4c76/82"\n    },\n    {\n      "part": 1,\n      "receipt_name": "scst_train_shard_receipt.json",\n      "receipt_sha256": "7ba08f77f38ce48fc8a8a023dbe70ffd6b9bed142f28d54711a9c3ebd0b98a12",\n      "archive_sha256": "b9cadfb1f55067b767ed1e32012d5b5bfc7f9f0000c9ff8df0c2ed9676ab461c",\n      "train_start": 40000,\n      "train_stop": 80000,\n      "ids_sha256": "570fab90751d77fda57aebe84a17537a7cba57d3244fc06a5cdcb55ff2c1d885",\n      "source_sha256": "b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef",\n      "notebook_version": "ducanh3/notebook7bc83c4c76/83"\n    },\n    {\n      "part": 2,\n      "receipt_name": "scst_train_shard_receipt.json",\n      "receipt_sha256": "3b8247a0e3fc24aff0a0b2125ce5fbd372976274c5fcf5e1f46315516824ad5c",\n      "archive_sha256": "a78982bb44fbb4ce5668d36f9df47de5da9536834ac379f123349a893fd0c9c8",\n      "train_start": 80000,\n      "train_stop": 113287,\n      "ids_sha256": "22d47e2713dbe61777a97a8104ba4bc2490e253c4ff8fd71d0e871d57b2d38fb",\n      "source_sha256": "b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef",\n      "notebook_version": "ducanh3/notebook7bc83c4c76/84"\n    }\n  ],\n  "ids_sha256": "6165b0fe756398f3625c4504ebe62f25f07657ce0cffc571a2ce7613ada6e263",\n  "source_sha256": "b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef"\n}\n', 'beam_policy.py': '"""USER-RUN differentiable beam histories. Importing this module needs no Torch."""\n\n\ndef train_beams(model, memory, width=5, maximum=20):\n    import torch\n    if width < 2 or maximum < 1:\n        raise ValueError(\'Need at least two beams and one action\')\n    b, n, d = memory.shape\n    vocab, device = model.head.out_features, memory.device\n    if width > vocab-2:\n        raise ValueError(\'Beam width exceeds valid vocabulary\')\n    seq = torch.ones((b, width, 1), dtype=torch.long, device=device)\n    scores = torch.full((b, width), -torch.inf, device=device)\n    scores[:, 0] = 0\n    done = torch.zeros((b, width), dtype=torch.bool, device=device)\n    history = memory.new_zeros((b, width, 0))\n    expanded = memory[:, None].expand(-1, width, -1, -1).reshape(b*width, n, d)\n    for _ in range(maximum):\n        logits = model.decode(expanded, seq.reshape(b*width, -1))[:, -1].float()\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError(\'Nonfinite beam logits\')\n        logp = logits.log_softmax(-1).view(b, width, vocab)\n        forbidden = torch.arange(vocab, device=device) < 2\n        logp = logp.masked_fill(forbidden, -torch.inf)\n        finished = torch.full_like(logp, -torch.inf)\n        finished[:, :, 0] = 0\n        logp = torch.where(done[:, :, None], finished, logp)\n        # Selection is discrete; selected path log probabilities retain their graph.\n        scores, flat = (scores[:, :, None]+logp.detach()).flatten(1).topk(width, dim=1)\n        parent, token = flat//vocab, flat % vocab\n        chosen = logp.gather(1, parent[:, :, None].expand(-1, -1, vocab)).gather(2, token[:, :, None]).squeeze(-1)\n        history = torch.cat((history.gather(1, parent[:, :, None].expand(-1, -1, history.shape[-1])), chosen[:, :, None]), -1)\n        seq = torch.cat((seq.gather(1, parent[:, :, None].expand(-1, -1, seq.shape[-1])), token[:, :, None]), -1)\n        done = done.gather(1, parent) | token.eq(2)\n        if done.all():\n            break\n    if history.shape[-1] < maximum:\n        padding = maximum-history.shape[-1]\n        history = torch.cat((history, history.new_zeros((b, width, padding))), -1)\n        seq = torch.cat((seq, seq.new_zeros((b, width, padding))), -1)\n    if not torch.isfinite(history).all():\n        raise FloatingPointError(\'Nonfinite selected beam log probabilities\')\n    return seq, history\n\n\ndef image_losses(logps, rewards):\n    if logps.ndim != 3 or rewards.shape != logps.shape[:2]:\n        raise ValueError(\'Expected image x beam x action and image x beam reward\')\n    advantage = rewards.detach()-rewards.detach().mean(-1, keepdim=True)\n    return -(logps.mean(-1)*advantage).mean(-1)\n\n\ndef rank_microbatch(start, rank, micro, real_images=113287):\n    """Two ranks x four images x four microbatches; dummy rows have weight zero."""\n    if rank not in (0, 1) or micro not in range(4) or not 0 <= start < real_images:\n        raise ValueError(\'Invalid epoch cursor/rank/microbatch\')\n    offsets = [start+micro*8+rank*4+i for i in range(4)]\n    return [i if i < real_images else 0 for i in offsets], [i < real_images for i in offsets]\n\n\ndef mechanism_guard(device):\n    """USER-RUN only: beam parent history, EOS and weighted-tail gradients."""\n    import torch\n    from patch_beam import beam_search\n    class Toy(torch.nn.Module):\n        def __init__(self, eos_first=False):\n            super().__init__()\n            self.head = torch.nn.Linear(3, 9)\n            self.eos_first = eos_first\n        def decode(self, memory, prefix):\n            x = memory.mean(1)[:, None, :].expand(-1, prefix.shape[1], -1)\n            # Deterministic prefix dependence tests parent-history gathering.\n            x = x+prefix.float()[:, :, None]*.03\n            logits = self.head(x)\n            if self.eos_first:\n                offset = torch.zeros_like(logits)\n                offset[:, :, 2] = 50\n                logits = logits+offset\n            return logits\n    for eos_first in (False, True):\n        toy = Toy(eos_first).to(device).eval()\n        memory = torch.randn(3, 4, 3, device=device)\n        seq, logps = train_beams(toy, memory, 5, 6)\n        with torch.no_grad():\n            expected, score = beam_search(toy, memory, 5, 6)\n        padded = torch.cat((expected, expected.new_zeros((3, 5, seq.shape[-1]-expected.shape[-1]))), -1)\n        torch.testing.assert_close(seq, padded, rtol=0, atol=0)\n        torch.testing.assert_close(logps.sum(-1), score, rtol=1e-5, atol=1e-5)\n        after_eos = seq[:, :, 1:].eq(2).cumsum(-1)-seq[:, :, 1:].eq(2).long() > 0\n        if torch.count_nonzero(logps.masked_select(after_eos)):\n            raise ValueError(\'Post-EOS actions affect loss\')\n        reward = torch.arange(5, device=device).float().expand(3, -1)\n        loss = image_losses(logps, reward).mean()\n        loss.backward()\n        if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in toy.parameters()):\n            raise ValueError(\'Beam history lost its gradient\')\n    # Compare the actual weighted image loss gradients to an explicit formula.\n    for count in (7, 32):\n        lp = torch.randn(32, 5, 20, device=device, requires_grad=True)\n        reward = torch.randn(32, 5, device=device)\n        reference = image_losses(lp[:count], reward[:count]).mean()\n        g = torch.autograd.grad(reference, lp, retain_graph=True)[0]\n        distributed = lp.sum()*0\n        for rank in (0, 1):\n            for micro in range(4):\n                rows, keep = rank_microbatch(0, rank, micro, count)\n                weights = torch.tensor(keep, device=device, dtype=torch.float32)\n                distributed = distributed+(image_losses(lp[rows], reward[rows])*weights).sum()/count\n        actual = torch.autograd.grad(distributed, lp)[0]\n        torch.testing.assert_close(actual, g, rtol=1e-6, atol=1e-7)\n    return dict(beam_path_and_score=True, selected_history_autograd=True, EOS_mask=True,\n                image_mean_reward_baseline=True, tail7_and_global32_weighted_gradient=True)\n', 'full_epoch_data.py': '"""Sealed metadata staging and readonly patch access; no model import."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\nimport numpy as np\n\nfrom real_bank_data import read, save, sha\n\n\ndef id_digest(ids):\n    return hashlib.sha256(json.dumps(ids, separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef prepare(cfg):\n    from scst_dataset_mount import materialize_part0\n    from scst_cache import mount\n    from semantic_data import ARRAYS\n    expected = read(Path(__file__).with_name(\'CACHE_PARTS.json\'))\n    dataset, out = Path(cfg[\'cache_root\']).resolve(), Path(cfg[\'output\'])\n    spec = read(dataset/\'cache_manifest.json\')\n    base = Path(cfg[\'metadata_root\'])\n    base.mkdir(parents=True)\n    part0 = base/\'part0\'\n    links = materialize_part0(dataset, part0, expected[\'parts\'], expected[\'ids_sha256\'])\n    mount(cfg, part0, out, linked_patches=links)\n    roots = [part0/\'train\']\n    with zipfile.ZipFile(dataset/\'cache_metadata_zip.bin\') as archive:\n        for part, exp in zip(spec[\'parts\'][1:], expected[\'parts\'][1:], strict=True):\n            number = exp[\'part\']\n            dest = base/f\'part{number}\'\n            dest.mkdir()\n            raw = archive.read(f"part{number}/{exp[\'receipt_name\']}")\n            if hashlib.sha256(raw).hexdigest() != exp[\'receipt_sha256\']:\n                raise ValueError(\'Changed original shard receipt\')\n            receipt = json.loads(raw)\n            if receipt[\'test_used\'] or receipt[\'source_sha256\'] != expected[\'source_sha256\'] or receipt[\'train_start\'] != exp[\'train_start\'] or receipt[\'train_stop\'] != exp[\'train_stop\']:\n                raise ValueError(\'Wrong shard scope\')\n            files = receipt[\'files_sha256\']\n            if set(files) != set(part[\'files\']):\n                raise ValueError(\'Incomplete shard mapping\')\n            for name, digest in files.items():\n                entry, target = part[\'files\'][name], dest/name\n                if not target.resolve().is_relative_to(dest.resolve()) or entry[\'sha256\'] != digest:\n                    raise ValueError(\'Invalid shard member\')\n                target.parent.mkdir(parents=True, exist_ok=True)\n                if name == \'train/pilot_patches.npy\':\n                    filename = f\'part{number}_train_pilot_patches.npy\'\n                    if entry.get(\'file\') != filename:\n                        raise ValueError(\'Wrong patch mapping\')\n                    source = (dataset/filename).resolve(strict=True)\n                    if not source.is_relative_to(dataset):\n                        raise ValueError(\'Patch escapes input dataset\')\n                    target.symlink_to(source)\n                else:\n                    member = f\'part{number}/{name}\'\n                    if entry.get(\'bundle\') != \'cache_metadata_zip.bin\' or entry.get(\'member\') != member:\n                        raise ValueError(\'Wrong metadata mapping\')\n                    with archive.open(member) as src, target.open(\'xb\') as dst:\n                        shutil.copyfileobj(src, dst, length=8*1024**2)\n                    if sha(target) != digest:\n                        raise ValueError(\'Changed shard metadata\')\n            roots.append(dest/\'train\')\n    all_ids = []\n    vector_digest = sha(roots[0]/\'sem_text_vectors.npy\')\n    for root, exp in zip(roots, expected[\'parts\'], strict=True):\n        size = exp[\'train_stop\']-exp[\'train_start\']\n        ids = np.load(root/\'pilot_ids.npy\', allow_pickle=False)\n        patches = np.load(root/\'pilot_patches.npy\', mmap_mode=\'r\', allow_pickle=False)\n        if ids.shape != (size,) or ids.dtype.kind not in \'iu\' or patches.shape != (size, 196, 768) or patches.dtype != np.float16:\n            raise ValueError(\'Wrong TRAIN shard headers\')\n        if patches.offset+patches.nbytes != (root/\'pilot_patches.npy\').stat().st_size or id_digest(ids.tolist()) != exp[\'ids_sha256\']:\n            raise ValueError(\'Wrong patch size or ordered IDs\')\n        if sha(root/\'sem_text_vectors.npy\') != vector_digest:\n            raise ValueError(\'Text dictionaries differ between shards\')\n        for name, (shape, dtype) in ARRAYS.items():\n            array = np.load(root/f\'sem_{name}.npy\', mmap_mode=\'r\', allow_pickle=False)\n            if array.shape != (size, *shape) or array.dtype != dtype:\n                raise ValueError(\'Wrong semantic array: \'+name)\n        all_ids.extend(ids.tolist())\n    valid = read(out/\'test_lock.json\')[\'expected_test_ids\']\n    if len(all_ids) != 113287 or len(set(all_ids)) != 113287:\n        raise ValueError(\'Wrong full TRAIN uniqueness\')\n    if len(all_ids) != 113287 or id_digest(all_ids) != expected[\'ids_sha256\'] or set(all_ids) & set(valid):\n        raise ValueError(\'Wrong TRAIN order or TRAIN/VALID overlap\')\n    cfg[\'train_roots\'] = list(map(str, roots))\n    cfg[\'train_stage\'] = str(roots[0])\n    cfg[\'cache_manifest_sha256\'] = sha(dataset/\'cache_manifest.json\')\n    save(out/\'full_epoch_cache.json\', dict(train_images=113287, valid_images=5000, test_used=False,\n        cache_manifest_sha256=cfg[\'cache_manifest_sha256\'], ordered_train_ids_sha256=id_digest(all_ids),\n        metadata_payloads_SHA_checked=True, original_source_receipts_checked=True,\n        patch_full_content_SHA_rehashed=False, patch_shape_size_and_readonly_dataset_mapping_checked=True,\n        patch_copies=False, train_roots=cfg[\'train_roots\']))\n\n\nclass FullTrain:\n    def __init__(self, roots):\n        from semantic_data import SemanticContext\n        self.roots = list(map(Path, roots))\n        self.patches = [np.load(p/\'pilot_patches.npy\', mmap_mode=\'r\', allow_pickle=False) for p in self.roots]\n        self.ids = [np.load(p/\'pilot_ids.npy\', allow_pickle=False) for p in self.roots]\n        self.context = [SemanticContext(p) for p in self.roots]\n        self.stops = np.cumsum([len(i) for i in self.ids])\n\n    def batch(self, rows, device):\n        import torch\n        p, x, ids = [], [], []\n        # Batch4; preserve row order even when crossing a shard boundary.\n        for row in rows:\n            part = int(np.searchsorted(self.stops, row, side=\'right\'))\n            offset = int(row-(self.stops[part-1] if part else 0))\n            x.append(np.asarray(self.patches[part][offset], np.float32))\n            ids.append(int(self.ids[part][offset]))\n            p.append(self.context[part].images([offset], device))\n        prompt = {name: torch.cat([item[name] for item in p]) for name in p[0]}\n        return torch.from_numpy(np.stack(x)).to(device), prompt, ids\n', 'PROTOCOL.md': "# Author-style beam SCST, one full TRAIN epoch — 25/11/2026\n\nUser requests one epoch following the authors' SCST mechanism, then reports 16h GPU available. Prepare a user-run Kaggle notebook; never run models/Torch/scorers locally. This supersedes the earlier assumption that no GPU remains for this particular trial; zero-text seed43/44 remain queued, not cancelled.\n\nQuestion: does changing the bounded SCST recipe to five-beam, within-image mean-reward baseline improve VALID CIDEr, while tracking CHAIR? Not a controlled attribution of the effect of beam baseline alone: duration, policy mode and optimizer rate also change versus the previous sampled/greedy experiment.\n\n## Fixed recipe before code\n\n- Start fresh from full-prompt seed42 best CE epoch9, SHA `b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef`, mounted at `/kaggle/input/datasets/ducanh3/upt-coco-full-cache`. Never start from SCST1500/3000, baseline or zero-text weights. No API/secret required for this mounted source.\n- Reuse `/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1`, original parts82/83/84 and all113287 unique TRAIN IDs in their sealed order. Read patch NPYs directly; no cache regeneration, full patch copies or encoder/detector inference. Materialize metadata only with all original SHA contracts. VALID5000 remains disjoint.\n- One epoch means exactly113287 real image events, once each; effective global image batch32,3541 optimizer steps, final real batch7. No duplicated/padded image contributes to the objective. Preserve unused final slots only to keep DDP collectives balanced.\n- Five-beam generation during **train mode with gradients**, max20, PAD/BOS forbidden as in existing decoder policy, EOS included and subsequent log probabilities zero. Fixed20-position denominator, matching the public `mean(log_probs,-1)` over a beam_len20 tensor. All five rewards from the same image define its baseline. Loss is negative mean-over-beams of advantage times mean token log probability. Pure SCST; no CE/CHAIR/object/CLIP reward or reranking.\n- Fresh Adam, lr4e-6,betas(.9,.99),eps1e-8,weight_decay0; FP32, no AMP/compile, no gradient clipping (finite norm monitored). Original encoder/cache architecture retained; all existing trainable captioner/alignment parameters learn, frozen buffers remain buffers.\n- T4x2 DDP. **Adaptation for memory:** four images/GPU/microbatch, four accumulation rounds gives global32, preserving the effective image batch. This is not an exact dropout/RNG replay of authors' single local batch16. No automatic batch/precision/optimizer fallback after OOM.\n- Reuse our sealed cached TRAIN CIDEr-D reward implementation/tokenization/DF. It is not a claim of byte-identical tokenizer/scorer reproduction of the authors' public code. Original backbone differs too. This tests an author-style beam SCST adaptation, not paper reproduction or a promise of +13CIDEr.\n- Endpoint fixed at one epoch. Compare endpoint VALID top1 beam5 with the already audited CE-best VALID reference; compute CIDEr/BLEU1/4/ROUGE-L/CHAIRs/i/Recall/NLL. No TEST, no rerank, no selection between partial checkpoints using TEST. Report all unfavorable metrics; no automatic second epoch.\n\n## Guard, budget, continuation\n\nHost only static/metadata/NumPy/mock checks. User-run preflight checks CUDA, checkpoint SHA, all source/cache/ID/vocabulary contracts, beam paths against existing eval beam search, EOS masking and autograd loss signs, per-image baseline, tail7 distributed weighting, finite gradients into core/M3, real peak memory, two warmup updates, one timed update in each of the three shards and one tail update (all six discarded). Use the slowest measured shard/rank update for the projection; this remains a small sample, not a completion guarantee. Training starts in a separate fresh process; discarded preflight updates never initialize the real run. If OOM/nonfinite/identity failure, stop with review, never change recipe silently.\n\nPreflight prints measured throughput and a projection, not a guaranteed ETA/quota bill. A parent invocation cap11.5h reserves time for validation/checkpoint packaging. It is not a claim that10/16 quota hours equal wall hours or that the epoch fits a session. If the measured epoch projection exceeds14h, stop before full training for review. Save atomic latest checkpoint every100 updates, full optimizer/RNG per rank/cursor/identity; interruption writes a final partial checkpoint. Resume only this exact recipe/source/seed/cache/software and cursor. Do not restart the epoch or reuse preflight state. Mounted RESUME_ROOT is supported; if only Save Version remains, user supplies actual version for a pinned API continuation later.\n\nIf a cap interrupts training, keep the entire output, not only review ZIP. Evaluation of the completed endpoint can run separately if its phase fails; never retrain to recover CPU scores.\n\n## Primary sources checked\n\n- Paper, objective III-D / implementation IV-B:15CE+30SCSTepochs and CIDEr-D reward, DOI https://doi.org/10.1109/TCSVT.2023.3343520.\n- Public `train_sc`: https://github.com/CrossmodalGroup/CSA-Net/blob/main/engine/caption_engine.py#L359-L457 — five beams, reward mean baseline, mean token logp, no CE term, `model.train()`.\n- https://github.com/CrossmodalGroup/CSA-Net/blob/main/configs/caption/coco_config_csa_net_end_to_end.yaml — beam5/max20,SCSTlr4e-6,15/30epochs.\n- Public train script reloads bestVALID each SCST epoch; this one-epoch adaptation begins at CEbest, with exact latest-state continuation inside that epoch. We do not copy the public per-epoch reload behavior into interruption recovery.\n\n## Input-failure amendment,25/11/2026\n\nUser log passes CUDA/NCCL2GPU but stops in checkpoint resolver before torch.load/model construction or any training update. The default upt-coco-full-cache path was historical, not confirmed for this run. Sheet Kết quả mới row210 records seed43 source179 and seed44 source236, but only bestepoch9 for seed42; no exact seed42 Save Version/Dataset was found in the searched registry/source rows. Do not invent an API version or silently switch seed. Prepare V2 to verify exact source bytes before pip/cache/CUDA, search mounted weight files independent of filename, and emit a CPU-only source report. Wrong or missing bytes must still stop. All training/beam/loss/model/cache recipes stay fixed pending the user source choice. The observed run has0 training updates; no optimizer continuation is needed for that failed run.\n\n## V2 proposed source replacement after Sheet lookup,25/11/2026\n\nUser asks which source is better after being shown the missing seed42 provenance. Recommend Full seed43 bestCEepoch8 from ducanh3/notebook7bc83c4c76/version179, SHA a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5, because its source and prior same-seed CE/SCST results are already audited; not because of TEST score. Prepare this concrete replacement for user-run review; no API/model execution by agent. Keep V1 notebook/source manifest and failure evidence intact.\n\nV2 bundles the already-audited seed43 VALID contract/cache-binding helper from the earlier completed SCST1500 notebook. Replace checkpoint/epoch/source identity, policyseed4342(+rank), output label and CE VALID reference consistently. CE VALID baseline is CIDEr121.777807/CHAIRs4.18%/CHAIRi3.004824%/NLL2.155262. Original cache82/83/84 receipts and frozen feature bytes remain unchanged; verify original source42 cache, then rebind only local VALID model metadata to source43. No algorithm/loss/optimizer/batch/one-epoch changes. Compare only to the same seed43 CE source; earlier seed42 bootstrap numbers are not reused as this pilot baseline.\n\nBlank CHECKPOINT_ROOT downloads only exact179 best.bin through pinned isolated Kaggle CLI using an existing Secret with access to ducanh3. No latest-version fallback or automatic switch to236. Optional mounted weights are accepted under any filename only if SHA exactly matches179. Verify source before CUDA/pip scorers/cache staging; preserve input source binary in Output/input_source. Private-source API access is not established by local tests.\n", 'checkpoint_input.py': '"""Check exact mounted checkpoint bytes without importing Torch or loading pickle."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef resolve_mounted(root, expected_sha, report_path=None):\n    root = Path(root)\n    files = [root] if root.is_file() else sorted(\n        (p for p in root.rglob(\'*\') if p.is_file() and p.suffix.lower() in (\'.bin\',\'.pt\',\'.pth\',\'.ckpt\')\n         and p.name != \'cache_metadata_zip.bin\'),\n        key=lambda p: (p.name != \'best.bin\', \'best\' not in p.name.lower(), str(p))) if root.is_dir() else []\n    report = dict(root=str(root), expected_sha256=expected_sha, model_loaded=False,\n                  checked=[], matched_file=None,\n                  extracted_torch_directories=[str(p.parent) for p in root.rglob(\'data.pkl\')] if root.is_dir() else [])\n    for path in files:\n        with path.open(\'rb\') as stream:\n            digest = hashlib.file_digest(stream, \'sha256\').hexdigest()\n        row = dict(path=str(path), bytes=path.stat().st_size, sha256=digest)\n        report[\'checked\'].append(row)\n        print(\'Checkpoint input:\', json.dumps(row), flush=True)\n        if digest == expected_sha:\n            report[\'matched_file\'] = str(path.resolve())\n            break\n    if report_path:\n        Path(report_path).write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    if report[\'matched_file\'] is None:\n        print(json.dumps(report,indent=2),flush=True)\n        hint = \' Torch archive appears extracted; attach the intact checkpoint file.\' if report[\'extracted_torch_directories\'] else \'\'\n        raise FileNotFoundError(\'No mounted checkpoint matches the locked source SHA; no model loaded.\'+hint)\n    return Path(report[\'matched_file\'])\n', 'run_epoch2.py': '"""USER-RUN two-GPU FP32 beam-mean SCST; no model import at module load."""\nimport argparse\nfrom contextlib import nullcontext\nfrom datetime import timedelta\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\nfrom real_bank_data import read, save, sha\nfrom beam_policy import image_losses, mechanism_guard, rank_microbatch, train_beams\nfrom full_epoch_data import FullTrain\nfrom speed_candidate import batch_rows, train_beams_fast\nfrom epoch2_contract import (PARENT_SHA, PARENT_VERSION, PARENT_STEPS, END_STEPS,\n    validate_parent, make_identity, validate_resume)\n\nSOURCE_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nIMAGES, GLOBAL_BATCH, STEPS = 113287, 32, 3541\n\n\n\n\n\ndef run(cfg, mode):\n    import torch\n    import torch.distributed as dist\n    from torch import nn\n    from cuda_runtime_guard import check_runtime\n    from csa_patch_model import CSAPatchCaptioner\n    from csa_per_caption_test_data import configure_valid\n    from patch_beam import text_from_ids\n    from scst_cache import load_reward\n\n    rank = int(os.environ[\'LOCAL_RANK\'])\n    out = Path(cfg[\'output\'])\n    check_runtime(out, rank)\n    if int(os.environ[\'WORLD_SIZE\']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError(\'Select GPU T4 x2\')\n    torch.set_num_threads(1)\n    torch.cuda.set_device(rank)\n    device = torch.device(\'cuda\', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.manual_seed(4342+rank)\n    torch.cuda.manual_seed_all(4342+rank)\n    np.random.seed(4342+rank)\n    dist.init_process_group(\'nccl\', timeout=timedelta(seconds=1800))\n    start = time.monotonic()\n    deadline = start+cfg[\'train_seconds\']\n    try:\n        configure_valid(cfg)\n        source = Path(cfg[\'checkpoint_root\'])\n        if sha(source) != PARENT_SHA:\n            raise ValueError(\'Need exact v199 full checkpoint including Adam/RNG\')\n        ck = torch.load(source, map_location=\'cpu\', weights_only=False)\n        expected = read(Path(__file__).with_name(\'PARENT_CHECKPOINT_RECEIPT.json\'))\n        validate_parent(ck, expected)\n        words = read(out/\'vocabulary.json\')\n        if len(words) != 10000:raise ValueError(\'Wrong vocabulary\')\n        identity = make_identity(cfg, expected, str(torch.__version__), np.__version__, sha(out/\'vocabulary.json\'))\n        model = CSAPatchCaptioner(len(words), ck[\'model\'][\'core.nouns\'], ck[\'model\'][\'core.verbs\'])\n        model.load_state_dict(ck[\'model\'], strict=True)\n        if sum(p.numel() for p in model.parameters()) != 53183736:\n            raise ValueError(\'Captioner architecture differs\')\n        model = model.to(device).train()\n        data = FullTrain(cfg[\'train_roots\'])\n        rewarder = load_reward(cfg[\'train_roots\'][0])\n        if set(map(int, np.concatenate(data.ids))) != set(rewarder.references):\n            raise ValueError(\'Cached TRAIN reward and feature cohorts differ\')\n\n        class Policy(nn.Module):\n            def __init__(self, captioner):\n                super().__init__()\n                self.captioner = captioner\n            def forward(self, patches, prompts):\n                return train_beams_fast(self.captioner, self.captioner.encode(patches, prompts))\n\n        if mode == \'preflight\':\n            guard = mechanism_guard(device)\n            # Real model eval path agrees with the existing decoder beam policy.\n            from patch_beam import beam_search\n            x, prompt, _ = data.batch([0], device)\n            model.eval()\n            with torch.no_grad():\n                memory = model.encode(x, prompt)\n                seq, logps = train_beams_fast(model, memory)\n                old, scores = beam_search(model, memory)\n                padded = torch.cat((old, old.new_zeros((1, 5, seq.shape[-1]-old.shape[-1]))), -1)\n                torch.testing.assert_close(seq, padded, atol=0, rtol=0)\n                torch.testing.assert_close(logps.sum(-1), scores, atol=3e-4, rtol=1e-4)\n            del x, prompt, memory, seq, logps, old, padded, scores\n            model.train()\n        else:\n            guard = {}\n        wrapped = nn.parallel.DistributedDataParallel(Policy(model), device_ids=[rank],\n            broadcast_buffers=False, find_unused_parameters=True)\n        optimizer = torch.optim.Adam(wrapped.parameters(), lr=4e-6, betas=(.9, .99), eps=1e-8, weight_decay=0)\n        cursor, updates = 0, PARENT_STEPS\n        prior = ck\n        resume_receipt = None\n        if cfg.get(\'resume_root\'):\n            root = Path(cfg[\'resume_root\'])\n            resume_receipt = read(root/\'checkpoint_receipt.json\')\n            validate_resume(resume_receipt, identity)\n            if sha(root/\'checkpoint.bin\') != resume_receipt[\'sha256\']:\n                raise ValueError(\'Changed epoch2 resume binary\')\n            prior = torch.load(root/\'checkpoint.bin\', map_location=\'cpu\', weights_only=False)\n            for key in (\'identity\',\'updates\',\'cursor_images\',\'epoch\',\'parent_sha256\',\'source_sha256\'):\n                if prior[key] != resume_receipt[key]:raise ValueError(\'Resume binary/receipt differ: \'+key)\n            cursor, updates = prior[\'cursor_images\'], prior[\'updates\']\n        model.load_state_dict(prior[\'model\'], strict=True)\n        optimizer.load_state_dict(prior[\'optimizer\'])\n        for group in optimizer.param_groups:\n            if (group[\'lr\'] != 4e-6 or tuple(group[\'betas\']) != (.9,.99)\n                    or group[\'eps\'] != 1e-8 or group[\'weight_decay\'] != 0):\n                raise ValueError(\'Changed parent Adam settings\')\n        if not optimizer.state:raise ValueError(\'Missing Adam moments\')\n        steps = [float(state[\'step\']) for state in optimizer.state.values()]\n        if not all(math.isfinite(x) and 0 < x <= updates for x in steps):\n            raise ValueError(\'Invalid Adam step counters\')\n        state = prior[\'rng\'][rank]\n        torch.set_rng_state(state[\'cpu\'])\n        torch.cuda.set_rng_state(state[\'cuda\'], device)\n        np.random.set_state(state[\'numpy\'])\n        if not torch.equal(torch.get_rng_state(),state[\'cpu\']) or not torch.equal(torch.cuda.get_rng_state(device),state[\'cuda\']):\n            raise ValueError(\'RNG restore failed\')\n        save(out/f\'continuity_rank{rank}.json\',dict(parent_version=PARENT_VERSION,parent_sha256=PARENT_SHA,\n            epoch=2,cursor_images=cursor,updates=updates,optimizer_loaded=True,optimizer_states=len(steps),\n            Adam_step_min=min(steps),Adam_step_max=max(steps),rank_RNG_restored=True,\n            preflight_only=mode==\'preflight\',resume=resume_receipt))\n        del state, prior, ck\n        save(out/f\'identity_rank{rank}.json\', identity)\n\n        def checkpoint_now():\n            rng = dict(cpu=torch.get_rng_state(), cuda=torch.cuda.get_rng_state(device), numpy=np.random.get_state())\n            states = [None, None]\n            dist.all_gather_object(states, rng)\n            if rank == 0:\n                target = out/\'checkpoint.bin\'\n                temp = out/\'checkpoint.tmp\'\n                torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                    updates=updates, cursor_images=cursor, source_sha256=SOURCE_SHA,\n                    identity=identity, epoch=2, parent_sha256=PARENT_SHA, rng=states), temp)\n                os.replace(temp, target)\n                save(out/\'checkpoint_receipt.json\', dict(sha256=sha(target), updates=updates,\n                    cursor_images=cursor, source_sha256=SOURCE_SHA, identity=identity,\n                    epoch=2, parent_sha256=PARENT_SHA, complete_epoch=cursor == IMAGES, test_used=False))\n            dist.barrier(device_ids=[rank])\n\n        def update(position, guard_gradients=False):\n            real = min(32, IMAGES-position)\n            optimizer.zero_grad(set_to_none=True)\n            summaries = torch.zeros(5, dtype=torch.float64, device=device)\n            for micro in range(4):\n                rows, keep = rank_microbatch(position, rank, micro)\n                weights = torch.tensor(keep, dtype=torch.float32, device=device)\n                x, prompt, ids = batch_rows(data, rows, device)\n                sync = wrapped.no_sync() if micro < 3 else nullcontext()\n                with sync:\n                    seq, logps = wrapped(x, prompt)\n                    captions = [text_from_ids(tokens, words) for tokens in seq.detach().reshape(20, 21).cpu().tolist()]\n                    ids5 = [image for image in ids for _ in range(5)]\n                    rewards = torch.from_numpy(rewarder.score(ids5, captions).reshape(4, 5)).to(device)\n                    if not torch.isfinite(rewards).all():\n                        raise FloatingPointError(\'Nonfinite CIDEr reward\')\n                    per_image = image_losses(logps, rewards)\n                    loss = (per_image*weights).sum()*(2/real)\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError(\'Nonfinite beam SCST loss\')\n                    loss.backward()\n                summaries[0] += (per_image.detach()*weights).sum()\n                summaries[1] += (rewards.mean(-1)*weights).sum()\n                summaries[2] += ((rewards-rewards.mean(-1, keepdim=True)).abs().mean(-1)*weights).sum()\n                summaries[3] += weights.sum()\n                del x, prompt, seq, logps, loss, rewards, per_image\n            norm = nn.utils.clip_grad_norm_(wrapped.parameters(), float(\'inf\'), error_if_nonfinite=True)\n            if guard_gradients:\n                for prefix in (\'core\', \'object_prompt\', \'relation_prompt\'):\n                    grads = [p.grad for n,p in model.named_parameters() if n.startswith(prefix) and p.grad is not None]\n                    if not grads or not all(torch.isfinite(g).all() for g in grads) or not any(torch.count_nonzero(g) for g in grads):\n                        raise ValueError(\'Missing finite/nonzero gradient group: \'+prefix)\n            optimizer.step()\n            summaries[4] = float(norm)\n            dist.all_reduce(summaries)\n            return summaries.cpu().tolist()\n\n        torch.cuda.reset_peak_memory_stats(device)\n        if mode == \'preflight\':\n            # Probe all three cache shards; none of these updates initializes training.\n            for position in (0, 32):\n                update(position)\n            timings = []\n            for position in (64, 40000, 80000):\n                torch.cuda.synchronize(device)\n                timed = time.monotonic()\n                update(position, guard_gradients=position == 64)\n                torch.cuda.synchronize(device)\n                timings.append(time.monotonic()-timed)\n            update(IMAGES-7)\n            elapsed = torch.tensor(max(timings), device=device)\n            dist.all_reduce(elapsed, op=dist.ReduceOp.MAX)\n            projected = float(elapsed)*STEPS\n            save(out/f\'beam_preflight_rank{rank}.json\', dict(passed=True, mechanism=guard,\n                real_eval_paths_match=True, real_gradient_groups_checked=[\'core\',\'object_prompt\',\'relation_prompt\'],\n                actual_tail7_update=True, discarded_optimizer_steps=6, shard_timing_seconds=timings,\n                seconds_per_global32_update=float(elapsed), projected_training_seconds=projected,\n                peak_allocated_bytes=torch.cuda.max_memory_allocated(device),\n                trainable_parameters=53183736, training_initializes_from_discarded_guard=False,\n                not_a_quota_or_completion_guarantee=True))\n            if projected > 14*3600:\n                raise RuntimeError(\'Measured full-epoch projection exceeds14h; preserve preflight for review, no full train\')\n            return\n\n        if mode == \'train\':checkpoint_now()\n        while cursor < IMAGES:\n            stop = torch.tensor(int(time.monotonic() > deadline-180), device=device)\n            dist.all_reduce(stop, op=dist.ReduceOp.MAX)\n            if int(stop):\n                checkpoint_now()\n                raise TimeoutError(\'Epoch partial; exact checkpoint/RNG/cursor preserved, resume without restart\')\n            stats = update(cursor)\n            cursor = min(IMAGES, cursor+32)\n            updates += 1\n            if updates % 25 == 0 or cursor == IMAGES:\n                if rank == 0:\n                    row = dict(updates=updates, cursor_images=cursor, SCST=stats[0]/stats[3],\n                        mean_beam_reward=stats[1]/stats[3], mean_absolute_advantage=stats[2]/stats[3],\n                        last_real_images=int(stats[3]), mean_grad_norm=stats[4]/2, seconds=time.monotonic()-start)\n                    with (out/\'training.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                        stream.write(json.dumps(row, allow_nan=False)+\'\\n\')\n                    print(\'Beam SCST\', row, flush=True)\n            if updates % 100 == 0 or cursor == IMAGES:\n                checkpoint_now()\n        if updates != END_STEPS:\n            raise ValueError(\'Epoch2 endpoint must reach cumulative7082 updates\')\n        if rank == 0:\n            # Existing VALID evaluator accepts this weights-only, receipt-sealed export.\n            target, tmp = out/\'pilot_checkpoint.bin\', out/\'pilot_checkpoint.tmp\'\n            torch.save(dict(model=model.state_dict(), updates=updates, source_sha256=SOURCE_SHA, epoch=2, parent_sha256=PARENT_SHA), tmp)\n            os.replace(tmp, target)\n            save(out/\'pilot_checkpoint_receipt.json\', dict(sha256=sha(target), updates=updates,\n                source_sha256=SOURCE_SHA, cursor_images=cursor, epoch=2, parent_sha256=PARENT_SHA, complete_epoch=True, identity=identity))\n            save(out/\'training_complete.json\', dict(complete=True, train_images=cursor, updates=updates,\n                epochs=2, epoch2_image_events=cursor, cumulative_image_events=2*IMAGES, parent_sha256=PARENT_SHA, source_sha256=SOURCE_SHA, test_used=False, reranking=False,\n                seconds=time.monotonic()-start, peak_allocated_bytes=torch.cuda.max_memory_allocated(device)))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'mode\', choices=(\'preflight\',\'train\'))\n    parser.add_argument(\'config\')\n    args = parser.parse_args()\n    run(read(args.config), args.mode)\n', 'speed_candidate.py': '"""Same-shape cached input batching and deferred finite-logit synchronization."""\nimport numpy as np\n\n\ndef batch_rows(data, rows, device):\n    import torch\n    rows=np.asarray(rows,dtype=np.int64)\n    if rows.ndim!=1 or len(rows)==0 or rows.min()<0 or rows.max()>=data.stops[-1]:\n        raise ValueError(\'Invalid cached image indices\')\n    parts=np.searchsorted(data.stops,rows,side=\'right\')\n    patches=[];prompts=[];ids=[];positions=[]\n    for part in np.unique(parts):\n        loc=np.flatnonzero(parts==part)\n        offsets=rows[loc]-(data.stops[part-1] if part else 0)\n        patches.append(np.array(data.patches[part][offsets],dtype=np.float32,copy=True))\n        ids.extend(map(int,data.ids[part][offsets]))\n        prompts.append(data.context[part].images(offsets,device))\n        positions.extend(loc.tolist())\n    if len(patches)==1:\n        return torch.from_numpy(patches[0]).to(device),prompts[0],ids\n    order=np.argsort(positions)\n    x=torch.from_numpy(np.concatenate(patches,axis=0)[order]).to(device)\n    index=torch.from_numpy(order).to(device)\n    prompt={key:torch.cat([p[key] for p in prompts]).index_select(0,index) for key in prompts[0]}\n    return x,prompt,[ids[i] for i in order]\n\n\ndef train_beams_fast(model, memory, width=5, maximum=20):\n    import torch\n    if width<2 or maximum<1:raise ValueError(\'Need at least two beams and one action\')\n    b,n,d=memory.shape\n    vocab,device=model.head.out_features,memory.device\n    if width>vocab-2:raise ValueError(\'Beam width exceeds valid vocabulary\')\n    seq=torch.ones((b,width,1),dtype=torch.long,device=device)\n    scores=torch.full((b,width),-torch.inf,device=device);scores[:,0]=0\n    done=torch.zeros((b,width),dtype=torch.bool,device=device)\n    history=memory.new_zeros((b,width,0))\n    expanded=memory[:,None].expand(-1,width,-1,-1).reshape(b*width,n,d)\n    forbidden=torch.arange(vocab,device=device)<2\n    finite=torch.ones((),dtype=torch.bool,device=device)\n    finished=None\n    for _ in range(maximum):\n        logits=model.decode(expanded,seq.reshape(b*width,-1))[:,-1].float()\n        finite.logical_and_(torch.isfinite(logits).all())\n        logp=logits.log_softmax(-1).view(b,width,vocab)\n        logp=logp.masked_fill(forbidden,-torch.inf)\n        if finished is None:\n            finished=torch.full_like(logp,-torch.inf);finished[:,:,0]=0\n        logp=torch.where(done[:,:,None],finished,logp)\n        scores,flat=(scores[:,:,None]+logp.detach()).flatten(1).topk(width,dim=1)\n        parent,token=flat//vocab,flat%vocab\n        chosen=logp.gather(1,parent[:,:,None].expand(-1,-1,vocab)).gather(2,token[:,:,None]).squeeze(-1)\n        history=torch.cat((history.gather(1,parent[:,:,None].expand(-1,-1,history.shape[-1])),chosen[:,:,None]),-1)\n        seq=torch.cat((seq.gather(1,parent[:,:,None].expand(-1,-1,seq.shape[-1])),token[:,:,None]),-1)\n        done=done.gather(1,parent)|token.eq(2)\n        if done.all():break\n    # Check every visited logit, including masked/unselected values, before backward.\n    if not finite:raise FloatingPointError(\'Nonfinite beam logits\')\n    if history.shape[-1]<maximum:\n        padding=maximum-history.shape[-1]\n        history=torch.cat((history,history.new_zeros((b,width,padding))),-1)\n        seq=torch.cat((seq,seq.new_zeros((b,width,padding))),-1)\n    if not torch.isfinite(history).all():raise FloatingPointError(\'Nonfinite selected beam log probabilities\')\n    return seq,history\n', 'PARENT_CHECKPOINT_RECEIPT.json': '{\n  "sha256": "5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c",\n  "updates": 3541,\n  "cursor_images": 113287,\n  "source_sha256": "a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5",\n  "identity": {\n    "method": "beam_mean_scst_epoch1_v1",\n    "source_sha256": "a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5",\n    "train_images": 113287,\n    "global_batch": 32,\n    "local_microbatch": 4,\n    "accumulation": 4,\n    "beams": 5,\n    "maximum": 20,\n    "denominator": 20,\n    "baseline": "image_mean_of_five_beam_rewards",\n    "mode": "train",\n    "CE_weight": 0,\n    "gradient_clipping": false,\n    "optimizer": "Adam",\n    "learning_rate": 4e-06,\n    "betas": [\n      0.9,\n      0.99\n    ],\n    "eps": 1e-08,\n    "weight_decay": 0,\n    "cache_manifest_sha256": "dc33aba97298104c2af37d7d8cd6ca05f40739063d0acfcd6820214f2d2e9417",\n    "source_hashes": {\n      "TEST199_LOCK.json": "a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3",\n      "TEST199_PROTOCOL.md": "2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988",\n      "TEST199_BUILD_V2_PROTOCOL.md": "7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8",\n      "SMOKE_SOURCE.json": "6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49",\n      "test199_data.py": "0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4",\n      "run_test199.py": "3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3",\n      "test_test199.py": "ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be",\n      "patch_model.py": "dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70",\n      "patch_beam.py": "53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be",\n      "patch_data.py": "3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5",\n      "full_data.py": "ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74",\n      "make_test199_notebook.py": "d2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64",\n      "data.py": "5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71",\n      "real_bank_data.py": "2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0",\n      "cuda_runtime_guard.py": "640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773",\n      "saved_quality_data.py": "f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e",\n      "score_prompt_test_extra.py": "8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7",\n      "semantic_test_data.py": "a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc",\n      "semantic_data.py": "a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540",\n      "semantic_patch_model.py": "10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3",\n      "test_semantic_test.py": "fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951",\n      "BANK_PROMPT_TEST_PROTOCOL.md": "56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f",\n      "bank_layers.py": "139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09",\n      "PROMPT_TEST_PREPARATION_RESULT.json": "d99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205",\n      "csa_patch_model.py": "99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220",\n      "memory_fusion.py": "a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087",\n      "csa_per_caption_test_data.py": "6338f2a4018143d45feedaf4356e9c51d7c90b2d8ec6c4c2e8881ae26ec2d759",\n      "run_csa_per_caption_test.py": "657bb4e60fdb5647e9130a74ca7607cd12249ee79c472e5e5f3ad04e95484a28",\n      "scst_cache.py": "9721401031272f50cd5c99b177d6177f33b0805cabde19eb936dbb622086dd0f",\n      "scst_dataset_mount.py": "e4574bc17a36f4d6b8fb51ad8fbc725db4269559d0a05f34a7d5ba1354ca35fb",\n      "captioning/scst.py": "3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e",\n      "captioning/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n      "CACHE_PARTS.json": "50632a1ea657604915fdfa9396320d5c1649df408a28b6b8003b0eab601f3519",\n      "beam_policy.py": "1f2ec8d95c38f91793a2b554cd6f585034871ccbb95d0ed5ad9521206b6ddf67",\n      "full_epoch_data.py": "6cf0ebbd2d9d8740aabef7f8fbe9fdc8a88263b2447a64ab43953deecb31f719",\n      "run_beam_epoch.py": "79027395920ab9e252ef4f29682efd14f4e84239e11bafc7695f51f23c832b10",\n      "PROTOCOL.md": "9b0f373fdc778cb1e56339716569d7674b67ba0eebff990fd929bc729297c2ac",\n      "checkpoint_input.py": "55e2d57313ef3e51dc3c3106c938db2ea7a44f058b0687580e5376e3117e004a"\n    },\n    "vocabulary_sha256": "cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd",\n    "torch": "2.11.0+cu128",\n    "numpy": "2.1.3",\n    "seed": 4342,\n    "XE_seed": 43,\n    "order": "original sealed cache, once, no shuffle/replacement",\n    "precision": "FP32",\n    "TF32": false\n  },\n  "complete_epoch": true,\n  "test_used": false\n}\n', 'epoch2_contract.py': '"""Metadata-only epoch transition and exact continuation contracts."""\nimport math\n\nPARENT_VERSION = \'ducanh3/notebook7bc83c4c76/199\'\nPARENT_SHA = \'5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c\'\nXE_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nIMAGES, GLOBAL_BATCH, PARENT_STEPS, END_STEPS = 113287, 32, 3541, 7082\n\n\ndef validate_parent(state, expected):\n    if (state[\'identity\'] != expected[\'identity\'] or state[\'updates\'] != PARENT_STEPS\n            or state[\'cursor_images\'] != IMAGES or state[\'source_sha256\'] != XE_SHA\n            or not expected[\'complete_epoch\'] or expected[\'sha256\'] != PARENT_SHA):\n        raise ValueError(\'Need complete exact epoch1/v199 state\')\n    if len(state[\'rng\']) != 2 or not {\'model\', \'optimizer\'} <= state.keys():\n        raise ValueError(\'Need full model/Adam/two-rank RNG, not pilot weights\')\n    for rng in state[\'rng\']:\n        if not {\'cpu\', \'cuda\', \'numpy\'} <= rng.keys():\n            raise ValueError(\'Missing rank RNG state\')\n\n\ndef make_identity(cfg, expected, torch_version, numpy_version, vocabulary_sha):\n    old = expected[\'identity\']\n    if (torch_version != old[\'torch\'] or numpy_version != old[\'numpy\']\n            or vocabulary_sha != old[\'vocabulary_sha256\']\n            or cfg[\'cache_manifest_sha256\'] != old[\'cache_manifest_sha256\']):\n        raise ValueError(\'Changed software, vocabulary or cache; no fallback\')\n    return dict(method=\'beam_mean_scst_epoch2_speed_v1\', epoch=2, seed=43,\n                parent_version=PARENT_VERSION, parent_sha256=PARENT_SHA, source_sha256=XE_SHA,\n                parent_identity=old, source_hashes=cfg[\'source_hashes\'],\n                train_images=IMAGES, cumulative_image_events=2*IMAGES,\n                parent_updates=PARENT_STEPS, endpoint_updates=END_STEPS,\n                order=\'same original sealed TRAIN order, once per epoch\',\n                optimization=\'audited batched prompt transfer/deferred finite-logit check\')\n\n\ndef validate_resume(receipt, identity):\n    cursor, step = receipt[\'cursor_images\'], receipt[\'updates\']\n    if (receipt[\'identity\'] != identity or receipt[\'epoch\'] != 2\n            or receipt[\'parent_sha256\'] != PARENT_SHA or receipt[\'source_sha256\'] != XE_SHA\n            or type(cursor) is not int or type(step) is not int or not 0 <= cursor <= IMAGES\n            or step != PARENT_STEPS+math.ceil(cursor/GLOBAL_BATCH)):\n        raise ValueError(\'Foreign epoch2 recipe, lineage or cursor\')\n    if cursor != IMAGES and cursor != (step-PARENT_STEPS)*GLOBAL_BATCH:\n        raise ValueError(\'Resume cursor is not an epoch2 optimizer boundary\')\n    if receipt[\'complete_epoch\'] != (cursor == IMAGES):\n        raise ValueError(\'Wrong epoch2 completeness\')\n\n\ndef validate_eval_binding(binding):\n    if (binding[\'epoch\'] != 2 or binding[\'updates\'] != END_STEPS or binding[\'seed\'] != 43\n            or binding[\'cursor_images\'] != IMAGES or binding[\'parent_sha256\'] != PARENT_SHA\n            or binding[\'source_sha256\'] != XE_SHA or not binding[\'complete_epoch\']):\n        raise ValueError(\'Refuse partial/foreign endpoint for TEST\')\n    digest = binding[\'sha256\']\n    if (len(digest) != 64 or any(c not in \'0123456789abcdef\' for c in digest)\n            or digest in (PARENT_SHA, \'3b03acc29f025b07dc358846b30618ae9c83f8634bc12bf7905e9d908b971f24\')):\n        raise ValueError(\'Wrong epoch2 endpoint SHA\')\n    return binding\n', 'EPOCH2_PROTOCOL.md': '# Beam SCST epoch 2 + TEST — quyết định 26/11/2026\n\nNgười dùng chốt chuẩn bị học tiếp epoch 2 và chấm TEST; tài khoản còn 6 giờ GPU. Agent chỉ chuẩn bị và kiểm tra static/metadata/mock, không chạy Kaggle, Torch, mô hình hay scorer local.\n\n## Câu hỏi và cấu hình đã chốt\n\nHọc thêm đúng một lượt 113.287 ảnh TRAIN có tiếp tục cải thiện CIDEr không, và CHAIR thay đổi thế nào? Không hứa tăng tiếp từ kết quả epoch 1. Đây là thử tối ưu tiếp cùng mô hình; không suy riêng đóng góp prompt từ lượt này.\n\n- Khởi đầu duy nhất: full `checkpoint.bin` của `ducanh3/notebook7bc83c4c76/199`, SHA256 `5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c`. Giữ model, Adam/moments/step và CPU/CUDA/NumPy RNG hai rank. Không dùng weights-only pilot hay output TEST200/benchmark.\n- Chuyển rõ từ epoch 1 đã đủ 3.541 update/cursor113.287 sang epoch 2 cursor0, cumulative update3.541. Endpoint epoch2: cumulative7.082 update, 226.574 image events; riêng epoch2 đúng113.287 ảnh không thay thế, thứ tự cache gốc như epoch1. Không shuffle mới, reset Adam hoặc reseed sau khôi phục RNG.\n- Giữ beam5/train-mode/mean reward baseline/20-token denominator, pure SCST CIDEr-D TRAIN, FP32/noTF32, T4×2 DDP, local4×accum4/global32, Adam4e-6/betas(.9,.99), không CE/clip gradients/AMP/compile/feature sharing. Dùng đúng hai tối ưu đã vượt guard: batched prompt transfer và deferred finite-logit checks; giữ finite loss/reward/grad guards.\n- Reuse cache82/83/84 đã đóng Dataset và metadata/VALID; không trích feature hay copy lại full patch arrays. Giữ nguyên model/core/M3/reward/decoder. Preflight6 update bỏ, sau đó process train riêng khôi phục lại trạng thái gốc. Giữ checkpoint atomic định kỳ100 update và tại deadline, identity epoch/cursor rõ ràng để resume chính xác.\n\n## Đánh giá và giới hạn\n\nEndpoint cố định cuối epoch 2; không chọn checkpoint dựa vào TEST. Chấm VALID5000 top1 bằng đường đã chạy của epoch1 để so đúng cùng seed43. Sau đó TEST5000 sinh beam5 một lần, chấm top1 và rerank cố định 0,3decoder/0,4CLIP/0,3object, đủ BLEU1/4, METEOR, ROUGE-L, CIDEr, SPICE, CHAIRs/i, thêm Recall/NLL. Báo cả hai chính sách dù bất lợi; so với kết quả epoch1 TEST output200 và CE179 cùng seed đã lưu. Không tìm trọng số theo TEST, không tự epoch3.\n\nTEST đã được dự án dùng trong phát triển; ghi rõ đây là đánh giá lặp lại, không gọi là holdout hoàn toàn chưa xem. CHAIR đo lỗi đối tượng, không chứng minh thuộc tính/quan hệ đúng. Một seed SCST chưa xác lập độ ổn định.\n\nHai namespace source riêng cho train/VALID và TEST để tránh csa_per_caption_test_data bị cache nhầm split. TEST checkpoint seal được tạo chỉ sau khi full epoch2 hoàn tất, dựa trên SHA thực tế export/receipt và kiểm tra lineage epoch2/7082/CE179/parent199. Không dùng SHA epoch1 cố định cho trọng số mới.\n\nCap mặc định5,75 giờ wall/phiên, không phải ETA hay cách tính GPU quota. Dự phòng100 phút sau train cho VALID/TEST/scorer; epoch2 train dự kiến gần3giờ từ speed sample, không bảo đảm đủ ngân sách. Nếu train hết cap, giữ full state và dừng, không chấm partial checkpoint. Nếu scorer lỗi/hết giờ, giữ caption/beam/CLIPscores/selection/basic metrics và weights để recovery CPU; không train/decode lại tự động. Notebook hỗ trợ resume bằng toàn bộ output epoch2 được gắn và kiểm tra identity/SHA; nếu chỉ có Save Version, cần source mới xác nhận để làm API resume riêng.\n\n## Dữ liệu cần gắn\n\n1. `ducanh3/csa-scst-full-karpathy-cache-v1`\n2. `vuthetam/mscoco-2014`\n3. `ducanh2403/visual-cache`\n4. `ducanh2403/csa-prompt-test-cache-v181`\n5. `ducanh2403/objectdetectionecache`\n\nInternet và Secret `codex` hoặc `KAGGLE_API_TOKEN` có quyền tải nguồn ducanh3. Không cần gắn output199 riêng; API tải đúng version/path và SHA trước setup mô hình.\n\n## Đã chuẩn bị và kiểm tra, 26/11/2026\n\n[Notebook](BEAM_SCST_EPOCH2_FROM_V199_VALID_TEST_RERANK_2GPU.ipynb), [source manifest](SOURCE_MANIFEST.json), [9 kiểm tra host](CHECKS.json). Bản sao ở Downloads được xác minh cùng SHA. Các cell và80 payload compile/hash PASS; closure các import runtime được kiểm tra riêng hai namespace. Metadata transition/resume/tail7 và endpoint binding có trường hợp từ chối sai nguồn/partial; AST update giữ nguyên toán học của epoch1 ngoài batch helper đã kiểm chứng. Model/reward/inference/scorer/selector và speed candidate giữ đúng bytes nguồn đã chạy.\n\nMô phỏng API acquisition chỉ nhận đúng v199 full checkpoint; không gọi API thật. Mô phỏng pipeline dùng file fixture (không phải weights/caption/metric thực): đường thành công tới TEST comparison, partial/timeout train chặn TEST, timeout scorer giữ weights và beam fixture. Không import Torch hoặc metric scorer local. Actual API authorization, load binary/Adam/RNG/CUDA, scorer assets và chất lượng/thời gian toàn lượt chưa được agent chạy.\n\nKiểm tra Torch/NumPy phiên Kaggle ngay sau CUDA guard, trước setup scorer/cache; không tự cài lại runtime khi khác nguồn v199. Java/scorer smoke chạy trước lượt huấn luyện chính để phát hiện lỗi môi trường sớm. Không có kết quả thực nghiệm mới, không ghi dòng metric mới vào Sheet hoặc sửa báo cáo Word chỉ vì đã chuẩn bị notebook.\n'}
TEST_PAYLOADS={'TEST199_LOCK.json': '{\n  "schema": "clip_patch_199_test_lock_v1",\n  "source_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/199",\n  "checkpoint_sha256": "5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7",\n  "training_commit": "9aaddf5694f72373b8ab2482fafe7d7992abbe5e",\n  "epoch": 10,\n  "updates": 177110,\n  "beam_width": 5,\n  "max_new_tokens": 20,\n  "length_normalization": false,\n  "reranking": false,\n  "expected_test_ids": [\n    42,\n    359,\n    636,\n    711,\n    810,\n    974,\n    1146,\n    1180,\n    1290,\n    1353,\n    1393,\n    1425,\n    1448,\n    1573,\n    1590,\n    1626,\n    1655,\n    1682,\n    1700,\n    1757,\n    2142,\n    2225,\n    2240,\n    2295,\n    2453,\n    2690,\n    2753,\n    2759,\n    2822,\n    2867,\n    2890,\n    3001,\n    3103,\n    3156,\n    3382,\n    3590,\n    3716,\n    3832,\n    4069,\n    4125,\n    4246,\n    4312,\n    4359,\n    4760,\n    4916,\n    4972,\n    5123,\n    5154,\n    5205,\n    5388,\n    5758,\n    6040,\n    6393,\n    6415,\n    6484,\n    6861,\n    6896,\n    6921,\n    7072,\n    7088,\n    7278,\n    7288,\n    7519,\n    7556,\n    7593,\n    7784,\n    7795,\n    8179,\n    8180,\n    8218,\n    8277,\n    8483,\n    8493,\n    8612,\n    8775,\n    8803,\n    8888,\n    9002,\n    9041,\n    9156,\n    9170,\n    9217,\n    9236,\n    9381,\n    9548,\n    9647,\n    9699,\n    9727,\n    9791,\n    9801,\n    9807,\n    9914,\n    10014,\n    10104,\n    10115,\n    10290,\n    10400,\n    10428,\n    10526,\n    10684,\n    10707,\n    10764,\n    10825,\n    10837,\n    10935,\n    11081,\n    11099,\n    11115,\n    11156,\n    11181,\n    11184,\n    11198,\n    11209,\n    11276,\n    11625,\n    12085,\n    12115,\n    12204,\n    12471,\n    12744,\n    12817,\n    12896,\n    12927,\n    13220,\n    13466,\n    13490,\n    13525,\n    13882,\n    13985,\n    14056,\n    14175,\n    14257,\n    14338,\n    14392,\n    14450,\n    14473,\n    14713,\n    14723,\n    14781,\n    15017,\n    15050,\n    15254,\n    15260,\n    15345,\n    15667,\n    16005,\n    16241,\n    16249,\n    16491,\n    16521,\n    16599,\n    16631,\n    16848,\n    16928,\n    16931,\n    17115,\n    17178,\n    17313,\n    17756,\n    17769,\n    17905,\n    17927,\n    17953,\n    18014,\n    18412,\n    18480,\n    18519,\n    18687,\n    18770,\n    18771,\n    18975,\n    19158,\n    19167,\n    19308,\n    19443,\n    19455,\n    19888,\n    19904,\n    20111,\n    20179,\n    20254,\n    20268,\n    20276,\n    20305,\n    20381,\n    20470,\n    20598,\n    20913,\n    20992,\n    21198,\n    21202,\n    21284,\n    21310,\n    21327,\n    21498,\n    21864,\n    21900,\n    21979,\n    22118,\n    22158,\n    22167,\n    22176,\n    22192,\n    22213,\n    22371,\n    22759,\n    22850,\n    22892,\n    22929,\n    23034,\n    23117,\n    23320,\n    23351,\n    23731,\n    23802,\n    23840,\n    24207,\n    24223,\n    24343,\n    24553,\n    24734,\n    24805,\n    25014,\n    25144,\n    25181,\n    25286,\n    25316,\n    25386,\n    25521,\n    25550,\n    25603,\n    25747,\n    25758,\n    25860,\n    25989,\n    26051,\n    26209,\n    26584,\n    26609,\n    26654,\n    26665,\n    26671,\n    26730,\n    26762,\n    26942,\n    27065,\n    27186,\n    27226,\n    27440,\n    27570,\n    27585,\n    27620,\n    27656,\n    27778,\n    27871,\n    28114,\n    28156,\n    28157,\n    28293,\n    28377,\n    28452,\n    28463,\n    28506,\n    28532,\n    28582,\n    28655,\n    28675,\n    28688,\n    28790,\n    28998,\n    29045,\n    29056,\n    29094,\n    29160,\n    29465,\n    29472,\n    29577,\n    29656,\n    30034,\n    30143,\n    30455,\n    30470,\n    30549,\n    30607,\n    30667,\n    30685,\n    30785,\n    30793,\n    30925,\n    30954,\n    31000,\n    31016,\n    31024,\n    31176,\n    31281,\n    31390,\n    31442,\n    31606,\n    31636,\n    31667,\n    31965,\n    31981,\n    32334,\n    32400,\n    32524,\n    32570,\n    32587,\n    32688,\n    32712,\n    32811,\n    32901,\n    32909,\n    32990,\n    33052,\n    33144,\n    33368,\n    33638,\n    33645,\n    33647,\n    33672,\n    33753,\n    33759,\n    33830,\n    33845,\n    33871,\n    33904,\n    33990,\n    34080,\n    34115,\n    34126,\n    34180,\n    34428,\n    34482,\n    34617,\n    34662,\n    34811,\n    34900,\n    34904,\n    35012,\n    35105,\n    35195,\n    35498,\n    35738,\n    35770,\n    35807,\n    35825,\n    35853,\n    35975,\n    36082,\n    36238,\n    36478,\n    36501,\n    36598,\n    36661,\n    37367,\n    37705,\n    37729,\n    37882,\n    38048,\n    38210,\n    38332,\n    38479,\n    38662,\n    38666,\n    38678,\n    38938,\n    39009,\n    39068,\n    39081,\n    39152,\n    39202,\n    39430,\n    39484,\n    39516,\n    39589,\n    39606,\n    39671,\n    39697,\n    39731,\n    39844,\n    39914,\n    40446,\n    40621,\n    40635,\n    40924,\n    40946,\n    41077,\n    41119,\n    41397,\n    41839,\n    41867,\n    42055,\n    42070,\n    42091,\n    42190,\n    42201,\n    42408,\n    42526,\n    42944,\n    42956,\n    42970,\n    43202,\n    43289,\n    43613,\n    43635,\n    43657,\n    43680,\n    44112,\n    44244,\n    44328,\n    44520,\n    44524,\n    44642,\n    44910,\n    44934,\n    45057,\n    45108,\n    45208,\n    45356,\n    45383,\n    45434,\n    45467,\n    45550,\n    45587,\n    45627,\n    45644,\n    45710,\n    45775,\n    45966,\n    46011,\n    46077,\n    46099,\n    46269,\n    46331,\n    46432,\n    46433,\n    46497,\n    46571,\n    46761,\n    46775,\n    46847,\n    47394,\n    47511,\n    47596,\n    47632,\n    47651,\n    47658,\n    47837,\n    47882,\n    47886,\n    47938,\n    48129,\n    48332,\n    48504,\n    48530,\n    48546,\n    48668,\n    48731,\n    48743,\n    48905,\n    48924,\n    49014,\n    49097,\n    49255,\n    49688,\n    49731,\n    49893,\n    49984,\n    50006,\n    50025,\n    50159,\n    50201,\n    50230,\n    50309,\n    50350,\n    50411,\n    50443,\n    50482,\n    51089,\n    51223,\n    51250,\n    51335,\n    51484,\n    51540,\n    51576,\n    51773,\n    51961,\n    52017,\n    52020,\n    52155,\n    52282,\n    52386,\n    52591,\n    52827,\n    53267,\n    53454,\n    53465,\n    53491,\n    53642,\n    53892,\n    54340,\n    54389,\n    54490,\n    54513,\n    54516,\n    54593,\n    54679,\n    54899,\n    54931,\n    54959,\n    55002,\n    55010,\n    55059,\n    55223,\n    55295,\n    55466,\n    55727,\n    55772,\n    55810,\n    56013,\n    56248,\n    56326,\n    56396,\n    56405,\n    56413,\n    56580,\n    56633,\n    56651,\n    56983,\n    57244,\n    57286,\n    57593,\n    57843,\n    58139,\n    58232,\n    58329,\n    58393,\n    58425,\n    58472,\n    58492,\n    58569,\n    58636,\n    58851,\n    59015,\n    59383,\n    59614,\n    59732,\n    59740,\n    60049,\n    60177,\n    60340,\n    60364,\n    60623,\n    60700,\n    60970,\n    61207,\n    61414,\n    61564,\n    61647,\n    61899,\n    62230,\n    62353,\n    62706,\n    62985,\n    63403,\n    63602,\n    63647,\n    63726,\n    64084,\n    64152,\n    64196,\n    64334,\n    64531,\n    64710,\n    65208,\n    65258,\n    65969,\n    65981,\n    66072,\n    66179,\n    66271,\n    66412,\n    66413,\n    66567,\n    66568,\n    66685,\n    66771,\n    66879,\n    67122,\n    67255,\n    67422,\n    67463,\n    67532,\n    67569,\n    67587,\n    67686,\n    67704,\n    67881,\n    68025,\n    68120,\n    68203,\n    68375,\n    68380,\n    68408,\n    68490,\n    68745,\n    68833,\n    68864,\n    68881,\n    69138,\n    69236,\n    69392,\n    69698,\n    69757,\n    69795,\n    69873,\n    69946,\n    70020,\n    70067,\n    70256,\n    70280,\n    70351,\n    70426,\n    71004,\n    71072,\n    71090,\n    71171,\n    71281,\n    71596,\n    71687,\n    71929,\n    72011,\n    72052,\n    72077,\n    72338,\n    72368,\n    72397,\n    72507,\n    72873,\n    73119,\n    73256,\n    73333,\n    73521,\n    73527,\n    73702,\n    74181,\n    74209,\n    74369,\n    74460,\n    74465,\n    74478,\n    74711,\n    74733,\n    74759,\n    74789,\n    75055,\n    75162,\n    75173,\n    75479,\n    75591,\n    75595,\n    75600,\n    75888,\n    75932,\n    75990,\n    76155,\n    76292,\n    76329,\n    76529,\n    76738,\n    76765,\n    76962,\n    77181,\n    77282,\n    77360,\n    77402,\n    77415,\n    77689,\n    77754,\n    77963,\n    77980,\n    78035,\n    78056,\n    78420,\n    78565,\n    78707,\n    78748,\n    78959,\n    78979,\n    79139,\n    79369,\n    79392,\n    79545,\n    79686,\n    79816,\n    79860,\n    79868,\n    79975,\n    79992,\n    80000,\n    80117,\n    80328,\n    80367,\n    80369,\n    80413,\n    80659,\n    80737,\n    80747,\n    81354,\n    81656,\n    81661,\n    81897,\n    81988,\n    81998,\n    82042,\n    82229,\n    82293,\n    82740,\n    82812,\n    82846,\n    82921,\n    82933,\n    82935,\n    83059,\n    83065,\n    83217,\n    83477,\n    83836,\n    83915,\n    83923,\n    84000,\n    84060,\n    84064,\n    84073,\n    84258,\n    84362,\n    84447,\n    84650,\n    84674,\n    84735,\n    84859,\n    84889,\n    84936,\n    85007,\n    85144,\n    85183,\n    85329,\n    85390,\n    85527,\n    85529,\n    85562,\n    85665,\n    85721,\n    85747,\n    86135,\n    86202,\n    86439,\n    86483,\n    86516,\n    86582,\n    86839,\n    86884,\n    87199,\n    87399,\n    87429,\n    87470,\n    87476,\n    87493,\n    87617,\n    87864,\n    87912,\n    88092,\n    88142,\n    88225,\n    88252,\n    88269,\n    88445,\n    88485,\n    88517,\n    88556,\n    88652,\n    88695,\n    88784,\n    88854,\n    89228,\n    89293,\n    89487,\n    89500,\n    89558,\n    89603,\n    89638,\n    89909,\n    90062,\n    90208,\n    90280,\n    90365,\n    90366,\n    90476,\n    90646,\n    90707,\n    91045,\n    91300,\n    91304,\n    91318,\n    91416,\n    91469,\n    91488,\n    91857,\n    92177,\n    92257,\n    92416,\n    92488,\n    92678,\n    92801,\n    93154,\n    93156,\n    93175,\n    93353,\n    93534,\n    93765,\n    93771,\n    93803,\n    93853,\n    93948,\n    93964,\n    94025,\n    94055,\n    94140,\n    94157,\n    94174,\n    94306,\n    94651,\n    94951,\n    95051,\n    95062,\n    95161,\n    95297,\n    95441,\n    95702,\n    95772,\n    95786,\n    96110,\n    96306,\n    96351,\n    96633,\n    96638,\n    96689,\n    96832,\n    96991,\n    97017,\n    97146,\n    97156,\n    97240,\n    97513,\n    97561,\n    97667,\n    97693,\n    97754,\n    97924,\n    97964,\n    98003,\n    98197,\n    98284,\n    98322,\n    98434,\n    98444,\n    99053,\n    99081,\n    99317,\n    99747,\n    99794,\n    99874,\n    99961,\n    100001,\n    100087,\n    100098,\n    100138,\n    100187,\n    100306,\n    100594,\n    100661,\n    100848,\n    100853,\n    100977,\n    101473,\n    101491,\n    101919,\n    101985,\n    102348,\n    102466,\n    102589,\n    102735,\n    102805,\n    103114,\n    103255,\n    103272,\n    103375,\n    103485,\n    103491,\n    103496,\n    103584,\n    104421,\n    104568,\n    104906,\n    105011,\n    105052,\n    105234,\n    105480,\n    105611,\n    105622,\n    105751,\n    105912,\n    105945,\n    106046,\n    106048,\n    106331,\n    106335,\n    106849,\n    106909,\n    106970,\n    107108,\n    107123,\n    107183,\n    107216,\n    107511,\n    107541,\n    107814,\n    107853,\n    108152,\n    108193,\n    108338,\n    108392,\n    108495,\n    108853,\n    109005,\n    109316,\n    109425,\n    109434,\n    109445,\n    109454,\n    109939,\n    110026,\n    110156,\n    110282,\n    110369,\n    110435,\n    110551,\n    110601,\n    110630,\n    110765,\n    110857,\n    110877,\n    110884,\n    111024,\n    111109,\n    111367,\n    111546,\n    111830,\n    111988,\n    112093,\n    112394,\n    112568,\n    112591,\n    112634,\n    112751,\n    112904,\n    112956,\n    112988,\n    113040,\n    113058,\n    113128,\n    113223,\n    113556,\n    113828,\n    113836,\n    113890,\n    113905,\n    114158,\n    114303,\n    114317,\n    114348,\n    114352,\n    114616,\n    114634,\n    114684,\n    114744,\n    114745,\n    114941,\n    115070,\n    115118,\n    115162,\n    115178,\n    115241,\n    115455,\n    115846,\n    116017,\n    116133,\n    116208,\n    116341,\n    116361,\n    116370,\n    116574,\n    116678,\n    116712,\n    116722,\n    117061,\n    117089,\n    117201,\n    117432,\n    117458,\n    117525,\n    117690,\n    117914,\n    117988,\n    118069,\n    118134,\n    118154,\n    118260,\n    118367,\n    118432,\n    119134,\n    119146,\n    119414,\n    119469,\n    119640,\n    120007,\n    120070,\n    120234,\n    120428,\n    120475,\n    120767,\n    120776,\n    120792,\n    120977,\n    121031,\n    121196,\n    121611,\n    121692,\n    121716,\n    121839,\n    121876,\n    122007,\n    122051,\n    122161,\n    122239,\n    122252,\n    122266,\n    122302,\n    122354,\n    122606,\n    122747,\n    122861,\n    122981,\n    123017,\n    123137,\n    123321,\n    123415,\n    123622,\n    123836,\n    123964,\n    124185,\n    124462,\n    124647,\n    124659,\n    124760,\n    124930,\n    124952,\n    125071,\n    125072,\n    125129,\n    125404,\n    125472,\n    125656,\n    125815,\n    125971,\n    125983,\n    126030,\n    126098,\n    126145,\n    126192,\n    126257,\n    126299,\n    126324,\n    126502,\n    126538,\n    126766,\n    126958,\n    127263,\n    127451,\n    127516,\n    127517,\n    127556,\n    127671,\n    127788,\n    127801,\n    127842,\n    127865,\n    128014,\n    128119,\n    128140,\n    128528,\n    128675,\n    128679,\n    128704,\n    128812,\n    129339,\n    129416,\n    129431,\n    129548,\n    129637,\n    129645,\n    129663,\n    129706,\n    129707,\n    129852,\n    129982,\n    130171,\n    130225,\n    130352,\n    130419,\n    130437,\n    130438,\n    130516,\n    130524,\n    130566,\n    130826,\n    130909,\n    131019,\n    131089,\n    131138,\n    131207,\n    131418,\n    131497,\n    131504,\n    131847,\n    132272,\n    132317,\n    132415,\n    132612,\n    132702,\n    132776,\n    132796,\n    133090,\n    133210,\n    133380,\n    133470,\n    133515,\n    133567,\n    133619,\n    133645,\n    133867,\n    134133,\n    134167,\n    134213,\n    134255,\n    134503,\n    134518,\n    134555,\n    134558,\n    134691,\n    134722,\n    134863,\n    134964,\n    135117,\n    135210,\n    135281,\n    135399,\n    135578,\n    135900,\n    136117,\n    136433,\n    136572,\n    136584,\n    136642,\n    136680,\n    136681,\n    136780,\n    136911,\n    137221,\n    137265,\n    137412,\n    137678,\n    137727,\n    137844,\n    137938,\n    137950,\n    137967,\n    138070,\n    138073,\n    138083,\n    138477,\n    138553,\n    138599,\n    138675,\n    138755,\n    138821,\n    138937,\n    138965,\n    138970,\n    139004,\n    139094,\n    139457,\n    139468,\n    139567,\n    139660,\n    140021,\n    140076,\n    140388,\n    140444,\n    140658,\n    140686,\n    140702,\n    140929,\n    140963,\n    141197,\n    141256,\n    141278,\n    141317,\n    141517,\n    141587,\n    141849,\n    141879,\n    141962,\n    142177,\n    142189,\n    142246,\n    142323,\n    142346,\n    142386,\n    142879,\n    142891,\n    143101,\n    143103,\n    143107,\n    143167,\n    143346,\n    143458,\n    143554,\n    143998,\n    144064,\n    144122,\n    144167,\n    144333,\n    144373,\n    144379,\n    144715,\n    144795,\n    144798,\n    144874,\n    145009,\n    145048,\n    145208,\n    145369,\n    145436,\n    145549,\n    145666,\n    145727,\n    145734,\n    145751,\n    145831,\n    145873,\n    146042,\n    146272,\n    146315,\n    146503,\n    146767,\n    146837,\n    146855,\n    146865,\n    146965,\n    147042,\n    147068,\n    147346,\n    147375,\n    147409,\n    147415,\n    147425,\n    147556,\n    147667,\n    147685,\n    147725,\n    147915,\n    147980,\n    148188,\n    148329,\n    148444,\n    148568,\n    148588,\n    148665,\n    148792,\n    148910,\n    148977,\n    149150,\n    149196,\n    149268,\n    149304,\n    149371,\n    149375,\n    149444,\n    149456,\n    149500,\n    150013,\n    150024,\n    150080,\n    150117,\n    150224,\n    150538,\n    150576,\n    150717,\n    150779,\n    151081,\n    151480,\n    151521,\n    151524,\n    151790,\n    151832,\n    151970,\n    152004,\n    152176,\n    152252,\n    152258,\n    152298,\n    152326,\n    152333,\n    152336,\n    152388,\n    152398,\n    152482,\n    152525,\n    152543,\n    152598,\n    152648,\n    152740,\n    153231,\n    153631,\n    153782,\n    153797,\n    153803,\n    153966,\n    154095,\n    154145,\n    154241,\n    154670,\n    154783,\n    154867,\n    154880,\n    154931,\n    154971,\n    155488,\n    155714,\n    155736,\n    155897,\n    156195,\n    156242,\n    156282,\n    156292,\n    156532,\n    156572,\n    156685,\n    156772,\n    156858,\n    156915,\n    157001,\n    157019,\n    157102,\n    157184,\n    157516,\n    157593,\n    157617,\n    157726,\n    157789,\n    157807,\n    157980,\n    158109,\n    158118,\n    158292,\n    158414,\n    158722,\n    158744,\n    159030,\n    159044,\n    159112,\n    159203,\n    159215,\n    159262,\n    159311,\n    159374,\n    159742,\n    160126,\n    160186,\n    160195,\n    160351,\n    160501,\n    160963,\n    161008,\n    161112,\n    161144,\n    161185,\n    161202,\n    161222,\n    161231,\n    161370,\n    161505,\n    161586,\n    161697,\n    161736,\n    161743,\n    161962,\n    162043,\n    162362,\n    162393,\n    162398,\n    162415,\n    162445,\n    162459,\n    162503,\n    162634,\n    162677,\n    162902,\n    162914,\n    163118,\n    163348,\n    163611,\n    163684,\n    163710,\n    163852,\n    164104,\n    164440,\n    164548,\n    164759,\n    164835,\n    165009,\n    165029,\n    165225,\n    165298,\n    165547,\n    165572,\n    165643,\n    165752,\n    166124,\n    166478,\n    166645,\n    167122,\n    167235,\n    167422,\n    167510,\n    167529,\n    167544,\n    167549,\n    167602,\n    167810,\n    167989,\n    168093,\n    168243,\n    168250,\n    168337,\n    168580,\n    168683,\n    168714,\n    168847,\n    168991,\n    169098,\n    169361,\n    169436,\n    169539,\n    169757,\n    169802,\n    169858,\n    170015,\n    170038,\n    170211,\n    170250,\n    170321,\n    170411,\n    170436,\n    170893,\n    171190,\n    171199,\n    171241,\n    171443,\n    171595,\n    171695,\n    171819,\n    172021,\n    172265,\n    172285,\n    172330,\n    172396,\n    172408,\n    172556,\n    172648,\n    172673,\n    172733,\n    172851,\n    172911,\n    172935,\n    173081,\n    173117,\n    173235,\n    173385,\n    173391,\n    173584,\n    173797,\n    173819,\n    174329,\n    174511,\n    174567,\n    174594,\n    174672,\n    175024,\n    175387,\n    175653,\n    175688,\n    175718,\n    175804,\n    175864,\n    175948,\n    176031,\n    176312,\n    176330,\n    176466,\n    176799,\n    176863,\n    176906,\n    177015,\n    177102,\n    177246,\n    177262,\n    177366,\n    177453,\n    177489,\n    177535,\n    177567,\n    177811,\n    177838,\n    177861,\n    177938,\n    178413,\n    178578,\n    178746,\n    178748,\n    178761,\n    178795,\n    178796,\n    178818,\n    179114,\n    179285,\n    179441,\n    179487,\n    179720,\n    179765,\n    179921,\n    179960,\n    180336,\n    180447,\n    180495,\n    180784,\n    180798,\n    181026,\n    181169,\n    181572,\n    181574,\n    181586,\n    181643,\n    181799,\n    181859,\n    181870,\n    181948,\n    182056,\n    182162,\n    182245,\n    182279,\n    182334,\n    182369,\n    182417,\n    182441,\n    182503,\n    182683,\n    182736,\n    182784,\n    183014,\n    183181,\n    183528,\n    183620,\n    183715,\n    183786,\n    183970,\n    184139,\n    184227,\n    184324,\n    184384,\n    184386,\n    184400,\n    184557,\n    184773,\n    184791,\n    184830,\n    184902,\n    185210,\n    185240,\n    185476,\n    185487,\n    185639,\n    185721,\n    185781,\n    185789,\n    185850,\n    185922,\n    185930,\n    185950,\n    186282,\n    186428,\n    186497,\n    186518,\n    186747,\n    186797,\n    186816,\n    187001,\n    187054,\n    187072,\n    187329,\n    187610,\n    187776,\n    187857,\n    188077,\n    188173,\n    188192,\n    188416,\n    188718,\n    188795,\n    188852,\n    188865,\n    189197,\n    189376,\n    189566,\n    189744,\n    189766,\n    189806,\n    189828,\n    190015,\n    190033,\n    190081,\n    190291,\n    190326,\n    190391,\n    190595,\n    190648,\n    190690,\n    190705,\n    190783,\n    191069,\n    191096,\n    191240,\n    191390,\n    191842,\n    191846,\n    191893,\n    192095,\n    192212,\n    192247,\n    192306,\n    192440,\n    192496,\n    192594,\n    192730,\n    192782,\n    192788,\n    192904,\n    192932,\n    193015,\n    193021,\n    193349,\n    193449,\n    193494,\n    193504,\n    193881,\n    193884,\n    194120,\n    194188,\n    194306,\n    194411,\n    194559,\n    194704,\n    194746,\n    194755,\n    194756,\n    195086,\n    195113,\n    195275,\n    195353,\n    195406,\n    195542,\n    195594,\n    195645,\n    195648,\n    195673,\n    195750,\n    195790,\n    195800,\n    195817,\n    196009,\n    196091,\n    196490,\n    196521,\n    196715,\n    196839,\n    196989,\n    196995,\n    197193,\n    197245,\n    197458,\n    197461,\n    197609,\n    197716,\n    197745,\n    198043,\n    198176,\n    198448,\n    198641,\n    199050,\n    199122,\n    199149,\n    199203,\n    199247,\n    199257,\n    199319,\n    199346,\n    199395,\n    199403,\n    199404,\n    199449,\n    199551,\n    199553,\n    199764,\n    199783,\n    199940,\n    199962,\n    199969,\n    200250,\n    200267,\n    200274,\n    200291,\n    200401,\n    200567,\n    200572,\n    200717,\n    200739,\n    200959,\n    200961,\n    201148,\n    201163,\n    201207,\n    201637,\n    201732,\n    201916,\n    201918,\n    202001,\n    202040,\n    202102,\n    202389,\n    202489,\n    202653,\n    202913,\n    202923,\n    202944,\n    203003,\n    203438,\n    203629,\n    203636,\n    203669,\n    203734,\n    204020,\n    204232,\n    204640,\n    204930,\n    204943,\n    205101,\n    205183,\n    205230,\n    205577,\n    205601,\n    205636,\n    205676,\n    205720,\n    205854,\n    205863,\n    205866,\n    206112,\n    206381,\n    206705,\n    206770,\n    206783,\n    206851,\n    206859,\n    207027,\n    207151,\n    207205,\n    207292,\n    207366,\n    207437,\n    207585,\n    207675,\n    207853,\n    207913,\n    208132,\n    208283,\n    208311,\n    208408,\n    208516,\n    208589,\n    208623,\n    208648,\n    208778,\n    208808,\n    208830,\n    208871,\n    209048,\n    209147,\n    209388,\n    209420,\n    209480,\n    209527,\n    209544,\n    209747,\n    209803,\n    209808,\n    209868,\n    209904,\n    209917,\n    209935,\n    210012,\n    210032,\n    210144,\n    210230,\n    210273,\n    210372,\n    210564,\n    210982,\n    211054,\n    211112,\n    211119,\n    211120,\n    211186,\n    211204,\n    211206,\n    211246,\n    211326,\n    211487,\n    211491,\n    211618,\n    211653,\n    211690,\n    211756,\n    212058,\n    212077,\n    212285,\n    212288,\n    212351,\n    212421,\n    212587,\n    212688,\n    212817,\n    213008,\n    213132,\n    213224,\n    213338,\n    213538,\n    213739,\n    213773,\n    214046,\n    214184,\n    214224,\n    214369,\n    214454,\n    214527,\n    214664,\n    214919,\n    214961,\n    215124,\n    215245,\n    215356,\n    215408,\n    215456,\n    215471,\n    215524,\n    215608,\n    215622,\n    215675,\n    215744,\n    215808,\n    215867,\n    216006,\n    216051,\n    216228,\n    216303,\n    216363,\n    216677,\n    216710,\n    216726,\n    217060,\n    217071,\n    217082,\n    217133,\n    217156,\n    217183,\n    217323,\n    217393,\n    217468,\n    217872,\n    217951,\n    217957,\n    218290,\n    218646,\n    218988,\n    218996,\n    219135,\n    219269,\n    219315,\n    219434,\n    219525,\n    219567,\n    219589,\n    219632,\n    219657,\n    219723,\n    219792,\n    219820,\n    219928,\n    220111,\n    220171,\n    220187,\n    220224,\n    220255,\n    220307,\n    220367,\n    220446,\n    220678,\n    220858,\n    221155,\n    221190,\n    221213,\n    221232,\n    221291,\n    221700,\n    221829,\n    221869,\n    222075,\n    222146,\n    222370,\n    222430,\n    222559,\n    222648,\n    222757,\n    222811,\n    222863,\n    222879,\n    222908,\n    223032,\n    223094,\n    223182,\n    223183,\n    223241,\n    223256,\n    223300,\n    223648,\n    223738,\n    223930,\n    224051,\n    224053,\n    224377,\n    224530,\n    224542,\n    224554,\n    224647,\n    224664,\n    224675,\n    224742,\n    224757,\n    224802,\n    224848,\n    224974,\n    225032,\n    225076,\n    225299,\n    225494,\n    225495,\n    225546,\n    225715,\n    225757,\n    225786,\n    225792,\n    225867,\n    225946,\n    226058,\n    226161,\n    226368,\n    226498,\n    226588,\n    226629,\n    226883,\n    227164,\n    227220,\n    227226,\n    227358,\n    227370,\n    227523,\n    227736,\n    227781,\n    227851,\n    227878,\n    227884,\n    228309,\n    228335,\n    228676,\n    228867,\n    228981,\n    229014,\n    229132,\n    229267,\n    229274,\n    229529,\n    229553,\n    229688,\n    229747,\n    229889,\n    229968,\n    230008,\n    230040,\n    230150,\n    230501,\n    230598,\n    230669,\n    230679,\n    230739,\n    230884,\n    230936,\n    230964,\n    230987,\n    231037,\n    231617,\n    231685,\n    231691,\n    231732,\n    231758,\n    231991,\n    232173,\n    232243,\n    232348,\n    232432,\n    232949,\n    233042,\n    233079,\n    233124,\n    233139,\n    233266,\n    233370,\n    233376,\n    233553,\n    233830,\n    233882,\n    233915,\n    233950,\n    233968,\n    233977,\n    234057,\n    234153,\n    234251,\n    234413,\n    234499,\n    234633,\n    234677,\n    234719,\n    234902,\n    234963,\n    235274,\n    235597,\n    235692,\n    235751,\n    235778,\n    235791,\n    235836,\n    235864,\n    236000,\n    236049,\n    236068,\n    236505,\n    236690,\n    236784,\n    236837,\n    237350,\n    237399,\n    237495,\n    237501,\n    237618,\n    237764,\n    237767,\n    237772,\n    237814,\n    238125,\n    238272,\n    238355,\n    238449,\n    238459,\n    238528,\n    238700,\n    238989,\n    239198,\n    239347,\n    239351,\n    239376,\n    239943,\n    240147,\n    240274,\n    240387,\n    240403,\n    240637,\n    240655,\n    240784,\n    240804,\n    241297,\n    241373,\n    241503,\n    241668,\n    242103,\n    242270,\n    242365,\n    242423,\n    242909,\n    242946,\n    243090,\n    243213,\n    243344,\n    243914,\n    243955,\n    243989,\n    244215,\n    244240,\n    244339,\n    244487,\n    244539,\n    244737,\n    245067,\n    245242,\n    245450,\n    245664,\n    246348,\n    246597,\n    246649,\n    246650,\n    246863,\n    246876,\n    247123,\n    247184,\n    247428,\n    247625,\n    247782,\n    247806,\n    248051,\n    248111,\n    248142,\n    248224,\n    248441,\n    248488,\n    248616,\n    248701,\n    248774,\n    248980,\n    249131,\n    249158,\n    249352,\n    249506,\n    249532,\n    249623,\n    249643,\n    249687,\n    249953,\n    249969,\n    250260,\n    250311,\n    250313,\n    250344,\n    250619,\n    251140,\n    251206,\n    251367,\n    251379,\n    251590,\n    251717,\n    251856,\n    251857,\n    251888,\n    251892,\n    252101,\n    252177,\n    252203,\n    252220,\n    252573,\n    252582,\n    252610,\n    252639,\n    252716,\n    252783,\n    252927,\n    253695,\n    253732,\n    253755,\n    253810,\n    253825,\n    253964,\n    254167,\n    254493,\n    254750,\n    254814,\n    254828,\n    254838,\n    254929,\n    254983,\n    255071,\n    255165,\n    255209,\n    255315,\n    255322,\n    255385,\n    255401,\n    255631,\n    255662,\n    255827,\n    256003,\n    256094,\n    256180,\n    256192,\n    256223,\n    256250,\n    256260,\n    256337,\n    256569,\n    256660,\n    256838,\n    256868,\n    256899,\n    256941,\n    256973,\n    257187,\n    257219,\n    257421,\n    257624,\n    257870,\n    257965,\n    257992,\n    258078,\n    258089,\n    258266,\n    258331,\n    258395,\n    258588,\n    258628,\n    258883,\n    258911,\n    258920,\n    258998,\n    259097,\n    259382,\n    259567,\n    259625,\n    259819,\n    259854,\n    259983,\n    259986,\n    260307,\n    260927,\n    260966,\n    261180,\n    261182,\n    261453,\n    261471,\n    261721,\n    261779,\n    261780,\n    261948,\n    262101,\n    262148,\n    262162,\n    262175,\n    262262,\n    262347,\n    262391,\n    262440,\n    262514,\n    262800,\n    262895,\n    263011,\n    263120,\n    263211,\n    263223,\n    263258,\n    263346,\n    263474,\n    263612,\n    263828,\n    263858,\n    263866,\n    264347,\n    264454,\n    265125,\n    265407,\n    265552,\n    265596,\n    265742,\n    265816,\n    265916,\n    266082,\n    266348,\n    266712,\n    266847,\n    266892,\n    266920,\n    266977,\n    267135,\n    267175,\n    267349,\n    267408,\n    267515,\n    267694,\n    267802,\n    267861,\n    267932,\n    268000,\n    268052,\n    268215,\n    268322,\n    268363,\n    268375,\n    268390,\n    268394,\n    268710,\n    268770,\n    268885,\n    268941,\n    268944,\n    269058,\n    269254,\n    269395,\n    269417,\n    269419,\n    269606,\n    269689,\n    269784,\n    269942,\n    270175,\n    270386,\n    270505,\n    270705,\n    270738,\n    270815,\n    271138,\n    271240,\n    271402,\n    271452,\n    271471,\n    271576,\n    271607,\n    271852,\n    271892,\n    271934,\n    272049,\n    272212,\n    272262,\n    272323,\n    273002,\n    273118,\n    273188,\n    273493,\n    273618,\n    273772,\n    273784,\n    273930,\n    274126,\n    274219,\n    274398,\n    274455,\n    274494,\n    275120,\n    275202,\n    275210,\n    275611,\n    275657,\n    275761,\n    275881,\n    276018,\n    276036,\n    276055,\n    276127,\n    276195,\n    276215,\n    276408,\n    276434,\n    276458,\n    277051,\n    277197,\n    277208,\n    277239,\n    277289,\n    277440,\n    277521,\n    277642,\n    277694,\n    278437,\n    278509,\n    278550,\n    278579,\n    278601,\n    278747,\n    278934,\n    279030,\n    279085,\n    279149,\n    279209,\n    279387,\n    279490,\n    279524,\n    279621,\n    279634,\n    279677,\n    279696,\n    279994,\n    280007,\n    280022,\n    280360,\n    280619,\n    280705,\n    280734,\n    280736,\n    280760,\n    280909,\n    281008,\n    281019,\n    281056,\n    281102,\n    281208,\n    281376,\n    281409,\n    281455,\n    281598,\n    281722,\n    281759,\n    281782,\n    281837,\n    281924,\n    281929,\n    281972,\n    282098,\n    282113,\n    282130,\n    282224,\n    282251,\n    282298,\n    282365,\n    282772,\n    282912,\n    283012,\n    283060,\n    283113,\n    283203,\n    283277,\n    283495,\n    283698,\n    283772,\n    283910,\n    283977,\n    284350,\n    284605,\n    285302,\n    285325,\n    285352,\n    285568,\n    285597,\n    285633,\n    285645,\n    285661,\n    285929,\n    286010,\n    286176,\n    286662,\n    286711,\n    286717,\n    286760,\n    286981,\n    287469,\n    287636,\n    287663,\n    287725,\n    287882,\n    287959,\n    288202,\n    288435,\n    288491,\n    288799,\n    288955,\n    289128,\n    289260,\n    289392,\n    289469,\n    289630,\n    289754,\n    289781,\n    289883,\n    290078,\n    290289,\n    290570,\n    290839,\n    290843,\n    290948,\n    291251,\n    291538,\n    291664,\n    291962,\n    291983,\n    292170,\n    292278,\n    292301,\n    292478,\n    292590,\n    292647,\n    292663,\n    292685,\n    292804,\n    292901,\n    292916,\n    293022,\n    293026,\n    293611,\n    293703,\n    293810,\n    293841,\n    293907,\n    294053,\n    294134,\n    294264,\n    294487,\n    294593,\n    294763,\n    294831,\n    294832,\n    294853,\n    294855,\n    294908,\n    295051,\n    295055,\n    295097,\n    295134,\n    295340,\n    295440,\n    295614,\n    295728,\n    295837,\n    296231,\n    296236,\n    296243,\n    296284,\n    296433,\n    296524,\n    296707,\n    296731,\n    297017,\n    297037,\n    297074,\n    297233,\n    297372,\n    297972,\n    298172,\n    298252,\n    298290,\n    298370,\n    298382,\n    298452,\n    298461,\n    298628,\n    298691,\n    298699,\n    298722,\n    298978,\n    299023,\n    299089,\n    299116,\n    299295,\n    299319,\n    299409,\n    299448,\n    299533,\n    299550,\n    299601,\n    299631,\n    299640,\n    299734,\n    299785,\n    299946,\n    299952,\n    300000,\n    300023,\n    300034,\n    300066,\n    300086,\n    300233,\n    300437,\n    300471,\n    300509,\n    300655,\n    300848,\n    301102,\n    301107,\n    301333,\n    301467,\n    301756,\n    301817,\n    301827,\n    301837,\n    301908,\n    301912,\n    302030,\n    302221,\n    302260,\n    302375,\n    302555,\n    302599,\n    302716,\n    302767,\n    302806,\n    302842,\n    302867,\n    303215,\n    303267,\n    303706,\n    303738,\n    303893,\n    304012,\n    304147,\n    304217,\n    304300,\n    304390,\n    304744,\n    304834,\n    305000,\n    305004,\n    305035,\n    305540,\n    305609,\n    305695,\n    305799,\n    305800,\n    305821,\n    306664,\n    306693,\n    306940,\n    307034,\n    307243,\n    307262,\n    307292,\n    307511,\n    307523,\n    307538,\n    307790,\n    307873,\n    307967,\n    307993,\n    308235,\n    308430,\n    308587,\n    308828,\n    308883,\n    308889,\n    308894,\n    309100,\n    309120,\n    309371,\n    309418,\n    309678,\n    309933,\n    310185,\n    310262,\n    310479,\n    310611,\n    310618,\n    310757,\n    310796,\n    310980,\n    311394,\n    311465,\n    311518,\n    311690,\n    311877,\n    311904,\n    311922,\n    311961,\n    312192,\n    312216,\n    312343,\n    312692,\n    312783,\n    312803,\n    312878,\n    313055,\n    313063,\n    313169,\n    313176,\n    313182,\n    313372,\n    313398,\n    313888,\n    314016,\n    314251,\n    314294,\n    314370,\n    314388,\n    314613,\n    314667,\n    314836,\n    314864,\n    314865,\n    314914,\n    315018,\n    315128,\n    315248,\n    315249,\n    315303,\n    315352,\n    315434,\n    315474,\n    315486,\n    315501,\n    315524,\n    315742,\n    315769,\n    315841,\n    315864,\n    315899,\n    315909,\n    316041,\n    316047,\n    316138,\n    316240,\n    316323,\n    316404,\n    316503,\n    316694,\n    316795,\n    317028,\n    317070,\n    317244,\n    317441,\n    317565,\n    317725,\n    317999,\n    318080,\n    318099,\n    318124,\n    318130,\n    318290,\n    318314,\n    318461,\n    318523,\n    318573,\n    318618,\n    318813,\n    319024,\n    319051,\n    319073,\n    319127,\n    319350,\n    319654,\n    319743,\n    319931,\n    319933,\n    320180,\n    320370,\n    320425,\n    320480,\n    320524,\n    320627,\n    320664,\n    321014,\n    321195,\n    321214,\n    321333,\n    321437,\n    321542,\n    321647,\n    321706,\n    321811,\n    321866,\n    321973,\n    322175,\n    322222,\n    322369,\n    322482,\n    322511,\n    322586,\n    322719,\n    322824,\n    322922,\n    322944,\n    322955,\n    323100,\n    323128,\n    323263,\n    323356,\n    323639,\n    324008,\n    324189,\n    324250,\n    324421,\n    324500,\n    324595,\n    324638,\n    324818,\n    324837,\n    325027,\n    325153,\n    325157,\n    325211,\n    325228,\n    325557,\n    325593,\n    325736,\n    325770,\n    325840,\n    325991,\n    326082,\n    326174,\n    326317,\n    326344,\n    326462,\n    326555,\n    326911,\n    326968,\n    327038,\n    327073,\n    327149,\n    327177,\n    327466,\n    327567,\n    327605,\n    327663,\n    327872,\n    327961,\n    328030,\n    328111,\n    328200,\n    328301,\n    328352,\n    328433,\n    328452,\n    328462,\n    329030,\n    329139,\n    329217,\n    329219,\n    329336,\n    329475,\n    329717,\n    329806,\n    330051,\n    330369,\n    330439,\n    330522,\n    330681,\n    330699,\n    330750,\n    330897,\n    331223,\n    331455,\n    331727,\n    331937,\n    332096,\n    332318,\n    332407,\n    332455,\n    332545,\n    332877,\n    333018,\n    333069,\n    333101,\n    333156,\n    333182,\n    333198,\n    333286,\n    333434,\n    333480,\n    333665,\n    333746,\n    333756,\n    333845,\n    334083,\n    334085,\n    334311,\n    334321,\n    334371,\n    334399,\n    334530,\n    334760,\n    334977,\n    335099,\n    335450,\n    335631,\n    335733,\n    335800,\n    335810,\n    335885,\n    335981,\n    336232,\n    336384,\n    336493,\n    336777,\n    336802,\n    337042,\n    337055,\n    337139,\n    337185,\n    337194,\n    337506,\n    337666,\n    337705,\n    337826,\n    338472,\n    338562,\n    338678,\n    338703,\n    339022,\n    339356,\n    339426,\n    339676,\n    339678,\n    339943,\n    340175,\n    340209,\n    340332,\n    340472,\n    340637,\n    340642,\n    340654,\n    340894,\n    341010,\n    341058,\n    341128,\n    341393,\n    341409,\n    341603,\n    341700,\n    341921,\n    342322,\n    342334,\n    342394,\n    342479,\n    342711,\n    342800,\n    342849,\n    343394,\n    343599,\n    343706,\n    343774,\n    343914,\n    343954,\n    344003,\n    344197,\n    344308,\n    344397,\n    344909,\n    344930,\n    345029,\n    345071,\n    345136,\n    345139,\n    345245,\n    345288,\n    345389,\n    345941,\n    346011,\n    346067,\n    346140,\n    346638,\n    346641,\n    346788,\n    346865,\n    346934,\n    347018,\n    347172,\n    347236,\n    347535,\n    347650,\n    347666,\n    347740,\n    347879,\n    347950,\n    348012,\n    348519,\n    348648,\n    348669,\n    348701,\n    348708,\n    348730,\n    348793,\n    348877,\n    348881,\n    348896,\n    348905,\n    348991,\n    349184,\n    349185,\n    349437,\n    349579,\n    349776,\n    349804,\n    350054,\n    350084,\n    350099,\n    350111,\n    350262,\n    350334,\n    350522,\n    350648,\n    350789,\n    350959,\n    350974,\n    351017,\n    351034,\n    351149,\n    351183,\n    351335,\n    351351,\n    351530,\n    351620,\n    352027,\n    352080,\n    352127,\n    352228,\n    352476,\n    352694,\n    352744,\n    352761,\n    352901,\n    352937,\n    353001,\n    353136,\n    353231,\n    353536,\n    353644,\n    353695,\n    353830,\n    354165,\n    354368,\n    354507,\n    354527,\n    354533,\n    354559,\n    354657,\n    354685,\n    354736,\n    354929,\n    355137,\n    355197,\n    355221,\n    355228,\n    355272,\n    355276,\n    355370,\n    355441,\n    355453,\n    355513,\n    355905,\n    355971,\n    356424,\n    356500,\n    356505,\n    356623,\n    356708,\n    356950,\n    357096,\n    357265,\n    357322,\n    357331,\n    357459,\n    357478,\n    357542,\n    357633,\n    357978,\n    358039,\n    358055,\n    358090,\n    358136,\n    358254,\n    358301,\n    358342,\n    358494,\n    358664,\n    358817,\n    358976,\n    358982,\n    359086,\n    359136,\n    359184,\n    359270,\n    359465,\n    359676,\n    359715,\n    359791,\n    359838,\n    359855,\n    360101,\n    360143,\n    360170,\n    360181,\n    360318,\n    360346,\n    360504,\n    360610,\n    360611,\n    360678,\n    360730,\n    360762,\n    360818,\n    360943,\n    360960,\n    361027,\n    361123,\n    361147,\n    361177,\n    361400,\n    361472,\n    361481,\n    361527,\n    361551,\n    361586,\n    361739,\n    361751,\n    361763,\n    361804,\n    361860,\n    361919,\n    361924,\n    361992,\n    362122,\n    362159,\n    362275,\n    362343,\n    362368,\n    362831,\n    363100,\n    363188,\n    363508,\n    363594,\n    363793,\n    364113,\n    364166,\n    364340,\n    364592,\n    364783,\n    365094,\n    365121,\n    365139,\n    365325,\n    365685,\n    365745,\n    365851,\n    365945,\n    366031,\n    366096,\n    366111,\n    366157,\n    366508,\n    366529,\n    366630,\n    366830,\n    366852,\n    366950,\n    367018,\n    367329,\n    367579,\n    367843,\n    368088,\n    368117,\n    368218,\n    368220,\n    368222,\n    368505,\n    368528,\n    368961,\n    368994,\n    369243,\n    369373,\n    369506,\n    369541,\n    369771,\n    369913,\n    369934,\n    369979,\n    370138,\n    370175,\n    370678,\n    370749,\n    370819,\n    370839,\n    371004,\n    371058,\n    371135,\n    371176,\n    371250,\n    371395,\n    371822,\n    371945,\n    371957,\n    371958,\n    372191,\n    372252,\n    372384,\n    372471,\n    372495,\n    372577,\n    372633,\n    373007,\n    373089,\n    373119,\n    373333,\n    373422,\n    373846,\n    374061,\n    374083,\n    374369,\n    374574,\n    374677,\n    375078,\n    375205,\n    375232,\n    375285,\n    375324,\n    375407,\n    375426,\n    375440,\n    375484,\n    375769,\n    375786,\n    375840,\n    375915,\n    376177,\n    376236,\n    376278,\n    376321,\n    376322,\n    376372,\n    376493,\n    376677,\n    376891,\n    376900,\n    377000,\n    377111,\n    377239,\n    377282,\n    377314,\n    377368,\n    377730,\n    377738,\n    377882,\n    377946,\n    377984,\n    378098,\n    378506,\n    378545,\n    378657,\n    378667,\n    378673,\n    378701,\n    378709,\n    378765,\n    378814,\n    378860,\n    378962,\n    378968,\n    379108,\n    379158,\n    379190,\n    379332,\n    379475,\n    379529,\n    379734,\n    379922,\n    380011,\n    380143,\n    380162,\n    380300,\n    380493,\n    380639,\n    380734,\n    380854,\n    381032,\n    381051,\n    381123,\n    381262,\n    381360,\n    381413,\n    381587,\n    382041,\n    382089,\n    382214,\n    382696,\n    382743,\n    382999,\n    383018,\n    383211,\n    383282,\n    383337,\n    383443,\n    383513,\n    383676,\n    384012,\n    384213,\n    384333,\n    384462,\n    384723,\n    384822,\n    384981,\n    385005,\n    385194,\n    385378,\n    385508,\n    385514,\n    385580,\n    385598,\n    386164,\n    386204,\n    386227,\n    386613,\n    386739,\n    386764,\n    386766,\n    386967,\n    387576,\n    387776,\n    388027,\n    388161,\n    388225,\n    388481,\n    388487,\n    388528,\n    388531,\n    388643,\n    388658,\n    388677,\n    388770,\n    388983,\n    388988,\n    389283,\n    389378,\n    389464,\n    389480,\n    389644,\n    390301,\n    390475,\n    390759,\n    390795,\n    391006,\n    391016,\n    391199,\n    391214,\n    391365,\n    391400,\n    391895,\n    391915,\n    391940,\n    392212,\n    392640,\n    392878,\n    392892,\n    393004,\n    393031,\n    393068,\n    393145,\n    393271,\n    393338,\n    393410,\n    393583,\n    393647,\n    393777,\n    393869,\n    393942,\n    394240,\n    394608,\n    394801,\n    395124,\n    395185,\n    395290,\n    395382,\n    395402,\n    395621,\n    395644,\n    395752,\n    396051,\n    396209,\n    396350,\n    396542,\n    396691,\n    396692,\n    396693,\n    396779,\n    397041,\n    397042,\n    397045,\n    397404,\n    398005,\n    398007,\n    398067,\n    398385,\n    398606,\n    398810,\n    398882,\n    399049,\n    399178,\n    399205,\n    399284,\n    399296,\n    399703,\n    399769,\n    399790,\n    399851,\n    399921,\n    399957,\n    399991,\n    400235,\n    400317,\n    400453,\n    400886,\n    400985,\n    401028,\n    401249,\n    401591,\n    401623,\n    401797,\n    401808,\n    401860,\n    401862,\n    401935,\n    402000,\n    402248,\n    402330,\n    402588,\n    402674,\n    402685,\n    402786,\n    402788,\n    403122,\n    403180,\n    403378,\n    403385,\n    403421,\n    403500,\n    403821,\n    403863,\n    403864,\n    403937,\n    404088,\n    404128,\n    404338,\n    404388,\n    404461,\n    404462,\n    404568,\n    404608,\n    404613,\n    404886,\n    404922,\n    404931,\n    405249,\n    405365,\n    405432,\n    405440,\n    405531,\n    405614,\n    405660,\n    405811,\n    405815,\n    406070,\n    406155,\n    406182,\n    406329,\n    406377,\n    406395,\n    406647,\n    406744,\n    406899,\n    406908,\n    406976,\n    407042,\n    407072,\n    407260,\n    407298,\n    407524,\n    407532,\n    407717,\n    407915,\n    407943,\n    408049,\n    408057,\n    408120,\n    408327,\n    408364,\n    408449,\n    408542,\n    408621,\n    408757,\n    408774,\n    408789,\n    408818,\n    408859,\n    408950,\n    409052,\n    409088,\n    409100,\n    409119,\n    409217,\n    409331,\n    409667,\n    409701,\n    410002,\n    410101,\n    410231,\n    410278,\n    410428,\n    410482,\n    410597,\n    410612,\n    410641,\n    410670,\n    410724,\n    410855,\n    410880,\n    410885,\n    410924,\n    410979,\n    411093,\n    411138,\n    411405,\n    411438,\n    411630,\n    411666,\n    411685,\n    411821,\n    411845,\n    411941,\n    411968,\n    412184,\n    412240,\n    412355,\n    412419,\n    412464,\n    412483,\n    412584,\n    412604,\n    412676,\n    412767,\n    412807,\n    412813,\n    412978,\n    413056,\n    413096,\n    413120,\n    413419,\n    413734,\n    413852,\n    413955,\n    414071,\n    414078,\n    414228,\n    414261,\n    414374,\n    414385,\n    414664,\n    414884,\n    414934,\n    415020,\n    415153,\n    415196,\n    415432,\n    415727,\n    415741,\n    415746,\n    415748,\n    415770,\n    415880,\n    416184,\n    416331,\n    416405,\n    416534,\n    416668,\n    416746,\n    416968,\n    416991,\n    417164,\n    417281,\n    417284,\n    417416,\n    417586,\n    417696,\n    418106,\n    418143,\n    418281,\n    418523,\n    418563,\n    419158,\n    419228,\n    419344,\n    419371,\n    419723,\n    419859,\n    419860,\n    419884,\n    419971,\n    420069,\n    420113,\n    420313,\n    420422,\n    420546,\n    420579,\n    420666,\n    420669,\n    420958,\n    420963,\n    421072,\n    421109,\n    421195,\n    421361,\n    421471,\n    421564,\n    421923,\n    421979,\n    421999,\n    422157,\n    422200,\n    422212,\n    422221,\n    422326,\n    422517,\n    422545,\n    422676,\n    422998,\n    423005,\n    423141,\n    423309,\n    423337,\n    423576,\n    423618,\n    423830,\n    423971,\n    424172,\n    424174,\n    424208,\n    424220,\n    424422,\n    424432,\n    424585,\n    424776,\n    424812,\n    424880,\n    424975,\n    424998,\n    425000,\n    425100,\n    425151,\n    425475,\n    425522,\n    425526,\n    425573,\n    425609,\n    425620,\n    425906,\n    426166,\n    426172,\n    426203,\n    426532,\n    426578,\n    426878,\n    427111,\n    427338,\n    427376,\n    427401,\n    427646,\n    427666,\n    427802,\n    427997,\n    428064,\n    428291,\n    428447,\n    428867,\n    429063,\n    429108,\n    429142,\n    429174,\n    429283,\n    429593,\n    429623,\n    429633,\n    429833,\n    429834,\n    429913,\n    430160,\n    430286,\n    430420,\n    430469,\n    430762,\n    430856,\n    430961,\n    431197,\n    431521,\n    431545,\n    431570,\n    431615,\n    431660,\n    431832,\n    431891,\n    431996,\n    432120,\n    432234,\n    432519,\n    432534,\n    432547,\n    432570,\n    432588,\n    432605,\n    432657,\n    432799,\n    432993,\n    433136,\n    433243,\n    433574,\n    433625,\n    433804,\n    433883,\n    433892,\n    433904,\n    433924,\n    434129,\n    434177,\n    434279,\n    434463,\n    434510,\n    434804,\n    434900,\n    434915,\n    435034,\n    435037,\n    435069,\n    435270,\n    435324,\n    435807,\n    435922,\n    436044,\n    436127,\n    436141,\n    436252,\n    436348,\n    436350,\n    436385,\n    436426,\n    436467,\n    436676,\n    436858,\n    436865,\n    436992,\n    437126,\n    437325,\n    437599,\n    437789,\n    438038,\n    438221,\n    438226,\n    438232,\n    438269,\n    438373,\n    438535,\n    438721,\n    438738,\n    438774,\n    438850,\n    438915,\n    438985,\n    438999,\n    439015,\n    439092,\n    439180,\n    439326,\n    439398,\n    439472,\n    439589,\n    439651,\n    439774,\n    439781,\n    439834,\n    440067,\n    440354,\n    440514,\n    440871,\n    440895,\n    441009,\n    441028,\n    441147,\n    441240,\n    441292,\n    441363,\n    441453,\n    441472,\n    441473,\n    441496,\n    441553,\n    442165,\n    442214,\n    442305,\n    442463,\n    442667,\n    442836,\n    442861,\n    442942,\n    442993,\n    443299,\n    443537,\n    443583,\n    443689,\n    443887,\n    443949,\n    443969,\n    444302,\n    444444,\n    444722,\n    444746,\n    444927,\n    445041,\n    445233,\n    445512,\n    445668,\n    445812,\n    445908,\n    446459,\n    446522,\n    446597,\n    446799,\n    447080,\n    447087,\n    447314,\n    447407,\n    447460,\n    447548,\n    447787,\n    447789,\n    448263,\n    448278,\n    448365,\n    448626,\n    448627,\n    448697,\n    448786,\n    448787,\n    448795,\n    448871,\n    448897,\n    448923,\n    449114,\n    449180,\n    449191,\n    449274,\n    449338,\n    449403,\n    449485,\n    449634,\n    449675,\n    449828,\n    449872,\n    449879,\n    450037,\n    450355,\n    450370,\n    450399,\n    450404,\n    450559,\n    450724,\n    450728,\n    450762,\n    450845,\n    451095,\n    451234,\n    451305,\n    451431,\n    451621,\n    451683,\n    451751,\n    451872,\n    451896,\n    451972,\n    451990,\n    452078,\n    452084,\n    452113,\n    452122,\n    452414,\n    452623,\n    452663,\n    452724,\n    452784,\n    452881,\n    452899,\n    453013,\n    453283,\n    453297,\n    453302,\n    453400,\n    453622,\n    453649,\n    453860,\n    453895,\n    453903,\n    454102,\n    454195,\n    454252,\n    454296,\n    454444,\n    454577,\n    454607,\n    454693,\n    454776,\n    454907,\n    454957,\n    454978,\n    455157,\n    455170,\n    455219,\n    455275,\n    455290,\n    455299,\n    455464,\n    455588,\n    455675,\n    455698,\n    455716,\n    455859,\n    455974,\n    456046,\n    456192,\n    456519,\n    456521,\n    456552,\n    456611,\n    456690,\n    457054,\n    457249,\n    457683,\n    457717,\n    457718,\n    457900,\n    458045,\n    458052,\n    458119,\n    458137,\n    458177,\n    458240,\n    458549,\n    458558,\n    458560,\n    458567,\n    458721,\n    458750,\n    458755,\n    459141,\n    459374,\n    459437,\n    459590,\n    459665,\n    459757,\n    459922,\n    460243,\n    460251,\n    460346,\n    460361,\n    460379,\n    460390,\n    460454,\n    460461,\n    460505,\n    460621,\n    460682,\n    461130,\n    461278,\n    461378,\n    461884,\n    462026,\n    462031,\n    462105,\n    462213,\n    462345,\n    462364,\n    462565,\n    462640,\n    462755,\n    462784,\n    462814,\n    462929,\n    462982,\n    463084,\n    463203,\n    463211,\n    463226,\n    463283,\n    463325,\n    463398,\n    463454,\n    463459,\n    463542,\n    463610,\n    463640,\n    463653,\n    463825,\n    463918,\n    464092,\n    464286,\n    464357,\n    465069,\n    465087,\n    465357,\n    465418,\n    465424,\n    465463,\n    465537,\n    465550,\n    465585,\n    465591,\n    465735,\n    465862,\n    465911,\n    465973,\n    466052,\n    466156,\n    466259,\n    466392,\n    466422,\n    466456,\n    466818,\n    466882,\n    466986,\n    467031,\n    467062,\n    467067,\n    467135,\n    467197,\n    467437,\n    467467,\n    467477,\n    467540,\n    467578,\n    467721,\n    467755,\n    467887,\n    468197,\n    468302,\n    468337,\n    468363,\n    468541,\n    468704,\n    468933,\n    468935,\n    468993,\n    468997,\n    469002,\n    469085,\n    469192,\n    469577,\n    469618,\n    469777,\n    469793,\n    469803,\n    469961,\n    469996,\n    470005,\n    470366,\n    470417,\n    470513,\n    470623,\n    470952,\n    470984,\n    471013,\n    471015,\n    471179,\n    471497,\n    471566,\n    471641,\n    471642,\n    471869,\n    472088,\n    472101,\n    472109,\n    472143,\n    472216,\n    472295,\n    472415,\n    472557,\n    472732,\n    472762,\n    472795,\n    472833,\n    472854,\n    473069,\n    473133,\n    473171,\n    473237,\n    473318,\n    473340,\n    473406,\n    473573,\n    473728,\n    473754,\n    474384,\n    474402,\n    474519,\n    474608,\n    474675,\n    474751,\n    474934,\n    474974,\n    475053,\n    475150,\n    475396,\n    475398,\n    475403,\n    475667,\n    475906,\n    475923,\n    476065,\n    476109,\n    476215,\n    476282,\n    476339,\n    476406,\n    476552,\n    476754,\n    476925,\n    476939,\n    477087,\n    477194,\n    477202,\n    477477,\n    477483,\n    477526,\n    477867,\n    477924,\n    478077,\n    478155,\n    478282,\n    478528,\n    478621,\n    478766,\n    478858,\n    478922,\n    479030,\n    479213,\n    479508,\n    479562,\n    479612,\n    479832,\n    479948,\n    480000,\n    480021,\n    480215,\n    480605,\n    480643,\n    480720,\n    480747,\n    480951,\n    481187,\n    481252,\n    481314,\n    481325,\n    481398,\n    481446,\n    482021,\n    482022,\n    482081,\n    482160,\n    482436,\n    482562,\n    482690,\n    482907,\n    482994,\n    483108,\n    483149,\n    483275,\n    483311,\n    483348,\n    483545,\n    483722,\n    483767,\n    484029,\n    484042,\n    484080,\n    484145,\n    484175,\n    484301,\n    484303,\n    484312,\n    484415,\n    484434,\n    484551,\n    484587,\n    484681,\n    484695,\n    484938,\n    485485,\n    485494,\n    485731,\n    486046,\n    486568,\n    486573,\n    487076,\n    487141,\n    487217,\n    487222,\n    487236,\n    487349,\n    487375,\n    487450,\n    487525,\n    487685,\n    487741,\n    487774,\n    487898,\n    487957,\n    488198,\n    488219,\n    488401,\n    488539,\n    488697,\n    488723,\n    488736,\n    489611,\n    489624,\n    489739,\n    489763,\n    489861,\n    490008,\n    490337,\n    490413,\n    490509,\n    490620,\n    490643,\n    490688,\n    490741,\n    490878,\n    491064,\n    491090,\n    491102,\n    491497,\n    491653,\n    491757,\n    491765,\n    491793,\n    491835,\n    491902,\n    491947,\n    491985,\n    492067,\n    492132,\n    492282,\n    492420,\n    492469,\n    492506,\n    492552,\n    492683,\n    492814,\n    492817,\n    493284,\n    493623,\n    493797,\n    494077,\n    494404,\n    494439,\n    494566,\n    494578,\n    494608,\n    494808,\n    494957,\n    495081,\n    495160,\n    495183,\n    495349,\n    495377,\n    495519,\n    495985,\n    496207,\n    496213,\n    496525,\n    496541,\n    496575,\n    496597,\n    496604,\n    497014,\n    497034,\n    497067,\n    497094,\n    497334,\n    497438,\n    497719,\n    497907,\n    498439,\n    498555,\n    498583,\n    498665,\n    499095,\n    499181,\n    499198,\n    499200,\n    499266,\n    499423,\n    499537,\n    499705,\n    499755,\n    499937,\n    500019,\n    500211,\n    500446,\n    500703,\n    500780,\n    500825,\n    500844,\n    501116,\n    501122,\n    501420,\n    501429,\n    501491,\n    501614,\n    501652,\n    502090,\n    502240,\n    502419,\n    502558,\n    502671,\n    502766,\n    502852,\n    502854,\n    502936,\n    502959,\n    503005,\n    503137,\n    503183,\n    503311,\n    503470,\n    503595,\n    503826,\n    503980,\n    504005,\n    504074,\n    504101,\n    504152,\n    504248,\n    504293,\n    504341,\n    504414,\n    504415,\n    504747,\n    504790,\n    504811,\n    504813,\n    504824,\n    504891,\n    504958,\n    505636,\n    505650,\n    505739,\n    506201,\n    506310,\n    506357,\n    506416,\n    506458,\n    506489,\n    506568,\n    506577,\n    506872,\n    507065,\n    507081,\n    507147,\n    507352,\n    507686,\n    507921,\n    508302,\n    508370,\n    508586,\n    508672,\n    508917,\n    509270,\n    509497,\n    509656,\n    509807,\n    509819,\n    510657,\n    510955,\n    511145,\n    511583,\n    512116,\n    512183,\n    512337,\n    512467,\n    512476,\n    512576,\n    512760,\n    512776,\n    512785,\n    512793,\n    513041,\n    513060,\n    513064,\n    513073,\n    513507,\n    513567,\n    513611,\n    513699,\n    514147,\n    514366,\n    514402,\n    514437,\n    514468,\n    514787,\n    515056,\n    515241,\n    515303,\n    515347,\n    515531,\n    515540,\n    515668,\n    515727,\n    515755,\n    515760,\n    516020,\n    516143,\n    516178,\n    516316,\n    516408,\n    516416,\n    516601,\n    516750,\n    516766,\n    516786,\n    516813,\n    516875,\n    517296,\n    517610,\n    517629,\n    517687,\n    517967,\n    518234,\n    518255,\n    518375,\n    518530,\n    518551,\n    518725,\n    518729,\n    518873,\n    519039,\n    519046,\n    519208,\n    519359,\n    519555,\n    519569,\n    519611,\n    519631,\n    519744,\n    519764,\n    519815,\n    520012,\n    520043,\n    520108,\n    520109,\n    520430,\n    520482,\n    520508,\n    520525,\n    520530,\n    520787,\n    520871,\n    520918,\n    520933,\n    521131,\n    521142,\n    521306,\n    521427,\n    521879,\n    522273,\n    522427,\n    522452,\n    522489,\n    522889,\n    522909,\n    523175,\n    523194,\n    523816,\n    523978,\n    524002,\n    524011,\n    524064,\n    524173,\n    524637,\n    524638,\n    525118,\n    525170,\n    525272,\n    525297,\n    525373,\n    525376,\n    525568,\n    525616,\n    525619,\n    525646,\n    525763,\n    525777,\n    525899,\n    525903,\n    526030,\n    526126,\n    526222,\n    526256,\n    526257,\n    526319,\n    526364,\n    526414,\n    526555,\n    526560,\n    526645,\n    526737,\n    526806,\n    526955,\n    526972,\n    527164,\n    527248,\n    527313,\n    527407,\n    527464,\n    527535,\n    527575,\n    527625,\n    527631,\n    527828,\n    527846,\n    527906,\n    527961,\n    527962,\n    528011,\n    528575,\n    528604,\n    528729,\n    528862,\n    528905,\n    528962,\n    528984,\n    529256,\n    529578,\n    529592,\n    529626,\n    529631,\n    529668,\n    529691,\n    529813,\n    529950,\n    529981,\n    530162,\n    530207,\n    530226,\n    530317,\n    530383,\n    530706,\n    530913,\n    531163,\n    531264,\n    531414,\n    531602,\n    531622,\n    531707,\n    531739,\n    531797,\n    531875,\n    532085,\n    532211,\n    532295,\n    532403,\n    532568,\n    532580,\n    532610,\n    532695,\n    532780,\n    532855,\n    532867,\n    533013,\n    533281,\n    533407,\n    533485,\n    533493,\n    533979,\n    534121,\n    534162,\n    534444,\n    534533,\n    534605,\n    534733,\n    534983,\n    535080,\n    535183,\n    535245,\n    535262,\n    535292,\n    535306,\n    535313,\n    535578,\n    535602,\n    535748,\n    536028,\n    536038,\n    536158,\n    536183,\n    536429,\n    536589,\n    536615,\n    536653,\n    536725,\n    536791,\n    537007,\n    537053,\n    537574,\n    537589,\n    537939,\n    537982,\n    537991,\n    538092,\n    538122,\n    538451,\n    538695,\n    538819,\n    538934,\n    538976,\n    539079,\n    539124,\n    539355,\n    539423,\n    539678,\n    539768,\n    539784,\n    539904,\n    539926,\n    540187,\n    540280,\n    540531,\n    540636,\n    540714,\n    540774,\n    540806,\n    540860,\n    541071,\n    541202,\n    541474,\n    541478,\n    541587,\n    541627,\n    541664,\n    541965,\n    542077,\n    542089,\n    542150,\n    542325,\n    542388,\n    542509,\n    542625,\n    542634,\n    542755,\n    543065,\n    543300,\n    543322,\n    543631,\n    544365,\n    544444,\n    544825,\n    544883,\n    544975,\n    545007,\n    545039,\n    545111,\n    545129,\n    545385,\n    545441,\n    545597,\n    545612,\n    545734,\n    545756,\n    545796,\n    545867,\n    546052,\n    546067,\n    546095,\n    546126,\n    546222,\n    546325,\n    546378,\n    546500,\n    546773,\n    546854,\n    546964,\n    546965,\n    546976,\n    547155,\n    547258,\n    547502,\n    547601,\n    547839,\n    548014,\n    548361,\n    548423,\n    548538,\n    548703,\n    548795,\n    549012,\n    549136,\n    549261,\n    549797,\n    550013,\n    550051,\n    550084,\n    550322,\n    550349,\n    550365,\n    550422,\n    550452,\n    550529,\n    550576,\n    550627,\n    550707,\n    550968,\n    551107,\n    551338,\n    551654,\n    551944,\n    551974,\n    551987,\n    552217,\n    552503,\n    552564,\n    552654,\n    552837,\n    553078,\n    553097,\n    553154,\n    553221,\n    553420,\n    553442,\n    553447,\n    553547,\n    553558,\n    553576,\n    553678,\n    553776,\n    553788,\n    553869,\n    553879,\n    553942,\n    553968,\n    553990,\n    554037,\n    554154,\n    554241,\n    554291,\n    554445,\n    554500,\n    554595,\n    554618,\n    554749,\n    554859,\n    555337,\n    555387,\n    555763,\n    556073,\n    556083,\n    556120,\n    556158,\n    556205,\n    556341,\n    556349,\n    556545,\n    556562,\n    556608,\n    556653,\n    556726,\n    556901,\n    557130,\n    557135,\n    557346,\n    557501,\n    557552,\n    557564,\n    557731,\n    557732,\n    557998,\n    558013,\n    558064,\n    558083,\n    558137,\n    558235,\n    558388,\n    558406,\n    558483,\n    558585,\n    559047,\n    559364,\n    559440,\n    559550,\n    559656,\n    559665,\n    559773,\n    559884,\n    560010,\n    560218,\n    560439,\n    560591,\n    560623,\n    560718,\n    560744,\n    560804,\n    560880,\n    560918,\n    561014,\n    561100,\n    561403,\n    561491,\n    561670,\n    561750,\n    561856,\n    561928,\n    561938,\n    562084,\n    562121,\n    562207,\n    562345,\n    562356,\n    562381,\n    562419,\n    562448,\n    562582,\n    562592,\n    562628,\n    562778,\n    562843,\n    562897,\n    563175,\n    563311,\n    563665,\n    563797,\n    563927,\n    564088,\n    564091,\n    564163,\n    564280,\n    564382,\n    564612,\n    565012,\n    565031,\n    565186,\n    565198,\n    565239,\n    565331,\n    565389,\n    566314,\n    566529,\n    566634,\n    566644,\n    566717,\n    567011,\n    567013,\n    567329,\n    567565,\n    567657,\n    567787,\n    567863,\n    567886,\n    567922,\n    567944,\n    568040,\n    568107,\n    568115,\n    568132,\n    568318,\n    568325,\n    568337,\n    568961,\n    569059,\n    569096,\n    569289,\n    569849,\n    569931,\n    569969,\n    569976,\n    570107,\n    570138,\n    570465,\n    570521,\n    570573,\n    570678,\n    570688,\n    570736,\n    570755,\n    570756,\n    570834,\n    571449,\n    571518,\n    571641,\n    571746,\n    571786,\n    571804,\n    571821,\n    571876,\n    571990,\n    572055,\n    572182,\n    572226,\n    572233,\n    572370,\n    572408,\n    572427,\n    572574,\n    572733,\n    572767,\n    572784,\n    573067,\n    573223,\n    573626,\n    573756,\n    573881,\n    573896,\n    573962,\n    574138,\n    574256,\n    574280,\n    574316,\n    574633,\n    574666,\n    574672,\n    574804,\n    575018,\n    575135,\n    575179,\n    575194,\n    575367,\n    575389,\n    575410,\n    575643,\n    575904,\n    575915,\n    575916,\n    576084,\n    576222,\n    576363,\n    576369,\n    576463,\n    576566,\n    576576,\n    576714,\n    576820,\n    577024,\n    577033,\n    577160,\n    577277,\n    577385,\n    577403,\n    577417,\n    577524,\n    577526,\n    577853,\n    577866,\n    578093,\n    578314,\n    578492,\n    578698,\n    578736,\n    578740,\n    578848,\n    578977,\n    578993,\n    579056,\n    579240,\n    579307,\n    579548,\n    579664,\n    579759,\n    580591,\n    580613,\n    580778,\n    580829,\n    580835,\n    580931,\n    581010,\n    581033,\n    581253,\n    581332,\n    581357,\n    581482,\n    581886,\n    581929\n  ],\n  "test_id_sha256": "18a173e42d695050f9700423634f3caee4401a03ba9481320f57044806a8bc95",\n  "selection": "checkpoint fixed by VALID before TEST; rank0 raw beam cumulative logprob",\n  "prior_project_test_exposure": true\n}\n', 'TEST199_PROTOCOL.md': '# Fixed checkpoint199 TEST5000 — 09/11/2026\n\nUser explicitly approved TEST. Freeze best=endepoch10 from ducanh2403/2-vi-cai-kia-qua-nhieu/199, SHA5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7. No training/SCST/optimizer restore or model selection. Same source model, beam5/max20/raw logprob/no lengthnormalization/rerank as VALID; priorprojectTESTexposure disclosed. Strict5000KarpathyTEST IDs frozen from historicalTESTsnapshot and COCOdatasetSHA; reference captions used only for metrics/NLL, never generation or selecting candidates. TRAIN-only vocabularyexactSHA same as199. No prompt/bank/detector/ROI needed.\n\nOnly originalvisual-cache H5 +dataset_coco.json. Sequential selectedTESTrows to temporary local196patchfloat16 array~1.4GiB, originalprofile/shape/IDs/filename/finite/duplicate coverage/bytes checks; no TRAIN/VALIDfeature reads. AllfullH5headerIDs may be inspected. Store references/vocabulary separately, then2T4 independentworkers fixed2500images/rank, batched8images beam5 fromexactcheckpoint; no testtuning. Save captions/IDs/rawscores beforeCPUmetrics; NLL usesallTESTreferencecaptionevents in separateteacherforced loop after captionssaved. Keep lock/inputseals.\n\nIf no optionallocalcheckpointroot, downloadONLYbest.bin fromexactown199 via enabledKaggleSecret codex or KAGGLE_API_TOKEN; pinnedCLIrevision/exactversionroutingmockpreflight, fileSHA check, no latest/foreignnotebook/cache download. No tokenlogging. Samepackage runtimeTorch2.11cu128 checked; do not reinstallTorch. GPUphase uses no optimizer/backward; CUDA2T4/NCCL guard. Author only NumPy/mock/AST; user runs realIO/CUDA/officialscorers.\n\nOne notebook(parent2hstopcap, notETA/billing): setup/test/data/strictcheckpointlock, metricenvironment/syntheticPTB/METEOR/SPICE smoke, full5000decode, teacherNLL, savecaptions. CPUphases basicBLEU1/4/CIDEr/ROUGE-L, METEOR, SPICE, CHAIRs/i/recall, independentphaseoutputs thencombine. Reuse proven pycocoevalcap1.2 Java8/CoreNLP3.6 environment helper. StanfordmodelarchiveSHA recorded notindependentlypinned, same limitation aspriorprojectevaluation; no fullCoreNLPsubsets claims. SPICE save perimageAllF only(finite), notundefinedNaNsubcategories. All8required beforecomplete; failurepreservesreview/savedcaptions so no needautomaticmodelrerun. No model/scorer runlocally. Keep downloadedweights/output; zip excludesbinary/features. No Word edits. Record newresults inexistingSheet onlyafter userrun.\n', 'TEST199_BUILD_V2_PROTOCOL.md': '# TEST199 V2 — build compatibility fix, 10/11/2026\n\nV1retry downloaded exact199/SHA5061e305..., completed5000TESTstage559.44s and metricenvironment/synthetic smoke; no captionworker completed. Bothranks stopped at exactTorchstringguard: trained2.11.0+cu128 vs current2.11.0+cu130. Earlier5data/static tests andCUDA2T4runtime passed. Data/model/secret are not the stoppingcause this time. Do not conflate this with stuckCancelling sessions.\n\nV2 changes only environment-compatibility policy: accept exactly observedTorch2.11.0 CUDAwheel buildscu128/cu130, require trainingversioncu128, rejectCPU/otherpublicreleases/unreviewedbuilds. This is not blanket removal of guards or an assurance of identical floatingpoint/beam results. Record bothversions/buildchanged incheckpointpreflight,eachworker andfinalresult. Same checkpointSHA,epoch10,sourcemodel/beam/rank0/no lengthnormalization/no rerank, noTESTtuning/training. Strictstate_dictload, finite/pathlikelihood rescoringonfirstGPUbeam batch andcomplete2×2500IDsremainmandatory. RuntimepreflightbeforeCoreNLPdownload/TESTstage catches futureunsupportedversions early. No Torch reinstall/downgrade oroptimizer use. CPUmetadataunit teststestedlocally, userrunsGPU/officialscorers.\n\nFreshoutputclip_patch_199_test5000_v2, same2hstopcap/twoinputs/T4x2/secretnames. Reuse exactalreadydownloaded199bestbin atknownoldWorkingpath ifpresentandSHAvalid, elseonlyown199download; no broadpathscan/latest fallback. ExistingcompleteTEST features remaininoldtmp but notautoresumed/copied acrossruns. Original5000TESTref/patchprofile/metadata sealsunchanged, onlyTESTfeaturesstaged~1.4GiB. Eightmetrics+Recall/NLLrequired; captions/phaseoutputspreservedonfailure. DeliverV2sourcepinnednotebook andrecordlimits. References: PyTorchstate_dict/map_location tutorial supportsportableloading generally, not bitwiseproof acrosstheseCUDAwheels: https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html .\n', 'SMOKE_SOURCE.json': '{\n  "archive_sha256": "d6353b2adca467be2f2435e12fe31e317c17b2e9267efeccb2415c02e4d9b45a",\n  "version": 192,\n  "feature_profile": {\n    "model_id": "openai/clip-vit-base-patch16",\n    "output": "vision_model.last_hidden_state",\n    "preprocessing": "bilinear",\n    "image_size": [\n      224,\n      224\n    ],\n    "antialias": true,\n    "dataset_json_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "model_revision": null,\n    "mean": [\n      0.48145466,\n      0.4578275,\n      0.40821073\n    ],\n    "std": [\n      0.26862954,\n      0.26130258,\n      0.27577711\n    ],\n    "compute_dtype": "float32",\n    "storage_dtype": "float16",\n    "autocast": false\n  },\n  "vocabulary_sha256": "cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd",\n  "raw_selected_row_sha256": {\n    "32682": "4f0972bede6f3bb2d0c08c47d765afa30057cc221140207bc7160a03d0f4a684",\n    "432898": "d89b6dc04255ca587f3f137e3a9196662a21627a2fb72e0f6f1a2b1632f00c6a",\n    "337246": "ee4d8be1fdd6324d73c82626caacbb13291e9d0be8c413ac65c1b31fdf61e123",\n    "411871": "60e6cbcb2e8ecfbb8e8fda6e1b7d0b1e03ab9e4e82178d137268a6fe91bf0660",\n    "31280": "5914069fda1b6e2c9958f5585826a8a4ba59c60a0f6dab6c63d405b585a08321",\n    "303260": "689df55ff0b69f08e4c7b630c790626dbc46519a6d8a72aee621e394e5a1d30f",\n    "478665": "9b95376163897b507cd9c483b57c966785988309b8d0b2d9da8a8e8b853c8aff",\n    "155540": "23ea87d964d0e3bf9934643ee2e8a2476a002e1991de66844c2e171f57c44c86",\n    "456236": "504a5b6d1c2566a61ff05a4dfa8075f63bfe80b455bda716947405b890a46af5",\n    "419116": "b2ad30e5f0102825028f3e7b70e949145848e6a5b360dc8483430e572837c05e",\n    "28646": "bd0baba12de7a000172abc624f369e201aded42ddd13fa1fb0ba2a379f293f1e",\n    "407121": "b6da110d89d9abff9b8c7d8da1864679bae822c694a36873ed2d005ef50ba196",\n    "482626": "06bb06afcc51ff92985509b8029cf599125176bfc22824e6f2e3c62d9624dff1",\n    "97599": "fcf34e149aad871d4e126b0eea222c2488036327c314c11fd7d24de1757dff5c",\n    "64183": "5bf007a2a52b9187b35ee12a7d28f85240b413422ae45126da5f128f8191d9bc",\n    "301082": "cefd9cdfc4fc68ba10e60681d4663a44de16a2df299ba5c0d2b703b54accc39a",\n    "472445": "1596f5ff8636e0529eca9be23d19b843d380c08c8cbde769eeb5d2d77915834d",\n    "429839": "a379e5940de7f0f5c2909903e1bb54b2687020baa2687fb3eb37e8c1adfd1559",\n    "389404": "07eef654ba4fd4a50905844c9d7e7fbd0469373446bad55f60189e55a394cb70",\n    "290659": "5770ddce8b60f020f48eee897eddd7e2418892e0880df87469ee27a7af193f0c",\n    "189504": "1a0835b57030ceef86516f456d03a84a4b9295b19053899c4e7b0340c16b5ee9",\n    "77479": "a12f2e4b4e22ce32f1f8ea4643dca24a2fb2d521da9ba62a67fc03aee7ac8f37",\n    "89781": "ede9ef7ad497d3fa0fc37935dc9002c70cdf14bd0e82f73802e2699ef8a86afb",\n    "363241": "e6da8fdb3ced00e02d3e7c118335e26bc02cea08bf8f3a7890b4416f7db85206",\n    "85914": "ff153115805e6dca1999bbdcd1a9268a0d18470ccb37b91a3e80f62f8031b2e0",\n    "486986": "b513f8beb6d3fd3f6e436d6b40ee80bb2d9a28869e70d6bdf2289f6a86e1512b",\n    "288340": "460f64ad77ad1303e7c8012385c0be56cbf151e7252fc57aeca14e33da3538a6",\n    "145665": "082240910a659472d8d1503b536c2f6baa868dc37203f098cbbbba1f6e8e8336",\n    "139871": "847040a30c148ea4f01591e745729e4d075f404b5494e3e9097903ff0aecc9fa",\n    "403943": "b25bb9e0c857ea39eb2e8d3634d75b9b7aa4d271d83d9b69998b6f89d65cffbb",\n    "285089": "a9fd8e346ec568a224e0a01e29b607c76b5f042045bdc66bae9739de897a6b47",\n    "272870": "b915d0fbe0a93d3675d8141353d387f7d39e1d1d8f2000bfac27c4e0904cee78",\n    "268254": "07a77188db83d8eddc8d47e83ae995ec6ec5b87bfffde835c65e332ca4a1fa28",\n    "147205": "f2d2e6b54e6065560932a338cce11c125198a2cdf42e0fb375643558c009d7c6",\n    "382399": "e650037023ee0db7429fc0ea749600a0fcc6ea0f7a69f58044b1181227013a70",\n    "509551": "38d0f717c1ad38243517160030693f45e45cb827d270de4fd3f1bd124d2ddf11",\n    "119752": "577f55b46c63b329501d7dda4c7b161f625523972131d52ed5115d30ab4089cd",\n    "18728": "a887339f668fc1b736467460b9a41c5dc70674067ddd80a052054aafe31c5b3f",\n    "38118": "001275230bcbb0c24018eea685c6e40b5481941cc4e0bf64bb978ca45cfb4a89",\n    "561958": "e9decb01c27886db3e3b243ce1fd272df01ef969ab00152fcf0f976c0f86eebc",\n    "69140": "f0896a56b0a095ff5b9559ca38017bdd6de6f1e11bdf6bf87cf1c3a4328332f9",\n    "370401": "2163a3a4c2e5643c4ed63c82b3b3df7d5fba2e32ade0769f52f7b1b85c9e6bf3",\n    "487688": "076b0a3d8b8e7bfd8f1ca8288d8de71df4a4d79ec54cbb872084a9bf91046344",\n    "80104": "8c5bf66413c21fa5729ad8e9df2584917451ea72b34e1e4036a1b66f86f7f27e",\n    "217154": "faf53359d1b8b4c332a661e43bc4dd9e76ec1e329f77588c19372ed5b6d6a5fe",\n    "383777": "32c9529c7f0a9e050636f58f1dcdf80ce1d099a861345c4f9d7bb1b730547d8f",\n    "153184": "2a98c73d16c2ea6e1ab4a9835b8290f70ecc2ad36105d7829072253d69dcec9c",\n    "476279": "7cfc9b15701d560be56a7a7ba2730efa19c0915dd0fbeb2fc018d3ffbce12cca",\n    "267035": "0b7dee801c80fcbcb357a603f836617f0c164ea6aa176c0f8dc1b67a2cc129cc",\n    "412285": "bed0a09b9e4e29fdc07feb0b401d50a3a7b4d3beaed79c449d36a21db1916448",\n    "565929": "e7fbfca42adcbbfa609256b22b3f14d184b73abd4667af6ed8f9aa45eb472567",\n    "536366": "b0f31343c9f89e3623f4abfb1843d9f16c970c5bfcfa35f12dfbee615b5ca5ee",\n    "53529": "526350c7cf239cf7080e0c77033e1c614cd720a721a8e5b37857cfbd548cbdef",\n    "370095": "0fa0a4e6f69f674f5d576fc985e23c17ecef61331ac7c13da5354dedd4ce8473",\n    "544201": "224ebbcc427b71a9f37d35921852c14eb50cec0d3dbd4f5093af11611c6dc8ea",\n    "250127": "940755f90a2216df6b238cf0baee6ab3b1676b4381afbcb0b77cb6841ae52154",\n    "286018": "8d7e5ff4be8db785d241e83296a2b35da5f16d76c2edf8ad9b87f0fd0f26bf5f",\n    "453138": "3bb160671644a71a7f83e47ecddd4cbd8a707c63939e11d4fe6e6cb04f5ede35",\n    "50960": "759230405b0f97f220ec373efd0cca40a14ed905b24f0724ed00aaf6b2e5c5ed",\n    "563816": "de8011585b27e6a7a8a4aa7cb8c9a575c2c918f620ffde69952ec8e3fb9a17cc",\n    "174070": "38ce4f6926e8a0d2265a4e60e8e782e7ace86020870abd7719e784a4be4b6e6d",\n    "227048": "6b965dd3ff339be2719fd2042cbbafe8d2046462982de66f677cf0f9a1a7d1e5",\n    "306264": "49d95a63c71231913b76c27a91842656ddaeb272be59b3c0c91a9f3a68068fcb",\n    "392506": "ffc232d28f84c8fcab9690e6bed0ecfb306444fa56bd5263af2e4c6f9c09bb14",\n    "478522": "0277c5202643818175bafe7ab9cd5a774cc722bb3cae6a925afd288b2f95fce2",\n    "482305": "c6b9641c33e001ed21e5b209ff61ad448703210d1179fa5d130590f249eff433",\n    "36660": "3c955f36af644e6b9fa1f9bb6777e1fc160d5c641ca06fca57748a95834c3f51",\n    "562054": "653a29c0a5fbcc4bfd38300db0b67ebe3e66b85c9cf1f6a72a48d770e87f8a7f",\n    "385036": "c442b80a951ad81476535b2e5320f42cb2ebac72c4f2cdd4cb7405e0ffbdbf69",\n    "88983": "5bae2afab82d78a76dbb513b51230b23b0d2190ac6f61232d74a21d5f55dd7a4",\n    "459967": "7b564fde4dbe14dc7efd3707f0969a69ef79be40df12d6f5d1f00752b4cbd1cb",\n    "78188": "d21a74c2be034ee627e291dbe41f2fe08b31df835f0f2a93eb2d7c9dea7251cb",\n    "31796": "286054c8d47c3c3c28e527da6d945b7c9fd365885ce9f3e9884a9dfdb0cd7b11",\n    "542934": "22ef616cb1d25196de45977bf4d93182d22d065cec801df776394e982c8282ee",\n    "53328": "ec3b47a11d8f4af37ff7ebe8f7841373ff84a036c5e61b147f808d4c20a55f1a",\n    "569464": "ae159349ac86d1225775a581eace05b22e66159a946282b081089c97fe875126",\n    "491107": "0d063f8fb7fa60154ce214a5eb2ecc74c2a335b43d10efee0a63ec64c3bd7f33",\n    "416718": "4aae966096065a6faacb5c85a7aff2a4de12a94f039a8713deb60a72ecfebe61",\n    "547348": "5ffa5553cc67f7625bdb78fd3110f57a9eeb8af9159b48ea3e06c748ed318595",\n    "316963": "60087279f2bce95ad97bc2afb3c61abaddb5009d29ca5e3f881052ba1d4a464d",\n    "241534": "bc0b9483f40c5d58139ecae5b9bdb7c37f76cdb608684d705b9796c6c52c96ca",\n    "466797": "8625d03a9c83aaa92de966151889291280177f9d0c3e80330c733c3245c985e9",\n    "179650": "cbe934031183aee846a69adc4364373ab975816c59d2f2caf887b59ade6d895c",\n    "171376": "84a2952378f31cb2dd6cb935bebe219b7450af3689fcdc729e19d2f6031a0bab",\n    "186361": "de2590e41e5de51972e92dc2fe1619846fb5ed19330001b9758c1f825c8e7e45",\n    "279065": "a28482bd2fbe26924d4dcc80d735ceac25d30b992520cec826a6284f2ebeb3b7",\n    "486769": "dd7893205a7cd98a73270bbfb779a84c50a055772b9385b34437e410e018d406",\n    "406077": "ee2b9a80957525c4be1c054520ee87ce46c42fdc2e049451fbc163efa4e73de8",\n    "393675": "3fbd0a08ef9c5476b65de9a20cb4d2212c22420f2a7620cab76df09c781cbf45",\n    "189404": "18dc511dea6b0f8f66ddeaff9f3f6b26c928291bec4c25ee0479d7f5fbb00352",\n    "225539": "e6bd24516d71042ecedadb6c624e6ec54741536d047e0f69802eba262591b509",\n    "259833": "5c98bd0b96058b3fc64cc70f2bbe8c6120a22c7a71f88b08a690230b687546fb",\n    "292687": "f450f30803a502456ddeac74abfd898aad3da34f51e4ad1cf860559f6ef558d9",\n    "74517": "8395fd7b519c94f6e239534132a60aad072c7743cc836d560e6a0fa6c5176101",\n    "211208": "060943ab2bbcb7431df8119117b2d5b6e795732b9bd282a07a78a3cdef398955",\n    "368764": "6bb9c9e3d7a5eb3e3ffa997fc1983c30df3ad9dc604dcc325805b079b2d2dd16",\n    "497202": "cac06747ddc81c708d036e1837b39bd541c5699326ad48b5fcdb201b1e38bd01",\n    "497194": "9eaf904f5e6ffbef6bf5ffa5668e6b6dfaad733cbe0b64ed9e4353bb6f4561df",\n    "356921": "2fd5e27440cfb784b3ca14078bc283145f306f074808f3f3f03cec9fdf4e107f",\n    "5253": "5fed259e1a3659302bbb3ee8df74814740ca88c4b19870e4207df2863b9ea842",\n    "359722": "38d782ef8a184a5b4dba8f29cfa9c0cf438e5d4562e49f04d4cb7323c9b2efa6",\n    "395080": "8df14202ff0389850117a50775232ba42c7d5a8a8d47f63f99c37fa5e182c0c5",\n    "464174": "8998e350b40e938ed9f1fbd60fc9ffbd62970d264b2e21962ffff72e6769f903",\n    "139124": "6661afac5fa65f09431deb907f5abacac611e8381ee0f5a154b4b97ae174917c",\n    "245373": "6bf9dbe1cd0ee154f3c170dd4adeeff58aa04e5b628363e25535d4358fa46702",\n    "459346": "a02e0ffd68574f4625a79cf5a2eded1f2a7a17956d6bfb902da0a7efc60bc860",\n    "43407": "5dd6840ada8fa0b501acded3d74535dbd9f73a3cd3e6ff3b1bf947db7d36025f",\n    "55683": "f00f2523752a5273261b472ce265902357951ce9ffe32b18dfe3b23a3c10f0f2",\n    "407349": "59b887aa59bde8d76c67083a6d30f0f61a49ab77f2b752d629098e8bfe7e5027",\n    "11146": "e111073e78e24c6c20e7ed56183ddc806fdf0ced6aa3399116040d7d8500d47d",\n    "263651": "6fa1196b96c263565e5e8f76f0f9ba41ebca368891de1a6f50f45f5d37f5dfe2",\n    "579999": "9de985d18b004a488116603a3cb9e7c5112650421aeba6bf69fb8115ba72ff62",\n    "33352": "79394e7529ab1bbed029706f1a1844a50e932949717277b8e8c9df002ae42d7d",\n    "322315": "4a04ba23de6c6464e25fb68f622a2f43cb60bd9c23548ce3bc2963c50235262a",\n    "231521": "0003718a0ff1cb129b9377214b5e3283e835b3873ccb09564c07c39bd0ee9877",\n    "495712": "0db186721fe287df0eafd939879206506c7e05d61a42c787a561e1949b9c32e0",\n    "369212": "bc08cae636606e45f805d1c2bad9b80f83f90686f4b647624037826a9eb5746d",\n    "105237": "fbcd2d82f2b37d270fe5bf287d19547d542ee61c07b45752c204fffd8ce864c1",\n    "435920": "05f9d38444da410b6e9eab98ae8f685812fc7be14e8e56f304776dd49304456f",\n    "463114": "e16c17b27ff9895711c61fb04c72d0961c14ea8b240154544be1d6ca7d2242f5",\n    "238757": "90ae7e5258ad4b099064d5cc371972af23d159b6ad64448064c92175e8231407",\n    "398350": "9c6f70d7e72a98e033556248b254d47571e431c3837bbfb6370e11a46ecc38cd",\n    "314398": "6e54d746a29ff0dba5e82aa7b7b94f45928e1388d61c245c0604a4e75fe454f7",\n    "168287": "aee2b04ad009674a9c44c14267caca84b6216f9bf5359d4b8bc40c594c0bc749",\n    "233477": "5fcc321c6730d01a37837123e80b795811811891e3f2dc6bce18d06d9f0ac4f3",\n    "111624": "22814586ba0f161d35719abf3685dafade49baa6a9ccdd6e29b8cf90fa94d47c",\n    "558467": "e53daa3f50d29e5bfe1ec1e76c605368ba99779dd88f21e205670d07149f808a",\n    "217583": "972c15c8c69f4a21174cf66f8c9274d1e3af126bd08dc3876d17e3e42ca0864f",\n    "307464": "6e55ddae11b94dae3cc4f1067f28762e2b975db7ed28a0bd9bd58f08ad36d761",\n    "59552": "12b54f59f37125fa7789ea3a87432d304ffeb35804d5d21d73d265c7fc0d5912",\n    "579487": "8e6f65a3c8f0245737f074fe74ab61ae26cd7054840d986524b0a17abe428723",\n    "55559": "803be71f1af01445123cf3483660fb33b9399d43ae8212657e304439365619e8",\n    "171235": "1ba179c48dc7a62511c04ec9118d68451734f5166e65fc466590c4fa8d1a8011",\n    "4739": "d126881cb4d4ad5b683fbbeeeb285578e69f6f26a3c6bb1ef7f7406177ac7373",\n    "327841": "313c12d2d62edf53757de271074fb2831b0feee469af1d3269ecbefd7db13e1e",\n    "362865": "bcac7359ec47abe2914615fbfd5d683bb9f7957eca8b0553b0603ddd54b8bd12",\n    "406807": "e5981d92ea1f470412c83c1b499d4c8534c85941c2f5961454917284cfceea4d",\n    "123412": "44b8a36d2e4ddb16b6bf69043f9a156bb0fd4431b42d02011e04c8734100d221",\n    "113521": "b257bca5cf25c2914fc01b035ca6d00c72d13ea7aaa28865660ef6991b1aa59a",\n    "535113": "c6226b18fdef1dc0c473db9f3bd57ca4b2565355722e985cf943203416a8842b",\n    "350842": "7fd1a882bdd8e2f711b5ad26d4b41df4cfe23d61a2685210e1cc8f473204ecf1",\n    "8965": "781d3cbf28b7e0695c18b431c5df9fa4ecbb0593c7f1e08549151989cce00a9d",\n    "224695": "6a97091dd2a20aa714a08b3577b7fe758670cc572e3530cf28c6a8971f203261",\n    "469330": "ea707cf0377e818e21858186ac93daa5b30ec9c7843aef4abdd7fd0195be3b32",\n    "380733": "5541aed252bdf0854d4e7ff7b6bffbf2e22c81ca70007dc11665479a5bc5418e",\n    "86654": "6146a3ee767e412dae76e08747ff8f562a2f56d46690fb864df1a93d77ea1265",\n    "524834": "10184f1af56d0ee9cf8014f64a6874bd3fbcbe9052b1bc3955e59e244bfbd625",\n    "78548": "e9ebd75e3dae6e53e89f6ce5f49339f89f77d34bfbd271370d63a4a445f00382",\n    "404361": "5adfce34fb5bdeca86677c8cf93ccc184d4e6b1bf6bef59cf39ca20b3ef57ae7",\n    "253965": "509f85733f7f4cec3dccd4af3a9d8a4acc3a1eccdee60b11fabe292ec672c834",\n    "211325": "55684be7e4281d3b5b8e831a4987e4cef197b66b0c067389e08617eeb8db0ed4",\n    "264511": "bc37b1fff14ba9c1f1931c7eee96d6bbe622f465774f8afac5809ce3fa6eead2",\n    "117313": "fafb2b65b5118fd092da1abf7d6f2838515208142cdfd640a3d25abe4e7c613d",\n    "152286": "0b7d627b3c0afbc51f242f80ca51150f19a2a865b91d5aa59acfb165df40aed2",\n    "10784": "df5d08e192fa176f411eddbd3241deb60dbf8e9b7294355db756168628578a6e",\n    "560202": "8c9734a9bf8e1000523b4502521133d2552fb59a5c9a6d0ce1418e8743296fd0",\n    "24072": "8afea17a960b65f86cf318c666a6d89ba464837b67bc03a1c54798937a2ce1f8",\n    "279753": "612bba5d9808482be78bf16ba8cbbb3cabb9e711d14946a55e43d28cc4124b7a",\n    "326454": "3a2beb93368b54160b2e72d95e1762a50841f3f57b8e25de84fc777e406cd459",\n    "115374": "690e459d0bae6a9181b55518cd013b6f09964cb90d911d282d498715f8586f4b",\n    "227207": "8c75307a0c8e0fd2f487801394e40555ddee74155749d6dfba089fc20b4ad1f3",\n    "240904": "8c16a2a3d6c56fec7dad6d3d2fa54d671a5f60831bf22ff29cfd3b605fded912",\n    "193709": "cf7902733b74dd735f19b7c63949e6a63a10279fc724bc45e01d7d237caa37ba",\n    "155999": "db99f6666ed948626ff9ca94de0194e9ffbaf9ff07551ee3d91f822900c9a980",\n    "19552": "9205f8a8ce4be3f6cc7d23f9d18904b19c5fa65b949ef570648fb0342135d1a9",\n    "171959": "0718103de104d41f85be7c4204ffe853e9be25b49bcac85b398ee1d5bba20e40",\n    "257690": "99afa3066a9d6a12a3dc3e0f61f1ca5201e64c3b2fd90f82ae9f01ea81665efb",\n    "164655": "f4cdd483a2607de5952d81abe726835b7383389442c96983edf5cfbafbf82875",\n    "137824": "b4a30482bc33ce62a267c27800676ce89c6262b955b3071d9df14a6d380a5c97",\n    "335534": "e4e0f8a7e941949172c82e0c11e734ec1783fd43d771d12d5416edbec65d0aeb",\n    "101904": "136768c0d4ccbc03f50bd4ed9f99a24a3a8679398de862b44461a85091b5a2f9",\n    "265594": "d60f8bf27b58d0aaa97ab154c68db345fa3c40c81ae242c54e5d4840b401e6e5",\n    "64036": "66a3532a8a00170ed703a775ec9c2319dff29350daa8d67cefc20df56aa2e568",\n    "439251": "3e2f2b06d0f7220f8d879322d22ad9acd6368f3c9e2513e0238a572a81988de4",\n    "278494": "47b59f6a2881eb7800722197fbeff640b15a3cdb36841bcc5948b60870f9e268",\n    "359156": "38e4636720465cd0db1b777e103576084c87be61c260e06bd90e9b5cf5b744ec",\n    "813": "da5ef9621fa605e9c4140c54a9985090bbae0aad74898cb07b080b18f5d7e140",\n    "264838": "4f455500ff17caaa0c75d77aea8e555384e0f33bc2f5b8233e91cc581e53b335",\n    "483305": "04572bb4ac0685ebfe5ae2516f95b37b728a33af81706e8bcc0433942db9d5c7",\n    "321355": "1fc5723a2d671a2cbc09fa1befe8b4eb1d2e84e260cb2fd2bad92da96920c521",\n    "181208": "38fed0968423114fbafc2c586e2a1a42671f208d5577e26621afd1dc3193a776",\n    "136668": "23e948daf896b9251092212724bba1fffe82d2e9128f790ecb8116ac16affefd",\n    "43522": "809d3e98020abf9cf28ce99d062129845ad648e3543dd73bbef6b3e10c35967f",\n    "191473": "99028c8ba241596742f0ff65a6106ddb5413eaa2515e51abc9ba7d15f6c8e595",\n    "546511": "d6b798e6a09f1b79f811f6be6bd20c372d72595d4df28f6fc2344f4b14ac4e72",\n    "517139": "49fa03f0e7bbb6863f01cf53a99a5249db68fe6533d4b7566279a17c733d7c2f",\n    "120670": "cce1f2ffa0d26a64626146e747187c501ce1f6fe88f6cbf3f3977bbedd171265",\n    "30508": "44660d6e08cfbda6d2fc92d7d1051910e8f72565256687781751b4d96e27134f",\n    "120376": "86fe41ad39a83ed6178214fb2266a72b63f7cd80d2cdc7c26a7700e016783dc7",\n    "365729": "3ec3e0defb1530990db938816d2554ae8803792041fb0b6e0d1e4196e2820163",\n    "210086": "db6de2befe3187ad14650a4b95798ac4445ba7fb6a8d35a96d0f9124b6164ffc",\n    "221685": "f3547bc0d50c213f8cf972cbd7ceb815da1ae93334aa6d512d98254e2ef6c211",\n    "352523": "bf9d65de0bdd004ad294d6ddfff1f1b692e96805dd9fa90d206bf4ee052fa058",\n    "274463": "f1be1082bc800b2f23cbf8380eae0c4d94dda82e02a877b9fe11aebac621ccc1",\n    "528563": "eee2cd5ca9e99240dfad7808775d65e1783f14831d9fee4f6bfbeca9f79adafc",\n    "68701": "cb266574e16e6b115f7b4bc6c821ac472171e38e718e3db0d5c9ab7df1207136",\n    "128695": "11707b3ba6526c3c0cbf78d7ca9aaad21517301e7c12683a27afd935107383c0",\n    "289591": "2bd84b4bfc7e380a3d7a8f5cf9a537390d80e71cbf51d64beb8f6b6346be308b",\n    "467502": "6d921c5a0fcae3763fd942727da078470adda92cc83dc706d73dba2cba21927c",\n    "139932": "25ee5860b12b0607d8b043d83342bfd402e9e464e46254e95cd979d7c899205d",\n    "96897": "396949c41f0e3dec799d7f5972682dc1961c5d4c1d71b28255404bb6335f88e5",\n    "420400": "67342100dde68202712ba843e250d26e5542f68c436c2d8c3e3652fcf7024774",\n    "265237": "e7594a46cefd002a46ffac9734b4745d01d6c3f5163ed052481fbf25683b0735",\n    "285486": "36596ab1d982f33b479f6def8c94a889d5665838af17f26f839b4d82e5bfccdd",\n    "474373": "4d6ef18d4c8f45bb2fde2da474bdb698f253819e98bf6f889bba63e468ad7a6e",\n    "215805": "433e57ddb152276a25cc44a0e7ecac4eddcfb5bbf8caa77ca1f3da926c4c02ce",\n    "309024": "97b168ace9f5fab89539fab8aa49b4f8b2e4983bcc98a95aa5e0cf7e03b8cbc3",\n    "310751": "1bbb7d4eaf75541a7f9778aef000824842f83de0f1f68c31e10544323211894d",\n    "202918": "c55da768e68f92a5f3b24b1bee417eb7b0e598e3272b5c83c2d24c5c8b90c752",\n    "48432": "a48d3e7652c5d39f9023503ba84847571d59a006d2ff382659392c514c4a3e89",\n    "31434": "635c6afb4aad6c99599bc9d6f5f404b147fc1008545df1ee7a2eb7d923b50070",\n    "120529": "1641e51fa6f9f0ff57e003dc2c38c6edf3ca599be1d84579a909ab63a1fdf79d",\n    "326533": "2cb43975310d4faf304c6edf0592c07f26e574713216ba367ed6be8c89e309af",\n    "388779": "7e719c3bf20c0d11dc64e100ae4eb8e887caf2227bc4cba503aa1bd841d73d03",\n    "279810": "879589c20efbc41634157f6dff652494e80abc67a862e646d6b88459055a85f5",\n    "362718": "0bfeb72953d5f50f42c04fc55a88ce2868c3a7144e7fe51a9a655c0b5a013a2d",\n    "280642": "b0832207bc65fbb395628cef07335b9cd149fd8af37868461d070a472d7078f7",\n    "533019": "fd3ae938dab4ca770bdae70109517e65cd97f272d411bea1a61522de083b5f50",\n    "244646": "bdfc06549972082fd2f034fd66f1fae055d7db6d654430678f8b8b004fb8d793",\n    "272653": "a5a12eb2f24e83c4dd95abdab37844a06ee65389ab5f59f3c5abe2da6bf7b3e5",\n    "155198": "0d46f5a028b01380b8ae307e6e9cfe7b6f6fc5526318f33623f2e9cf115e80fa",\n    "499913": "03c214fbed87a845c6a72393564e1c3db924af88a5e263682b2977ce42948ce8",\n    "378911": "7195b91a872d412651ce971eaed1a1fb3bd27375bc8b9be9bb9bf5fcb1b7c128",\n    "441544": "b9b43a9e89e788d165bd9af7c16eeef70a115ba85ac5d09fa0171e9f4d6510f1",\n    "502748": "1e93412ac2376517780fe1dc32208805ae35839dcca843775d5d484ead5189a8",\n    "363951": "12bd9289cf80055058ee4e28c4e9b5dc8dfd5a21bea764cc0d3f0349c74a8fca",\n    "534443": "7709d6517a5663b9256617be43423f8a1f5007f655b21e412c4b2f5c70464b61",\n    "481810": "8fdae48c50c02a621eb78c80758bafe4c654a9f3f8436e1e6f6ef01cda389ecb",\n    "80084": "cf7fb3705a16258c357c0f19b033ef1087d58119c4ec1cef482493d8cd835c21",\n    "104047": "dd8fbd0f25debcd7201346d5d9726fa64f353e8ddcecf13369afb3f834ba18e3",\n    "364508": "751a0b7bc43ef3059e3121861d971f3303c2a7f4f5b276d16ca3c6bd650915da",\n    "389463": "e2a6d6c2498931351d2a297bceface58f40ff5cd776bd9b34ed0f74871e63cd7",\n    "551364": "4270511196de99fc8af0fbb8b153d6f38682f05892736d9825af5b48176f4c15",\n    "98915": "d7c8c2787ec9c8c1612cc8d03bdf6e28a6e883893ce66bb3f109bed664d9cb84",\n    "124501": "830e655d6b7009957288a85630e3fbb6a12ea08b774d8098947f650bcc159b28",\n    "564870": "8a859e1765dde7e6ddd70a67eea4a38480c7ad6b71193be84bdd0174e1a26dc2",\n    "90359": "554822fe63f0d15342730bdc13d5ca2eb67ca7b57cf4eed87a2a0be660766676",\n    "459328": "4dae4247f1d6dbb48fc15b7b45dc2285f82e82e104a1535e0c5f81a374a83fb8",\n    "549366": "67d2d90dba293c145a3e0579bab8caaa2db1fda41c839969413ccf0547491b7c",\n    "286492": "48e6db001fd929193b3e5b7dc4742ff3b9257c082f96477785ad6eabc593a179",\n    "412680": "95e13575633c3887100ff588166d795a5ca0d11e42f81658382e0d856e41e2a7",\n    "21204": "93892a0dcb29eeb1c62584f5a9ab33b983ca39f1823bdf2257dcfc2998ec6b1d",\n    "504888": "b9f3c97cc6ed82c7e31d10b6c55e5eddf75a176f12d2e73fa486c39172547fd2",\n    "480575": "81e3a9e5976d2de66824499b386c852f39a525ad47b3173448812877ed1a15b8",\n    "38628": "8214e538094c1503b962000205e912b1969cbb75192568baffb068b558cbf75c",\n    "188375": "be115512826f0472f81d768e629c8ab6c3fe7d1a0f224b2453471f3b9f57bd0e",\n    "94549": "ca2f2d3e44dc68c9a949db0a654e54ad5240bc9e55068437f46b65ce2e9bc291",\n    "156943": "266b92daf557112a6cb77308323038cf8962443b27276ae0e19858e929111b38",\n    "499997": "0ef21190d80183275b2e26e0c6392565f261e5655a55bde9c1fe2a3586301b03",\n    "455328": "524cfa1c030f05c0fd5c488fbb68848e11882bb0be37e3b34e7c9db517eedf24",\n    "150981": "307429a56936fb1be5369c9e1f10fd6e69af299a1532f2952ba74a23bbae76a6",\n    "239504": "f03693e11fcf2784d3dc6cfdd196397cc283392acfa2d266fae05e158b234148",\n    "319731": "df5a06025b554cbd12e866715cb0dfaea78d2e3f24e95ef6319826808eac0815",\n    "52929": "813cb2627d2bfb32d064bbb910c615161010c0758841e2175d1482b405bfee78",\n    "95178": "9acf4ffc1a10e28c8f001c3d9b9aa9584be27704f8a424734122abd49a15bf4c",\n    "104588": "094574f2af03d16942b841094e6c5904c1aefb39aecf62e06d821c995c00bfa5"\n  }\n}\n', 'test199_data.py': '"""Fixed TEST199 metadata/staging contracts; no neural imports."""\nfrom collections import Counter\nimport hashlib\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom full_data import encode_tokens\nfrom patch_data import DATA_SHA,COUNTS\n\nHERE=Path(__file__).resolve().parent;TEST_SIZE=5000\nCHECKPOINT_SHA=\'5061e305d98edf54930a200090a41c983c354d1ba2dfcf700b715cf20a5b80b7\'\nVERSION=\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\'\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu128\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):\n        raise RuntimeError(\'Unreviewed Torch runtime; expected training2.11.0+cu128 and CUDA inference2.11.0+cu128/cu130. Preserve log; do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Observed CUDA builds of the same publicTorch2.11.0; strictweights and GPU beam-path checks required.\',\n        bitwise_cross_build_equivalence_established=False)\n\ndef validate_lock(lock):\n    if lock[\'source_version\']!=VERSION or lock[\'checkpoint_sha256\']!=CHECKPOINT_SHA or lock[\'epoch\']!=10 or lock[\'updates\']!=177110:raise ValueError(\'Wrong fixed199 checkpoint\')\n    if lock[\'beam_width\']!=5 or lock[\'max_new_tokens\']!=20 or lock[\'length_normalization\'] or lock[\'reranking\']:raise ValueError(\'Changed TEST decoding rule\')\n    ids=lock[\'expected_test_ids\']\n    if len(ids)!=TEST_SIZE or ids!=sorted(set(ids)) or any(type(x)!=int for x in ids):raise ValueError(\'Wrong fixed TEST cohort\')\n    import json\n    if hashlib.sha256(json.dumps(ids,separators=(\',\',\':\')).encode()).hexdigest()!=lock[\'test_id_sha256\']:raise ValueError(\'Changed cohort SHA\')\n    return lock\n\ndef lock():return validate_lock(read(HERE/\'TEST199_LOCK.json\'))\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted([*root.rglob(\'best.bin\'),*root.rglob(\'checkpoint.bin\')],key=lambda p:(p.name!=\'best.bin\',str(p)))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Need exact199 checkpoint bytes; no latest/other version fallback\')\n    return matches[0]\n\ndef selected_test_rows(ds,locked):\n    if dict(Counter(r[\'split\'] for r in ds[\'images\']))!=COUNTS:raise ValueError(\'Wrong Karpathy split counts\')\n    if len({int(r[\'cocoid\']) for r in ds[\'images\']})!=len(ds[\'images\']):raise ValueError(\'Duplicate source COCO IDs\')\n    rows=sorted([r for r in ds[\'images\'] if r[\'split\']==\'test\'],key=lambda r:int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in rows]!=locked[\'expected_test_ids\']:raise ValueError(\'Wrong TEST IDs\')\n    return rows\n\ndef prepare(cfg,locked=None,weight_path=None):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False);locked=lock() if locked is None else locked;weight_path=checkpoint(cfg[\'checkpoint_root\']) if weight_path is None else weight_path\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO dataset JSON\')\n    ds=read(cfg[\'dataset_json\']);rows=selected_test_rows(ds,locked);wanted={int(r[\'cocoid\']):i for i,r in enumerate(rows)};words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    gold=read(HERE/\'SMOKE_SOURCE.json\');save(out/\'vocabulary.json\',words)\n    if sha(out/\'vocabulary.json\')!=gold[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from199\')\n    save(out/\'test_references.json\',[dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]) for r in rows])\n    sequences=[];caption_image=[]\n    for i,r in enumerate(rows):\n        for s in r[\'sentences\']:sequences.append(encode_tokens(s[\'tokens\'],lookup));caption_image.append(i)\n    np.save(stage/\'sequences.npy\',np.asarray(sequences,np.int32));np.save(stage/\'caption_image.npy\',np.asarray(caption_image,np.int32));np.save(stage/\'ids.npy\',np.asarray([r[\'cocoid\'] for r in rows],np.int64))\n    patches=np.lib.format.open_memmap(stage/\'patches.npy\',mode=\'w+\',dtype=np.float16,shape=(len(rows),196,768));seen=set();sources=[]\n    try:\n        for path in sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\')):\n            with h5py.File(path,\'r\',rdcc_nbytes=2*1024**2) as f:\n                text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n                import json\n                if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n                meta=json.loads(text(f.attrs[\'metadata_json\']))\n                if {k:meta.get(k) for k in gold[\'feature_profile\']}!=gold[\'feature_profile\']:raise ValueError(\'Feature profile differs from199\')\n                ids=f[\'imgids\'][:];features=f[\'features\']\n                if features.shape!=(len(ids),197,768) or features.dtype!=np.float16:raise ValueError(\'Invalid original H5 shape/dtype\')\n                positions=[i for i,x in enumerate(ids) if int(x) in wanted]\n                for start in range(0,len(positions),128):\n                    pos=positions[start:start+128];raw=features[pos]\n                    if not np.isfinite(raw).all():raise ValueError(\'Nonfinite TEST patches\')\n                    for j,p in enumerate(pos):\n                        coco=int(ids[p]);image=wanted[coco]\n                        if coco in seen or text(f[\'filename\'][p])!=rows[image][\'filename\']:raise ValueError(\'Repeated TEST ID/filename mismatch\')\n                        values=raw[j,1:];patches[image]=values\n                        if not np.array_equal(patches[image].view(np.uint16),values.view(np.uint16)):raise ValueError(\'TEST patch bytes changed\')\n                        seen.add(coco)\n                    print(\'TEST patches staged\',len(seen),\'/\',len(rows),round(time.monotonic()-started),\'s\',flush=True)\n                sources.append(dict(path=str(path),bytes=path.stat().st_size,selected_TEST_rows=len(positions)))\n        if seen!=set(wanted):raise ValueError(\'Missing TEST features\')\n        patches.flush()\n    finally:patches._mmap.close()\n    receipt=dict(status=\'PATCH_TEST199_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),feature_profile=gold[\'feature_profile\'],sources=sources,\n        dataset_sha256=DATA_SHA,training=False,TRAIN_VALID_features_read=False,seconds=time.monotonic()-started,\n        stage_files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},\n        output_files_sha256={n:sha(out/n) for n in (\'vocabulary.json\',\'test_references.json\')})\n    save(out/\'stage_receipt.json\',receipt);save(out/\'test_lock.json\',dict(**locked,checkpoint=str(weight_path),stage_receipt_sha256=sha(out/\'stage_receipt.json\')))\n\ndef load(cfg,verify=True,validator=None):\n    out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);r=read(out/\'stage_receipt.json\');locked=(validator or validate_lock)(read(out/\'test_lock.json\'))\n    if r[\'status\']!=\'PATCH_TEST199_STAGE_COMPLETE\' or r[\'images\']!=TEST_SIZE or r[\'TRAIN_VALID_features_read\']:raise ValueError(\'Invalid TEST stage scope\')\n    if locked[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\'):raise ValueError(\'Changed TEST receipt\')\n    if verify:\n        for n,h in r[\'stage_files_sha256\'].items():\n            if Path(n).name!=n or sha(stage/n)!=h:raise ValueError(\'Changed staged TEST \'+n)\n        for n,h in r[\'output_files_sha256\'].items():\n            if Path(n).name!=n or sha(out/n)!=h:raise ValueError(\'Changed TEST output \'+n)\n    return {n:np.load(stage/(n+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for n in (\'patches\',\'ids\',\'sequences\',\'caption_image\')},read(out/\'vocabulary.json\'),r,locked\n', 'run_test199.py': '"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import semantic_test_data as module\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from semantic_patch_model import SemanticPatchCaptioner\n            from semantic_data import SemanticContext\n            model=SemanticPatchCaptioner(len(words),ck[\'model\'][\'noun_bank\'],ck[\'model\'][\'verb_bank\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'TEST decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==d.CHECKPOINT_SHA for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong TEST references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=tok.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete TEST scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CLIP_PATCH_BANK_PROMPT_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n', 'test_test199.py': '"""TEST199 NumPy/mock metadata checks; never import Torch/scorers."""\nimport ast\nimport copy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport test199_data as d\nimport run_test199 as run\n\nclass Checks(unittest.TestCase):\n    def test_observed_cuda_build_change_requires_same_framework_release(self):\n        self.assertFalse(d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu128\')[\'cuda_wheel_changed\'])\n        policy=d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu130\');self.assertTrue(policy[\'cuda_wheel_changed\']);self.assertFalse(policy[\'bitwise_cross_build_equivalence_established\'])\n        for current in (\'2.11.0+cpu\',\'2.11.0\',\'2.12.0+cu130\',\'2.11.0+cu129\'):\n            with self.assertRaises(RuntimeError):d.runtime_policy(\'2.11.0+cu128\',current)\n        with self.assertRaises(RuntimeError):d.runtime_policy(\'2.10.0+cu128\',\'2.11.0+cu130\')\n    def test_frozen_checkpoint_and_decoding_rule(self):\n        gold=d.lock();self.assertEqual(len(gold[\'expected_test_ids\']),5000)\n        for key,value in [(\'beam_width\',4),(\'max_new_tokens\',21),(\'reranking\',True),(\'source_version\',d.VERSION.replace(\'/199\',\'/200\')),(\'checkpoint_sha256\',\'0\'*64)]:\n            wrong=copy.deepcopy(gold);wrong[key]=value\n            with self.assertRaises(ValueError):d.validate_lock(wrong)\n    def test_checkpoint_accepts_exact_file_or_both_names_and_rejects_other(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/\'best.bin\').write_bytes(b\'fixture\');(root/\'checkpoint.bin\').write_bytes(b\'fixture\')\n            with patch.object(d,\'CHECKPOINT_SHA\',d.sha(root/\'best.bin\')):\n                self.assertEqual(d.checkpoint(root),root/\'best.bin\');self.assertEqual(d.checkpoint(root/\'checkpoint.bin\'),root/\'checkpoint.bin\')\n                (root/\'best.bin\').write_bytes(b\'wrong\');(root/\'checkpoint.bin\').write_bytes(b\'wrong\')\n                with self.assertRaises(ValueError):d.checkpoint(root)\n    def test_only_TEST_features_exact_bytes_vocab_and_seal(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);out=root/\'out\';out.mkdir();rawdir=root/\'raw\';rawdir.mkdir();(rawdir/\'one.h5\').write_bytes(b\'fake header\')\n            rows=[dict(cocoid=(i+1)*10,split=s,filename=f\'{i}.jpg\',sentences=[dict(tokens=[\'a\',\'dog\'],raw=\'a dog\')]) for i,s in enumerate((\'train\',\'test\',\'train\',\'val\',\'test\'))]\n            dataset=root/\'coco.json\';d.save(dataset,dict(images=rows));(root/\'best.bin\').write_bytes(b\'fixed checkpoint\')\n            words=[\'<pad>\',\'<bos>\',\'<eos>\',\'<unk>\',\'a\',\'dog\'];d.save(root/\'expected_vocab.json\',words);profile=dict(model_id=\'openai/clip-vit-base-patch16\')\n            locked=d.lock();locked.update(checkpoint_sha256=d.sha(root/\'best.bin\'),expected_test_ids=[20,50],test_id_sha256=hashlib.sha256(b\'[20,50]\').hexdigest())\n            raw=np.full((5,197,768),np.nan,np.float16);raw[1]=3;raw[4]=7;raw[:,0]=19;reads=[]\n            class Features:\n                shape=raw.shape;dtype=raw.dtype\n                def __getitem__(self,pos):\n                    if set(pos)-{1,4}:raise AssertionError(\'TRAIN/VALID row read\')\n                    reads.extend(pos);return raw[pos]\n            class Handle(dict):\n                attrs=dict(complete=True,id_key=\'coco_id\',metadata_json=json.dumps(profile))\n                def __enter__(self):return self\n                def __exit__(self,*a):pass\n            h=Handle(features=Features(),imgids=np.arange(1,6)*10,filename=np.array([f\'{i}.jpg\'.encode() for i in range(5)]))\n            original=d.read\n            def read(p):return dict(feature_profile=profile,vocabulary_sha256=d.sha(root/\'expected_vocab.json\')) if Path(p).name==\'SMOKE_SOURCE.json\' else original(p)\n            cfg=dict(output=str(out),stage=str(root/\'stage\'),dataset_json=str(dataset),visual_cache=str(rawdir),checkpoint_root=str(root/\'best.bin\'))\n            with patch.object(d,\'TEST_SIZE\',2),patch.object(d,\'CHECKPOINT_SHA\',locked[\'checkpoint_sha256\']),patch.object(d,\'DATA_SHA\',d.sha(dataset)),patch.object(d,\'COUNTS\',dict(train=2,val=1,test=2)),patch.object(d,\'lock\',return_value=locked),patch.object(d,\'vocabulary\',return_value=words),patch.object(d,\'read\',side_effect=read),patch.dict(sys.modules,{\'h5py\':SimpleNamespace(File=lambda *a,**k:h)}):\n                d.prepare(cfg);data,got,receipt,_=d.load(cfg)\n                try:\n                    self.assertEqual(reads,[1,4]);self.assertEqual(got,words);np.testing.assert_array_equal(data[\'patches\'][0],raw[1,1:]);np.testing.assert_array_equal(data[\'patches\'][1],raw[4,1:]);self.assertFalse(receipt[\'TRAIN_VALID_features_read\'])\n                finally:\n                    for v in data.values():v._mmap.close()\n                (out/\'test_references.json\').write_bytes(b\'changed\')\n                with self.assertRaises(ValueError):d.load(cfg)\n    def test_all_eight_metrics_required_before_complete(self):\n        with tempfile.TemporaryDirectory() as td:\n            out=Path(td);d.save(out/\'scoring_lock.json\',dict(files_sha256={}));d.save(out/\'chair_counts.json\',{});d.save(out/\'environment.json\',dict(fixture=True))\n            parts={\'basic\':dict(BLEU_1=.7,BLEU_4=.3,CIDEr=1.1,ROUGE_L=.5),\'meteor\':dict(METEOR=.2),\'spice\':dict(SPICE=.15),\'chair\':dict(CHAIRs=.05,CHAIRi=.03,Recall=.4)}\n            for mode,metrics in parts.items():d.save(out/(mode+\'.json\'),dict(images=5000,metrics=metrics,input_sha256={}))\n            with patch.object(run,\'inputs\',return_value=(out,[],[],[dict(NLL_sum=6,NLL_tokens=3,runtime_policy=d.runtime_policy(\'2.11.0+cu128\',\'2.11.0+cu130\'))])),patch(\'builtins.print\'):\n                run.combine(dict(output=str(out)));self.assertEqual(d.read(out/\'result.json\')[\'metrics\'][\'NLL\'],2.)\n                parts[\'spice\']={};d.save(out/\'spice.json\',dict(images=5000,metrics={},input_sha256={}))\n                with self.assertRaises(ValueError):run.combine(dict(output=str(out)))\n    def test_source_parse_no_training_no_neural_import_during_audit(self):\n        folder=Path(__file__).parent\n        for n in (\'test199_data.py\',\'run_test199.py\'):ast.parse((folder/n).read_text(encoding=\'utf-8\'))\n        source=(folder/\'run_test199.py\').read_text(encoding=\'utf-8\')\n        for bad in (\'backward(\', \'torch.optim\', \'optimizer.step\', \'model.train(\'):self.assertNotIn(bad,source)\n        self.assertNotIn(\'torch\',sys.modules);self.assertNotIn(\'pycocoevalcap\',sys.modules)\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'patch_model.py': '"""User-run neural baseline: one patch memory, standard causal decoder."""\nimport math\nimport torch\nfrom torch import nn\n\nclass PatchCaptioner(nn.Module):\n    def __init__(self,vocab,dim=512,heads=8,layers=4,hidden=2048,dropout=.1,maximum=64):\n        super().__init__();self.dim=dim\n        self.project=nn.Sequential(nn.Linear(768,dim),nn.LayerNorm(dim))\n        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)\n        p=torch.arange(maximum,dtype=torch.float32)[:,None];scale=torch.exp(torch.arange(0,dim,2)*(-math.log(10000)/dim))\n        pos=torch.zeros(maximum,dim);pos[:,0::2]=torch.sin(p*scale);pos[:,1::2]=torch.cos(p*scale)\n        self.register_buffer(\'positions\',pos,persistent=True);self.dropout=nn.Dropout(dropout)\n        layer=nn.TransformerDecoderLayer(dim,heads,hidden,dropout,batch_first=True,norm_first=False)\n        self.decoder=nn.TransformerDecoder(layer,layers,norm=nn.LayerNorm(dim));self.head=nn.Linear(dim,vocab)\n        for module in self.modules():\n            if isinstance(module,nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:nn.init.zeros_(module.bias)\n            elif isinstance(module,nn.Embedding):nn.init.normal_(module.weight,std=.02);module.weight.data[0].zero_()\n            elif isinstance(module,nn.MultiheadAttention):nn.init.xavier_uniform_(module.in_proj_weight);nn.init.zeros_(module.in_proj_bias)\n\n    def encode(self,patches):\n        if patches.ndim!=3 or patches.shape[1:]!=(196,768):raise ValueError(\'Wrong patch shape\')\n        return self.project(patches)\n\n    def decode(self,memory,prefix):\n        if prefix.ndim!=2 or len(prefix)!=len(memory):raise ValueError(\'Wrong memory/prefix shape\')\n        if prefix.shape[1]>len(self.positions) or (prefix==0).all(1).any():raise ValueError(\'Empty or too long prefix\')\n        text=self.dropout(self.embedding(prefix)*math.sqrt(self.dim)+self.positions[:prefix.shape[1]])\n        causal=torch.ones(prefix.shape[1],prefix.shape[1],dtype=torch.bool,device=prefix.device).triu(1)\n        output=self.decoder(text,memory,tgt_mask=causal,tgt_key_padding_mask=prefix==0)\n        return self.head(output)\n\n    def forward(self,patches,prefix):return self.decode(self.encode(patches),prefix)\n\n    @torch.no_grad()\n    def greedy(self,patches,maximum=20):\n        ids=torch.ones(len(patches),1,dtype=torch.long,device=patches.device);done=torch.zeros(len(patches),dtype=torch.bool,device=patches.device)\n        for _ in range(maximum):\n            word=self(patches,ids)[:,-1].argmax(-1);word=word.masked_fill(done,0)\n            ids=torch.cat((ids,word[:,None]),1);done|=word==2\n            if done.all():break\n        return ids\n', 'patch_beam.py': '"""User-run batched beam search; no model import at module load."""\ndef beam_search(model,memory,width=5,maximum=20):\n    import torch\n    if model.training or not 1<=width<=model.head.out_features-2:raise ValueError(\'Beam requires eval model/valid width\')\n    b,n,d=memory.shape;vocab=model.head.out_features;device=memory.device\n    seq=torch.ones((b,width,1),dtype=torch.long,device=device);score=torch.full((b,width),-torch.inf,device=device);score[:,0]=0\n    done=torch.zeros((b,width),dtype=torch.bool,device=device);expanded=memory[:,None].expand(-1,width,-1,-1).reshape(b*width,n,d)\n    for _ in range(maximum):\n        logits=model.decode(expanded,seq.reshape(b*width,-1))[:,-1].float();logp=logits.log_softmax(-1).view(b,width,vocab)\n        logp[:,:,:2]=-torch.inf;logp=logp.masked_fill(done[:,:,None],-torch.inf);logp[:,:,0]=torch.where(done,torch.zeros_like(score),logp[:,:,0])\n        score,flat=(score[:,:,None]+logp).flatten(1).topk(width,dim=1)\n        parent=flat//vocab;token=flat%vocab\n        seq=torch.cat((seq.gather(1,parent[:,:,None].expand(-1,-1,seq.shape[-1])),token[:,:,None]),-1)\n        done=done.gather(1,parent)|(token==2)\n        if done.all():break\n    return seq,score\n\ndef check_paths(model,memory,seq,scores):\n    import torch\n    from torch.nn import functional as F\n    b,k,length=seq.shape;vocab=model.head.out_features\n    if not torch.isfinite(scores).all() or (scores[:,:-1]<scores[:,1:]).any() or not (seq[:,:,0]==1).all():raise ValueError(\'Invalid beam paths/scores\')\n    target=seq[:,:,1:];before_eos=(target==2).cumsum(-1)-(target==2).long();mask=before_eos==0\n    if ((target<2)&mask).any() or ((target!=0)&~mask).any():raise ValueError(\'Invalid BOS/PAD/EOS path\')\n    mem=memory[:,None].expand(-1,k,-1,-1).reshape(b*k,*memory.shape[1:]);prefix=seq[:,:,:-1].reshape(b*k,-1)\n    logits=model.decode(mem,prefix).float();logp=F.log_softmax(logits,-1);logp[:,:,:2]=-torch.inf\n    # Exclude post-EOS PAD without multiplying negative infinity by zero.\n    per=logp.gather(-1,target.reshape(b*k,-1,1)).squeeze(-1).masked_fill(~mask.reshape(b*k,-1),0).sum(-1).view(b,k)\n    torch.testing.assert_close(scores,per,rtol=1e-4,atol=2e-4)\n\ndef text_from_ids(ids,words):\n    out=[]\n    for word in ids[1:]:\n        word=int(word)\n        if word==2:break\n        if word>1:out.append(words[word])\n    return \' \'.join(out)\n', 'patch_data.py': '"""TRAIN-only raw-patch smoke stage; importing never imports a neural library."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom data import vocabulary,encode\nfrom real_bank_data import sha,save,read\n\nDATA_SHA=\'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d\'\nCOUNTS=dict(train=82783,restval=30504,val=5000,test=5000)\n\ndef selected_rows(dataset,count=256):\n    from collections import Counter\n    rows=dataset[\'images\']\n    if dict(Counter(r[\'split\'] for r in rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    ids=[int(r[\'cocoid\']) for r in rows]\n    if len(set(ids))!=len(ids):raise ValueError(\'Duplicate COCO ID\')\n    pool=[r for r in rows if r[\'split\'] in (\'train\',\'restval\')]\n    if not 1<=count<=len(pool):raise ValueError(\'Invalid TRAIN smoke size\')\n    return sorted(pool,key=lambda r:(hashlib.sha256(f"patch-baseline/42/{r[\'cocoid\']}".encode()).hexdigest(),int(r[\'cocoid\'])))[:count]\n\ndef check_features(raw):\n    if raw.ndim!=3 or raw.shape[1:]!=(197,768) or not np.isfinite(raw).all():raise ValueError(\'Expected finite raw [B,197,768]\')\n    if raw.dtype.kind!=\'f\':raise ValueError(\'Expected floating raw features\')\n    return raw[:,1:,:].astype(np.float16)\n\ndef batch(data,events):\n    events=np.asarray(events,dtype=np.int64)\n    if events.ndim!=1 or not len(events) or np.any(events<0) or np.any(events>=len(data[\'sequences\'])):raise ValueError(\'Invalid caption event\')\n    seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n    return np.asarray(data[\'patches\'][data[\'caption_image\'][events]],np.float32),seq\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False)\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    dataset=read(cfg[\'dataset_json\']);rows=selected_rows(dataset);words=vocabulary(dataset)\n    if len(words)!=10000:raise ValueError(\'Wrong TRAIN vocabulary size\')\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)};found=set();profile=None;shards=[]\n    patches=np.empty((len(rows),196,768),np.float16);row_hashes={}\n    paths=sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))\n    if not paths:raise FileNotFoundError(\'Add raw visual-cache Dataset; ROI134 cannot replace raw patches\')\n    for path in paths:\n        if time.monotonic()-started>cfg[\'prepare_seconds\']:raise TimeoutError(\'Raw-cache scan/staging cap exceeded\')\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not bool(f.attrs.get(\'complete\',False)) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete or ambiguous cache IDs\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']))\n            expected=dict(model_id=\'openai/clip-vit-base-patch16\',output=\'vision_model.last_hidden_state\',preprocessing=\'bilinear\',image_size=[224,224],antialias=True,dataset_json_sha256=DATA_SHA)\n            if any(meta.get(k)!=v for k,v in expected.items()):raise ValueError(\'Raw feature profile differs\')\n            np.testing.assert_allclose(meta[\'mean\'],[.48145466,.4578275,.40821073],rtol=0,atol=1e-8)\n            np.testing.assert_allclose(meta[\'std\'],[.26862954,.26130258,.27577711],rtol=0,atol=1e-8)\n            current={k:meta.get(k) for k in (*expected,\'model_revision\',\'mean\',\'std\',\'compute_dtype\',\'storage_dtype\',\'autocast\')}\n            if profile is not None and current!=profile:raise ValueError(\'Mixed raw extraction metadata\')\n            profile=current;ids=f[\'imgids\'][:];features=f[\'features\'];names=f[\'filename\']\n            if features.shape!=(len(ids),197,768) or features.dtype.kind!=\'f\' or features.dtype.itemsize not in (2,4) or len(names)!=len(ids):raise ValueError(\'Raw cache shape/dtype differs\')\n            positions=[i for i,v in enumerate(ids) if int(v) in wanted]\n            if positions:\n                raw=features[positions];values=check_features(raw)\n                for pos,full,value in zip(positions,raw,values):\n                    coco=int(ids[pos]);j=wanted[coco]\n                    if coco in found or text(names[pos])!=rows[j][\'filename\']:raise ValueError(\'Duplicate/mismatched selected raw ID\')\n                    found.add(coco);patches[j]=value;row_hashes[str(coco)]=hashlib.sha256(np.ascontiguousarray(full).tobytes()).hexdigest()\n            shards.append(dict(name=path.name,bytes=path.stat().st_size,shape=list(features.shape),selected_rows=len(positions)))\n        print(\'Raw patch scan:\',path.name,\'TRAIN extracted\',len(found),\'/\',len(rows),flush=True)\n    if found!=set(wanted):raise ValueError(\'Missing selected TRAIN features\')\n    sequences=[];caption_image=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:\n            sequences.append(encode(sentence[\'tokens\'],words));caption_image.append(j)\n    arrays=dict(patches=patches,sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(caption_image,np.int32),coco_ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64))\n    for k,v in arrays.items():np.save(stage/(k+\'.npy\'),v)\n    save(stage/\'vocabulary.json\',words)\n    manifest=dict(status=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),split=\'TRAIN_only\',test_used=False,validation_used=False,\n        dataset_sha256=DATA_SHA,feature_profile=profile,shards=shards,raw_selected_row_sha256=row_hashes,\n        files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},seconds=time.monotonic()-started,\n        limits=\'Inherited producer/revision/preprocessing; no full H5 payload hash or new CLIP extraction. Only selected TRAIN feature rows read.\')\n    save(stage/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\' or m[\'images\']!=256 or m[\'test_used\'] or m[\'validation_used\']:raise ValueError(\'Wrong stage scope\')\n    for n,h in m[\'files_sha256\'].items():\n        if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'patches\',\'sequences\',\'caption_image\',\'coco_ids\')}\n    return data,read(root/\'vocabulary.json\'),m\n', 'full_data.py': '"""Full metadata, schedules and lazy raw H5 reader; no neural imports."""\nfrom collections import OrderedDict,Counter\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport unicodedata\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom patch_data import DATA_SHA,COUNTS,check_features\n\nTRAIN_IMAGES=113287;TRAIN_EVENTS=566747;VALID_IMAGES=5000;VALID_EVENTS=25010\nSTEPS=(TRAIN_EVENTS+31)//32;EPOCHS=10\n\ndef encode_tokens(tokens,lookup):\n    if not isinstance(tokens,list) or not all(isinstance(t,str) and t for t in tokens):raise ValueError(\'Invalid caption tokens\')\n    x=[lookup.get(unicodedata.normalize(\'NFKC\',t).lower(),3) for t in tokens[:52]]\n    return [1]+x+[2]+[0]*(52-len(x))\n\ndef learning_rate(step):\n    if type(step)!=int or not 1<=step<=EPOCHS*STEPS:raise ValueError(\'Invalid update\')\n    if step<=STEPS:return 8e-6+(1e-4-8e-6)*(step-1)/(STEPS-1)\n    ratio=(step-STEPS)/(9*STEPS)\n    return 1e-5+.5*(1e-4-1e-5)*(1+math.cos(math.pi*ratio))\n\ndef order_for_epoch(data,epoch):\n    if type(epoch)!=int or not 1<=epoch<=10:raise ValueError(\'Invalid epoch\')\n    image=np.asarray(data[\'caption_image\'][:TRAIN_EVENTS])\n    if (np.diff(image)<0).any() or len(np.unique(image))!=TRAIN_IMAGES:raise ValueError(\'TRAIN caption mapping differs\')\n    if image.min()!=0 or image.max()!=TRAIN_IMAGES-1:raise ValueError(\'Heldout/foreign TRAIN image index\')\n    return np.random.default_rng(42+epoch).permutation(TRAIN_EVENTS)\n\ndef validate_state(s):\n    epoch=s[\'epoch\'];cursor=s[\'batch_cursor\'];phase=s[\'phase\']\n    if type(epoch)!=int or type(cursor)!=int or not 1<=epoch<=11 or not 0<=cursor<=STEPS or phase not in (\'train\',\'validate\'):raise ValueError(\'Invalid cursor\')\n    if epoch==11 and (cursor or phase!=\'train\'):raise ValueError(\'Invalid completed cursor\')\n    if phase==\'validate\' and cursor!=STEPS:raise ValueError(\'Premature validation\')\n    if s[\'updates\']!=(epoch-1)*STEPS+cursor or s[\'caption_events\']!=(epoch-1)*TRAIN_EVENTS+min(cursor*32,TRAIN_EVENTS):raise ValueError(\'Exposure mismatch\')\n    if [r[\'epoch\'] for r in s[\'history\']]!=list(range(1,epoch)):raise ValueError(\'Incomplete history\')\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();root=Path(cfg[\'stage\']);root.mkdir(parents=True,exist_ok=False);out=Path(cfg[\'output\'])\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    ds=read(cfg[\'dataset_json\']);all_rows=ds[\'images\']\n    if dict(Counter(r[\'split\'] for r in all_rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    mapping={int(r[\'cocoid\']):r for r in all_rows}\n    if len(mapping)!=len(all_rows):raise ValueError(\'Duplicate COCO ID\')\n    train=[r for r in all_rows if r[\'split\'] in (\'train\',\'restval\')];val=[r for r in all_rows if r[\'split\']==\'val\'];rows=train+val\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)}\n    index=np.full((len(rows),2),-1,np.int32);seen=set();profiles=[];sources=[]\n    for number,path in enumerate(sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))):\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']));gold=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'feature_profile\']\n            profile={k:meta.get(k) for k in gold}\n            if profile!=gold:raise ValueError(\'Cache extraction profile differs from smoke192\')\n            ids=f[\'imgids\'][:];names=f[\'filename\'][:];features=f[\'features\']\n            if features.shape!=(len(ids),197,768) or features.dtype!=np.dtype(\'float16\') or len(names)!=len(ids):raise ValueError(\'Raw H5 shape differs\')\n            for i,(coco,name) in enumerate(zip(ids,names)):\n                coco=int(coco)\n                if coco in seen or coco not in mapping or text(name)!=mapping[coco][\'filename\']:raise ValueError(\'H5 ID/filename mismatch\')\n                seen.add(coco)\n                if coco in wanted:index[wanted[coco]]=(number,i)\n            sources.append(dict(path=str(path.resolve()),bytes=path.stat().st_size,shape=list(features.shape),dtype=str(features.dtype),chunks=features.chunks,compression=features.compression))\n        print(\'Full H5 metadata:\',number+1,\'mapped\',int((index[:,0]>=0).sum()),\'/\',len(rows),flush=True)\n    if seen!=set(mapping) or (index<0).any() or len(train)!=TRAIN_IMAGES or len(val)!=VALID_IMAGES:raise ValueError(\'Missing/foreign raw cache coverage\')\n    words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    sequences=[];images=[];refs=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:sequences.append(encode_tokens(sentence[\'tokens\'],lookup));images.append(j)\n        if j>=TRAIN_IMAGES:refs.append(dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]))\n        if (j+1)%20000==0:print(\'Caption metadata:\',j+1,\'/\',len(rows),flush=True)\n    arrays=dict(sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(images,np.int32),ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64),locations=index)\n    if arrays[\'sequences\'].shape!=(TRAIN_EVENTS+VALID_EVENTS,54) or int((arrays[\'caption_image\']<TRAIN_IMAGES).sum())!=TRAIN_EVENTS:raise ValueError(\'Wrong full caption count\')\n    for n,v in arrays.items():np.save(root/(n+\'.npy\'),v)\n    save(root/\'vocabulary.json\',words);save(root/\'validation_references.json\',refs)\n    if sha(root/\'vocabulary.json\')!=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from smoke\')\n    manifest=dict(status=\'FULL_PATCH_METADATA_COMPLETE\',schema=\'raw_h5_patch_stream_v1\',dataset_sha256=DATA_SHA,sources=sources,feature_profile=gold,\n        train_images=TRAIN_IMAGES,train_caption_events=TRAIN_EVENTS,valid_images=VALID_IMAGES,valid_caption_events=VALID_EVENTS,\n        files_sha256={p.name:sha(p) for p in root.iterdir() if p.is_file()},test_features_used=False,feature_arrays_staged=False,seconds=time.monotonic()-started)\n    save(root/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root,verify=True):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'FULL_PATCH_METADATA_COMPLETE\' or m[\'train_caption_events\']!=TRAIN_EVENTS or m[\'test_features_used\']:raise ValueError(\'Wrong stage scope\')\n    if verify:\n        for n,h in m[\'files_sha256\'].items():\n            if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage file \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'sequences\',\'caption_image\',\'ids\',\'locations\')}\n    return data,read(root/\'vocabulary.json\'),m\n\nclass FeatureReader:\n    def __init__(self,root,capacity=256):\n        self.data,_,self.meta=load_stage(root,verify=False);self.handles={};self.cache=OrderedDict();self.capacity=capacity\n        self.golden=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'raw_selected_row_sha256\']\n        self.local=None;self.local_rows=None\n        if self.meta.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            spec=self.meta[\'local_features\']\n            if spec[\'file\']!=\'local_patches.npy\' or spec[\'rows\']!=\'local_rows.npy\':raise ValueError(\'Unexpected local payload names\')\n            self.local=np.load(Path(root)/spec[\'file\'],mmap_mode=\'r\',allow_pickle=False)\n            self.local_rows=np.load(Path(root)/spec[\'rows\'],mmap_mode=\'r\',allow_pickle=False)\n            if self.local.shape!=(len(self.data[\'ids\']),196,768) or self.local.dtype!=np.float16 or self.local_rows.shape!=(len(self.data[\'ids\']),):raise ValueError(\'Invalid local patch mapping/shape\')\n    def get(self,image):\n        image=int(image)\n        if image in self.cache:self.cache.move_to_end(image);return self.cache[image]\n        if self.local is not None:\n            row=int(self.local_rows[image])\n            if not 0<=row<len(self.local):raise ValueError(\'Invalid local row\')\n            value=self.local[row]\n            if not np.isfinite(value).all():raise ValueError(\'Nonfinite local feature\')\n            self.cache[image]=value\n            if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n            return value\n        import h5py\n        file,row=map(int,self.data[\'locations\'][image]);source=self.meta[\'sources\'][file]\n        if file not in self.handles:\n            path=Path(source[\'path\'])\n            if path.stat().st_size!=source[\'bytes\']:raise ValueError(\'Raw file size changed\')\n            self.handles[file]=h5py.File(path,\'r\',rdcc_nbytes=2*1024**2)\n        raw=self.handles[file][\'features\'][row];coco=str(int(self.data[\'ids\'][image]))\n        if coco in self.golden and hashlib.sha256(np.ascontiguousarray(raw).tobytes()).hexdigest()!=self.golden[coco]:raise ValueError(\'Smoke raw row replay differs \'+coco)\n        value=check_features(raw[None])[0];self.cache[image]=value\n        if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n        return value\n    def close(self):\n        self.cache.clear()\n        for f in self.handles.values():f.close()\n        self.handles.clear()\n        if self.local is not None:self.local._mmap.close();self.local=None\n        if self.local_rows is not None:self.local_rows._mmap.close();self.local_rows=None\n\nclass CaptionBatches:\n    def __init__(self,root,events,rank=None,cursor=0,batch_size=32):\n        self.root=str(root);self.events=np.asarray(events,np.int64);self.rank=rank;self.cursor=cursor;self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.events)+self.batch_size-1)//self.batch_size-self.cursor\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;ev=self.events[start:start+self.batch_size]\n        if self.rank is not None:ev=ev[self.rank*16:(self.rank+1)*16]\n        if not len(ev):raise ValueError(\'Empty distributed caption batch\')\n        d=self.reader.data;image=d[\'caption_image\'][ev];x=np.stack([self.reader.get(i) for i in image]).astype(np.float32)\n        seq=np.asarray(d[\'sequences\'][ev],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return x,seq,ev.copy()\n\nclass ImageBatches:\n    def __init__(self,root,images,batch_size=8):self.root=str(root);self.images=np.asarray(images,np.int64);self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.images)+self.batch_size-1)//self.batch_size\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        indices=self.images[item*self.batch_size:(item+1)*self.batch_size]\n        return np.stack([self.reader.get(i) for i in indices]).astype(np.float32),indices.copy()\n\ndef loader(dataset,workers=2):\n    from torch.utils.data import DataLoader\n    return DataLoader(dataset,batch_size=None,num_workers=workers,pin_memory=True,multiprocessing_context=\'spawn\' if workers else None,prefetch_factor=2 if workers else None)\n', 'make_test199_notebook.py': '"""Author fixed199 TEST notebook; never submit/execute inference."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nHERE=Path(__file__).resolve().parent;ROOT=HERE.parents[1];PREFIX=\'experiments/clip_patch_baseline/\'\nsys.path.insert(0,str(ROOT/\'experiments/csa_paper\'))\nfrom make_smoke_notebook import RUNNER,cell\n\ndef make(revision,semantic=False):\n    revision=subprocess.check_output([\'git\',\'rev-parse\',revision],cwd=ROOT,text=True).strip()\n    mapping={PREFIX+n:n for n in (\'TEST199_LOCK.json\',\'TEST199_PROTOCOL.md\',\'TEST199_BUILD_V2_PROTOCOL.md\',\'SMOKE_SOURCE.json\',\'test199_data.py\',\'run_test199.py\',\'test_test199.py\',\'patch_model.py\',\'patch_beam.py\',\'patch_data.py\',\'full_data.py\',\'make_test199_notebook.py\')}\n    mapping.update({\'experiments/csa_paper/\'+n:n for n in (\'data.py\',\'real_bank_data.py\',\'cuda_runtime_guard.py\',\'saved_quality_data.py\',\'score_prompt_test_extra.py\')})\n    if semantic:\n        mapping.update({PREFIX+n:n for n in (\'semantic_test_data.py\',\'semantic_data.py\',\'semantic_patch_model.py\',\'test_semantic_test.py\',\'BANK_PROMPT_TEST_PROTOCOL.md\')})\n        mapping.update({\'experiments/csa_paper/model.py\':\'bank_layers.py\',\'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json\':\'PROMPT_TEST_PREPARATION_RESULT.json\'})\n    hashes={n:hashlib.sha256(subprocess.check_output([\'git\',\'show\',revision+\':\'+n],cwd=ROOT)).hexdigest() for n in mapping}\n    setup=f"""import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil\nfrom pathlib import Path\nREVISION={revision!r}\nMAPPING={mapping!r}\nHASHES={hashes!r}\n"""+"""VISUAL_CACHE=Path(\'/kaggle/input/datasets/ducanh2403/visual-cache\')\nCOCO_JSON=Path(\'/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json\')\nCHECKPOINT_ROOT=\'\' # Optional exact199 mounted file/root. Blank downloads ONLY own199/best.bin.\nSECRET_NAMES=(\'codex\',\'KAGGLE_API_TOKEN\') # Enable the matching Kaggle Secret; token is never logged.\nWALL_HOURS=2 # STOP cap including CPU metrics, not ETA or billing.\nOUTPUT=Path(\'/kaggle/working/clip_patch_199_test5000_v2\')\nfor p in (VISUAL_CACHE,COCO_JSON):\n    if not p.exists():raise FileNotFoundError(\'Attach/correct original inputs: \'+str(p))\nif OUTPUT.exists():raise RuntimeError(\'Preserve existing captions/output; use fresh session/output path\')\nif min(shutil.disk_usage(\'/tmp\').free,shutil.disk_usage(\'/kaggle/working\').free)<6*1024**3:raise RuntimeError(\'Need6GiB free for TEST patches/checkpoint/CPU scorer assets\')\nOUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix=\'clip-patch199-test-\',dir=\'/tmp\'));DEPS=SOURCE/\'deps\';CLI=SOURCE/\'cli\';CHAIR=SOURCE/\'chair\';LOG=OUTPUT/\'notebook.log\'\nPARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]\nos.environ[\'PYTHONPATH\']=str(SOURCE);os.environ[\'OMP_NUM_THREADS\']=\'1\'\nCONFIG=SOURCE/\'config.json\'\ncfg=dict(output=str(OUTPUT),stage=str(SOURCE/\'test_patches\'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)\ndef write_config():\n    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/\'protocol.json\').write_text(json.dumps(cfg,indent=2))\nwrite_config()\ndef redact(line):\n    for token in TOKENS:line=line.replace(token,\'[REDACTED]\')\n    return line\ndef download(url,expected,path):\n    remaining=PARENT_DEADLINE-time.monotonic()\n    if remaining<=0:raise TimeoutError(\'Parent deadline before download\')\n    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()\n    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError(\'Changed pinned input: \'+path.name)\n    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)\n"""\n    runner=RUNNER.replace(\'def run(args, timeout=1800):\',\'def run(args, timeout=1800, environment=None):\').replace(\'cwd=SOURCE, stdout=\',\'cwd=SOURCE, env=environment, stdout=\')\n    runner=runner.replace("print(\'Running:\',", "if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError(\'TEST parent cap\')\\n    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())\\n    print(\'Running:\',").replace("else: print(line, end=\'\', flush=True);", "else: line=redact(line); print(line, end=\'\', flush=True);").replace(\'Smoke active;\',\'TEST/CPU metrics active;\').replace(\'Smoke wall cap exceeded\',\'TEST parent/phase cap exceeded\')\n    routing="""import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,\'build_kaggle_client\') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',path=directory)\n    request=endpoint.call_args.args[0]\n    assert (request.user_name,request.kernel_slug,request.version_label)==(\'ducanh2403\',\'2-vi-cai-kia-qua-nhieu\',\'v199\')\nprint(\'Exact own199 API routing mock PASS; no network.\',flush=True)\n"""\n    if semantic:\n        routing=routing.replace(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',\'ducanh2403/kltnnnnnff/9\').replace("(\'ducanh2403\',\'2-vi-cai-kia-qua-nhieu\',\'v199\')","(\'ducanh2403\',\'kltnnnnnff\',\'v9\')").replace(\'Exact own199\',\'Exact own9\')\n    execution=\'ROUTING_PREFLIGHT=\'+repr(routing)+\'\\n\'+"""complete=False\ntry:\n    for path,name in MAPPING.items():download(\'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/\'+REVISION+\'/\'+path,HASHES[path],SOURCE/name)\n    (OUTPUT/\'source_receipt.json\').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))\n    run([sys.executable,\'-u\',SOURCE/\'test_test199.py\'],120)\n    run([sys.executable,\'-u\',SOURCE/\'cuda_runtime_guard.py\',\'--output\',OUTPUT],60)\n    sys.path.insert(0,str(SOURCE))\n    from test199_data import checkpoint,VERSION,CHECKPOINT_SHA\n    existing=Path(\'/kaggle/working/clip_patch_199_test5000_v1/inputs199/clip_patch_baseline_full_xe10_v3/best.bin\')\n    available=CHECKPOINT_ROOT or (str(existing) if existing.is_file() else \'\')\n    if available:\n        weights=checkpoint(available);origin=\'mounted or existing exact199 bytes\'\n    else:\n        run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--target\',CLI,\'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip\'],600)\n        env=os.environ.copy();env[\'PYTHONPATH\']=str(CLI)\n        run([sys.executable,\'-u\',\'-c\',ROUTING_PREFLIGHT],120,env)\n        from kaggle_secrets import UserSecretsClient\n        token=None\n        for name in SECRET_NAMES:\n            try:value=UserSecretsClient().get_secret(name).strip()\n            except Exception:continue\n            if value:token=value;break\n        if not token:raise RuntimeError(\'Enable own Kaggle API Secret codex or KAGGLE_API_TOKEN; no token is printed.\')\n        TOKENS.append(token)\n        for key in (\'KAGGLE_API_TOKEN\',\'KAGGLE_USERNAME\',\'KAGGLE_KEY\'):env.pop(key,None)\n        env[\'KAGGLE_API_TOKEN\']=token;env[\'PYTHONUNBUFFERED\']=\'1\'\n        target=OUTPUT/\'inputs199\';target.mkdir()\n        # CLI main entry works even on builds without kaggle.__main__.\n        run([sys.executable,\'-u\',\'-c\',\'from kaggle.cli import main; main()\',\'kernels\',\'output\',VERSION,\'-p\',str(target),\'--file-pattern\',r\'(^|/)best[.]bin$\'],900,env)\n        weights=checkpoint(target);origin=\'exact own SaveVersion199 API\'\n    cfg[\'checkpoint_root\']=str(weights);write_config()\n    (OUTPUT/\'checkpoint_source.json\').write_text(json.dumps(dict(version=VERSION,sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'checkpoint_preflight\',\'--config\',CONFIG],120)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'pycocoevalcap==1.2\',\'nltk==3.9.2\',\'future\'],600)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85\'],600)\n    os.environ[\'PYTHONPATH\']=str(DEPS)+os.pathsep+str(SOURCE)\n    run([sys.executable,\'-c\',"import nltk;from pattern.en import singularize;assert singularize(\'umbrellas\')==\'umbrella\';assert nltk.download(\'punkt\');assert nltk.download(\'punkt_tab\')"],300)\n    base=\'https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/\'\n    for name,digest in {\'chair.py\':\'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d\',\'chair.pkl\':\'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\'}.items():download(base+name,digest,CHAIR/name)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'environment\',\'--config\',CONFIG],1200)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'smoke\',\'--config\',CONFIG],600)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'prepare\',\'--config\',CONFIG],1800)\n    cfg[\'worker_seconds\']=min(1800,PARENT_DEADLINE-time.monotonic()-1800);write_config()\n    if cfg[\'worker_seconds\']<300:raise TimeoutError(\'Insufficient budget before TEST inference; no automatic extension\')\n    launch=[sys.executable,\'-u\',\'-m\',\'torch.distributed.run\',\'--standalone\',\'--nnodes=1\',\'--nproc_per_node=2\',\'--max_restarts=0\']\n    run([*launch,SOURCE/\'run_test199.py\',\'worker\',\'--config\',CONFIG],cfg[\'worker_seconds\']+30)\n    run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',\'tokenize\',\'--config\',CONFIG],180)\n    for mode,cap in ((\'basic\',180),(\'meteor\',600),(\'spice\',1800),(\'chair\',600),(\'combine\',120)):\n        run([sys.executable,\'-u\',SOURCE/\'run_test199.py\',mode,\'--config\',CONFIG],cap)\n    complete=json.loads((OUTPUT/\'result.json\').read_text())[\'status\']==\'CLIP_PATCH_199_TEST5000_COMPLETE\'\nexcept BaseException:\n    (OUTPUT/\'failure.txt\').write_text(redact(traceback.format_exc()));raise\nfinally:\n    (OUTPUT/\'run_status.json\').write_text(json.dumps(dict(complete=complete,training=False,test_tuning=False,source_version=\'199\')))\n    review=OUTPUT.with_name(OUTPUT.name+\'_review.zip\')\n    with zipfile.ZipFile(review,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in OUTPUT.rglob(\'*\'):\n            if p.is_file() and p.suffix in (\'.json\',\'.jsonl\',\'.log\',\'.txt\'):z.write(p,str(p.relative_to(OUTPUT)))\n    print(\'SEND REVIEW:\',review,flush=True);print(\'KEEP OUTPUT:\',OUTPUT,flush=True)\n    print(\'TEST COMPLETE; no training/tuning.\' if complete else \'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.\',flush=True)\n"""\n    intro="""# Baseline199 — TEST5000 V2, sửa tương thích CUDA\n\n**T4×2**, InternetON, Save & Run All. Chỉ gắn `ducanh2403/visual-cache` và `vuthetam/mscoco-2014`. Không cần detector/ROI/bank/prompt/testcachecũ. Chỉ stage5000TESTpatch~1,4GiB, không chép full33GiB và không chạyCLIP lại.\n\nCheckpointbestepoch10 cốđịnh từ **ducanh2403/2-vi-cai-kia-qua-nhieu/199**, SHA5061e305...; CHECKPOINT_ROOT tùychọn nếuđãcófile/DS, đểtrống thì tảiONLYbest.bin từđúng199. Bật Secret **codex** hoặc **KAGGLE_API_TOKEN** của tàikhoảnđó. Khônglatest/phiênbảnkhác/foreignnotebook API; khônglộtoken.\n\nBeam5/max20/rawlogprob/rank0 giữnhưVALID, khôngrerank/lengthnormalization/training/tuning. CaptionthamchiếuTEST chỉdùng chấmđiểm vàNLL saukhisinhcaption. Haiworker2500ảnh/rank, lưucaptiontrướcCPUmetrics. ChấmBLEU-1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall,NLL); thiếuphase thìkhôngđánhcomplete. Giữảnh/split/model/sốcaptioncốđịnh; priorprojectTESTexposure vẫnđượcghi.\n\nV2 cho phép Torch2.11.0 CUDAcu128/cu130 đã quan sát, ghi rõ khác biệtbuild và không cam kết caption giốngbitwise. Kiểm checkpoint/runtime trước khi stage; strictweights và GPUbeam-pathcheck vẫn bắt buộc. Không cài lại Torch. Nếu filecheckpoint199 cũ còn trongWorking, tự dùng lại sau kiểmSHA.\n\nCap2giờgồmcảdownload/scorerCPU, khôngphảiETA/quota. StanfordCoreNLP/JavaMeteorspicecóthểcầnthờigian. MỗiCPUphase lưukếtquảriêng; nếulỗi, giữoutputcaption đểphụchồiCPU, khôngtựchạymôhìnhlại. Gửi **clip_patch_199_test5000_v2_review.zip**. Tácgiảchỉkiểmmetadata/NumPy/mock/AST; chưa chạyCUDA/scorerslocal.\n"""\n    if semantic:\n        setup=setup.replace("CHECKPOINT_ROOT=\'\'", "PROMPT_TEST_ROOT=Path(\'/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181\')\\nCHECKPOINT_ROOT=\'\'")\n        setup=setup.replace(\'exact199\',\'exact bank+prompt best9\').replace(\'own199\',\'own kltnnnnnff/9\').replace(\'clip_patch_199_test5000_v2\',\'clip_patch_bank_prompt_test5000_v1\')\n        setup=setup.replace(\'for p in (VISUAL_CACHE,COCO_JSON):\',\'for p in (VISUAL_CACHE,COCO_JSON,PROMPT_TEST_ROOT):\')\n        setup=setup.replace(\'cfg=dict(output=\',"cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=")\n        execution=execution.replace(\'ducanh2403/2-vi-cai-kia-qua-nhieu/199\',\'ducanh2403/kltnnnnnff/9\')\n        execution=execution.replace("\'2-vi-cai-kia-qua-nhieu\', \'v199\'", "\'kltnnnnnff\', \'v9\'")\n        execution=execution.replace(\'from test199_data import checkpoint,VERSION,CHECKPOINT_SHA\',\'from semantic_test_data import checkpoint,VERSION,CHECKPOINT_SHA,prompt_inputs\')\n        execution=execution.replace("Path(\'/kaggle/working/clip_patch_199_test5000_v1/inputs199/clip_patch_baseline_full_xe10_v3/best.bin\')", "Path(\'/kaggle/working/clip_patch_bank_prompt_xe10_v1/best.bin\')")\n        execution=execution.replace("SOURCE/\'test_test199.py\'],120)","SOURCE/\'test_test199.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_semantic_test.py\'],120)")\n        execution=execution.replace("cfg[\'checkpoint_root\']=str(weights);write_config()", "cfg[\'checkpoint_root\']=str(weights);write_config()\\n    prompt_inputs(cfg) # exact TEST181 seal/scope check before expensive setup/stage")\n        execution=execution.replace("TOKENS.append(token)","if token.startswith(\'sk-\'):raise RuntimeError(\'Use a Kaggle API Secret, not an OpenAI key.\')\\n        TOKENS.append(token)")\n        execution=execution.replace(\'exact199 bytes\',\'exact best9 bytes\').replace(\'exact own SaveVersion199 API\',\'exact own kltnnnnnff/9 API\')\n        execution=execution.replace(\'CLIP_PATCH_199_TEST5000_COMPLETE\',\'CLIP_PATCH_BANK_PROMPT_TEST5000_COMPLETE\').replace("source_version=\'199\'","source_version=\'ducanh2403/kltnnnnnff/9\'")\n        intro=\'\'\'# Bank + prompt, best epoch9 — TEST5000\n\n**T4 ×2**, Internet ON, Save & Run All. Attach THREE Datasets:\n- `ducanh2403/visual-cache`\n- `vuthetam/mscoco-2014`\n- `ducanh2403/csa-prompt-test-cache-v181`\n\nEnable your **Kaggle API** Secret `codex` or `KAGGLE_API_TOKEN` (not an OpenAI key). Blank CHECKPOINT_ROOT downloads only best.bin from **ducanh2403/kltnnnnnff/9**, exact SHA7c11be52. User confirmed notebook slug/version; default owner is own ducanh2403. Missing access fails early; no latest/other checkpoint fallback. Alternatively mount the exact knownSHA file/root and set CHECKPOINT_ROOT. No API token printed.\n\nBest epoch9 is159399updates; the full experiment ran10epochs but epoch10 has different weights. Keep epoch9 chosen on VALID before TEST. Model frozen, no training/selection/reranking. Frozen TRAIN noun/verb banks are inside the checkpoint; no bank130 or prompt175 dataset needed. TEST181 only supplies258 fixed prompt embeddings +instance links. Uses exact same ROI-box→14×14patch pooling/masks as training; old ROI feature arrays and old181 vocabulary are NOT used. No CLIP/UPT inference.\n\nOnly5000TEST patches staged (~1.4GiB), no full33GiB copy or TRAIN/VALID feature reads. Same5000IDs/rawreferences, TRAIN vocabulary and beam5/max20/rawlogprob/rank0 as baseline199. Captions saved before teacherNLL and CPUscoring. Metrics: BLEU1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall,NLL). Complete only when all eight metrics finished. Previous project TEST exposure disclosed; no TEST tuning.\n\nParent2h STOP cap inherited from baselineTEST, not ETA/billing. Latest answer confirmed cache path but did not supply remaining GPU hours; check account quota before launch. GPUweights/runtime/firstbeam-path guards before full decode. No Torch reinstall. Preserve caption/output on errors; no automatic model rerun. Send **clip_patch_bank_prompt_test5000_v1_review.zip**. Author checks NumPy/mock/AST only; models/CUDA/scorers run only by user.\n\'\'\'\n    nb=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(display_name=\'Python 3\',language=\'python\',name=\'python3\'),kaggle=dict(isInternetEnabled=True,isGpuEnabled=True)),cells=[cell(\'markdown\',intro),cell(\'code\',setup),cell(\'code\',runner),cell(\'code\',execution)])\n    p=HERE/(\'CLIP_PATCH_BANK_PROMPT_TEST5000_2GPU.ipynb\' if semantic else \'CLIP_PATCH_199_TEST5000_V2_2GPU.ipynb\');p.write_text(json.dumps(nb,ensure_ascii=False,indent=1)+\'\\n\',encoding=\'utf-8\');print(p.name,p.stat().st_size,\'bytes\',len(mapping),\'pins\',revision)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--revision\',required=True);p.add_argument(\'--semantic\',action=\'store_true\');a=p.parse_args();make(a.revision,a.semantic)\n', 'data.py': '"""Pure NumPy/data contracts; importing this file never imports a model."""\nfrom collections import Counter\nimport re\nimport unicodedata\n\nimport numpy as np\n\nPAD, BOS, EOS, UNK = range(4)\nSPECIALS = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\n\n\ndef roi_pool(patches, objects, width, height, cap=150):\n    """Pixel xyxy -> fractional-area-weighted raw patch features, no class inputs."""\n    patches = np.asarray(patches, dtype=np.float32)\n    if patches.ndim != 3 or patches.shape[0] != patches.shape[1]:\n        raise ValueError(\'Expected square patch grid [G,G,C], without CLS\')\n    if not np.isfinite(patches).all() or not np.isfinite([width, height]).all():\n        raise ValueError(\'Nonfinite patch or dimensions\')\n    if width <= 0 or height <= 0 or cap < 1 or not isinstance(objects, list):\n        raise ValueError(\'Invalid dimensions, cap, or detector record\')\n    if not objects:\n        return patches.mean(axis=(0, 1), keepdims=False)[None], np.array([[0, 0, width, height]], np.float32), [\'full_frame_fallback\']\n    checked = []\n    for index, obj in enumerate(objects):\n        box = np.asarray(obj[\'bbox\'], dtype=np.float32)\n        score = float(obj[\'conf\'])\n        if box.shape != (4,) or not np.isfinite(box).all() or not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Malformed detection\')\n        box = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if box[2] <= box[0] or box[3] <= box[1]:\n            raise ValueError(\'Nonpositive detection area; do not turn corruption into empty detection\')\n        checked.append((-score, index, box))\n    selected = sorted(checked, key=lambda item: item[:2])[:cap]\n    grid = patches.shape[0]\n    axis = np.arange(grid, dtype=np.float32)\n    pooled, boxes = [], []\n    for _, _, box in selected:\n        x0, y0, x1, y1 = box / np.array([width, height, width, height]) * grid\n        wx = np.maximum(0, np.minimum(axis + 1, x1) - np.maximum(axis, x0))\n        wy = np.maximum(0, np.minimum(axis + 1, y1) - np.maximum(axis, y0))\n        weights = (wy[:, None] * wx[None, :]).astype(np.float32)\n        pooled.append(np.einsum(\'ij,ijc->c\', weights, patches) / weights.sum())\n        boxes.append(box)\n    return np.stack(pooled), np.stack(boxes), [\'predicted_roi\'] * len(pooled)\n\n\ndef train_captions(dataset):\n    for image in dataset[\'images\']:\n        if image[\'split\'] in (\'train\', \'restval\'):\n            for sentence in image[\'sentences\']:\n                tokens = sentence[\'tokens\']\n                if not isinstance(tokens, list) or not all(isinstance(t, str) and t for t in tokens):\n                    raise ValueError(\'Malformed caption tokens\')\n                yield [unicodedata.normalize(\'NFKC\', t).lower() for t in tokens]\n\n\ndef vocabulary(dataset, limit=10000):\n    if limit < 4:\n        raise ValueError(\'Vocabulary limit includes four specials\')\n    counts = Counter(t for caption in train_captions(dataset) for t in caption if t not in SPECIALS)\n    return SPECIALS + sorted(counts, key=lambda word: (-counts[word], word))[:limit - 4]\n\n\ndef encode(tokens, words, maximum=54):\n    if maximum < 2:\n        raise ValueError(\'Sequence must retain BOS/EOS\')\n    lookup = {word: i for i, word in enumerate(words)}\n    content = [lookup.get(unicodedata.normalize(\'NFKC\', t).lower(), UNK) for t in tokens[:maximum - 2]]\n    return [BOS] + content + [EOS] + [PAD] * (maximum - len(content) - 2)\n\n\ndef bank_words(tagged_lemmas, minimum=5):\n    """Input must come from pinned POS model applied to TRAIN captions only."""\n    counts = {\'noun\': Counter(), \'verb\': Counter()}\n    for lemma, pos in tagged_lemmas:\n        word = unicodedata.normalize(\'NFKC\', lemma).lower()\n        kind = \'noun\' if pos in (\'NOUN\', \'PROPN\') else \'verb\' if pos == \'VERB\' else None\n        if kind and re.fullmatch(\'[a-z]+\', word):\n            counts[kind][word] += 1\n    return {kind: [(w, c[w]) for w in sorted(c, key=lambda w: (-c[w], w)) if c[w] >= minimum]\n            for kind, c in counts.items()}\n', 'real_bank_data.py': '"""Safe real-input contracts. No neural imports or execution."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\ndef validate_arrays(a, images, vocab):\n    n = len(images); total = len(a[\'features\'])\n    shapes = dict(features=(total,768), boxes=(total,4), scores=(total,), offsets=(n+1,), ids=(n,))\n    types = dict(features=np.float16, boxes=np.float32, scores=np.float32, offsets=np.int64, ids=np.int64)\n    if set(a) != set(shapes): raise ValueError(\'Region members differ\')\n    for key in shapes:\n        if a[key].shape != shapes[key] or a[key].dtype != types[key] or not np.isfinite(a[key]).all():\n            raise ValueError(\'Invalid region array: \'+key)\n    offsets = a[\'offsets\']\n    if offsets[0] != 0 or offsets[-1] != total or not ((np.diff(offsets)>=1)&(np.diff(offsets)<=150)).all():\n        raise ValueError(\'Invalid offsets\')\n    for i, m in enumerate(images):\n        if int(a[\'ids\'][i]) != m[\'coco_id\']: raise ValueError(\'Image order differs\')\n        b = a[\'boxes\'][offsets[i]:offsets[i+1]]\n        if len(b)!=len(m[\'region_kinds\']) or m[\'width\']<=0 or m[\'height\']<=0: raise ValueError(\'Region metadata differs\')\n        if not ((b[:,0]>=0)&(b[:,1]>=0)&(b[:,2]<=m[\'width\'])&(b[:,3]<=m[\'height\'])&(b[:,2]>b[:,0])&(b[:,3]>b[:,1])).all():\n            raise ValueError(\'Invalid boxes\')\n        if any(k not in (\'predicted_roi\',\'full_frame_fallback\') for k in m[\'region_kinds\']): raise ValueError(\'Unknown ROI kind\')\n        for seq in m[\'sequences\']:\n            if len(seq)!=54 or seq[0]!=1 or seq.count(2)!=1 or any(type(x)!=int or x<0 or x>=len(vocab) for x in seq):\n                raise ValueError(\'Invalid target sequence\')\n            end=seq.index(2)\n            if 0 in seq[:end] or any(seq[end+1:]): raise ValueError(\'Invalid EOS/PAD sequence\')\n\ndef load_prepared(root):\n    root=Path(root); p=read(HERE/\'REAL_BANK_INPUT_PACKAGE.json\')\n    if not root.is_dir():\n        raise FileNotFoundError(\'Add Input csa-real-v126-lemma-fixed or correct PREPARED_ROOT; directory not mounted: \'+str(root))\n    if not (root/\'corrected_assets_receipt.json\').is_file():\n        candidates=list(root.rglob(\'corrected_assets_receipt.json\'))\n        if len(candidates)!=1:\n            raise ValueError(f\'Expected one corrected_assets_receipt.json under {root}; found {len(candidates)}. Add the combined csa-real-v126-lemma-fixed Dataset, not only the semantic-bank Dataset.\')\n        root=candidates[0].parent\n    receipt=read(root/\'corrected_assets_receipt.json\')\n    for field, expected in [(\'original_files_sha256\',p[\'original_files_sha256\']),(\'corrected_overlay_sha256\',p[\'overlay_sha256\'])]:\n        if receipt.get(field)!=expected: raise ValueError(\'Input receipt differs: \'+field)\n    if receipt.get(\'selected_bank_file\')!=p[\'selected_bank_file\'] or receipt.get(\'original_kernel_version\')!=p[\'original_kernel_version\']:\n        raise ValueError(\'Selected bank/version differs\')\n    for name, expected in {**p[\'original_files_sha256\'],**p[\'overlay_sha256\']}.items():\n        if not (root/name).is_file() or sha(root/name)!=expected: raise ValueError(\'Missing/changed input: \'+name)\n    with np.load(root/\'regions.npz\', allow_pickle=False) as f: arrays={k:f[k] for k in f.files}\n    metadata=read(root/\'pilot_metadata.json\'); vocab=read(root/\'vocabulary.json\'); bank=read(root/p[\'selected_bank_file\'])\n    images=metadata[\'images\']\n    if len(images)!=320 or sum(x[\'group\']==\'train\' for x in images)!=256 or sum(x[\'group\']==\'val\' for x in images)!=64:\n        raise ValueError(\'Pilot groups differ\')\n    validate_arrays(arrays,images,vocab)\n    if len(bank[\'noun\'])!=5269 or len(bank[\'verb\'])!=1594 or len(vocab)!=10000: raise ValueError(\'Bank/vocabulary size differs\')\n    for group in (\'train\',\'val\'):\n        if [images[i][\'coco_id\'] for i in p[group+\'_indices\']]!=p[group+\'_ids\']: raise ValueError(\'Smoke selection differs\')\n    return root,p,arrays,images,vocab,bank\n\ndef batch(arrays, images, indices):\n    lengths=[int(arrays[\'offsets\'][i+1]-arrays[\'offsets\'][i]) for i in indices]\n    x=np.zeros((len(indices),max(lengths),768),np.float32); valid=np.zeros(x.shape[:2],bool)\n    seq=np.array([images[i][\'sequences\'][0] for i in indices],np.int64)\n    seq=seq[:,:int((seq!=0).sum(1).max())]\n    for j,i in enumerate(indices):\n        lo,hi=arrays[\'offsets\'][i:i+2]; x[j,:lengths[j]]=arrays[\'features\'][lo:hi]; valid[j,:lengths[j]]=True\n    return x,valid,seq\n', 'cuda_runtime_guard.py': '"""Metadata-only CUDA preflight; Torch is imported only in the user-run probe."""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef validate_runtime(info):\n    if not info[\'cuda_build\'] or not info[\'set_device_symbol\']:\n        raise RuntimeError(\'Loaded PyTorch lacks CUDA support. Select Accelerator GPU T4 x2 and restart the Kaggle session. If T4 x2 was already selected, send cuda_runtime.json; do not blindly reinstall Torch.\')\n    if not info[\'cuda_available\'] or info[\'device_count\']!=2:\n        raise RuntimeError(\'Exactly two CUDA GPUs must be visible. Select GPU T4 x2, restart the session and check Save Version settings. See cuda_runtime.json.\')\n    if not info[\'nccl_available\']:raise RuntimeError(\'This PyTorch runtime lacks NCCL. Send cuda_runtime.json before changing packages.\')\n\n\ndef check_runtime(output,rank=None):\n    import torch\n    import torch.distributed as dist\n    info=dict(torch_version=str(torch.__version__),torch_file=str(torch.__file__),cuda_build=torch.version.cuda,\n        set_device_symbol=hasattr(torch._C,\'_cuda_setDevice\'),cuda_available=bool(torch.cuda.is_available()),\n        device_count=int(torch.cuda.device_count()),nccl_available=bool(dist.is_nccl_available()),models_executed=False)\n    path=Path(output)/(\'cuda_runtime.json\' if rank is None else f\'cuda_runtime_rank{rank}.json\')\n    path.write_text(json.dumps(info,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(\'CUDA runtime:\',json.dumps(info),flush=True)\n    validate_runtime(info)\n    print(\'CUDA runtime PASS: two GPUs and NCCL; no model loaded.\',flush=True)\n    return info\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args();check_runtime(a.output)\n', 'saved_quality_data.py': '"""Saved text contracts only; no models or metric dependencies."""\nimport hashlib\nimport json\nimport math\n\ndef check_rows(rows,expected):\n    if len(rows)!=len(expected) or len({x[\'coco_id\'] for x in rows})!=len(rows) or {x[\'coco_id\'] for x in rows}!=set(expected):\n        raise ValueError(\'Missing/duplicate/foreign caption IDs\')\n    if any(not isinstance(x[\'caption\'],str) for x in rows):raise ValueError(\'Non-text caption\')\n\ndef common_refs(references,bundle):\n    all_refs={x[\'coco_id\']:x[\'references\'] for x in references}\n    ids=bundle[\'cohort_ids\']\n    if len(ids)!=len(set(ids)) or len(ids)!=4000:raise ValueError(\'Expected4000unique IDs\')\n    if not set(ids)<=set(all_refs):raise ValueError(\'Old4000 not contained in new5000\')\n    selected=[dict(coco_id=i,references=all_refs[i][:5]) for i in sorted(ids)]\n    if any(len(x[\'references\'])!=5 for x in selected):raise ValueError(\'Missing reference\')\n    digest=hashlib.sha256(json.dumps(selected,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n    if digest!=bundle[\'reference_sha256\']:raise ValueError(\'Old/new reference identities differ\')\n    order=bundle.get(\'legacy_order\',sorted(ids))\n    if len(order)!=len(ids) or set(order)!=set(ids):raise ValueError(\'Wrong legacy input order\')\n    by_id={x[\'coco_id\']:x for x in selected}\n    return [by_id[i] for i in order]\n\ndef chair_counts(chair,rows):\n    sentences=chair[\'sentences\'];ids={x[\'coco_id\'] for x in rows};mapping={x[\'coco_id\']:x[\'caption\'] for x in rows}\n    if len(sentences)!=len(ids) or {x[\'image_id\'] for x in sentences}!=ids:raise ValueError(\'CHAIR ID coverage\')\n    if any(x[\'caption\']!=mapping[x[\'image_id\']] for x in sentences):raise ValueError(\'CHAIR captions differ\')\n    counts=dict(images=len(ids),hallucinated_captions=sum(bool(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        hallucinated_mentions=sum(len(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        object_mentions=sum(len(x[\'mscoco_generated_words\']) for x in sentences),\n        recalled_unique=sum(len(set(x[\'mscoco_generated_words\'])&set(x[\'mscoco_gt_words\'])) for x in sentences),\n        reference_unique=sum(len(set(x[\'mscoco_gt_words\'])) for x in sentences))\n    metric=dict(CHAIRs=counts[\'hallucinated_captions\']/len(ids),CHAIRi=counts[\'hallucinated_mentions\']/max(1,counts[\'object_mentions\']),\n                Recall=counts[\'recalled_unique\']/max(1,counts[\'reference_unique\']))\n    if any(not math.isfinite(v) or abs(v-chair[\'overall_metrics\'][k])>1e-12 for k,v in metric.items()):raise ValueError(\'CHAIR count aggregation differs\')\n    return metric,counts\n\ndef align_cider_scores(references,values,rows):\n    if len(references)!=len(values):raise ValueError(\'CIDEr score/reference length mismatch\')\n    indexed={r[\'coco_id\']:float(v) for r,v in zip(references,values)}\n    if len(indexed)!=len(references) or {r[\'coco_id\'] for r in rows}!=set(indexed):raise ValueError(\'CIDEr score IDs differ\')\n    return [indexed[r[\'coco_id\']] for r in rows]\n', 'score_prompt_test_extra.py': '"""USER-RUN CPU scorers on exact saved TEST captions; no model imports."""\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport tarfile\nimport urllib.request\nimport zipfile\n\nHERE = Path(__file__).resolve().parent\nJRE_URL = (\'https://github.com/adoptium/temurin8-binaries/releases/download/\'\n           \'jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz\')\nJRE_SHA = \'6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e\'\nKEYS = (\'Bleu_1\', \'Bleu_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\', \'CHAIRs\', \'CHAIRi\')\n\n\ndef read(p):\n    return json.loads(Path(p).read_text(encoding=\'utf-8\'))\n\n\ndef sha(p):\n    with Path(p).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef save(p, value):\n    p = Path(p)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_suffix(p.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    os.replace(tmp, p)\n\n\ndef download(url, dest):\n    print(\'Downloading:\', url, flush=True)\n    total = 0\n    with urllib.request.urlopen(url, timeout=90) as r, Path(dest).open(\'wb\') as f:\n        while chunk := r.read(1024 * 1024):\n            f.write(chunk)\n            total += len(chunk)\n            if total % (25 * 1024 * 1024) == 0:\n                print(\'Downloaded\', total // (1024 * 1024), \'MiB\', flush=True)\n\n\ndef verify(root, arm, manifest):\n    """Accept nesting, never a different payload/version or incomplete cohort."""\n    spec = manifest[\'arms\'][arm]\n    files = {}\n    for name, digest in spec[\'payload_sha256\'].items():\n        matches = [p for p in Path(root).rglob(name) if p.is_file() and sha(p) == digest]\n        if len(matches) != 1:\n            raise ValueError(f\'Need one exact {name} from {spec["version"]}; found {len(matches)}\')\n        files[name] = matches[0]\n    result, lock = read(files[\'result.json\']), read(files[\'test_lock.json\'])\n    if (result[\'status\'] != \'PROMPT_TEST5000_COMPLETE\' or result[\'arm\'] != arm\n            or result[\'images\'] != manifest[\'images\'] or result[\'training\']\n            or result[\'checkpoint_sha256\'] != spec[\'checkpoint_sha256\']\n            or lock[\'checkpoint_sha256\'] != spec[\'checkpoint_sha256\']\n            or sha(files[\'test_references.json\']) != lock[\'references_sha256\']\n            or read(files[\'run_status.json\']) != dict(complete=True, arm=arm, training=False, test_tuning=False)):\n        raise ValueError(\'Wrong source identity/status\')\n    refs = read(files[\'test_references.json\'])\n    ids = {r[\'coco_id\'] for r in refs}\n    if len(refs) != manifest[\'images\'] or len(ids) != len(refs):\n        raise ValueError(\'Duplicate/missing TEST reference IDs\')\n    if any(type(i) is not int for i in ids) or any(\n            not isinstance(r[\'references\'], list) or len(r[\'references\']) < 5\n            or any(not isinstance(t, str) or not t.strip() for t in r[\'references\']) for r in refs):\n        raise ValueError(\'Wrong reference structure\')\n    captions = {}\n    for policy in manifest[\'policies\']:\n        rows = read(files[policy + \'_captions.json\'])\n        if (len(rows) != len(ids) or {r[\'coco_id\'] for r in rows} != ids\n                or any(not isinstance(r[\'caption\'], str) or not r[\'caption\'].strip() for r in rows)\n                or result[\'metrics\'][policy] != spec[\'metrics\'][policy]):\n            raise ValueError(\'Changed/incomplete captions or historical metrics: \' + policy)\n        captions[policy] = rows\n    return refs, captions, files\n\n\ndef environment(out):\n    from importlib.metadata import version\n    if version(\'pycocoevalcap\') != \'1.2\':\n        raise ValueError(\'Expected pycocoevalcap==1.2\')\n    assets = out / \'assets\'\n    assets.mkdir()\n    archive = assets / \'java8.tar.gz\'\n    download(JRE_URL, archive)\n    if sha(archive) != JRE_SHA:\n        raise ValueError(\'Changed pinned Java8 asset\')\n    with tarfile.open(archive) as t:\n        t.extractall(assets / \'jre\', filter=\'data\')\n    java = list((assets / \'jre\').rglob(\'bin/java\'))\n    if len(java) != 1:\n        raise ValueError(\'Missing pinned Java executable\')\n    spice_dir = Path(importlib.util.find_spec(\'pycocoevalcap.spice.spice\').origin).parent\n    lib = spice_dir / \'lib\'\n    lib.mkdir(exist_ok=True)\n    archive = assets / \'corenlp.zip\'\n    download(\'https://downloads.cs.stanford.edu/nlp/software/stanford-corenlp-full-2015-12-09.zip\', archive)\n    for suffix in (\'.jar\', \'-models.jar\'):\n        name = \'stanford-corenlp-3.6.0\' + suffix\n        member = \'stanford-corenlp-full-2015-12-09/\' + name\n        with zipfile.ZipFile(archive) as z, z.open(member) as src, (lib / name).open(\'wb\') as dst:\n            shutil.copyfileobj(src, dst)\n    package = spice_dir.parent\n    save(out / \'environment.json\', dict(pycocoevalcap=version(\'pycocoevalcap\'), java=str(java[0]),\n        java_archive_sha256=JRE_SHA, corenlp_archive_sha256=sha(archive),\n        corenlp_version=\'3.6.0\', model_digest_pinned_independently=False,\n        evaluator_files={str(p.relative_to(package)): sha(p) for p in sorted(package.rglob(\'*\'))\n                         if p.is_file() and p.suffix in (\'.py\', \'.jar\', \'.gz\')}))\n\n\ndef java_path(out):\n    env = read(out / \'environment.json\')\n    os.environ[\'PATH\'] = str(Path(env[\'java\']).parent) + os.pathsep + os.environ[\'PATH\']\n    os.environ[\'JAVA_HOME\'] = str(Path(env[\'java\']).parent.parent)\n\n\ndef checked_point(value, length, scores):\n    value = float(value)\n    if not math.isfinite(value) or len(scores) != length:\n        raise ValueError(\'Nonfinite metric or incomplete per-image coverage\')\n    return value\n\n\ndef smoke(out):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    tok = PTBTokenizer()\n    refs = tok.tokenize({1: [dict(caption=\'a person riding a bicycle on a road\')]})\n    hyp = tok.tokenize({1: [dict(caption=\'a person riding a bicycle on a road\')]})\n    points = {\'Bleu_1\': float(Bleu(4).compute_score(refs, hyp)[0][0])}\n    for name, factory in [(\'METEOR\', Meteor), (\'SPICE\', Spice)]:\n        scorer = factory()\n        try:\n            point, scores = scorer.compute_score(refs, hyp)\n            points[name] = checked_point(point, 1, scores)\n            if points[name] <= 0:\n                raise ValueError(\'Exact-match smoke has no positive score: \' + name)\n        finally:\n            del scorer\n    save(out / \'smoke.json\', dict(passed=True, scores=points))\n    print(\'Actual CPU PTB/BLEU/METEOR/SPICE smoke PASS\', points, flush=True)\n\n\ndef tokenize(out, root, arm, manifest):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    refs, captions, files = verify(root, arm, manifest)\n    tok = PTBTokenizer()\n    reference = tok.tokenize({r[\'coco_id\']: [dict(caption=t) for t in r[\'references\']] for r in refs})\n    save(out / \'references_ptb.json\', reference)\n    for policy, rows in captions.items():\n        save(out / (policy + \'_ptb.json\'), tok.tokenize({r[\'coco_id\']: [dict(caption=r[\'caption\'])] for r in rows}))\n    save(out / \'input_receipt.json\', dict(arm=arm, source_version=manifest[\'arms\'][arm][\'version\'],\n        images=len(refs), payload_sha256={n: sha(p) for n, p in files.items()},\n        tokenized_sha256={p.name: sha(p) for p in out.glob(\'*_ptb.json\')},\n        training=False, decoding=False, reranking=False))\n\n\ndef metric(out, arm, manifest, mode):\n    receipt = read(out / \'input_receipt.json\')\n    if receipt[\'arm\'] != arm or receipt[\'payload_sha256\'] != manifest[\'arms\'][arm][\'payload_sha256\']:\n        raise ValueError(\'Input receipt changed\')\n    for n, digest in receipt[\'tokenized_sha256\'].items():\n        if sha(out / n) != digest:\n            raise ValueError(\'Tokenized captions changed\')\n    reference = read(out / \'references_ptb.json\')\n    for policy in manifest[\'policies\']:\n        print(\'CPU scoring:\', policy, mode, flush=True)\n        hyp = read(out / (policy + \'_ptb.json\'))\n        if len(reference) != manifest[\'images\'] or set(reference) != set(hyp):\n            raise ValueError(\'Metric cohort differs\')\n        if mode == \'basic\':\n            from pycocoevalcap.bleu.bleu import Bleu\n            from pycocoevalcap.cider.cider import Cider\n            from pycocoevalcap.rouge.rouge import Rouge\n            b, _ = Bleu(4).compute_score(reference, hyp)\n            c, cs = Cider().compute_score(reference, hyp)\n            r, rs = Rouge().compute_score(reference, hyp)\n            scores = dict(Bleu_1=float(b[0]), Bleu_4=float(b[3]),\n                CIDEr=checked_point(c, len(hyp), cs), ROUGE_L=checked_point(r, len(hyp), rs))\n            for k in (\'Bleu_4\', \'CIDEr\', \'ROUGE_L\'):\n                if abs(scores[k] - manifest[\'arms\'][arm][\'metrics\'][policy][k]) > 1e-7:\n                    raise ValueError(\'Historical metric replay differs: \' + policy + \'/\' + k)\n        else:\n            if mode == \'meteor\':\n                from pycocoevalcap.meteor.meteor import Meteor\n                scorer = Meteor()\n            elif mode == \'spice\':\n                from pycocoevalcap.spice.spice import Spice\n                scorer = Spice()\n            else:\n                raise ValueError(mode)\n            try:\n                point, per_image = scorer.compute_score(reference, hyp)\n                scores = {mode.upper(): checked_point(point, len(hyp), per_image)}\n            finally:\n                del scorer\n        if any(not math.isfinite(v) for v in scores.values()):\n            raise ValueError(\'Nonfinite metric\')\n        save(out / f\'{policy}_{mode}.json\', dict(images=len(hyp), metrics=scores,\n            tokenized_sha256=receipt[\'tokenized_sha256\']))\n\n\ndef combine(out, arm, manifest):\n    points = {}\n    receipt = read(out / \'input_receipt.json\')\n    for policy in manifest[\'policies\']:\n        row = {}\n        for mode in (\'basic\', \'meteor\', \'spice\'):\n            part = read(out / f\'{policy}_{mode}.json\')\n            if part[\'images\'] != 5000 or part[\'tokenized_sha256\'] != receipt[\'tokenized_sha256\']:\n                raise ValueError(\'Partial/changed metric phase\')\n            row.update(part[\'metrics\'])\n        row.update({k: manifest[\'arms\'][arm][\'metrics\'][policy][k] for k in (\'CHAIRs\', \'CHAIRi\')})\n        if set(row) != set(KEYS) or not all(math.isfinite(v) for v in row.values()):\n            raise ValueError(\'Need all eight metrics\')\n        points[policy] = row\n    result = dict(status=\'PROMPT_TEST_EIGHT_METRICS_COMPLETE\', arm=arm, images=5000,\n        source_version=manifest[\'arms\'][arm][\'version\'], metrics=points,\n        display_x100={p: {k: 100*v for k, v in row.items()} for p, row in points.items()},\n        chair_provenance=\'Reused audited scores, exact caption/reference/result SHA verified; not recomputed.\',\n        inputs=receipt, environment=read(out / \'environment.json\'), training=False, decoding=False,\n        reranking=False, test_tuning=False, prior_project_test_exposure=True)\n    save(out / \'result.json\', result)\n    with (out / \'metrics_x100.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=[\'arm\', \'policy\', *KEYS])\n        w.writeheader()\n        for policy, row in result[\'display_x100\'].items():\n            w.writerow(dict(arm=arm, policy=policy, **row))\n    print(\'COMPLETE — saved captions only:\', result[\'display_x100\'], flush=True)\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'environment\', \'smoke\', \'tokenize\', \'basic\', \'meteor\', \'spice\', \'combine\'])\n    p.add_argument(\'--arm\', choices=[\'CONTROL\', \'PROMPT\'], required=True)\n    p.add_argument(\'--input\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    args = p.parse_args()\n    manifest = read(HERE / \'PROMPT_TEST_EXTRA_INPUTS.json\')\n    if args.mode == \'environment\':\n        environment(args.output)\n    else:\n        java_path(args.output)\n        if args.mode == \'smoke\':\n            smoke(args.output)\n        elif args.mode == \'tokenize\':\n            tokenize(args.output, args.input, args.arm, manifest)\n        elif args.mode == \'combine\':\n            combine(args.output, args.arm, manifest)\n        else:\n            metric(args.output, args.arm, manifest, args.mode)\n', 'semantic_test_data.py': '"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS,record_arrays,checked_root\n\nHERE=Path(__file__).resolve().parent\nCHECKPOINT_SHA=\'7c11be52fce9cd90dadc55fa514d4017d540f40f630232a68bfe4306c754158b\'\nVERSION=\'ducanh2403/kltnnnnnff/9\'\n\n\ndef validate_lock(value):\n    baseline=shared.lock()\n    if any(value[k]!=baseline[k] for k in (\'expected_test_ids\',\'test_id_sha256\',\'beam_width\',\'max_new_tokens\',\'length_normalization\',\'reranking\')):raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\']!=VERSION or value[\'checkpoint_sha256\']!=CHECKPOINT_SHA or value[\'epoch\']!=9 or value[\'updates\']!=159399 or value[\'method\']!=\'bank_prompt_patch_v1\':raise ValueError(\'Need fixed bank+prompt best9, not endpoint10\')\n    return value\n\n\ndef lock():\n    value=shared.lock();value.update(schema=\'bank_prompt_best9_test_v1\',source_version=VERSION,checkpoint_sha256=CHECKPOINT_SHA,\n        training_commit=\'f5f6eb05d7c89917b0719637dae4e948ad73f531\',epoch=9,updates=159399,method=\'bank_prompt_patch_v1\')\n    return validate_lock(value)\n\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Missing exact version9 best.bin SHA7c11be52; no final/latest fallback\')\n    return matches[0]\n\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu130\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Same publicTorch2.11.0 observed CUDA wheels; strictstate and actual GPU beam-path check required.\',bitwise_cross_build_equivalence_established=False)\n\n\ndef validate_checkpoint_state(ck):\n    s,i=ck[\'state\'],ck[\'identity\'];expected=lock()\n    if s[\'epoch\']!=10 or s[\'batch_cursor\']!=0 or s[\'phase\']!=\'train\' or s[\'updates\']!=159399 or s[\'caption_events\']!=5100723 or s[\'best_epoch\']!=9 or len(s[\'history\'])!=9:raise ValueError(\'Wrong best9 snapshot cursor\')\n    if i[\'source\'][\'commit\']!=expected[\'training_commit\'] or i[\'method\']!=expected[\'method\'] or i[\'epochs\']!=10 or i[\'global_batch\']!=32:raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if not {\'noun_bank\',\'verb_bank\'}<=set(ck[\'model\']):raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\n\ndef prompt_inputs(cfg):\n    inventory=read(HERE/\'PROMPT_TEST_PREPARATION_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    # The old ROI feature array/vocabulary are not inputs to the new patch model.\n    wanted={n:inventory[n] for n in (\'receipt.json\',\'summary.json\',\'prompt_links.jsonl\',\'prompt_texts.json\',\'prompt_embeddings.npy\')}\n    root=checked_root(cfg[\'prompt_test_root\'],\'receipt.json\',wanted);texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if texts!=sorted(set(texts)) or len(texts)!=258 or vectors.shape!=(258,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,rtol=0,atol=2e-6):raise ValueError(\'Changed TEST181 text space\')\n    rows=[json.loads(line) for line in (root/\'prompt_links.jsonl\').read_text(encoding=\'utf-8\').splitlines()]\n    expected=lock()[\'expected_test_ids\']\n    if len(rows)!=5000 or sorted(r[\'coco_id\'] for r in rows)!=expected or any(r[\'group\']!=\'test\' for r in rows):raise ValueError(\'Wrong/duplicate prompt TEST IDs\')\n    return rows,texts,vectors,wanted\n\n\ndef prepare(cfg):\n    rows,texts,vectors,seals=prompt_inputs(cfg)\n    for row in rows:record_arrays(row,texts)\n    shared.prepare(cfg,locked=lock(),weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg,rows,texts,vectors,seals)\n\n\ndef stage_prompts(cfg,rows,texts,vectors,seals):\n    encoded=[record_arrays(row,texts) for row in rows]\n    stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);ids=np.load(stage/\'ids.npy\',allow_pickle=False);lookup={r[\'coco_id\']:j for j,r in enumerate(rows)}\n    order=[lookup[int(i)] for i in ids]\n    for key in ARRAYS:np.save(stage/(\'sem_\'+key+\'.npy\'),np.stack([encoded[j][key] for j in order]))\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    receipt=read(out/\'stage_receipt.json\');receipt[\'method\']=\'bank_prompt_patch_v1\';receipt[\'prompt181_files_sha256\']=seals\n    receipt[\'stage_files_sha256\'].update({p.name:sha(p) for p in stage.glob(\'sem_*.npy\')});save(out/\'stage_receipt.json\',receipt)\n    saved=read(out/\'test_lock.json\');saved[\'stage_receipt_sha256\']=sha(out/\'stage_receipt.json\');save(out/\'test_lock.json\',saved)\n    save(out/\'semantic_test_stage.json\',dict(images=5000,objects=sum(len(r[\'objects\']) for r in rows),relations=sum(len(r[\'relations\']) for r in rows),\n        text_vectors=258,geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\',TRAIN_VALID_features_read=False,old_ROI_features_used=False,old_vocabulary_used=False))\n\n\ndef load(cfg,verify=True):\n    result=shared.load(cfg,verify=verify,validator=validate_lock)\n    if result[2].get(\'method\')!=\'bank_prompt_patch_v1\':raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n', 'semantic_data.py': '"""Pure NumPy fixed-bank/prompt175 to patch-grid preparation; no models."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import load_stage,TRAIN_IMAGES,VALID_IMAGES\n\nHERE=Path(__file__).resolve().parent\nARRAYS=dict(object_text_id=((8,),np.int16),object_box=((8,4),np.float32),object_weights=((8,196),np.float32),object_valid=((8,),bool),\n    relation_text_id=((3,),np.int16),subject_weights=((3,196),np.float32),target_weights=((3,196),np.float32),relation_geometry=((3,8),np.float32),relation_valid=((3,),bool))\n\ndef box_weights(value):\n    b=np.asarray(value,np.float64)\n    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError(\'Invalid normalized xyxy box\')\n    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)\n    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)\n    if weights.sum()<=0:raise ValueError(\'Box has no patch support\')\n    return (weights/weights.sum()).astype(np.float32)\n\ndef record_arrays(row,texts):\n    expected={\'coco_id\',\'filename\',\'group\',\'roi_count\',\'source_mode\',\'objects\',\'relations\',\'counts\'}\n    if set(row)!=expected or len(row[\'objects\'])>8 or len(row[\'relations\'])>3:raise ValueError(\'Unexpected prompt fields/caps\')\n    out={k:np.zeros(shape,dtype=dtype) for k,(shape,dtype) in ARRAYS.items()};lookup={t:i for i,t in enumerate(texts)};objects={}\n    for slot,obj in enumerate(row[\'objects\']):\n        instance=obj[\'instance_id\']\n        if not instance or instance in objects or obj[\'text\'] not in lookup:raise ValueError(\'Invalid/duplicate object instance/text\')\n        if not 0<=obj[\'roi\']<row[\'roi_count\']:raise ValueError(\'Invalid original ROI provenance\')\n        # Pool the SAME matched YOLO rectangle used by legacy ROI features, not the ROI index.\n        w=box_weights(obj[\'roi_bbox\']);box_weights(obj[\'bbox\'])\n        objects[instance]=w;out[\'object_weights\'][slot]=w;out[\'object_box\'][slot]=obj[\'bbox\'];out[\'object_text_id\'][slot]=lookup[obj[\'text\']];out[\'object_valid\'][slot]=True\n    for slot,rel in enumerate(row[\'relations\']):\n        s,o=rel[\'subject_instance\'],rel[\'object_instance\']\n        if s==o or s not in objects or o not in objects or rel[\'text\'] not in lookup:raise ValueError(\'Missing/collapsed relation endpoints/text\')\n        g=np.asarray(rel[\'geometry\'],np.float32)\n        if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid relation geometry\')\n        out[\'subject_weights\'][slot]=objects[s];out[\'target_weights\'][slot]=objects[o];out[\'relation_text_id\'][slot]=lookup[rel[\'text\']];out[\'relation_geometry\'][slot]=g;out[\'relation_valid\'][slot]=True\n    return out\n\ndef checked_root(mount,receipt_name,expected):\n    found=[p.parent for p in Path(mount).rglob(receipt_name) if sha(p)==expected[receipt_name]]\n    if len(found)!=1:raise ValueError(\'Need one exact semantic cache root: \'+receipt_name)\n    root=found[0]\n    for n,h in expected.items():\n        if Path(n).name!=n or not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed semantic payload \'+n)\n    return root\n\ndef fixed_banks(mount):\n    package=read(HERE/\'SAVE_BANK_V130_PACKAGE.json\');root=checked_root(mount,\'cache_receipt.json\',package[\'files\']);bank=read(HERE/\'bank_words_corrected.json\');manifest=read(root/\'cache_manifest.json\')\n    if sha(HERE/\'bank_words_corrected.json\')!=\'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac\' or bank[\'source\']!=\'TRAIN only, train+restval\' or manifest[\'source_bank_sha256\']!=sha(HERE/\'bank_words_corrected.json\'):raise ValueError(\'Bank supervision/provenance changed\')\n    arrays={}\n    with np.load(root/\'semantic_banks.npz\',allow_pickle=False) as f:\n        for kind in (\'noun\',\'verb\'):\n            words=[x[\'word\'] for x in bank[kind]];v=f[kind+\'_vectors\']\n            if f[kind+\'_words\'].tolist()!=words or v.shape!=(len(words),512) or v.dtype!=np.float32 or not np.isfinite(v).all() or not np.allclose(np.linalg.norm(v,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Bad bank vector order/space\')\n            arrays[kind]=v.copy()\n    return arrays,package,manifest\n\ndef checked_inputs(cfg):\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'];root=checked_root(cfg[\'prompt_root\'],\'cache_receipt.json\',inv[\'cache_files\']);cohort=read(root/\'cohort.json\');texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False);prompt_manifest=read(root/\'cache_manifest.json\')\n    if len(texts)!=410 or texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Wrong frozen175 text vectors\')\n    banks,bank_package,bank_manifest=fixed_banks(cfg[\'bank_root\'])\n    if (prompt_manifest[\'model\'],prompt_manifest[\'revision\'])!=(bank_manifest[\'model\'],bank_manifest[\'revision\']):raise ValueError(\'Bank/prompt CLIP text spaces differ\')\n    if cohort[\'test\'] or len(cohort[\'train\'])!=TRAIN_IMAGES or len(cohort[\'val\'])!=VALID_IMAGES or len(set(cohort[\'train\']+cohort[\'val\']))!=TRAIN_IMAGES+VALID_IMAGES:raise ValueError(\'Prompt source scope differs\')\n    return inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package\n\n\ndef stage_semantics(cfg,deadline):\n    started=time.monotonic();stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);data,_,meta=load_stage(stage,verify=False);count=TRAIN_IMAGES+VALID_IMAGES\n    if len(data[\'ids\'])!=count or meta.get(\'semantic\'):raise ValueError(\'Expected new full patch stage\')\n    inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package=checked_inputs(cfg)\n    if set(cohort[\'train\'])!=set(map(int,data[\'ids\'][:TRAIN_IMAGES])) or set(cohort[\'val\'])!=set(map(int,data[\'ids\'][TRAIN_IMAGES:])):raise ValueError(\'Prompt/full patch split differs\')\n    arrays={k:np.lib.format.open_memmap(stage/(\'sem_\'+k+\'.npy\'),mode=\'w+\',dtype=dtype,shape=(count,)+shape) for k,(shape,dtype) in ARRAYS.items()}\n    for value in arrays.values():value[:]=0\n    index={int(i):j for j,i in enumerate(data[\'ids\'])};seen=set();totals={g:Counter() for g in (\'train\',\'val\')}\n    try:\n        with (root/\'prompt_links.jsonl\').open(encoding=\'utf-8\') as stream:\n            for line in stream:\n                if time.monotonic()>deadline:raise TimeoutError(\'Semantic stage cap; not complete training input\')\n                row=json.loads(line);i=row[\'coco_id\'];j=index.get(i)\n                if type(i)!=int or i in seen or j is None or row[\'group\']!=(\'train\' if j<TRAIN_IMAGES else \'val\'):raise ValueError(\'Foreign/repeated/leaked prompt image\')\n                encoded=record_arrays(row,texts)\n                for key,a in encoded.items():arrays[key][j]=a\n                seen.add(i);totals[row[\'group\']].update(images=1,objects=len(row[\'objects\']),relations=len(row[\'relations\']),with_object=bool(row[\'objects\']),with_relation=bool(row[\'relations\']))\n                if len(seen)%10000==0:print(\'Semantic patch inputs\',len(seen),\'/\',count,flush=True)\n        if seen!=set(index):raise ValueError(\'Missing semantic coverage\')\n        for value in arrays.values():value.flush()\n    finally:\n        for value in arrays.values():value._mmap.close()\n        for value in data.values():value._mmap.close()\n    for kind,v in banks.items():np.save(stage/(\'sem_\'+kind+\'_bank.npy\'),v)\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    names=[\'sem_\'+n+\'.npy\' for n in ARRAYS]+[\'sem_noun_bank.npy\',\'sem_verb_bank.npy\',\'sem_text_vectors.npy\']\n    seals={n:sha(stage/n) for n in names}\n    if time.monotonic()>deadline:raise TimeoutError(\'Semantic sealing cap\')\n    meta[\'files_sha256\'].update(seals);meta[\'semantic\']=dict(method=\'bank_prompt_patch_v1\',prompt_cache=inv[\'cache_files\'],bank_cache=bank_package[\'files\'],text_model=prompt_manifest[\'model\'],text_revision=prompt_manifest[\'revision\'],geometry=\'14x14 fractional area on matched ROI box; Resize224x224, no center crop\',counts={g:dict(c) for g,c in totals.items()},seconds=time.monotonic()-started,TRAIN_VALID_only=True,test_used=False,files_sha256=seals)\n    save(stage/\'manifest.json\',meta);save(out/\'stage_summary.json\',meta);save(out/\'semantic_stage.json\',meta[\'semantic\'])\n    print(\'Semantic stage COMPLETE; no CLIP/UPT/model execution.\',flush=True)\n\nclass SemanticContext:\n    def __init__(self,stage):\n        self.arrays={k:np.load(Path(stage)/(\'sem_\'+k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in ARRAYS}\n        self.text=np.load(Path(stage)/\'sem_text_vectors.npy\',mmap_mode=\'r\',allow_pickle=False)\n    def images(self,indices,device):\n        import torch\n        p={k:np.array(v[indices],copy=True) for k,v in self.arrays.items() if k not in (\'object_text_id\',\'relation_text_id\')}\n        for role in (\'object\',\'relation\'):\n            p[role+\'_text\']=self.text[self.arrays[role+\'_text_id\'][indices]]*p[role+\'_valid\'][...,None]\n        return {k:torch.from_numpy(np.ascontiguousarray(v)).to(device) for k,v in p.items()}\n    def events(self,data,events,device):return self.images(data[\'caption_image\'][events],device)\n', 'semantic_patch_model.py': '"""User-run bank/prompt patch memory; unchanged baseline causal decoder."""\nimport torch\nfrom torch import nn\nfrom patch_model import PatchCaptioner\nfrom bank_layers import Config,SemanticAggregation,Alignment\nfrom memory_fusion import fuse_memory\n\n\nclass BankMemory(nn.Module):\n    def __init__(self):\n        super().__init__();self.aggregate=SemanticAggregation(Config(vocab=10000,dropout=0))\n        self.align=Alignment();self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,bank):\n        prototypes,_=self.aggregate(bank)\n        selected,_=self.align(visual,prototypes,torch.ones(visual.shape[:2],dtype=torch.bool,device=visual.device))\n        return visual+self.attention(visual,selected,selected,need_weights=False)[0]\n\n\nclass PromptMemory(nn.Module):\n    def __init__(self,input_dim):\n        super().__init__();self.slot=nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,512))\n        self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,slot_inputs,support,valid):\n        slots=self.slot(slot_inputs)\n        allowed=support.transpose(1,2).gt(0)&valid[:,None,:]\n        active=allowed.any(-1)\n        # Give inactive queries one finite key, then zero their entire residual.\n        safe=allowed.clone();safe[:,:,0]|=~active\n        mask=(~safe)[:,None].expand(-1,8,-1,-1).reshape(-1,visual.shape[1],slots.shape[1])\n        delta=self.attention(visual,slots,slots,attn_mask=mask,need_weights=False)[0]\n        return visual+delta.masked_fill(~active[...,None],0)\n\n\nclass SemanticPatchCaptioner(nn.Module):\n    def __init__(self,vocab,nouns,verbs,strength_gate=False):\n        super().__init__()\n        # Construct the baseline FIRST; never reinitialize its modules recursively.\n        self.base=PatchCaptioner(vocab)\n        self.register_buffer(\'noun_bank\',nouns);self.register_buffer(\'verb_bank\',verbs)\n        self.object_bank=BankMemory();self.relation_bank=BankMemory()\n        self.object_prompt=PromptMemory(512+512+4)\n        self.relation_prompt=PromptMemory(512+512+512+8)\n        self.gate=nn.Linear(1024,512);nn.init.zeros_(self.gate.weight);nn.init.zeros_(self.gate.bias)\n        self.strength_gate=nn.Linear(1024,1) if strength_gate else None\n        if self.strength_gate is not None:\n            nn.init.zeros_(self.strength_gate.weight);nn.init.zeros_(self.strength_gate.bias)\n    @property\n    def project(self):return self.base.project\n    @property\n    def head(self):return self.base.head\n    def memories(self,patches,p,visual=None):\n        if visual is None:visual=self.base.encode(patches)\n        mo=self.object_bank(visual,self.noun_bank);mr=self.relation_bank(mo,self.verb_bank)\n        obj=torch.cat((p[\'object_text\'],torch.bmm(p[\'object_weights\'],mo),p[\'object_box\']),-1)\n        rel=torch.cat((p[\'relation_text\'],torch.bmm(p[\'subject_weights\'],mo),torch.bmm(p[\'target_weights\'],mo),p[\'relation_geometry\']),-1)\n        vo=self.object_prompt(mo,obj,p[\'object_weights\'],p[\'object_valid\'])\n        vr=self.relation_prompt(mr,rel,p[\'subject_weights\']+p[\'target_weights\'],p[\'relation_valid\'])\n        return vo,vr\n    def encode(self,patches,p):\n        if self.strength_gate is not None:return self.encode_with_gate(patches,p)[0]\n        vo,vr=self.memories(patches,p);gate=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        return gate*vo+(1-gate)*vr\n    def encode_with_gate(self,patches,p):\n        if self.strength_gate is None:raise ValueError(\'Strength-gated model required\')\n        visual=self.base.encode(patches);vo,vr=self.memories(patches,p,visual)\n        branch=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        delta=branch*(vo-visual)+(1-branch)*(vr-visual)\n        strength=self.strength_gate(torch.cat((visual,delta),-1)).sigmoid()\n        return fuse_memory(visual,vo,vr,branch,strength),strength\n    def decode(self,memory,prefix):return self.base.decode(memory,prefix)\n    def forward(self,patches,prefix,p):return self.decode(self.encode(patches,p),prefix)\n', 'test_semantic_test.py': '"""Author-safe fixed bank/prompt TEST checks; no Torch/model/scorer execution."""\nimport ast\nimport copy\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport semantic_test_data as d\nfrom semantic_data import SemanticContext\nimport run_test199 as runner\n\n\ndef row(coco):\n    return dict(coco_id=coco,filename=f\'{coco}.jpg\',group=\'test\',roi_count=9,source_mode=\'object_only\',counts={},relations=[],\n        objects=[dict(instance_id=\'person:4\',roi=7,text=\'a person\',bbox=[0,0,.5,1],roi_bbox=[0,0,.5,1])])\n\n\nclass Checks(unittest.TestCase):\n    def test_fixed_checkpoint_epoch9_and_decode(self):\n        lock=d.lock();self.assertEqual(lock[\'updates\'],159399);self.assertEqual(len(lock[\'expected_test_ids\']),5000)\n        self.assertEqual(lock[\'expected_test_ids\'],d.shared.lock()[\'expected_test_ids\'])\n        for k,v in ((\'epoch\',10),(\'updates\',177110),(\'beam_width\',4),(\'reranking\',True),(\'checkpoint_sha256\',\'0\'*64),(\'expected_test_ids\',[1])):\n            bad=copy.deepcopy(lock);bad[k]=v\n            with self.assertRaises(ValueError):d.validate_lock(bad)\n\n    def test_best_snapshot_not_final_epoch10(self):\n        ck=dict(state=dict(epoch=10,batch_cursor=0,phase=\'train\',updates=159399,caption_events=5100723,best_epoch=9,history=[{}]*9),\n            identity=dict(source=dict(commit=d.lock()[\'training_commit\']),method=\'bank_prompt_patch_v1\',epochs=10,global_batch=32),model=dict(noun_bank=None,verb_bank=None))\n        d.validate_checkpoint_state(ck)\n        bad=copy.deepcopy(ck);bad[\'state\'].update(epoch=11,updates=177110)\n        with self.assertRaises(ValueError):d.validate_checkpoint_state(bad)\n        bad=copy.deepcopy(ck);bad[\'identity\'][\'method\']=\'plain_patch_v1\'\n        with self.assertRaises(ValueError):d.validate_checkpoint_state(bad)\n\n    def test_exact_weights_lookup_and_runtime(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/\'best.bin\').write_bytes(b\'best9\');(root/\'checkpoint.bin\').write_bytes(b\'final10\')\n            with patch.object(d,\'CHECKPOINT_SHA\',d.sha(root/\'best.bin\')):\n                self.assertEqual(d.checkpoint(root),root/\'best.bin\')\n                with self.assertRaises(ValueError):d.checkpoint(root/\'checkpoint.bin\')\n        for v in (\'2.11.0+cpu\',\'2.12.0+cu130\'):\n            with self.assertRaises(RuntimeError):d.runtime_policy(\'2.11.0+cu130\',v)\n        self.assertFalse(d.runtime_policy(\'2.11.0+cu130\',\'2.11.0+cu130\')[\'cuda_wheel_changed\'])\n\n    def test_stage_geometry_joins_logical_TEST_ids_and_seals(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);stage=root/\'stage\';stage.mkdir();out=root/\'out\';out.mkdir();np.save(stage/\'ids.npy\',np.array([20,10]))\n            d.save(out/\'stage_receipt.json\',dict(stage_files_sha256={}));d.save(out/\'test_lock.json\',{})\n            a=row(10);b=row(20);b[\'objects\'][0].update(roi_bbox=[.5,0,1,1])\n            vectors=np.zeros((1,512),np.float32);vectors[0,0]=1\n            d.stage_prompts(dict(stage=str(stage),output=str(out)),[a,b],[\'a person\'],vectors,{\'fixture\':\'hash\'})\n            context=SemanticContext(stage)\n            left=context.arrays[\'object_weights\'][0,0].reshape(14,14);self.assertEqual(float(left[:,:7].sum()),0)\n            self.assertAlmostEqual(float(left[:,7:].sum()),1,places=6)\n            np.testing.assert_array_equal(context.text,vectors)\n            receipt=d.read(out/\'stage_receipt.json\');self.assertEqual(receipt[\'method\'],\'bank_prompt_patch_v1\')\n            for n,h in receipt[\'stage_files_sha256\'].items():self.assertEqual(d.sha(stage/n),h)\n            self.assertEqual(d.read(out/\'test_lock.json\')[\'stage_receipt_sha256\'],d.sha(out/\'stage_receipt.json\'))\n            for v in (*context.arrays.values(),context.text):v._mmap.close()\n\n    def test_prompt181_payloads_and_ID_scope_without_ROI_or_old_vocab(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);ids=d.lock()[\'expected_test_ids\'];rows=[row(i) for i in ids]\n            d.save(root/\'receipt.json\',{});d.save(root/\'summary.json\',{})\n            texts=[\'a person\']+[f\'z{i:03}\' for i in range(257)];d.save(root/\'prompt_texts.json\',texts)\n            vectors=np.zeros((258,512),np.float32);vectors[:,0]=1;np.save(root/\'prompt_embeddings.npy\',vectors)\n            def write_rows():\n                (root/\'prompt_links.jsonl\').write_text(\'\'.join(__import__(\'json\').dumps(r)+\'\\n\' for r in rows),encoding=\'utf-8\')\n            write_rows();inventory=dict(cache_files={p.name:d.sha(p) for p in root.iterdir()});original=d.read\n            def reader(path):return dict(transport_inventory=inventory) if Path(path).name==\'PROMPT_TEST_PREPARATION_RESULT.json\' else original(path)\n            with patch.object(d,\'read\',side_effect=reader):\n                got,_,_,_=d.prompt_inputs(dict(prompt_test_root=str(root)));self.assertEqual(len(got),5000)\n                rows[0][\'coco_id\']=rows[1][\'coco_id\'];write_rows();inventory[\'cache_files\'][\'prompt_links.jsonl\']=d.sha(root/\'prompt_links.jsonl\')\n                with self.assertRaises(ValueError):d.prompt_inputs(dict(prompt_test_root=str(root)))\n            self.assertFalse((root/\'test_regions.npz\').exists());self.assertFalse((root/\'vocabulary.json\').exists())\n\n    def test_shared_scorer_complete_requires_all8_for_new_method(self):\n        with tempfile.TemporaryDirectory() as td:\n            out=Path(td);d.save(out/\'scoring_lock.json\',dict(files_sha256={}));d.save(out/\'chair_counts.json\',{});d.save(out/\'environment.json\',{})\n            parts={\'basic\':dict(BLEU_1=.7,BLEU_4=.3,CIDEr=1.2,ROUGE_L=.5),\'meteor\':dict(METEOR=.2),\'spice\':dict(SPICE=.2),\'chair\':dict(CHAIRs=.04,CHAIRi=.03,Recall=.4)}\n            for mode,m in parts.items():d.save(out/(mode+\'.json\'),dict(images=5000,metrics=m,input_sha256={}))\n            with patch.object(runner,\'inputs\',return_value=(out,[],[],[dict(NLL_sum=6,NLL_tokens=3,runtime_policy=d.runtime_policy(\'2.11.0+cu130\',\'2.11.0+cu130\'))])),patch(\'builtins.print\'):\n                runner.combine(dict(output=str(out),semantic=True));self.assertEqual(d.read(out/\'result.json\')[\'epoch\'],9)\n                self.assertEqual(d.read(out/\'result.json\')[\'checkpoint_sha256\'],d.CHECKPOINT_SHA)\n                d.save(out/\'spice.json\',dict(images=5000,metrics={},input_sha256={}))\n                with self.assertRaises(ValueError):runner.combine(dict(output=str(out),semantic=True))\n\n    def test_no_training_or_neural_import_source(self):\n        here=Path(__file__).parent\n        for n in (\'semantic_test_data.py\',\'run_test199.py\',\'test199_data.py\',\'make_test199_notebook.py\'):ast.parse((here/n).read_text(encoding=\'utf-8\'))\n        source=(here/\'run_test199.py\').read_text(encoding=\'utf-8\')\n        for bad in (\'backward(\', \'optimizer.step\', \'torch.optim\', \'model.train(\'):self.assertNotIn(bad,source)\n        self.assertNotIn(\'torch\',sys.modules);self.assertNotIn(\'pycocoevalcap\',sys.modules)\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'BANK_PROMPT_TEST_PROTOCOL.md': '# Bank + prompt — fixed epoch9 TEST5000, 10/11/2026\n\nUser approved TEST evaluation after reviewing VALID. Freeze best.bin epoch9 SHA7c11be52fce9cd90dadc55fa514d4017d540f40f630232a68bfe4306c754158b, training sourcef5f6eb0, SaveVersion9 at notebook kltnnnnnff (user confirmed slug/version, owner defaults to own ducanh2403; editable source route, exact weight SHA mandatory). Best snapshot has159399updates/9completedepochs even though run finished10epochs; do not demand177110updates or load finalepoch10.\n\nSame5000KarpathyTEST IDs/raw references/TRAIN vocabulary/CLIP-B16 patch bytes as baseline199, beam5/max20/rawlogprob/rank0/no lengthnormalization. One fixed learned model, no zero/shuffle branches or reranking/grid/selection/training. Three datasets: visual-cache, mscoco-2014 and csa-prompt-test-cache-v181 user reconfirmed. TRAIN bank vectors are checkpoint buffers; no bank130/prompt175 datasets needed at inference. Decode only uses image patches and frozen TEST181 object/relation phrases + geometric links. Remap stored matched ROI boxes to fractional14x14 patch pools/support with exact training record_arrays; resolve endpoints by instance IDs. Do not use old ROI feature arrays or old181 vocabulary, which differs from new baseline vocabulary. Verify needed181 payloads against historical seals and258x512 normalized vectors; all5000IDs exactlyonce/groupTEST. One unseen phrase is encoded by the same frozen text encoder in181, no caption-reference prompt creation or new encoder execution.\n\nReuse proven TEST199 patch staging/CPU scorer phases; allow explicit lock/validator injection while keeping baseline defaults/tests. New model constructed with noun/verb bank buffers from exact checkpoint and strict fullstate load. Verify saved state/method/source/core initialization/semantic source metadata and CUDA2.11.0observedcu128/cu130; training wascu130. Beam-path check on actual GPU firstbatch. Captions persisted before teacherNLL and CPU BLEU1/4,METEOR,ROUGE-L,CIDEr,SPICE,CHAIRs/i (+Recall). Onlycomplete if allphases/5000IDs/seals pass. TESTreferences onlyforNLL/scoring, nevercaptionselection. Previous project TEST exposure disclosed; no tuning on this run.\n\nFresh notebook default parent2h stopcap as baselineTEST, not ETA/billing. Current remaining GPU hours not supplied in latest answer; no increase in previous TEST cap or training authorized. Source checkpoint/API preflight and TESTprompt cache checks precede expensive stage. T4x2, exactown APIsecret unless user mounts knownSHA weights. Never use chat-exposed OpenAI token; Kaggle Secret is separate. No localTorch/models/autograd/scorers/Kaggle runs; NumPy/mock/AST/source closure author checks only. Preserve captions/checkpoints/log on failure, no automatic retries/redecode. Keep source version9 best+latest; no Word edits.\n', 'bank_layers.py': '"""CSA equations/conventions from IMPLEMENTATION_PROTOCOL; self-written, no C0."""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass Config:\n    vocab: int\n    dim: int = 512\n    heads: int = 8\n    hidden: int = 2048\n    prototypes: int = 150\n    layers: int = 3\n    dropout: float = .2\n    raw_dim: int = 768\n\n\ndef zero_pad(x, valid):\n    return x.masked_fill(~valid[..., None], 0)\n\n\nclass FFN(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(c.dim, c.hidden), nn.ReLU(), nn.Dropout(c.dropout),\n                                 nn.Linear(c.hidden, c.dim), nn.Dropout(c.dropout))\n        self.norm = nn.LayerNorm(c.dim, eps=1e-5)\n\n    def forward(self, x):\n        return self.norm(x + self.net(x))\n\n\nclass SemanticAggregation(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.wa = nn.Linear(c.dim, c.prototypes, bias=False)\n        self.mlp = nn.Sequential(nn.Linear(c.prototypes, c.prototypes), nn.ReLU(),\n                                 nn.Linear(c.prototypes, c.prototypes))\n        self.wb = nn.Linear(c.dim, c.dim, bias=False)\n\n    def forward(self, bank):\n        weights = self.mlp(self.wa(bank)).float().softmax(dim=0)\n        return weights.T.to(bank.dtype) @ self.wb(bank), weights\n\n\nclass Alignment(nn.Module):\n    def forward(self, visual, prototypes, valid):\n        if prototypes.ndim == 2:\n            prototypes = prototypes.unsqueeze(0).expand(visual.shape[0], -1, -1)\n        cosine = torch.bmm(F.normalize(prototypes.float(), dim=-1, eps=1e-6),\n                           F.normalize(visual.float(), dim=-1, eps=1e-6).transpose(1, 2))\n        weights = (cosine / .1).masked_fill(~valid[:, None, :], -torch.inf).softmax(dim=-1)\n        relevance = (weights * cosine).sum(dim=-1)\n        selected = relevance.softmax(dim=-1)[..., None] * prototypes\n        return selected.to(visual.dtype), weights\n\n\nclass Fusion(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.align = Alignment()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.cross_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm_s, self.norm_c = nn.LayerNorm(c.dim), nn.LayerNorm(c.dim)\n        self.ff_s, self.ff_c = FFN(c), FFN(c)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate = nn.Linear(2 * c.dim, 1, bias=False)\n\n    def forward(self, visual, prototypes, valid):\n        selected, _ = self.align(visual, prototypes, valid)\n        context = self.self_attn(visual, visual, visual, key_padding_mask=~valid, need_weights=False)[0]\n        own = zero_pad(self.norm_s(visual + self.drop(context)), valid)\n        own = zero_pad(self.ff_s(own), valid)\n        context = self.cross_attn(own, selected, selected, need_weights=False)[0]\n        cross = zero_pad(self.norm_c(own + self.drop(context)), valid)\n        cross = zero_pad(self.ff_c(cross), valid)\n        gate = self.gate(torch.cat((own, cross), dim=-1)).sigmoid()\n        return zero_pad(gate * own + (1 - gate) * cross, valid), selected\n\n\nclass DecoderLayer(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.object_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.relation_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm = nn.LayerNorm(c.dim)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate_o, self.gate_r = nn.Linear(2 * c.dim, c.dim), nn.Linear(2 * c.dim, c.dim)\n        self.ffn = FFN(c)\n\n    def forward(self, x, previous_word, vo, vr, text_valid, region_valid):\n        causal = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).triu(1)\n        h = self.self_attn(x, x, x, attn_mask=causal, key_padding_mask=~text_valid, need_weights=False)[0]\n        h = zero_pad(self.norm(x + self.drop(h)), text_valid)\n        co = self.object_attn(h, vo, vo, key_padding_mask=~region_valid, need_weights=False)[0]\n        cr = self.relation_attn(h, vr, vr, key_padding_mask=~region_valid, need_weights=False)[0]\n        go = self.gate_o(torch.cat((co, previous_word), dim=-1)).sigmoid()\n        gr = self.gate_r(torch.cat((cr, previous_word), dim=-1)).sigmoid()\n        return zero_pad(self.ffn(go * co + gr * cr), text_valid)\n\n\nclass CSACaptioner(nn.Module):\n    def __init__(self, c, nouns, verbs):\n        super().__init__()\n        if c.dim % 2 or c.dim % c.heads or c.vocab < 4 or c.layers < 1 or c.prototypes < 1:\n            raise ValueError(\'Invalid model dimensions\')\n        for bank in (nouns, verbs):\n            if bank.ndim != 2 or bank.shape[1] != c.dim or not len(bank) or not torch.isfinite(bank).all():\n                raise ValueError(\'Empty, nonfinite or incompatible semantic bank\')\n        self.c = c\n        self.register_buffer(\'nouns\', nouns.detach().float().clone())\n        self.register_buffer(\'verbs\', verbs.detach().float().clone())\n        self.project = nn.Linear(c.raw_dim, c.dim)\n        self.noun_agg, self.verb_agg = SemanticAggregation(c), SemanticAggregation(c)\n        self.object1, self.object2, self.relation = Fusion(c), Fusion(c), Fusion(c)\n        self.embedding = nn.Embedding(c.vocab, c.dim, padding_idx=0)\n        self.decoder = nn.ModuleList([DecoderLayer(c) for _ in range(c.layers)])\n        self.head = nn.Linear(c.dim, c.vocab)\n        self.apply(self._init)\n        for module in self.modules():\n            if isinstance(module, nn.MultiheadAttention):\n                nn.init.xavier_uniform_(module.in_proj_weight)\n                nn.init.zeros_(module.in_proj_bias)\n        nn.init.normal_(self.embedding.weight, std=c.dim ** -.5)\n        with torch.no_grad():\n            self.embedding.weight[0].zero_()\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, nn.Linear):\n            nn.init.xavier_uniform_(module.weight)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.LayerNorm):\n            nn.init.ones_(module.weight)\n            nn.init.zeros_(module.bias)\n\n    def encode(self, regions, valid):\n        if valid.dtype != torch.bool or regions.ndim != 3 or valid.shape != regions.shape[:2] or not valid.any(dim=1).all():\n            raise ValueError(\'Every image requires at least one valid region\')\n        if not torch.isfinite(regions).all():\n            raise ValueError(\'Nonfinite visual feature\')\n        v = zero_pad(self.project(regions), valid)\n        nouns, _ = self.noun_agg(self.nouns)\n        verbs, _ = self.verb_agg(self.verbs)\n        v, selected = self.object1(v, nouns, valid)\n        vo, _ = self.object2(v, selected, valid)\n        vr, _ = self.relation(vo, verbs, valid)\n        return vo, vr\n\n    def decode(self, ids, vo, vr, valid):\n        if ids.ndim != 2 or ids.shape[1] == 0 or not (ids[:, 0] == 1).all():\n            raise ValueError(\'Prefix must start with BOS, with nonempty time dimension\')\n        raw = self.embedding(ids)\n        position = torch.arange(ids.shape[1], device=ids.device, dtype=torch.float32)[:, None]\n        frequency = torch.exp(torch.arange(0, self.c.dim, 2, device=ids.device).float() * (-math.log(10000) / self.c.dim))\n        pe = torch.empty(ids.shape[1], self.c.dim, device=ids.device)\n        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)\n        text_valid = ids != 0\n        x = zero_pad(raw * math.sqrt(self.c.dim) + pe.to(raw.dtype), text_valid)\n        for layer in self.decoder:\n            x = layer(x, raw, vo, vr, text_valid, valid)\n        return self.head(x)\n\n    def forward(self, regions, valid, ids):\n        return self.decode(ids, *self.encode(regions, valid), valid)\n\n\ndef xe_loss(logits, target):\n    mask = target != 0\n    if not mask.any():\n        raise ValueError(\'XE batch has no target tokens\')\n    nll = F.cross_entropy(logits.flatten(0, 1), target.flatten(), reduction=\'none\', ignore_index=0).view_as(target)\n    return (nll * mask).sum() / mask.sum()\n\n\n@torch.no_grad()\ndef beam_search(model, vo, vr, valid, beams=5, maximum=20):\n    """Per-image beams; no hidden length normalization, BOS/PAD cannot be emitted."""\n    if model.training or maximum < 1 or not 1 <= beams <= model.c.vocab - 2:\n        raise ValueError(\'Beam search requires eval mode and valid width/length\')\n    results = []\n    for i in range(len(vo)):\n        seq = torch.ones(1, 1, dtype=torch.long, device=vo.device)\n        scores = torch.zeros(1, device=vo.device)\n        finished = torch.zeros(1, device=vo.device, dtype=torch.bool)\n        for _ in range(maximum):\n            count = len(seq)\n            logits = model.decode(seq, vo[i:i+1].expand(count, -1, -1),\n                                  vr[i:i+1].expand(count, -1, -1), valid[i:i+1].expand(count, -1))[:, -1]\n            logp = logits.float().log_softmax(-1)\n            logp[:, :2] = -torch.inf\n            logp[finished] = -torch.inf\n            logp[finished, 0] = 0\n            best, flat = (scores[:, None] + logp).flatten().topk(beams)\n            parent, token = flat // model.c.vocab, flat % model.c.vocab\n            seq = torch.cat((seq[parent], token[:, None]), dim=1)\n            finished = finished[parent] | (token == 2)\n            scores = best\n            if finished.all():\n                break\n        results.append((seq, scores))\n    return results\n\n\ndef beam_mean_loss(model, regions, valid, sequences, rewards, maximum=20):\n    """Gradient rescore; rewards [B,K] are externally computed CIDEr-D and detached."""\n    b, k, length = sequences.shape\n    if rewards.shape != (b, k) or length < 2 or length > maximum + 1:\n        raise ValueError(\'Invalid beam/reward shape\')\n    flat = sequences.reshape(b * k, length)\n    logits = model(regions.repeat_interleave(k, 0), valid.repeat_interleave(k, 0), flat[:, :-1])\n    target = flat[:, 1:]\n    # Include first EOS, exclude every later position (even malformed non-PAD tokens).\n    eos_before = (target == 2).cumsum(dim=1) - (target == 2).long()\n    mask = (target != 0) & (eos_before == 0)\n    logp = logits.float().log_softmax(-1).gather(-1, target[..., None]).squeeze(-1)\n    score = (logp * mask).sum(-1).reshape(b, k) / maximum\n    advantage = rewards.detach() - rewards.detach().mean(dim=1, keepdim=True)\n    return -(advantage * score).mean()\n', 'PROMPT_TEST_PREPARATION_RESULT.json': '{\n  "status": "TEST_PREPARATION_METADATA_REVIEW_PASS",\n  "checks": 48,\n  "summary": {\n    "status": "PROMPT_TEST_PREPARATION_COMPLETE",\n    "test_images": 5000,\n    "caption_generation": false,\n    "caption_training": false,\n    "caption_scoring": false,\n    "with_objects": 4865,\n    "with_relations": 1136,\n    "objects": 14593,\n    "relations": 2228,\n    "texts": 258,\n    "reused_texts": 257,\n    "new_texts": [\n      "a person hugging an elephant"\n    ],\n    "source175_files": {\n      "cohort.json": "d97eaa8cecd07a4de825e3be99926f751f610e103ff9582c2b482e53d2c9980d",\n      "prompt_texts.json": "35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2",\n      "prompt_links.jsonl": "e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0",\n      "summary.json": "28d650c5403fcf851e9a8f5883824ceba2d46144a9d2222be36cb056e07b5122",\n      "review_examples.json": "17b4ed35145338ac3fe65b28b293e4a11d8ac39fba1199ff3a46b29219329df7",\n      "prompt_embeddings.npy": "f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c",\n      "mapping_receipt.json": "134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d",\n      "reuse_report.json": "de76c927e7839bfb553ca09ea3373474ba2afeae6b12b7f1e9959d8ed41358ef",\n      "cache_manifest.json": "05f40a246b19cb37717a4aadb7bcb20f2e0dd0ebabe60e983d83b9f328e9f56a",\n      "cache_receipt.json": "72c532cd73b832c5cdf8f0ff747a882413e006fa84670f61a6a54abd272613cb",\n      "run_status.json": "fc1f7837f14785888c4ff30c193ca71b72ce34931f93c918cc23a9945089d8e9",\n      "source_receipt.json": "7df1b07aa9e8ef821e4608f406c355142d3435dbd7aa5bbf805daf4e6afd5365",\n      "notebook.log": "5c656006b117092d6cc19ef472d18e7fb264a6f1a00876be16f694af87facd74",\n      "cache_plan.json": "4f1fd4c729fe72db06de781ad2a99af8755eef225cae23ecd989778438aeb057"\n    },\n    "new_encoding": {\n      "spec": {\n        "id": "openai/clip-vit-base-patch16",\n        "revision": "57c216476eefef5ab752ec549e440a49ae4ae5f3",\n        "files_sha256": {\n          "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n          "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n          "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n          "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n          "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n          "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n          "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n          "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n        }\n      },\n      "files_sha256": {\n        "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n        "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n        "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n        "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n        "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n        "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n        "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n        "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n      },\n      "loading_info": {\n        "missing_keys": [],\n        "unexpected_keys": [],\n        "mismatched_keys": [],\n        "error_msgs": []\n      },\n      "max_tokens": 7,\n      "seconds": 14.177083713999991\n    }\n  },\n  "transport_inventory": {\n    "cache_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/181",\n    "cache_files": {\n      "test_regions.npz": "7fb385e1a03c5241c4d65cc240d0434e041284a9a829198c5167f0725be57b1a",\n      "test_metadata.json": "4271b46b7ab67117433bfdf84f6eb9ba66515ada9b3c4ae4d2e507bc2cede018",\n      "vocabulary.json": "924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7",\n      "test_receipt.json": "7a212068c7e73f6f0791906c1ec81d56e3178fd62317bdf0a5e528350d95055d",\n      "prompt_links.jsonl": "54e25cd17c9842c07fb4ef0b3da4ed4ad276571d3711d51bf0b3ac1a86cc3c55",\n      "prompt_texts.json": "afdad5d42734adfbd38883f1fa7a22aaba1b76ee8cd7510635fd3e8b22d44553",\n      "prompt_embeddings.npy": "7119776b70380376a343dd9130bed6d2e5fc19594d83cf4bdd724a7343a4fb35",\n      "summary.json": "3c9b863b73d9efdeda7f5979d25c829c74d805505a856c965141dc275a0e8847",\n      "source_coverage.json": "b5f66a8bcb40027dcdd11b0d71c92551d68a11f91a6daf56e1157cdd0b5a7a3c",\n      "receipt.json": "f86708aa99a3a07087a55164cd3fa099d01e01eef15a91da38540dc28a5c3742"\n    },\n    "source_review_zip_sha256": "44f752a96e5fd228f704f45a0cc7de0160b97a4636ad958ec4c59a2e77b1de8a"\n  },\n  "limitations": [\n    "NPZ/metadata/vector binaries absent review ZIP; their payload hashes are user-run evidence.",\n    "Slot packing used dummy vectors; real258x512 vector validation occurs during transport.",\n    "No author models/scorers/API; no new TEST captions or accuracy scores."\n  ]\n}\n', 'csa_patch_model.py': '"""Paper-structure CSA encoder/parallel decoder; CLIP-patch + UPT extensions disclosed."""\nimport torch\nfrom torch import nn\nfrom bank_layers import CSACaptioner, Config\nfrom semantic_patch_model import PromptMemory\n\n\nclass CSAPatchCaptioner(nn.Module):\n    def __init__(self, vocab, nouns, verbs):\n        super().__init__()\n        self.core = CSACaptioner(Config(vocab=vocab), nouns, verbs)\n        self.object_prompt = PromptMemory(512 + 512 + 4)\n        self.relation_prompt = PromptMemory(512 + 512 + 512 + 8)\n\n    @property\n    def head(self):\n        return self.core.head\n\n    def streams(self, patches, p):\n        if patches.ndim != 3 or patches.shape[1:] != (196, 768):\n            raise ValueError(\'Expected196CLIPpatches, not ROI indices/features\')\n        valid = torch.ones(patches.shape[:2], dtype=torch.bool, device=patches.device)\n        vo, vr = self.core.encode(patches, valid)\n        obj = torch.cat((p[\'object_text\'], torch.bmm(p[\'object_weights\'], vo), p[\'object_box\']), -1)\n        rel = torch.cat((p[\'relation_text\'], torch.bmm(p[\'subject_weights\'], vo),\n                         torch.bmm(p[\'target_weights\'], vo), p[\'relation_geometry\']), -1)\n        return (self.object_prompt(vo, obj, p[\'object_weights\'], p[\'object_valid\']),\n                self.relation_prompt(vr, rel, p[\'subject_weights\'] + p[\'target_weights\'], p[\'relation_valid\']))\n\n    def encode(self, patches, p):\n        # Packaging for existing beam expansion only; decode restores both streams.\n        return torch.cat(self.streams(patches, p), dim=1)\n\n    def decode(self, memory, prefix):\n        if memory.ndim != 3 or memory.shape[1:] != (392, 512):\n            raise ValueError(\'Expected packaged pair of196x512memories\')\n        vo, vr = memory.split(196, dim=1)\n        valid = torch.ones(vo.shape[:2], dtype=torch.bool, device=vo.device)\n        return self.core.decode(prefix, vo, vr, valid)\n\n    def forward(self, patches, prefix, p):\n        return self.decode(self.encode(patches, p), prefix)\n', 'memory_fusion.py': '"""Shared neural/NumPy fusion algebra and saved gate diagnostics, no Torch import."""\nimport math\nimport numpy as np\n\n\ndef fuse_memory(visual,objects,relations,branch,strength):\n    delta=branch*(objects-visual)+(1-branch)*(relations-visual)\n    return visual+strength*delta\n\n\ndef update_gate_stats(state,values):\n    a=np.asarray(values,dtype=np.float64).reshape(-1)\n    if not len(a) or not np.isfinite(a).all() or (a<0).any() or (a>1).any():raise ValueError(\'Invalid semantic strength\')\n    state=state or dict(count=0,total=0.,minimum=1.,maximum=0.,below_point1=0,above_point9=0)\n    return dict(count=state[\'count\']+len(a),total=state[\'total\']+float(a.sum()),minimum=min(state[\'minimum\'],float(a.min())),maximum=max(state[\'maximum\'],float(a.max())),\n        below_point1=state[\'below_point1\']+int((a<.1).sum()),above_point9=state[\'above_point9\']+int((a>.9).sum()))\n\n\ndef merge_gate_stats(states):\n    if not states or any(s[\'count\']<=0 or not math.isfinite(s[\'total\']) for s in states):raise ValueError(\'Missing semantic gate diagnostics\')\n    count=sum(s[\'count\'] for s in states)\n    return dict(patches=count,mean=sum(s[\'total\'] for s in states)/count,minimum=min(s[\'minimum\'] for s in states),maximum=max(s[\'maximum\'] for s in states),\n        fraction_below_point1=sum(s[\'below_point1\'] for s in states)/count,fraction_above_point9=sum(s[\'above_point9\'] for s in states)/count,\n        interpretation=\'Learned residual strength, not a calibrated confidence or grounding accuracy.\')\n', 'csa_per_caption_test_data.py': '"""Fixed beam SCST v199 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read, save, sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS, record_arrays, checked_root\nHERE = Path(__file__).resolve().parent\nfrom epoch2_contract import validate_eval_binding, END_STEPS, PARENT_SHA\nEPOCH2_BINDING = validate_eval_binding(read(HERE/\'EPOCH2_EVAL_LOCK.json\'))\nCHECKPOINT_SHA = EPOCH2_BINDING[\'sha256\']\nXE_SOURCE_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nVERSION = EPOCH2_BINDING[\'version\']\n\ndef validate_lock(value):\n    baseline = shared.lock()\n    if any((value[k] != baseline[k] for k in (\'expected_test_ids\', \'test_id_sha256\', \'beam_width\', \'max_new_tokens\', \'length_normalization\', \'reranking\'))):\n        raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\'] != VERSION or value[\'checkpoint_sha256\'] != CHECKPOINT_SHA or value[\'epoch\'] != 2 or (value[\'updates\'] != END_STEPS) or (value[\'method\'] != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'):\n        raise ValueError(\'Wrong frozen beam SCST epoch2 TEST lock\')\n    return value\n\ndef lock():\n    value = shared.lock()\n    value.update(schema=\'csa_per_caption_best9_test_v1\', source_version=VERSION, checkpoint_sha256=CHECKPOINT_SHA, training_commit=\'5eb012279e6fd0b5e47a56710ed5e3efa7449d2f\', epoch=2, updates=END_STEPS, method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\')\n    return validate_lock(value)\n\ndef checkpoint(root):\n    from checkpoint_input import resolve_mounted\n    return resolve_mounted(root, CHECKPOINT_SHA)\n\ndef runtime_policy(training, current):\n    if training != \'2.11.0+cu128\' or current not in (\'2.11.0+cu128\', \'2.11.0+cu130\'):\n        raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training, inference_torch=current, cuda_wheel_changed=training != current, policy=\'Same publicTorch2.11.0 observed CUDA wheels; strictstate and actual GPU beam-path check required.\', bitwise_cross_build_equivalence_established=False)\n\ndef validate_checkpoint_state(ck):\n    if ck[\'updates\'] != END_STEPS or ck[\'source_sha256\'] != XE_SOURCE_SHA or ck[\'epoch\'] != 2 or ck[\'parent_sha256\'] != PARENT_SHA:\n        raise ValueError(\'Wrong beam SCST epoch2 endpoint lineage\')\n    if not {\'core.nouns\', \'core.verbs\'} <= set(ck[\'model\']):\n        raise ValueError(\'Missing frozen TRAIN banks\')\n    return lock()\n\ndef prompt_inputs(cfg):\n    inventory = read(HERE / \'PROMPT_TEST_PREPARATION_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    wanted = {n: inventory[n] for n in (\'receipt.json\', \'summary.json\', \'prompt_links.jsonl\', \'prompt_texts.json\', \'prompt_embeddings.npy\')}\n    root = checked_root(cfg[\'prompt_test_root\'], \'receipt.json\', wanted)\n    texts = read(root / \'prompt_texts.json\')\n    vectors = np.load(root / \'prompt_embeddings.npy\', allow_pickle=False)\n    if texts != sorted(set(texts)) or len(texts) != 258 or vectors.shape != (258, 512) or (vectors.dtype != np.float32) or (not np.isfinite(vectors).all()) or (not np.allclose(np.linalg.norm(vectors, axis=1), 1, rtol=0, atol=2e-06)):\n        raise ValueError(\'Changed TEST181 text space\')\n    rows = [json.loads(line) for line in (root / \'prompt_links.jsonl\').read_text(encoding=\'utf-8\').splitlines()]\n    expected = lock()[\'expected_test_ids\']\n    if len(rows) != 5000 or sorted((r[\'coco_id\'] for r in rows)) != expected or any((r[\'group\'] != \'test\' for r in rows)):\n        raise ValueError(\'Wrong/duplicate prompt TEST IDs\')\n    return (rows, texts, vectors, wanted)\n\ndef prepare(cfg):\n    rows, texts, vectors, seals = prompt_inputs(cfg)\n    for row in rows:\n        record_arrays(row, texts)\n    shared.prepare(cfg, locked=lock(), weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg, rows, texts, vectors, seals)\n\ndef stage_prompts(cfg, rows, texts, vectors, seals):\n    encoded = [record_arrays(row, texts) for row in rows]\n    stage = Path(cfg[\'stage\'])\n    out = Path(cfg[\'output\'])\n    ids = np.load(stage / \'ids.npy\', allow_pickle=False)\n    lookup = {r[\'coco_id\']: j for j, r in enumerate(rows)}\n    order = [lookup[int(i)] for i in ids]\n    for key in ARRAYS:\n        np.save(stage / (\'sem_\' + key + \'.npy\'), np.stack([encoded[j][key] for j in order]))\n    np.save(stage / \'sem_text_vectors.npy\', vectors)\n    receipt = read(out / \'stage_receipt.json\')\n    receipt[\'method\'] = \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'\n    receipt[\'prompt181_files_sha256\'] = seals\n    receipt[\'stage_files_sha256\'].update({p.name: sha(p) for p in stage.glob(\'sem_*.npy\')})\n    save(out / \'stage_receipt.json\', receipt)\n    saved = read(out / \'test_lock.json\')\n    saved[\'stage_receipt_sha256\'] = sha(out / \'stage_receipt.json\')\n    save(out / \'test_lock.json\', saved)\n    save(out / \'semantic_test_stage.json\', dict(images=5000, objects=sum((len(r[\'objects\']) for r in rows)), relations=sum((len(r[\'relations\']) for r in rows)), text_vectors=258, geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\', TRAIN_VALID_features_read=False, old_ROI_features_used=False, old_vocabulary_used=False))\n\ndef load(cfg, verify=True):\n    result = shared.load(cfg, verify=verify, validator=validate_lock)\n    if result[2].get(\'method\') != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':\n        raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n', 'run_csa_per_caption_test.py': '"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import csa_per_caption_test_data as module\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(\'2.11.0+cu128\',str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from csa_patch_model import CSAPatchCaptioner\n            from semantic_data import SemanticContext\n            model=CSAPatchCaptioner(len(words),ck[\'model\'][\'core.nouns\'],ck[\'model\'][\'core.verbs\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];beams=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,seqs,values in zip(index.tolist(),seq.cpu().tolist(),scores.cpu().tolist()):\n                    beams.append(dict(coco_id=int(data[\'ids\'][i]),candidates=[\n                        dict(ids=ids,caption=text_from_ids(ids,words),logprob=float(value))\n                        for ids,value in zip(seqs,values)]))\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'TEST decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            save(out/f\'beams_rank{rank}.json\',beams)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),beams_sha256=sha(out/f\'beams_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==d.CHECKPOINT_SHA for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong TEST references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=tok.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete TEST scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CSA_PER_CAPTION_BEST9_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n', 'captioning/__init__.py': '', 'checkpoint_input.py': '"""Check exact mounted checkpoint bytes without importing Torch or loading pickle."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef resolve_mounted(root, expected_sha, report_path=None):\n    root = Path(root)\n    files = [root] if root.is_file() else sorted(\n        (p for p in root.rglob(\'*\') if p.is_file() and p.suffix.lower() in (\'.bin\',\'.pt\',\'.pth\',\'.ckpt\')\n         and p.name != \'cache_metadata_zip.bin\'),\n        key=lambda p: (p.name != \'best.bin\', \'best\' not in p.name.lower(), str(p))) if root.is_dir() else []\n    report = dict(root=str(root), expected_sha256=expected_sha, model_loaded=False,\n                  checked=[], matched_file=None,\n                  extracted_torch_directories=[str(p.parent) for p in root.rglob(\'data.pkl\')] if root.is_dir() else [])\n    for path in files:\n        with path.open(\'rb\') as stream:\n            digest = hashlib.file_digest(stream, \'sha256\').hexdigest()\n        row = dict(path=str(path), bytes=path.stat().st_size, sha256=digest)\n        report[\'checked\'].append(row)\n        print(\'Checkpoint input:\', json.dumps(row), flush=True)\n        if digest == expected_sha:\n            report[\'matched_file\'] = str(path.resolve())\n            break\n    if report_path:\n        Path(report_path).write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    if report[\'matched_file\'] is None:\n        print(json.dumps(report,indent=2),flush=True)\n        hint = \' Torch archive appears extracted; attach the intact checkpoint file.\' if report[\'extracted_torch_directories\'] else \'\'\n        raise FileNotFoundError(\'No mounted checkpoint matches the locked source SHA; no model loaded.\'+hint)\n    return Path(report[\'matched_file\'])\n', 'csa_per_caption_rerank_data.py': '"""Prepare and check inputs for the frozen CSA-only rerank replay."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom captioning.object_consistency import index_detections\nfrom patch_beam import text_from_ids\nfrom real_bank_data import read, save, sha\nfrom test199_data import DATA_SHA\n\nHERE = Path(__file__).resolve().parent\nPOLICIES = (\'decoder_rank0\', \'clip_object_fixed\')\nDETECTION_SHA = \'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c\'\n\n\ndef prepare(cfg):\n    out = Path(cfg[\'output\'])\n    test = Path(cfg[\'test_output\'])\n    source = read(test/\'test_lock.json\')\n    if sha(test/\'test_references.json\')!=\'311ff29fc3dab1dde650a01b141c2498fbc44a4d7f96f63059dfffe0bdd6e776\':raise ValueError(\'TEST references differ from same-seed CE\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if source[\'checkpoint_sha256\'] != replay[\'checkpoint_sha256\']:\n        raise ValueError(\'Changed CSA checkpoint\')\n    if sha(cfg[\'dataset_json\']) != DATA_SHA or sha(cfg[\'detections\']) != DETECTION_SHA:\n        raise ValueError(\'Changed COCO metadata or detector\')\n    records = sorted((r for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'test\'),\n                     key=lambda r: int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in records] != source[\'expected_test_ids\']:\n        raise ValueError(\'Wrong TEST image cohort\')\n    detections = index_detections(read(cfg[\'detections\']))\n    root = Path(cfg[\'dataset_json\']).parent.resolve()\n    names = {r[\'filename\'] for r in records}\n    paths = {}\n    for p in root.rglob(\'*.jpg\'):\n        if p.name in names:\n            if p.name in paths or not p.resolve().is_relative_to(root):\n                raise ValueError(\'Ambiguous TEST JPEG: \'+p.name)\n            paths[p.name] = p\n    if set(paths) != names or not names <= detections.keys():\n        raise ValueError(\'Missing TEST JPEGs or detector rows\')\n    inputs = []\n    for row in records:\n        name = row[\'filename\']\n        path = paths[name]\n        inputs.append(dict(coco_id=int(row[\'cocoid\']), image_path=str(path),\n                           image_sha256=sha(path), detected=detections[name]))\n    save(out/\'inference_inputs.json\', inputs)\n    (out/\'test_references.json\').write_bytes((test/\'test_references.json\').read_bytes())\n    save(out/\'test_lock.json\', dict(\n        checkpoint_sha256=replay[\'checkpoint_sha256\'], images=5000,\n        clip=policy[\'clip\'], selector=policy[\'frozen_selector\'],\n        inference_sha256=sha(out/\'inference_inputs.json\'),\n        references_sha256=sha(out/\'test_references.json\')))\n\n\ndef checked_lock(out):\n    out = Path(out)\n    lock = read(out/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if (lock[\'clip\'], lock[\'selector\'], lock[\'checkpoint_sha256\'], lock[\'images\']) != (\n            policy[\'clip\'], policy[\'frozen_selector\'], replay[\'checkpoint_sha256\'], 5000):\n        raise ValueError(\'Changed rerank policy or cohort\')\n    if sha(out/\'inference_inputs.json\') != lock[\'inference_sha256\']:\n        raise ValueError(\'Changed inference inputs\')\n    return lock\n\n\ndef read_beams(out, lock):\n    out = Path(out)\n    test = out.parent\n    inputs = read(out/\'inference_inputs.json\')\n    words = read(test/\'vocabulary.json\')\n    by_id = {}\n    for rank in (0, 1):\n        path = test/f\'beams_rank{rank}.json\'\n        receipt = read(test/f\'worker{rank}.json\')\n        if (not receipt[\'complete\'] or receipt[\'images\'] != 2500\n                or receipt[\'checkpoint_sha256\'] != lock[\'checkpoint_sha256\']\n                or sha(path) != receipt[\'beams_sha256\']\n                or sha(test/f\'captions_rank{rank}.json\') != receipt[\'caption_sha256\']):\n            raise ValueError(\'Incomplete CSA beam replay\')\n        rows = read(path)\n        original = read(test/f\'captions_rank{rank}.json\')\n        if len(original) != 2500 or [r[\'coco_id\'] for r in rows] != [r[\'coco_id\'] for r in inputs[rank::2]]:\n            raise ValueError(\'Wrong TEST beam coverage\')\n        for row, top in zip(rows, original):\n            beams = row[\'candidates\']\n            if (len(beams) != 5 or row[\'coco_id\'] != top[\'coco_id\']\n                    or beams[0] != dict(ids=top[\'ids\'], caption=top[\'caption\'], logprob=top[\'score\'])):\n                raise ValueError(\'Top-1 does not match beam candidates\')\n            if any(c[\'caption\'] != text_from_ids(c[\'ids\'], words) for c in beams):\n                raise ValueError(\'Changed beam text\')\n            by_id[row[\'coco_id\']] = row\n    return [by_id[row[\'coco_id\']] for row in inputs]\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    prepare(read(parser.parse_args().config))\n', 'run_csa_per_caption_rerank.py': "import argparse, hashlib, io, json, os, time\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom csa_per_caption_rerank_data import checked_lock, read_beams, POLICIES\nfrom captioning.multiscore_reranking import rerank_multiscore\nfrom score_prompt_test_extra import java_path\n\ndef fixed_rank(row, clipscore, detected):\n    if len(row['candidates']) != 5 or len(clipscore) != 5 or any((not np.isfinite(x) or not 0 <= x <= 2.50001 for x in clipscore)):\n        raise ValueError('Invalid TEST CLIP scores/width')\n    scores = [c['logprob'] for c in row['candidates']]\n    if not all(np.isfinite(scores)) or scores != sorted(scores, reverse=True):\n        raise ValueError('Invalid beam logprobs')\n    enriched = [dict(c, clipscore=float(s)) for c, s in zip(row['candidates'], clipscore)]\n    chosen, _ = rerank_multiscore(dict(candidates=enriched), detected, 0.4, 0.3)\n    return int(chosen['original_rank'])\n\ndef prefetch(cfg):\n    from huggingface_hub import snapshot_download\n    out = Path(cfg['output'])\n    locked = checked_lock(out)['clip']\n    root = Path(snapshot_download(locked['id'], revision=locked['revision'], allow_patterns=list(locked['files_sha256']), cache_dir=cfg['model_cache']))\n    for n, h in locked['files_sha256'].items():\n        if sha(root / n) != h:\n            raise ValueError('CLIP assets differ from VALID164: ' + n)\n    save(out / 'clip_assets.json', dict(**locked, path=str(root)))\n    print('CLIP model/processor match frozen VALID164 bytes; no BLIP download.', flush=True)\n\ndef text_preflight(cfg):\n    from transformers import CLIPProcessor\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    rows = read_beams(out, lock)\n    model = read(out / 'clip_assets.json')\n    processor = CLIPProcessor.from_pretrained(model['path'], local_files_only=True, use_fast=False)\n    lengths = [len(v) for v in processor.tokenizer([c['caption'] for r in rows for c in r['candidates']], truncation=False)['input_ids']]\n    if len(lengths) != 25000 or max(lengths) > 77:\n        raise ValueError('Would truncate TEST candidates')\n    save(out / 'token_coverage.json', dict(images=5000, captions=25000, max_tokens=max(lengths), limit=77))\n\ndef worker(cfg):\n    import torch\n    import torch.nn.functional as F\n    from PIL import Image\n    from transformers import CLIPModel, CLIPProcessor\n    rank = int(os.environ['LOCAL_RANK'])\n    if int(os.environ['WORLD_SIZE']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError('Select T4x2')\n    torch.cuda.set_device(rank)\n    device = torch.device('cuda', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    asset = read(out / 'clip_assets.json')\n    for n, h in lock['clip']['files_sha256'].items():\n        if sha(Path(asset['path']) / n) != h:\n            raise ValueError('Changed CLIP model/processor')\n    base = CLIPModel.from_pretrained(asset['path'], local_files_only=True, use_safetensors=False, attn_implementation='eager').float().to(device).eval().requires_grad_(False)\n    processor = CLIPProcessor.from_pretrained(asset['path'], local_files_only=True, use_fast=False)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256']:\n        raise ValueError('Changed TEST inference inputs')\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    started = time.monotonic()\n    count = 0\n    path = out / f'scores_rank{rank}.jsonl'\n    with path.open('x', encoding='utf-8') as log, torch.inference_mode():\n        for row, beam in list(zip(inputs, beams))[rank::2]:\n            if time.monotonic() - started > cfg['phase_seconds']:\n                raise TimeoutError('TEST CLIP scoring cap; preserve partial scores')\n            raw = Path(row['image_path']).read_bytes()\n            if hashlib.sha256(raw).hexdigest() != row['image_sha256']:\n                raise ValueError('Changed raw TEST image')\n            with Image.open(io.BytesIO(raw)) as image:\n                image = image.convert('RGB')\n            pixels = processor(images=image, return_tensors='pt')['pixel_values'].to(device)\n            text = processor.tokenizer([c['caption'] for c in beam['candidates']], padding=True, truncation=False, return_tensors='pt').to(device)\n            image_features = F.normalize(base.get_image_features(pixel_values=pixels).float(), dim=-1)\n            text_features = F.normalize(base.get_text_features(**text).float(), dim=-1)\n            clipscore = (2.5 * (image_features * text_features).sum(-1).clamp_min(0)).cpu().tolist()\n            chosen = fixed_rank(beam, clipscore, row['detected'])\n            log.write(json.dumps(dict(coco_id=row['coco_id'], image_sha256=row['image_sha256'], clipscore=clipscore, selected_rank=chosen), allow_nan=False) + '\\n')\n            log.flush()\n            count += 1\n            if count % 100 == 0:\n                print('TEST CLIP GPU', rank, count, '/2500', flush=True)\n            if count == 25 and (time.monotonic() - started) / count * (2500 - count) > cfg['phase_seconds'] - (time.monotonic() - started):\n                raise TimeoutError('Measured TEST CLIP projection exceeds cap')\n    save(out / f'worker{rank}.json', dict(complete=True, images=count, scored_pairs=count * 5, scores_sha256=sha(path), seconds=time.monotonic() - started, torch_version=torch.__version__, training=False, clip_revision=lock['clip']['revision']))\n\ndef score(cfg):\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    java_path(Path(cfg['test_output']))\n\n    import subprocess, sys\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from saved_quality_data import chair_counts\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256'] or sha(out / 'test_references.json') != lock['references_sha256']:\n        raise ValueError('Changed locked inputs')\n    scored = {}\n    for rank in (0, 1):\n        w = read(out / f'worker{rank}.json')\n        path = out / f'scores_rank{rank}.jsonl'\n        if not w['complete'] or w['images'] != 2500 or w['scored_pairs'] != 12500 or (sha(path) != w['scores_sha256']):\n            raise ValueError('Incomplete TEST scoring worker')\n        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]\n        if [r['coco_id'] for r in rows] != [r['coco_id'] for r in inputs[rank::2]]:\n            raise ValueError('Wrong TEST score IDs')\n        scored.update({r['coco_id']: r for r in rows})\n    if len(scored) != 5000:\n        raise ValueError('Missing TEST scores')\n    chosen = []\n    for row, beam in zip(inputs, beams):\n        s = scored[row['coco_id']]\n        if s['image_sha256'] != row['image_sha256']:\n            raise ValueError('Wrong image-score identity')\n        k = fixed_rank(beam, s['clipscore'], row['detected'])\n        if k != s['selected_rank']:\n            raise ValueError('Fixed selection replay differs')\n        chosen.append(k)\n    save(out / 'selected_ranks.json', chosen)\n    refs = read(out / 'test_references.json')\n    ptb = PTBTokenizer()\n    ref = ptb.tokenize({r['coco_id']: [dict(caption=c) for c in r['references']] for r in refs})\n    chair = Path(cfg['chair'])\n    metrics = {}\n    flags = {}\n    if sha(chair / 'chair.pkl') != '73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':\n        raise ValueError('Changed CHAIR cache')\n    for policy in POLICIES:\n        selected = [dict(coco_id=r['coco_id'], caption=r['candidates'][0 if policy == 'decoder_rank0' else k]['caption']) for r, k in zip(beams, chosen)]\n        hyp = ptb.tokenize({r['coco_id']: [dict(caption=r['caption'])] for r in selected})\n        c, per = Cider().compute_score(ref, hyp)\n        b, _ = Bleu(4).compute_score(ref, hyp)\n        r, _ = Rouge().compute_score(ref, hyp)\n        ci = out / (policy + '_chair_input.json')\n        co = out / (policy + '_chair_output.json')\n        save(ci, [dict(image_id=r['coco_id'], caption=r['caption']) for r in selected])\n        subprocess.run([sys.executable, '-u', chair / 'chair.py', '--cap_file', ci, '--cache', chair / 'chair.pkl', '--save_path', co], cwd=chair, check=True, timeout=300)\n        ch = read(co)\n        m, counts = chair_counts(ch, selected)\n        save(out / (policy + '_captions.json'), selected)\n        save(out / (policy + '_cider_per_image.json'), [dict(coco_id=int(i), CIDEr=float(v)) for i,v in zip(ref,per)])\n        partial=dict(CIDEr=float(c),BLEU_1=float(b[0]),BLEU_4=float(b[3]),ROUGE_L=float(r),**m,counts=counts)\n        save(out / (policy + '_basic_chair.json'),partial)\n        meteor = Meteor()\n        try:\n            meteor_score, meteor_per = meteor.compute_score(ref, hyp)\n        finally:\n            del meteor\n        save(out / (policy + '_meteor.json'),dict(METEOR=float(meteor_score),per=[float(v) for v in meteor_per]))\n        spice_score, spice_per = Spice().compute_score(ref, hyp)\n        if len(meteor_per) != 5000 or len(spice_per) != 5000 or not np.isfinite([meteor_score, spice_score]).all():\n            raise ValueError('Incomplete METEOR/SPICE')\n        metrics[policy] = dict(CIDEr=float(c), BLEU_1=float(b[0]), BLEU_4=float(b[3]), METEOR=float(meteor_score), SPICE=float(spice_score), ROUGE_L=float(r), **m, counts=counts, mean_words=sum((len(r['caption'].split()) for r in selected)) / 5000, short_le2=sum((len(r['caption'].split()) <= 2 for r in selected)))\n        save(out / (policy + '_captions.json'), selected)\n        save(out / (policy + '_cider_per_image.json'), [dict(coco_id=int(i), CIDEr=float(v)) for i, v in zip(ref, per)])\n        flags[policy] = {r['image_id']: bool(r['mscoco_hallucinated_words']) for r in ch['sentences']}\n    a, b = [flags[p] for p in POLICIES]\n    changes = dict(changed_captions=sum((k != 0 for k in chosen)), rank_histogram={str(k): chosen.count(k) for k in range(5)}, CHAIR_repairs=sum((a[i] and (not b[i]) for i in a)), CHAIR_new_flags=sum((not a[i] and b[i] for i in a)))\n    result = dict(status='CSA_PER_CAPTION_FIXED_RERANK_TEST5000_COMPLETE', images=5000, test_used=True, training=False, checkpoint_sha256=lock['checkpoint_sha256'], metrics=metrics, changes=changes, selector=lock['selector'], tuned_on_test=False, oracle_used_for_selection=False, prior_project_test_exposure=True, selection='both prelocked policies reported; no TEST-based fallback', NLL_note='same frozen captioner; reranking changes output only, no target-conditioned pass')\n    save(out / 'result.json', result)\n    print(json.dumps(result, indent=2), flush=True)\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('mode', choices=('prefetch', 'text_preflight', 'worker', 'score'))\n    parser.add_argument('config')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n", 'captioning/object_consistency.py': '"""Rule-based COCO object consistency scoring for caption candidates."""\nimport re\nfrom pathlib import PurePosixPath\n\n\nCOCO_LABELS = (\n    \'person\', \'bicycle\', \'car\', \'motorcycle\', \'airplane\', \'bus\', \'train\', \'truck\',\n    \'boat\', \'traffic light\', \'fire hydrant\', \'stop sign\', \'parking meter\', \'bench\',\n    \'bird\', \'cat\', \'dog\', \'horse\', \'sheep\', \'cow\', \'elephant\', \'bear\', \'zebra\',\n    \'giraffe\', \'backpack\', \'umbrella\', \'handbag\', \'tie\', \'suitcase\', \'frisbee\',\n    \'skis\', \'snowboard\', \'sports ball\', \'kite\', \'baseball bat\', \'baseball glove\',\n    \'skateboard\', \'surfboard\', \'tennis racket\', \'bottle\', \'wine glass\', \'cup\',\n    \'fork\', \'knife\', \'spoon\', \'bowl\', \'banana\', \'apple\', \'sandwich\', \'orange\',\n    \'broccoli\', \'carrot\', \'hot dog\', \'pizza\', \'donut\', \'cake\', \'chair\', \'couch\',\n    \'potted plant\', \'bed\', \'dining table\', \'toilet\', \'tv\', \'laptop\', \'mouse\',\n    \'remote\', \'keyboard\', \'cell phone\', \'microwave\', \'oven\', \'toaster\', \'sink\',\n    \'refrigerator\', \'book\', \'clock\', \'vase\', \'scissors\', \'teddy bear\',\n    \'hair drier\', \'toothbrush\',\n)\n\nSPECIAL_ALIASES = {\n    \'person\': (\'people\', \'man\', \'men\', \'woman\', \'women\', \'boy\', \'boys\', \'girl\',\n               \'girls\', \'child\', \'children\', \'kid\', \'kids\', \'rider\', \'riders\'),\n    \'bicycle\': (\'bike\', \'bikes\'),\n    \'car\': (\'cars\', \'vehicle\', \'vehicles\', \'automobile\', \'automobiles\'),\n    \'motorcycle\': (\'motorbike\', \'motorbikes\'),\n    \'airplane\': (\'plane\', \'planes\', \'aircraft\', \'jet\', \'jets\'),\n    \'bus\': (\'buses\',),\n    \'boat\': (\'ship\', \'ships\'),\n    \'traffic light\': (\'traffic signal\', \'traffic signals\'),\n    \'handbag\': (\'purse\', \'purses\'),\n    \'suitcase\': (\'luggage\',),\n    \'sports ball\': (\'ball\', \'balls\'),\n    \'baseball glove\': (\'glove\', \'gloves\'),\n    \'tennis racket\': (\'racket\', \'rackets\', \'racquet\', \'racquets\'),\n    \'couch\': (\'sofa\', \'sofas\'),\n    \'potted plant\': (\'plant\', \'plants\'),\n    \'dining table\': (\'table\', \'tables\'),\n    \'tv\': (\'television\', \'televisions\', \'monitor\', \'monitors\'),\n    \'cell phone\': (\'phone\', \'phones\', \'mobile phone\', \'mobile phones\'),\n    \'refrigerator\': (\'fridge\', \'fridges\'),\n    \'donut\': (\'doughnut\', \'doughnuts\'),\n    \'teddy bear\': (\'stuffed bear\', \'stuffed bears\'),\n    \'hair drier\': (\'hair dryer\', \'hair dryers\'),\n}\n\n\ndef _pluralize(phrase):\n    words = phrase.split()\n    last = words[-1]\n    if last.endswith(\'y\') and len(last) > 1 and last[-2] not in \'aeiou\':\n        words[-1] = last[:-1] + \'ies\'\n    elif last.endswith((\'s\', \'x\', \'ch\', \'sh\')):\n        words[-1] = last + \'es\'\n    else:\n        words[-1] = last + \'s\'\n    return \' \'.join(words)\n\n\ndef _alias_table():\n    aliases = []\n    for label in COCO_LABELS:\n        values = {label, _pluralize(label), *SPECIAL_ALIASES.get(label, ())}\n        for value in values:\n            aliases.append((tuple(value.split()), label))\n    return sorted(aliases, key=lambda item: (-len(item[0]), -len(\' \'.join(item[0]))))\n\n\nALIASES = _alias_table()\n\n\ndef extract_caption_objects(caption):\n    """Return non-overlapping COCO classes explicitly mentioned in a caption."""\n    tokens = re.findall(r"[a-z0-9]+", caption.lower())\n    occupied, labels = set(), []\n    for alias, label in ALIASES:\n        width = len(alias)\n        for start in range(len(tokens) - width + 1):\n            positions = set(range(start, start + width))\n            if positions & occupied or tuple(tokens[start:start + width]) != alias:\n                continue\n            occupied.update(positions)\n            if label not in labels:\n                labels.append(label)\n    return labels\n\n\ndef index_detections(source, objects_field=\'objects\', name_key=\'label\', confidence_key=\'conf\',\n                     min_confidence=0.5):\n    """Index maximum accepted detector confidence by filename and COCO label."""\n    if not isinstance(source, dict):\n        raise ValueError(\'Detection source must be a filename-to-entry mapping.\')\n    indexed = {}\n    for key, entry in source.items():\n        filename = PurePosixPath(str(key).replace(\'\\\\\', \'/\')).name\n        if filename in indexed:\n            raise ValueError(f\'Ambiguous detection filename: {filename}\')\n        objects = entry if isinstance(entry, list) else entry.get(objects_field)\n        if not isinstance(objects, list):\n            raise ValueError(f\'{filename}: expected object list in {objects_field!r}\')\n        accepted = {}\n        for detection in objects:\n            if not isinstance(detection, dict):\n                continue\n            label = detection.get(name_key)\n            confidence = detection.get(confidence_key)\n            if not isinstance(label, str) or not isinstance(confidence, (int, float)):\n                continue\n            label = \' \'.join(label.lower().strip().split())\n            if label in COCO_LABELS and confidence >= min_confidence:\n                accepted[label] = max(accepted.get(label, 0.0), float(confidence))\n        indexed[filename] = accepted\n    return indexed\n\n\ndef object_consistency(caption, detected, hallucination_penalty=1.0):\n    """Score supported mentions and penalize unsupported COCO-class mentions."""\n    mentioned = extract_caption_objects(caption)\n    supported = [label for label in mentioned if label in detected]\n    hallucinated = [label for label in mentioned if label not in detected]\n    if mentioned:\n        support = sum(detected[label] for label in supported) / len(mentioned)\n        hallucination_rate = len(hallucinated) / len(mentioned)\n        score = support - hallucination_penalty * hallucination_rate\n    else:\n        support, hallucination_rate, score = 0.0, 0.0, 0.0\n    return {\n        \'score\': float(score),\n        \'mentioned\': mentioned,\n        \'supported\': supported,\n        \'hallucinated\': hallucinated,\n        \'support\': float(support),\n        \'hallucination_rate\': float(hallucination_rate),\n    }\n\n\ndef rerank_record(record, detected, weight, hallucination_penalty=1.0):\n    candidates = record.get(\'candidates\', [])\n    if not candidates:\n        raise ValueError(f\'Image {record.get("image_id")}: no candidates\')\n    language = [float(candidate[\'logprob\']) for candidate in candidates]\n    low, high = min(language), max(language)\n    enriched = []\n    for rank, (candidate, score) in enumerate(zip(candidates, language)):\n        language_score = 1.0 if high == low else (score - low) / (high - low)\n        consistency = object_consistency(\n            candidate[\'caption\'], detected, hallucination_penalty)\n        object_score = ((consistency[\'score\'] + hallucination_penalty) /\n                        (1.0 + hallucination_penalty))\n        combined = (1.0 - weight) * language_score + weight * object_score\n        enriched.append({\n            **candidate,\n            \'original_rank\': rank,\n            \'language_score\': float(language_score),\n            \'object_score\': float(object_score),\n            \'combined_score\': float(combined),\n            \'object_consistency\': consistency,\n        })\n    selected = max(enriched, key=lambda item: (item[\'combined_score\'],\n                                                -item[\'original_rank\']))\n    return selected, enriched\n', 'captioning/multiscore_reranking.py': '"""Candidate re-ranking with decoder, CLIP image-text and object scores."""\nfrom .object_consistency import object_consistency\n\n\ndef _minmax(values):\n    low, high = min(values), max(values)\n    if high == low:\n        return [1.0] * len(values)\n    return [(value - low) / (high - low) for value in values]\n\n\ndef rerank_multiscore(record, detected, clip_weight, object_weight,\n                      hallucination_penalty=1.0):\n    """Return the best candidate under weights that sum with decoder to one."""\n    if clip_weight < 0 or object_weight < 0 or clip_weight + object_weight > 1:\n        raise ValueError(\'Require nonnegative CLIP/OCC weights with sum <= 1.\')\n    candidates = record.get(\'candidates\', [])\n    if not candidates:\n        raise ValueError(f\'Image {record.get("image_id")}: no candidates\')\n    if any(\'clipscore\' not in candidate for candidate in candidates):\n        raise ValueError(f\'Image {record.get("image_id")}: candidate lacks clipscore\')\n\n    # Beam candidates are ordered by the accumulated raw log-probability in\n    # inference.py. Use that same score so auxiliary weights of zero reproduce\n    # rank 0 exactly; avg_logprob would silently introduce length normalization.\n    language = [float(candidate[\'logprob\']) for candidate in candidates]\n    clip = [float(candidate[\'clipscore\']) for candidate in candidates]\n    language_scores, clip_scores = _minmax(language), _minmax(clip)\n    decoder_weight = 1.0 - clip_weight - object_weight\n    enriched = []\n    for rank, candidate in enumerate(candidates):\n        consistency = object_consistency(\n            candidate[\'caption\'], detected, hallucination_penalty)\n        object_score = ((consistency[\'score\'] + hallucination_penalty) /\n                        (1.0 + hallucination_penalty))\n        combined = (decoder_weight * language_scores[rank] +\n                    clip_weight * clip_scores[rank] +\n                    object_weight * object_score)\n        enriched.append({\n            **candidate,\n            \'original_rank\': rank,\n            \'decoder_score\': float(language_scores[rank]),\n            \'clip_score_normalized\': float(clip_scores[rank]),\n            \'object_score\': float(object_score),\n            \'combined_score\': float(combined),\n            \'object_consistency\': consistency,\n        })\n    selected = max(enriched, key=lambda item: (item[\'combined_score\'],\n                                                -item[\'original_rank\']))\n    return selected, enriched\n', 'RERANK_TEST_INPUTS.json': '{\n  "checkpoint_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/161",\n  "checkpoint_sha256": "58ce5dc0e30ac26add9bac8e24a2e9e03b91532bfdeece588997395bf9cfc16b",\n  "test_cache_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/143",\n  "test_receipt": {\n    "status": "KARPATHY_TEST_ROI_COMPLETE",\n    "images": 5000,\n    "group": "test",\n    "dataset_json_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "detector_sha256": "498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c",\n    "feature_profile": {\n      "model_id": "openai/clip-vit-base-patch16",\n      "output": "vision_model.last_hidden_state",\n      "image_size": [\n        224,\n        224\n      ],\n      "preprocessing": "bilinear",\n      "antialias": true,\n      "dataset_json_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n      "mean": [\n        0.48145466,\n        0.4578275,\n        0.40821073\n      ],\n      "std": [\n        0.26862954,\n        0.26130258,\n        0.27577711\n      ],\n      "model_revision": null,\n      "compute_dtype": "float32",\n      "storage_dtype": "float16",\n      "autocast": false\n    },\n    "full_h5_payload_hashed": false,\n    "neural_extraction": false,\n    "training_images": 0,\n    "validation_images": 0,\n    "files": {\n      "test_regions.npz": "7fb385e1a03c5241c4d65cc240d0434e041284a9a829198c5167f0725be57b1a",\n      "test_metadata.json": "4271b46b7ab67117433bfdf84f6eb9ba66515ada9b3c4ae4d2e507bc2cede018",\n      "vocabulary.json": "924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7"\n    },\n    "test_id_order_sha256": "e4105a76a6d00c2993aee4f1fc1f846d124ddd98f2883521ea9cd4e2b0a922af",\n    "seconds": 1384.8423266129998\n  },\n  "clip": {\n    "id": "openai/clip-vit-base-patch16",\n    "revision": "57c216476eefef5ab752ec549e440a49ae4ae5f3",\n    "files_sha256": {\n      "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n      "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n      "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n      "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n      "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n      "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n      "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n      "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n    }\n  },\n  "frozen_selector": {\n    "decoder_weight": 0.3,\n    "clip_weight": 0.4,\n    "object_weight": 0.3,\n    "min_detection_confidence": 0.5,\n    "hallucination_penalty": 1.0,\n    "tie_break": "earlier beam"\n  },\n  "valid_result_source": "ducanh2403/2-vi-cai-kia-qua-nhieu/164",\n  "source_note": "TEST receipt previously audited143; CLIP asset hashes user-run164 evidence; no author model execution"\n}\n', 'TEST_PROTOCOL.md': '# Beam SCST v199 — TEST5000 + fixed rerank,26/11/2026\n\nUser explicitly requests TEST now and includes reranking. Freeze endpoint before TEST: ducanh3/notebook7bc83c4c76/version199, root beam_scst_epoch1_seed43_v179_v2_run1/pilot_checkpoint.bin, SHA 3b03acc29f025b07dc358846b30618ae9c83f8634bc12bf7905e9d908b971f24. This is Full seed43 after one113287TRAIN-image epoch/3541updates from CE179best8/SHAa51e7ec2…, not resume checkpoint.bin or an earlier SCST1500 model.\n\nGenerate beam5/max20/FP32/T4x2 once, save25000 candidates and top1 before scoring. Report both decoder_rank0 and clip_object_fixed with all8 caption/hallucination metrics plus Recall/NLL. Preserve the successful frozen selector, normalization, CLIP asset SHA, detector SHA and thresholds: decoder0.3/CLIP0.4/object0.3; no tuning, oracle or TEST-based fallback. Compare each policy with the already-audited same seed43 CE179 TEST/rerank results, not seed42/VALID.\n\nReuse the proven standalone SCST1500 TEST/rerank scaffold, changing only source/lineage labels to the completed beam-SCST endpoint. Bundle the exact captioner/bank/decoder/beam bytes used by v199, sealed TEST181 semantic cache contracts and prior evaluator/rerank code. Endpoint binary structure contains model/updates/source_sha256; do not demand sampled-SCST parent1000 metadata that this endpoint never wrote. Verify exact SHA before torch.load/scorerpip/cache preparation. Pin actual API version199 routing and one endpoint pattern, authorized Secret codex or KAGGLE_API_TOKEN; optional exact mounted bytes allowed. No latest/final/other-version fallback.\n\nInputs: visual-cache, mscoco-2014, csa-prompt-test-cache-v181, objectdetectionecache. No training cache or checkpoint Dataset required. Full TEST JPEGs support frozen CLIP reranking; image-specific prompt/geometry cache supports the captioner. No model training, TEST optimizer or prompt regeneration. Preserve entire output with candidates, CLIP scores, selections and intermediate metric files so CPU scorer recovery never requires model inference again.\n\nThree-hour parent stop cap inherited from the successful TEST scaffold is not ETA, remaining quota or billing. User runs Kaggle. Agent performs syntax/hash/import-closure/metadata/mock checks only, never local Torch/model/scorer/API. TEST has been exposed in earlier project development; this is a fixed-checkpoint evaluation, not a pristine unseen holdout or a controlled beam-baseline-only ablation. Zero-text43/44 remain queued. No Word/Sheet result edits for a prepared notebook.\n', 'epoch2_contract.py': '"""Metadata-only epoch transition and exact continuation contracts."""\nimport math\n\nPARENT_VERSION = \'ducanh3/notebook7bc83c4c76/199\'\nPARENT_SHA = \'5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c\'\nXE_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nIMAGES, GLOBAL_BATCH, PARENT_STEPS, END_STEPS = 113287, 32, 3541, 7082\n\n\ndef validate_parent(state, expected):\n    if (state[\'identity\'] != expected[\'identity\'] or state[\'updates\'] != PARENT_STEPS\n            or state[\'cursor_images\'] != IMAGES or state[\'source_sha256\'] != XE_SHA\n            or not expected[\'complete_epoch\'] or expected[\'sha256\'] != PARENT_SHA):\n        raise ValueError(\'Need complete exact epoch1/v199 state\')\n    if len(state[\'rng\']) != 2 or not {\'model\', \'optimizer\'} <= state.keys():\n        raise ValueError(\'Need full model/Adam/two-rank RNG, not pilot weights\')\n    for rng in state[\'rng\']:\n        if not {\'cpu\', \'cuda\', \'numpy\'} <= rng.keys():\n            raise ValueError(\'Missing rank RNG state\')\n\n\ndef make_identity(cfg, expected, torch_version, numpy_version, vocabulary_sha):\n    old = expected[\'identity\']\n    if (torch_version != old[\'torch\'] or numpy_version != old[\'numpy\']\n            or vocabulary_sha != old[\'vocabulary_sha256\']\n            or cfg[\'cache_manifest_sha256\'] != old[\'cache_manifest_sha256\']):\n        raise ValueError(\'Changed software, vocabulary or cache; no fallback\')\n    return dict(method=\'beam_mean_scst_epoch2_speed_v1\', epoch=2, seed=43,\n                parent_version=PARENT_VERSION, parent_sha256=PARENT_SHA, source_sha256=XE_SHA,\n                parent_identity=old, source_hashes=cfg[\'source_hashes\'],\n                train_images=IMAGES, cumulative_image_events=2*IMAGES,\n                parent_updates=PARENT_STEPS, endpoint_updates=END_STEPS,\n                order=\'same original sealed TRAIN order, once per epoch\',\n                optimization=\'audited batched prompt transfer/deferred finite-logit check\')\n\n\ndef validate_resume(receipt, identity):\n    cursor, step = receipt[\'cursor_images\'], receipt[\'updates\']\n    if (receipt[\'identity\'] != identity or receipt[\'epoch\'] != 2\n            or receipt[\'parent_sha256\'] != PARENT_SHA or receipt[\'source_sha256\'] != XE_SHA\n            or type(cursor) is not int or type(step) is not int or not 0 <= cursor <= IMAGES\n            or step != PARENT_STEPS+math.ceil(cursor/GLOBAL_BATCH)):\n        raise ValueError(\'Foreign epoch2 recipe, lineage or cursor\')\n    if cursor != IMAGES and cursor != (step-PARENT_STEPS)*GLOBAL_BATCH:\n        raise ValueError(\'Resume cursor is not an epoch2 optimizer boundary\')\n    if receipt[\'complete_epoch\'] != (cursor == IMAGES):\n        raise ValueError(\'Wrong epoch2 completeness\')\n\n\ndef validate_eval_binding(binding):\n    if (binding[\'epoch\'] != 2 or binding[\'updates\'] != END_STEPS or binding[\'seed\'] != 43\n            or binding[\'cursor_images\'] != IMAGES or binding[\'parent_sha256\'] != PARENT_SHA\n            or binding[\'source_sha256\'] != XE_SHA or not binding[\'complete_epoch\']):\n        raise ValueError(\'Refuse partial/foreign endpoint for TEST\')\n    digest = binding[\'sha256\']\n    if (len(digest) != 64 or any(c not in \'0123456789abcdef\' for c in digest)\n            or digest in (PARENT_SHA, \'3b03acc29f025b07dc358846b30618ae9c83f8634bc12bf7905e9d908b971f24\')):\n        raise ValueError(\'Wrong epoch2 endpoint SHA\')\n    return binding\n', 'EPOCH2_PROTOCOL.md': '# Beam SCST epoch 2 + TEST — quyết định 26/11/2026\n\nNgười dùng chốt chuẩn bị học tiếp epoch 2 và chấm TEST; tài khoản còn 6 giờ GPU. Agent chỉ chuẩn bị và kiểm tra static/metadata/mock, không chạy Kaggle, Torch, mô hình hay scorer local.\n\n## Câu hỏi và cấu hình đã chốt\n\nHọc thêm đúng một lượt 113.287 ảnh TRAIN có tiếp tục cải thiện CIDEr không, và CHAIR thay đổi thế nào? Không hứa tăng tiếp từ kết quả epoch 1. Đây là thử tối ưu tiếp cùng mô hình; không suy riêng đóng góp prompt từ lượt này.\n\n- Khởi đầu duy nhất: full `checkpoint.bin` của `ducanh3/notebook7bc83c4c76/199`, SHA256 `5b8d33fc0b26f2c77ad1cee4a58d3e4a718f13107a2d2bd9a0884c427faed88c`. Giữ model, Adam/moments/step và CPU/CUDA/NumPy RNG hai rank. Không dùng weights-only pilot hay output TEST200/benchmark.\n- Chuyển rõ từ epoch 1 đã đủ 3.541 update/cursor113.287 sang epoch 2 cursor0, cumulative update3.541. Endpoint epoch2: cumulative7.082 update, 226.574 image events; riêng epoch2 đúng113.287 ảnh không thay thế, thứ tự cache gốc như epoch1. Không shuffle mới, reset Adam hoặc reseed sau khôi phục RNG.\n- Giữ beam5/train-mode/mean reward baseline/20-token denominator, pure SCST CIDEr-D TRAIN, FP32/noTF32, T4×2 DDP, local4×accum4/global32, Adam4e-6/betas(.9,.99), không CE/clip gradients/AMP/compile/feature sharing. Dùng đúng hai tối ưu đã vượt guard: batched prompt transfer và deferred finite-logit checks; giữ finite loss/reward/grad guards.\n- Reuse cache82/83/84 đã đóng Dataset và metadata/VALID; không trích feature hay copy lại full patch arrays. Giữ nguyên model/core/M3/reward/decoder. Preflight6 update bỏ, sau đó process train riêng khôi phục lại trạng thái gốc. Giữ checkpoint atomic định kỳ100 update và tại deadline, identity epoch/cursor rõ ràng để resume chính xác.\n\n## Đánh giá và giới hạn\n\nEndpoint cố định cuối epoch 2; không chọn checkpoint dựa vào TEST. Chấm VALID5000 top1 bằng đường đã chạy của epoch1 để so đúng cùng seed43. Sau đó TEST5000 sinh beam5 một lần, chấm top1 và rerank cố định 0,3decoder/0,4CLIP/0,3object, đủ BLEU1/4, METEOR, ROUGE-L, CIDEr, SPICE, CHAIRs/i, thêm Recall/NLL. Báo cả hai chính sách dù bất lợi; so với kết quả epoch1 TEST output200 và CE179 cùng seed đã lưu. Không tìm trọng số theo TEST, không tự epoch3.\n\nTEST đã được dự án dùng trong phát triển; ghi rõ đây là đánh giá lặp lại, không gọi là holdout hoàn toàn chưa xem. CHAIR đo lỗi đối tượng, không chứng minh thuộc tính/quan hệ đúng. Một seed SCST chưa xác lập độ ổn định.\n\nHai namespace source riêng cho train/VALID và TEST để tránh csa_per_caption_test_data bị cache nhầm split. TEST checkpoint seal được tạo chỉ sau khi full epoch2 hoàn tất, dựa trên SHA thực tế export/receipt và kiểm tra lineage epoch2/7082/CE179/parent199. Không dùng SHA epoch1 cố định cho trọng số mới.\n\nCap mặc định5,75 giờ wall/phiên, không phải ETA hay cách tính GPU quota. Dự phòng100 phút sau train cho VALID/TEST/scorer; epoch2 train dự kiến gần3giờ từ speed sample, không bảo đảm đủ ngân sách. Nếu train hết cap, giữ full state và dừng, không chấm partial checkpoint. Nếu scorer lỗi/hết giờ, giữ caption/beam/CLIPscores/selection/basic metrics và weights để recovery CPU; không train/decode lại tự động. Notebook hỗ trợ resume bằng toàn bộ output epoch2 được gắn và kiểm tra identity/SHA; nếu chỉ có Save Version, cần source mới xác nhận để làm API resume riêng.\n\n## Dữ liệu cần gắn\n\n1. `ducanh3/csa-scst-full-karpathy-cache-v1`\n2. `vuthetam/mscoco-2014`\n3. `ducanh2403/visual-cache`\n4. `ducanh2403/csa-prompt-test-cache-v181`\n5. `ducanh2403/objectdetectionecache`\n\nInternet và Secret `codex` hoặc `KAGGLE_API_TOKEN` có quyền tải nguồn ducanh3. Không cần gắn output199 riêng; API tải đúng version/path và SHA trước setup mô hình.\n\n## Đã chuẩn bị và kiểm tra, 26/11/2026\n\n[Notebook](BEAM_SCST_EPOCH2_FROM_V199_VALID_TEST_RERANK_2GPU.ipynb), [source manifest](SOURCE_MANIFEST.json), [9 kiểm tra host](CHECKS.json). Bản sao ở Downloads được xác minh cùng SHA. Các cell và80 payload compile/hash PASS; closure các import runtime được kiểm tra riêng hai namespace. Metadata transition/resume/tail7 và endpoint binding có trường hợp từ chối sai nguồn/partial; AST update giữ nguyên toán học của epoch1 ngoài batch helper đã kiểm chứng. Model/reward/inference/scorer/selector và speed candidate giữ đúng bytes nguồn đã chạy.\n\nMô phỏng API acquisition chỉ nhận đúng v199 full checkpoint; không gọi API thật. Mô phỏng pipeline dùng file fixture (không phải weights/caption/metric thực): đường thành công tới TEST comparison, partial/timeout train chặn TEST, timeout scorer giữ weights và beam fixture. Không import Torch hoặc metric scorer local. Actual API authorization, load binary/Adam/RNG/CUDA, scorer assets và chất lượng/thời gian toàn lượt chưa được agent chạy.\n\nKiểm tra Torch/NumPy phiên Kaggle ngay sau CUDA guard, trước setup scorer/cache; không tự cài lại runtime khi khác nguồn v199. Java/scorer smoke chạy trước lượt huấn luyện chính để phát hiện lỗi môi trường sớm. Không có kết quả thực nghiệm mới, không ghi dòng metric mới vào Sheet hoặc sửa báo cáo Word chỉ vì đã chuẩn bị notebook.\n'}
TRAIN_SHA256={'TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'bank_layers.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205', 'csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'csa_per_caption_test_data.py': '6338f2a4018143d45feedaf4356e9c51d7c90b2d8ec6c4c2e8881ae26ec2d759', 'run_csa_per_caption_test.py': '657bb4e60fdb5647e9130a74ca7607cd12249ee79c472e5e5f3ad04e95484a28', 'scst_cache.py': '9721401031272f50cd5c99b177d6177f33b0805cabde19eb936dbb622086dd0f', 'scst_dataset_mount.py': 'e4574bc17a36f4d6b8fb51ad8fbc725db4269559d0a05f34a7d5ba1354ca35fb', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'CACHE_PARTS.json': '50632a1ea657604915fdfa9396320d5c1649df408a28b6b8003b0eab601f3519', 'beam_policy.py': '1f2ec8d95c38f91793a2b554cd6f585034871ccbb95d0ed5ad9521206b6ddf67', 'full_epoch_data.py': '6cf0ebbd2d9d8740aabef7f8fbe9fdc8a88263b2447a64ab43953deecb31f719', 'PROTOCOL.md': '9b0f373fdc778cb1e56339716569d7674b67ba0eebff990fd929bc729297c2ac', 'checkpoint_input.py': '55e2d57313ef3e51dc3c3106c938db2ea7a44f058b0687580e5376e3117e004a', 'run_epoch2.py': '16ad65b1294bc60874189d74119d3ad7251a905efd769313e7d39e83ca09cb49', 'speed_candidate.py': '9e2a95b270fd271f7a1648d4c2fa65ec4ef3a30fbbb6b5fa278744e313417290', 'PARENT_CHECKPOINT_RECEIPT.json': 'c240d7a49c05377d65cb8135604ac03395f9d48c9a3bb952670a8fe527279592', 'epoch2_contract.py': '45bd1dd68e2397481536d29a0c10e31fc479595f01684017217fc35423e17d8a', 'EPOCH2_PROTOCOL.md': '5a4a6b177691330c03c7e72b2e2bf29a4dce0ee1d7cd4374a2d761cc829edd8d'}
TEST_SHA256={'TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'bank_layers.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205', 'csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'csa_per_caption_test_data.py': 'd88d7928595ef3ebf5ef76adb9cf7f6d6cf96943501444617c8ac5f53d8d316d', 'run_csa_per_caption_test.py': '769ec0beb3c01ee73317705a8f13865a3d9f36f877038f5b5228915128be9617', 'captioning/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'checkpoint_input.py': '55e2d57313ef3e51dc3c3106c938db2ea7a44f058b0687580e5376e3117e004a', 'csa_per_caption_rerank_data.py': '66ef05d0dd6c5b397d970a1ca0ac9439c3504482a54e52817b54113c5c9d4cc6', 'run_csa_per_caption_rerank.py': '7c63455bc116bd5657786e0979c60b97ae9fd813e081680e2f776f41e7426eed', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'TEST_PROTOCOL.md': 'b395ecdefae5d4dac5f157295f144e8dcfc216d8aeedc95ead4da1817ca1533a', 'epoch2_contract.py': '45bd1dd68e2397481536d29a0c10e31fc479595f01684017217fc35423e17d8a', 'EPOCH2_PROTOCOL.md': '5a4a6b177691330c03c7e72b2e2bf29a4dce0ee1d7cd4374a2d761cc829edd8d'}
REFERENCES={'CE': {'decoder_rank0': {'BLEU_1': 0.7776121266986261, 'BLEU_4': 0.37790280281231137, 'METEOR': 0.28813366890256553, 'ROUGE_L': 0.579128129683382, 'CIDEr': 1.2198431675475085, 'SPICE': 0.21825731146122443, 'CHAIRs': 0.0406, 'CHAIRi': 0.02803483941208492, 'Recall': 0.43712688936872857, 'NLL': 2.178126289902086}, 'clip_object_fixed': {'BLEU_1': 0.7790993525432683, 'BLEU_4': 0.3741711101557513, 'METEOR': 0.2941087006127335, 'ROUGE_L': 0.582496122998859, 'CIDEr': 1.2436387660847645, 'SPICE': 0.22570564007184174, 'CHAIRs': 0.0366, 'CHAIRi': 0.02450915799183028, 'Recall': 0.4493204623396418, 'NLL': 2.178126289902086}}, 'epoch1_TEST': {'decoder_rank0': {'CIDEr': 1.2960632849121856, 'BLEU_1': 0.8019781362146072, 'BLEU_4': 0.4026386985775675, 'METEOR': 0.28823099135395064, 'SPICE': 0.22656002578284543, 'ROUGE_L': 0.591988513596983, 'CHAIRs': 0.0506, 'CHAIRi': 0.033435426237876466, 'Recall': 0.4660231169820907, 'counts': {'images': 5000, 'hallucinated_captions': 253, 'hallucinated_mentions': 262, 'object_mentions': 7836, 'recalled_unique': 7338, 'reference_unique': 15746}, 'mean_words': 8.534, 'short_le2': 0, 'NLL': 2.978882952490194}, 'clip_object_fixed': {'CIDEr': 1.3017854425760478, 'BLEU_1': 0.808696662698238, 'BLEU_4': 0.39617604337966505, 'METEOR': 0.2917378536353819, 'SPICE': 0.2324931457754211, 'ROUGE_L': 0.5898779488681541, 'CHAIRs': 0.0368, 'CHAIRi': 0.02484709480122324, 'Recall': 0.4687539692620348, 'counts': {'images': 5000, 'hallucinated_captions': 184, 'hallucinated_mentions': 195, 'object_mentions': 7848, 'recalled_unique': 7381, 'reference_unique': 15746}, 'mean_words': 8.9746, 'short_le2': 0, 'NLL': 2.978882952490194}}, 'epoch1_VALID': {'BLEU_1': 0.8059457192271925, 'BLEU_4': 0.40330547656335064, 'CIDEr': 1.2938949542216254, 'ROUGE_L': 0.597202426454193, 'CHAIRs': 0.0508, 'CHAIRi': 0.033758536271099086, 'Recall': 0.4575590155700653, 'NLL': 2.9460973352896422}, 'selector': {'decoder_weight': 0.3, 'clip_weight': 0.4, 'object_weight': 0.3, 'min_detection_confidence': 0.5, 'hallucination_penalty': 1.0, 'tie_break': 'earlier beam'}}
ROUTING_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output(SOURCE_VERSION,path=directory)\n    request=endpoint.call_args.args[0]\n    owner,slug,version=SOURCE_VERSION.split('/')\n    assert (request.user_name,request.kernel_slug,request.version_label)==(owner,slug,'v'+version)\nprint('Exact TEST source API routing PASS',flush=True)\n"


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Epoch2 train/evaluation active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False;training_complete=False;test_started=False;phase='source'
try:
    for label,root,payloads,hashes in (('train',SOURCE,TRAIN_PAYLOADS,TRAIN_SHA256),
                                      ('test',TEST_SOURCE,TEST_PAYLOADS,TEST_SHA256)):
        preserved=OUTPUT/'source'/label;preserved.mkdir(parents=True)
        for name,source in payloads.items():
            raw=source.encode('utf-8')
            if hashlib.sha256(raw).hexdigest()!=hashes[name]:raise ValueError('Changed embedded '+label+'/'+name)
            for dest in (root,preserved):
                path=dest/name
                if not path.resolve().is_relative_to(dest.resolve()):raise ValueError('Unsafe source path')
                path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(train=TRAIN_SHA256,test=TEST_SHA256),indent=2))
    cfg['source_hashes']=TRAIN_SHA256;write_config()
    sys.path.insert(0,str(SOURCE))
    from checkpoint_input import resolve_mounted
    def checkpoint(root):return resolve_mounted(root,CHECKPOINT_SHA,OUTPUT/'checkpoint_input_report.json')
    if CHECKPOINT_ROOT:
        weights=checkpoint(CHECKPOINT_ROOT);origin='mounted exact source bytes'
    else:
        from kaggle_secrets import UserSecretsClient
        token=None
        for secret_name in SECRET_NAMES:
            try:value=UserSecretsClient().get_secret(secret_name).strip()
            except Exception:continue
            if value:token=value;break
        if not token:raise RuntimeError('Enable Secret codex or KAGGLE_API_TOKEN with access to exact ducanh3/version199')
        TOKENS.append(token)
        run([sys.executable,'-m','pip','install','-q','--target',CLI,
             'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],600)
        api_env=os.environ.copy()
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):api_env.pop(key,None)
        api_env.update(PYTHONPATH=str(CLI),KAGGLE_API_TOKEN=token,PYTHONUNBUFFERED='1')
        run([sys.executable,'-c','SOURCE_VERSION='+repr(SOURCE_VERSION)+'\n'+ROUTING_PREFLIGHT],120,api_env)
        target=OUTPUT/'input_source';target.mkdir()
        run([sys.executable,'-u','-c','from kaggle.cli import main; main()','kernels','output',SOURCE_VERSION,
             '-p',str(target),'--file-pattern','(^|/)beam_scst_epoch1_seed43_v179_v2_run1/checkpoint[.]bin$'],1200,api_env)
        weights=checkpoint(target);origin='exact Save Version API download'
    cfg['checkpoint_root']=str(weights);write_config()
    (OUTPUT/'checkpoint_source.json').write_text(json.dumps(dict(seed=43,version=SOURCE_VERSION,epoch=1,
        sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    shutil.copyfile(OUTPUT/'cuda_runtime.json',OUTPUT/'training_runtime.json')
    from importlib.metadata import version
    expected_software=json.loads(TRAIN_PAYLOADS['PARENT_CHECKPOINT_RECEIPT.json'])['identity']
    if (json.loads((OUTPUT/'training_runtime.json').read_text())['torch_version']!=expected_software['torch']
            or version('numpy')!=expected_software['numpy']):
        raise RuntimeError('Training software differs from v199. Preserve log; no automatic reinstall or fallback.')
    phase='setup'
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],300)
    from full_epoch_data import prepare
    prepare(cfg)
    from csa_per_caption_test_data import bind_verified_cache
    bind_verified_cache(cfg,'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef')
    write_config()
    if hashlib.sha256(DETECTIONS.read_bytes()).hexdigest()!='498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c':
        raise ValueError('Changed frozen rerank detector cache')
    test_out=OUTPUT/'test';test_out.mkdir()
    test_cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=str(test_out),
        stage=str(WORK/'test_patches'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),
        checkpoint_root=str(OUTPUT/'pilot_checkpoint.bin'),chair=str(CHAIR),worker_seconds=900)
    test_config=WORK/'test_config.json';test_config.write_text(json.dumps(test_cfg,indent=2))
    def test_env():
        env=os.environ.copy();env['PYTHONPATH']=str(DEPS)+os.pathsep+str(TEST_SOURCE);return env
    evaluator=TEST_SOURCE/'run_csa_per_caption_test.py'
    # Check Java/metrics before spending the production epoch; no checkpoint/model needed.
    run([sys.executable,'-u',evaluator,'environment','--config',test_config],1200,test_env())
    run([sys.executable,'-u',evaluator,'smoke','--config',test_config],600,test_env())
    environment=json.loads((test_out/'environment.json').read_text())
    os.environ['PATH']=str(Path(environment['java']).parent)+os.pathsep+os.environ['PATH']
    os.environ['JAVA_HOME']=str(Path(environment['java']).parent.parent)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    phase='preflight'
    cfg['train_seconds']=min(1200,PARENT_DEADLINE-time.monotonic()-EVALUATION_RESERVE_SECONDS);write_config()
    if cfg['train_seconds']<300:raise TimeoutError('Insufficient budget before training preflight')
    run([*launch,SOURCE/'run_epoch2.py','preflight',CONFIG],cfg['train_seconds']+30)
    guards=[json.loads((OUTPUT/f'beam_preflight_rank{rank}.json').read_text()) for rank in (0,1)]
    if not all(g['passed'] for g in guards):raise ValueError('Incomplete epoch2 preflight')
    cfg['train_seconds']=PARENT_DEADLINE-time.monotonic()-EVALUATION_RESERVE_SECONDS;write_config()
    if cfg['train_seconds']<600:raise TimeoutError('Insufficient budget to start epoch2')
    print('Epoch2 guard PASS; full-epoch train projection hours:',round(max(g['projected_training_seconds'] for g in guards)/3600,2),
          '; available train hours:',round(cfg['train_seconds']/3600,2),'; not ETA/quota guarantee',flush=True)
    phase='train'
    run([*launch,SOURCE/'run_epoch2.py','train',CONFIG],cfg['train_seconds']+60)
    trained=json.loads((OUTPUT/'training_complete.json').read_text())
    if (not trained['complete'] or trained['epochs']!=2 or trained['train_images']!=113287
            or trained['updates']!=7082 or trained['cumulative_image_events']!=226574):
        raise ValueError('Refuse incomplete epoch2 endpoint')
    training_complete=True
    from epoch2_contract import validate_resume,validate_eval_binding
    full=json.loads((OUTPUT/'checkpoint_receipt.json').read_text())
    endpoint=json.loads((OUTPUT/'pilot_checkpoint_receipt.json').read_text())
    validate_resume(full,full['identity']);validate_resume(endpoint,full['identity'])
    if any(endpoint[k]!=full[k] for k in ('identity','epoch','updates','cursor_images','parent_sha256','source_sha256','complete_epoch')):
        raise ValueError('Epoch2 full/export receipt mismatch')
    if hashlib.sha256((OUTPUT/'pilot_checkpoint.bin').read_bytes()).hexdigest()!=endpoint['sha256']:
        raise ValueError('Changed endpoint export')
    binding=validate_eval_binding(dict(seed=43,version=OUTPUT.name+'/epoch2_endpoint',
        **{k:endpoint[k] for k in ('sha256','epoch','updates','cursor_images','parent_sha256','source_sha256','complete_epoch')}))
    (OUTPUT/'evaluation_binding.json').write_text(json.dumps(binding,indent=2))
    for name,value in (('EPOCH2_EVAL_LOCK.json',binding),('REPLAY_RECEIPTS.json',dict(checkpoint_sha256=binding['sha256']))):
        raw=(json.dumps(value,indent=2)+'\n').encode()
        for root in (TEST_SOURCE,OUTPUT/'source'/'test'):(root/name).write_bytes(raw)
    (OUTPUT/'runtime_input_receipt.json').write_text(json.dumps({n:hashlib.sha256((TEST_SOURCE/n).read_bytes()).hexdigest()
        for n in ('EPOCH2_EVAL_LOCK.json','REPLAY_RECEIPTS.json')},indent=2))
    phase='VALID'
    cfg.update(evaluate_pilot=True,eval_update=7082,worker_seconds=min(900,PARENT_DEADLINE-time.monotonic()-3600));write_config()
    if cfg['worker_seconds']<300:raise TimeoutError('Epoch2 saved; VALID needs recovery')
    valid_evaluator=SOURCE/'run_csa_per_caption_test.py'
    run([*launch,valid_evaluator,'worker','--config',CONFIG],cfg['worker_seconds']+30)
    for mode,cap in (('tokenize',180),('basic',240),('chair',600)):
        run([sys.executable,'-u',valid_evaluator,mode,'--config',CONFIG],cap)
    metrics={}
    for mode in ('basic','chair'):metrics.update(json.loads((OUTPUT/(mode+'.json')).read_text())['metrics'])
    workers=[json.loads((OUTPUT/f'worker{r}.json').read_text()) for r in (0,1)]
    metrics['NLL']=sum(w['NLL_sum'] for w in workers)/sum(w['NLL_tokens'] for w in workers)
    valid_result=dict(split='val',images=5000,epoch=2,updates=7082,checkpoint_sha256=binding['sha256'],
        metrics=metrics,epoch1_reference=REFERENCES['epoch1_VALID'],
        delta_vs_epoch1={k:v-REFERENCES['epoch1_VALID'][k] for k,v in metrics.items()},
        endpoint_fixed_before_TEST=True)
    (OUTPUT/'valid_result.json').write_text(json.dumps(valid_result,indent=2,allow_nan=False))
    print('VALID:',json.dumps(valid_result,indent=2),flush=True)
    phase='TEST';test_started=True
    run([sys.executable,'-u',evaluator,'checkpoint_preflight','--config',test_config],120,test_env())
    run([sys.executable,'-u',evaluator,'prepare','--config',test_config],1800,test_env())
    test_cfg['worker_seconds']=min(900,PARENT_DEADLINE-time.monotonic()-2400)
    if test_cfg['worker_seconds']<300:raise TimeoutError('Endpoint saved; TEST inference needs recovery')
    test_config.write_text(json.dumps(test_cfg,indent=2));(test_out/'protocol.json').write_text(test_config.read_text())
    run([*launch,evaluator,'worker','--config',test_config],test_cfg['worker_seconds']+30,test_env())
    rerank_out=test_out/'rerank';rerank_out.mkdir()
    rerank_cfg=dict(output=str(rerank_out),test_output=str(test_out),dataset_json=str(COCO_JSON),
        detections=str(DETECTIONS),chair=str(CHAIR),model_cache=str(WORK/'hf_cache'),phase_seconds=1200)
    rerank_config=WORK/'rerank_config.json';rerank_config.write_text(json.dumps(rerank_cfg,indent=2))
    (rerank_out/'protocol.json').write_text(rerank_config.read_text())
    run([sys.executable,'-u',TEST_SOURCE/'csa_per_caption_rerank_data.py',rerank_config],600,test_env())
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],300)
    reranker=TEST_SOURCE/'run_csa_per_caption_rerank.py'
    run([sys.executable,'-u',reranker,'prefetch',rerank_config],900,test_env())
    run([sys.executable,'-u',reranker,'text_preflight',rerank_config],180,test_env())
    run([*launch,reranker,'worker',rerank_config],1230,test_env())
    phase='CPU_metrics'
    run([sys.executable,'-u',reranker,'score',rerank_config],5400,test_env())
    result=json.loads((rerank_out/'result.json').read_text())
    if (result['status']!='CSA_PER_CAPTION_FIXED_RERANK_TEST5000_COMPLETE' or result['images']!=5000
            or result['checkpoint_sha256']!=binding['sha256'] or result['selector']!=REFERENCES['selector']):
        raise ValueError('Incomplete or changed TEST comparison')
    workers=[json.loads((test_out/f'worker{r}.json').read_text()) for r in (0,1)]
    nll=sum(w['NLL_sum'] for w in workers)/sum(w['NLL_tokens'] for w in workers)
    for m in result['metrics'].values():m['NLL']=nll
    keys=('BLEU_1','BLEU_4','METEOR','ROUGE_L','CIDEr','SPICE','CHAIRs','CHAIRi','Recall','NLL')
    comparison=dict(seed=43,split='test',images=5000,epoch=2,updates=7082,checkpoint=binding,
        CE_metrics=REFERENCES['CE'],epoch1_metrics=REFERENCES['epoch1_TEST'],epoch2_metrics=result['metrics'],
        delta_vs_epoch1={p:{k:m[k]-REFERENCES['epoch1_TEST'][p][k] for k in keys} for p,m in result['metrics'].items()},
        delta_vs_CE={p:{k:m[k]-REFERENCES['CE'][p][k] for k in keys} for p,m in result['metrics'].items()},
        checkpoint_selection='fixed epoch2 endpoint, no TEST selection',test_tuning=False,prior_project_test_exposure=True)
    result.update(status='BEAM_SCST_EPOCH2_TEST5000_RERANK_COMPLETE',epochs_SCST=2,updates=7082,seed=43)
    (rerank_out/'result.json').write_text(json.dumps(result,indent=2,allow_nan=False))
    (test_out/'result.json').write_text(json.dumps(result,indent=2,allow_nan=False))
    (OUTPUT/'comparison.json').write_text(json.dumps(comparison,indent=2,allow_nan=False))
    (OUTPUT/'result.json').write_text(json.dumps(dict(status='BEAM_SCST_EPOCH2_VALID_TEST_COMPLETE',
        training=trained,VALID=valid_result,TEST=result),indent=2,allow_nan=False))
    print('TEST comparison:',json.dumps(comparison,indent=2),flush=True);complete=True;phase='complete'
except BaseException:
    (OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,phase=phase,training_complete=training_complete,
        requested_epoch=2,requested_epoch2_images=113287,requested_cumulative_updates=7082,
        parent_source=SOURCE_VERSION,test_started=test_started,test_tuning=False,reranking=True),indent=2))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT.rglob('*'):
            if path.is_file() and path.suffix in ('.json','.jsonl','.txt','.log','.py','.md'):
                archive.write(path,str(path.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    for name in ('checkpoint.bin','pilot_checkpoint.bin'):
        if (OUTPUT/name).is_file():print('SAVED WEIGHTS:',OUTPUT/name,flush=True)
    print('EPOCH2 + VALID + TEST COMPLETE' if complete else 'INCOMPLETE; preserve weights/captions/scores. No automatic restart, partial ranking or recipe fallback.',flush=True)
